In [ ]:
"""
Fetch daily PM2.5 data for Almaty, Kazakhstan from the OpenAQ v3 API.

SETUP
-----
1. pip install requests pandas
2. Paste your OpenAQ API key into API_KEY below.
3. python fetch_almaty_pm25.py

OUTPUT
------
almaty_locations.csv  - all monitoring stations found near Almaty
almaty_pm25_daily.csv - daily PM2.5 averages, one row per sensor per day
"""

import time
import requests
import pandas as pd

# ---------------------------------------------------------------- CONFIG
API_KEY = "PASTE_YOUR_OPENAQ_KEY_HERE"

LAT, LON = 43.222, 76.851      # Almaty city centre
RADIUS_M = 25000               # 25 km - OpenAQ maximum
DATE_FROM = "2016-01-01"
DATE_TO = "2026-07-01"

BASE = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": API_KEY}


def get(url, params=None):
    """GET with basic retry and rate-limit courtesy."""
    for attempt in range(4):
        r = requests.get(url, headers=HEADERS, params=params, timeout=60)
        if r.status_code == 200:
            time.sleep(0.3)
            return r.json()
        if r.status_code == 429:            # rate limited
            print("   rate limited, waiting 20s...")
            time.sleep(20)
            continue
        print(f"   HTTP {r.status_code}: {r.text[:300]}")
        return None
    return None


# ------------------------------------------------- STEP 1: find stations
print("Step 1: finding monitoring stations near Almaty...")

data = get(
    f"{BASE}/locations",
    {"coordinates": f"{LAT},{LON}", "radius": RADIUS_M, "limit": 1000},
)

if not data or not data.get("results"):
    raise SystemExit("No locations returned. Check your API key is valid.")

locations = data["results"]
print(f"   found {len(locations)} stations\n")

# Flatten station metadata so you can eyeball what exists
rows = []
pm25_sensors = []

for loc in locations:
    for sensor in loc.get("sensors", []):
        param = sensor.get("parameter", {})
        rows.append({
            "location_id": loc.get("id"),
            "location_name": loc.get("name"),
            "provider": (loc.get("provider") or {}).get("name"),
            "latitude": (loc.get("coordinates") or {}).get("latitude"),
            "longitude": (loc.get("coordinates") or {}).get("longitude"),
            "first_seen": (loc.get("datetimeFirst") or {}).get("utc"),
            "last_seen": (loc.get("datetimeLast") or {}).get("utc"),
            "sensor_id": sensor.get("id"),
            "parameter": param.get("name"),
            "units": param.get("units"),
        })
        if param.get("name") == "pm25":
            pm25_sensors.append({
                "sensor_id": sensor.get("id"),
                "location_name": loc.get("name"),
                "location_id": loc.get("id"),
            })

pd.DataFrame(rows).to_csv("almaty_locations.csv", index=False)
print("   saved almaty_locations.csv")
print(f"   PM2.5 sensors found: {len(pm25_sensors)}")
for s in pm25_sensors:
    print(f"      sensor {s['sensor_id']}  -  {s['location_name']}")
print()

if not pm25_sensors:
    raise SystemExit("No PM2.5 sensors found. Inspect almaty_locations.csv.")


# ------------------------------------------- STEP 2: pull daily averages
print("Step 2: pulling daily PM2.5 averages (this may take a few minutes)...")

all_rows = []

for s in pm25_sensors:
    sid = s["sensor_id"]
    print(f"   sensor {sid} ({s['location_name']})...")
    page = 1
    got = 0

    while True:
        payload = get(
            f"{BASE}/sensors/{sid}/days",
            {
                "datetime_from": DATE_FROM,
                "datetime_to": DATE_TO,
                "limit": 1000,
                "page": page,
            },
        )
        if not payload:
            break

        results = payload.get("results", [])
        if not results:
            break

        for rec in results:
            period = rec.get("period") or {}
            dt_from = period.get("datetimeFrom") or {}
            all_rows.append({
                "sensor_id": sid,
                "location_name": s["location_name"],
                "location_id": s["location_id"],
                "date_utc": dt_from.get("utc"),
                "date_local": dt_from.get("local"),
                "pm25": rec.get("value"),
                "n_obs": (rec.get("coverage") or {}).get("observedCount"),
                "coverage_pct": (rec.get("coverage") or {}).get("percentComplete"),
            })

        got += len(results)
        if len(results) < 1000:
            break
        page += 1

    print(f"      {got} daily records")

if not all_rows:
    raise SystemExit("No measurements returned. Sensors may have no data in range.")


# ------------------------------------------------------ STEP 3: tidy up
df = pd.DataFrame(all_rows)
df["date"] = pd.to_datetime(df["date_local"], errors="coerce", utc=True).dt.date
df = df.dropna(subset=["date", "pm25"]).sort_values(["sensor_id", "date"])
df.to_csv("almaty_pm25_daily.csv", index=False)

print("\n" + "=" * 55)
print("DONE - saved almaty_pm25_daily.csv")
print("=" * 55)
print(f"Rows:            {len(df):,}")
print(f"Date range:      {df['date'].min()}  ->  {df['date'].max()}")
print(f"Sensors:         {df['sensor_id'].nunique()}")
print(f"Mean PM2.5:      {df['pm25'].mean():.1f}")
print("\nRecords per year:")
print(pd.to_datetime(df["date"]).dt.year.value_counts().sort_index().to_string())


Step 1: finding monitoring stations near Almaty...
   found 194 stations

   saved almaty_locations.csv
   PM2.5 sensors found: 194
      sensor 25903  -  Almaty
      sensor 8967269  -  Almaty Arena
      sensor 8967449  -  Magnum Besagash
      sensor 8967213  -  CHP
      sensor 8967295  -  School 137
      sensor 8967305  -  Kokzhiyek-63
      sensor 8967291  -  EkoPost
      sensor 8966905  -  Alatau
      sensor 8966963  -  Elaman 105 St.
      sensor 8967169  -  Atyrau 54
      sensor 8967457  -  Shamiyeva St.
      sensor 8966913  -  Kokkainar Micro
      sensor 8967351  -  Suiynbay-55
      sensor 8967245  -  KBTU
      sensor 8967201  -  Iliyski trakt
      sensor 8967337  -  School 187
      sensor 8966985  -  169 School
      sensor 8967401  -  Ritz Palace
      sensor 8966911  -  Mamyr-3
      sensor 8966973  -  Gymnasium 39
      sensor 8967397  -  Hospital-7
      sensor 8967179  -  Ryskulova 81
      sensor 8967281  -  School 77
      sensor 8967097  -  School 190
     

In [ ]:
import pandas as pd

df = pd.read_csv("almaty_pm25_daily.csv")
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

# What is sensor 25903?
loc = pd.read_csv("almaty_locations.csv")
print(loc[loc.sensor_id == 25903].T)

# Coverage of the backbone sensor by year
back = df[df.sensor_id == 25903]
print("\nSensor 25903 days per year:")
print(back.groupby("year").size().to_string())
print(f"\nRange: {back.date.min().date()} to {back.date.max().date()}")
print(f"Mean PM2.5: {back.pm25.mean():.1f}, max: {back.pm25.max():.1f}")

# Any other sensor with real history?
depth = df.groupby("sensor_id")["date"].agg(["min","max","count"])
print("\nSensors starting before 2024:")
print(depth[depth["min"] < "2024-01-01"].to_string())


                                  0
location_id                    8876
location_name                Almaty
provider                     AirNow
latitude                  43.252855
longitude                  76.93118
first_seen     2020-04-09T17:00:00Z
last_seen      2025-11-14T11:00:00Z
sensor_id                     25903
parameter                      pm25
units                         µg/m³

Sensor 25903 days per year:
year
2020    254
2021    289
2022    223
2023    315
2024    353
2025    317

Range: 2020-04-08 to 2025-11-13
Mean PM2.5: 29.6, max: 229.0

Sensors starting before 2024:
                 min        max  count
sensor_id                             
25903     2020-04-08 2025-11-13   1751


In [ ]:
import requests, pandas as pd, numpy as np

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 43.2529, "longitude": 76.9312,   # the AirNow monitor site
    "start_date": "2020-01-01", "end_date": "2025-12-31",
    "hourly": "temperature_2m,precipitation,wind_speed_10m,"
              "wind_direction_10m,boundary_layer_height,surface_pressure",
    "timezone": "Asia/Almaty",
}
h = requests.get(url, params=params, timeout=120).json()["hourly"]
w = pd.DataFrame(h)
w["time"] = pd.to_datetime(w["time"])
w["date"] = w["time"].dt.date

# wind direction is circular - must vector-average, not arithmetic-average
rad = np.deg2rad(w["wind_direction_10m"])
w["u"] = -w["wind_speed_10m"] * np.sin(rad)
w["v"] = -w["wind_speed_10m"] * np.cos(rad)

daily = w.groupby("date").agg(
    temp=("temperature_2m", "mean"),
    temp_min=("temperature_2m", "min"),
    precip=("precipitation", "sum"),
    wind_speed=("wind_speed_10m", "mean"),
    blh=("boundary_layer_height", "mean"),
    blh_min=("boundary_layer_height", "min"),
    pressure=("surface_pressure", "mean"),
    u=("u", "mean"), v=("v", "mean"),
).reset_index()

daily["wind_dir"] = (np.rad2deg(np.arctan2(-daily.u, -daily.v)) + 360) % 360
daily.to_csv("almaty_weather_daily.csv", index=False)

print(daily.shape)
print(daily.head())
print(daily.describe().round(2))


(2192, 11)
         date      temp  temp_min  precip  wind_speed        blh  blh_min  \
0  2020-01-01 -1.866667      -5.6     0.4    5.150000  26.875000     20.0   
1  2020-01-02 -3.354167      -9.7     0.0    6.550000  25.625000     20.0   
2  2020-01-03 -2.250000      -9.7     0.0    6.675000  32.916667     20.0   
3  2020-01-04 -1.175000      -3.4     0.4    3.945833  27.083333     20.0   
4  2020-01-05 -3.145833      -7.7     0.0    3.695833  26.250000     20.0   

     pressure         u         v    wind_dir  
0  928.295833 -1.149619 -3.730992   17.125477  
1  925.016667  0.296249 -4.055739  355.822288  
2  926.770833  0.396664 -1.278606  342.764469  
3  928.712500 -1.337735 -2.249762   30.736201  
4  924.508333 -0.319697 -0.024513   85.615458  
          temp  temp_min   precip  wind_speed      blh  blh_min  pressure  \
count  2192.00   2192.00  2192.00     2192.00  2011.00  2011.00   2192.00   
mean     11.20      6.15     1.44        5.58   258.00    21.08    926.08   
std    

In [10]:
"""
Stage 1: build both BLH constructions from the CDS files, splice the 2024 gap,
and let the first stage decide which one the paper should use.

Run in Colab with Drive mounted. The six blh_20XX.nc files and the panel must
be in pollution_research/.

WHAT THIS DOES
  A. Builds calendar-day and 07:00-17:59 windowed BLH for all 2,192 days.
  B. Saves both, so nothing has to be recomputed later.
  C. Merges each onto the existing panel and runs the full first stage.
  D. Reports F, partial R-squared and the BLH elasticity side by side.

WHAT IT DOES NOT DO
  It does not touch the second stage. The comparison is on instrument
  relevance only, which is the thing that is legitimate to choose on -- and
  it still has to be disclosed in the paper, alongside the losing
  construction as a robustness row.

CAVEAT ON THE COMPARISON
  This runs on the 926-day panel, so both constructions are evaluated on the
  SAME days -- the gap is not yet spliced in. That is deliberate: it isolates
  the construction question from the sample-size question. The splice happens
  in step B and gets used once the construction is chosen.
"""

import glob
import numpy as np
import pandas as pd
import xarray as xr
import statsmodels.api as sm

DIR = "/content/drive/MyDrive/pollution_research"
LAT, LON = 43.2529, 76.9312

# ---------------------------------------------------------------------------
# A. Build both constructions
# ---------------------------------------------------------------------------

ds = xr.open_mfdataset(sorted(glob.glob(f"{DIR}/blh_*.nc")), combine="by_coords")
VAR = "blh" if "blh" in ds.data_vars else list(ds.data_vars)[0]
TIME = "valid_time" if "valid_time" in ds.coords else "time"

# Bilinear matched Open-Meteo marginally better than nearest, so use it
pt = ds[VAR].interp(latitude=LAT, longitude=LON)
h = pt.to_dataframe(name="blh").reset_index()[[TIME, "blh"]]
h.columns = ["utc", "blh"]
h["utc"] = pd.to_datetime(h["utc"])
h = h.dropna().sort_values("utc")

# Almaty unified its time zones on 1 March 2024: UTC+6 -> UTC+5
off = np.where(h["utc"] < pd.Timestamp("2024-03-01"), 6, 5)
h["local"] = h["utc"] + pd.to_timedelta(off, unit="h")
h["date"] = h["local"].dt.normalize()
h["hour"] = h["local"].dt.hour

# Calendar day: all 24 hours
cal = (h.groupby("date")
         .agg(blh_cal=("blh", "mean"), blh_cal_min=("blh", "min"), n=("blh", "size"))
         .reset_index())
cal = cal[cal["n"] == 24].drop(columns="n")

# Window: 07:00-17:59, eleven hours
w = h[(h["hour"] >= 7) & (h["hour"] <= 17)]
win = (w.groupby("date")
         .agg(blh_win=("blh", "mean"), blh_win_min=("blh", "min"), n=("blh", "size"))
         .reset_index())
win = win[win["n"] == 11].drop(columns="n")

both = cal.merge(win, on="date", how="outer").sort_values("date")
both.to_csv(f"{DIR}/blh_cds_both_constructions.csv", index=False)

print(f"built {len(cal)} calendar-day and {len(win)} windowed days")
print(f"calendar mean {cal.blh_cal.mean():.1f} m,  window mean {win.blh_win.mean():.1f} m")
gap = both[(both.date >= "2024-01-02") & (both.date <= "2024-06-30")]
print(f"2024 gap coverage: {gap.blh_cal.notna().sum()} of 181 days\n")

# ---------------------------------------------------------------------------
# C. First stage under each construction, on the SAME 926 days
# ---------------------------------------------------------------------------

panel = pd.read_csv(f"{DIR}/panel_window_final.csv")
panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()

CONTROLS = ["temp", "precip", "wind_speed", "cloud", "radiation", "lockdown", "qantar"]
FE = ([f"mo_{i}" for i in range(2, 13)]
      + [f"dw_{i}" for i in range(1, 5)]
      + [f"yr_{y}" for y in range(2021, 2026)])

def first_stage(blh_col, label):
    d = panel.merge(both[["date", blh_col]], on="date", how="inner").dropna(
        subset=[blh_col, "log_pm25", "u", "v", "pressure"] + CONTROLS + FE)

    d = d.assign(log_blh_new=np.log(d[blh_col]))
    excluded = ["log_blh_new", "u", "v", "pressure"]
    X = sm.add_constant(d[excluded + CONTROLS + FE])
    y = d["log_pm25"]

    fit = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

    # Joint robust Wald on the excluded instruments, expressed as an F
    R = np.zeros((len(excluded), X.shape[1]))
    for i, name in enumerate(excluded):
        R[i, list(X.columns).index(name)] = 1
    wald = fit.wald_test(R, scalar=True)
    F = float(wald.statistic) / len(excluded)

    # Partial R-squared: residualise treatment and instruments on controls
    Z = sm.add_constant(d[CONTROLS + FE])
    y_r = sm.OLS(y, Z).fit().resid
    inst_r = np.column_stack([sm.OLS(d[c], Z).fit().resid for c in excluded])
    partial = sm.OLS(y_r, sm.add_constant(inst_r)).fit().rsquared

    print(f"--- {label} ---")
    print(f"  N                    : {len(d)}")
    print(f"  robust first-stage F : {F:.1f}")
    print(f"  partial R-squared    : {partial:.4f}")
    print(f"  log BLH coefficient  : {fit.params['log_blh_new']:.4f} "
          f"(SE {fit.bse['log_blh_new']:.4f}, t {fit.tvalues['log_blh_new']:.2f})")
    print(f"  u / v / pressure t   : {fit.tvalues['u']:.2f} / "
          f"{fit.tvalues['v']:.2f} / {fit.tvalues['pressure']:.2f}")
    print()
    return F, partial

print("=" * 64)
print("FIRST STAGE COMPARISON  (same 926 days, only BLH construction differs)")
print("=" * 64)
f_cal, r_cal = first_stage("blh_cal", "CALENDAR-DAY BLH")
f_win, r_win = first_stage("blh_win", "WINDOWED BLH (07:00-17:59)")

print("=" * 64)
winner = "CALENDAR-DAY" if f_cal > f_win else "WINDOWED"
print(f"Stronger first stage: {winner}")
print(f"  F        {f_cal:.1f} vs {f_win:.1f}")
print(f"  partial R2  {r_cal:.4f} vs {r_win:.4f}")
print("=" * 64)
print("""
Read it this way. A clear win for calendar-day supports keeping the existing
met construction, correcting Section 3.2 to describe it accurately, and
splicing only the 2024 gap. A clear win for windowed argues for pulling u, v
and pressure on the window too -- nine more CDS requests, not twenty-four.
If the two are close, sample size is doing more work than construction, and
the cheaper path is the better one.
""")

built 2190 calendar-day and 2192 windowed days
calendar mean 251.1 m,  window mean 458.3 m
2024 gap coverage: 180 of 181 days

FIRST STAGE COMPARISON  (same 926 days, only BLH construction differs)
--- CALENDAR-DAY BLH ---
  N                    : 926
  robust first-stage F : 67.6
  partial R-squared    : 0.2724
  log BLH coefficient  : -0.6097 (SE 0.0433, t -14.07)
  u / v / pressure t   : 3.72 / -2.61 / -4.55

--- WINDOWED BLH (07:00-17:59) ---
  N                    : 926
  robust first-stage F : 56.4
  partial R-squared    : 0.2560
  log BLH coefficient  : -0.5304 (SE 0.0421, t -12.61)
  u / v / pressure t   : 3.64 / -2.37 / -4.20

Stronger first stage: CALENDAR-DAY
  F        67.6 vs 56.4
  partial R2  0.2724 vs 0.2560

Read it this way. A clear win for calendar-day supports keeping the existing
met construction, correcting Section 3.2 to describe it accurately, and
splicing only the 2024 gap. A clear win for windowed argues for pulling u, v
and pressure on the window too -- nine 

In [ ]:
import pandas as pd
kase = pd.read_csv("KASE Historical Data.csv")
print(kase.shape)
print(kase.dtypes)
print(kase.head(10))
print(kase.tail(5))

FileNotFoundError: [Errno 2] No such file or directory: 'KASE Historical Data.csv'

In [ ]:
from google.colab import files
uploaded = files.upload()
print(uploaded.keys())


In [ ]:
import pandas as pd
import numpy as np

kase = pd.read_csv("KASE Historical Data.csv")
print("RAW:")
print(kase.dtypes)
print(kase.head(3))
print("-" * 55)

# --- normalise column names
kase.columns = [c.strip().lower().replace(".", "").replace(" ", "_")
                for c in kase.columns]

# --- dates: Investing.com uses MM/DD/YYYY
kase["date"] = pd.to_datetime(kase["date"], format="%m/%d/%Y", errors="coerce")
if kase["date"].isna().mean() > 0.5:          # fallback if format differs
    kase["date"] = pd.to_datetime(kase["date"], dayfirst=True, errors="coerce")

# --- prices: strip commas, convert to float
for col in ["price", "open", "high", "low"]:
    if col in kase.columns:
        kase[col] = (kase[col].astype(str)
                     .str.replace(",", "", regex=False)
                     .str.replace("%", "", regex=False))
        kase[col] = pd.to_numeric(kase[col], errors="coerce")

# --- volume: handles "1.24M", "470.11K", "-", "0"
def parse_vol(v):
    s = str(v).strip().replace(",", "")
    if s in ("-", "", "nan", "0"):
        return np.nan
    mult = {"K": 1e3, "M": 1e6, "B": 1e9}
    if s[-1].upper() in mult:
        try:
            return float(s[:-1]) * mult[s[-1].upper()]
        except ValueError:
            return np.nan
    return pd.to_numeric(s, errors="coerce")

if "vol" in kase.columns:
    kase["volume"] = kase["vol"].apply(parse_vol)

# --- sort oldest-first and compute returns ourselves
kase = kase.dropna(subset=["date", "price"]).sort_values("date").reset_index(drop=True)
kase["ret"] = kase["price"].pct_change() * 100          # simple return, %
kase["logret"] = np.log(kase["price"]).diff() * 100     # log return, %

kase = kase[["date", "price", "ret", "logret"] +
            (["volume"] if "volume" in kase.columns else [])]
kase.to_csv("kase_clean.csv", index=False)

print("CLEANED:")
print(f"Rows:       {len(kase):,}")
print(f"Range:      {kase.date.min().date()} -> {kase.date.max().date()}")
print(f"Price NaN:  {kase.price.isna().sum()}")
print(f"\nReturns (%):")
print(kase["logret"].describe().round(3).to_string())
print(f"\nTrading days per year:")
print(kase.groupby(kase.date.dt.year).size().to_string())
if "volume" in kase.columns:
    print(f"\nVolume missing: {kase.volume.isna().mean():.1%}")
print(kase.head())


In [ ]:
import pandas as pd, numpy as np

# ---------- load
pm = pd.read_csv("almaty_pm25_daily.csv")
wx = pd.read_csv("almaty_weather_daily.csv")
ks = pd.read_csv("kase_clean.csv")

for d in (pm, wx, ks):
    d["date"] = pd.to_datetime(d["date"])

# ---------- backbone pollution series: US Embassy / AirNow monitor
pm = pm[pm.sensor_id == 25903][["date", "pm25", "n_obs", "coverage_pct"]]
pm = pm.rename(columns={"pm25": "pm25_raw"})

# drop days with poor within-day coverage (unreliable daily mean)
before = len(pm)
pm = pm[pm["n_obs"] >= 18]                      # >=18 of 24 hours
print(f"Dropped {before - len(pm)} low-coverage days, {len(pm)} remain")

# ---------- merge
df = ks.merge(pm, on="date", how="inner").merge(wx, on="date", how="inner")

# ---------- sample window (post-COVID, within pollution coverage)
df = df[(df.date >= "2022-01-01") & (df.date <= "2025-11-13")].copy()

# ---------- transforms
df["log_pm25"] = np.log(df["pm25_raw"])
df["log_blh"]  = np.log(df["blh"])
df["dow"]      = df.date.dt.dayofweek
df["month"]    = df.date.dt.month
df["year"]     = df.date.dt.year
df["t"]        = np.arange(len(df))             # linear time trend

df = df.dropna(subset=["logret", "log_pm25", "log_blh", "temp", "precip", "wind_speed"])
df.to_csv("panel.csv", index=False)

# ---------- report
print(f"\nPANEL: {len(df)} trading days, {df.date.min().date()} -> {df.date.max().date()}")
print(f"\nPM2.5   mean {df.pm25_raw.mean():.1f}  median {df.pm25_raw.median():.1f}  max {df.pm25_raw.max():.0f}")
print(f"BLH     mean {df.blh.mean():.0f}m  min {df.blh.min():.0f}m")
print(f"Return  mean {df.logret.mean():.3f}%  std {df.logret.std():.3f}%")

# ---------- FIRST STAGE: does inversion predict pollution?
import statsmodels.formula.api as smf
fs = smf.ols("log_pm25 ~ log_blh + temp + precip + wind_speed + C(month) + C(dow)",
             data=df).fit()
print("\n--- FIRST STAGE ---")
print(f"log_blh coef: {fs.params['log_blh']:.4f}  (se {fs.bse['log_blh']:.4f})")
print(f"t-stat: {fs.tvalues['log_blh']:.2f}   F on instrument: {fs.tvalues['log_blh']**2:.1f}")
print(f"R-squared: {fs.rsquared:.3f}")

# ---------- naive OLS: pollution vs returns
ols = smf.ols("logret ~ log_pm25 + temp + precip + wind_speed + C(month) + C(dow)",
              data=df).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print("\n--- NAIVE OLS (Newey-West) ---")
print(f"log_pm25 coef: {ols.params['log_pm25']:.4f}  (se {ols.bse['log_pm25']:.4f})")
print(f"t-stat: {ols.tvalues['log_pm25']:.2f}   p: {ols.pvalues['log_pm25']:.3f}")

# ---------- return autocorrelation check (thin market)
print(f"\nReturn AR(1): {df.logret.autocorr(1):.3f}")


In [ ]:
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
import pandas as pd, numpy as np

df = pd.read_csv("panel.csv")
df["date"] = pd.to_datetime(df["date"])

# REDUCED FORM: does inversion predict returns directly?
rf = smf.ols("logret ~ log_blh + temp + precip + wind_speed + C(month) + C(dow)",
             data=df).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print("--- REDUCED FORM ---")
print(f"log_blh: {rf.params['log_blh']:.4f} (se {rf.bse['log_blh']:.4f})  "
      f"t={rf.tvalues['log_blh']:.2f}  p={rf.pvalues['log_blh']:.3f}")

# IV
d = df.copy()
for m in sorted(d.month.unique())[1:]:
    d[f"m{m}"] = (d.month == m).astype(int)
for w in sorted(d.dow.unique())[1:]:
    d[f"d{w}"] = (d.dow == w).astype(int)
ctrl = ["temp", "precip", "wind_speed"] + [f"m{m}" for m in sorted(d.month.unique())[1:]] \
       + [f"d{w}" for w in sorted(d.dow.unique())[1:]]
d["const"] = 1

iv = IV2SLS(d["logret"], d[["const"] + ctrl], d["log_pm25"], d["log_blh"]).fit(cov_type="kernel")
print("\n--- IV (2SLS) ---")
print(iv.summary.tables[1])

# Does pollution move VOLATILITY rather than returns?
d["absret"] = d.logret.abs()
vol = smf.ols("absret ~ log_pm25 + temp + precip + wind_speed + C(month) + C(dow)",
              data=d).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"\n--- |RETURN| ---")
print(f"log_pm25: {vol.params['log_pm25']:.4f} (se {vol.bse['log_pm25']:.4f})  "
      f"p={vol.pvalues['log_pm25']:.3f}")


In [ ]:
!pip install linearmodels -q


In [ ]:
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
import pandas as pd, numpy as np

df = pd.read_csv("panel.csv")
df["date"] = pd.to_datetime(df["date"])

# REDUCED FORM: does inversion predict returns directly?
rf = smf.ols("logret ~ log_blh + temp + precip + wind_speed + C(month) + C(dow)",
             data=df).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print("--- REDUCED FORM ---")
print(f"log_blh: {rf.params['log_blh']:.4f} (se {rf.bse['log_blh']:.4f})  "
      f"t={rf.tvalues['log_blh']:.2f}  p={rf.pvalues['log_blh']:.3f}")

# IV
d = df.copy()
for m in sorted(d.month.unique())[1:]:
    d[f"m{m}"] = (d.month == m).astype(int)
for w in sorted(d.dow.unique())[1:]:
    d[f"d{w}"] = (d.dow == w).astype(int)
ctrl = ["temp", "precip", "wind_speed"] + [f"m{m}" for m in sorted(d.month.unique())[1:]] \
       + [f"d{w}" for w in sorted(d.dow.unique())[1:]]
d["const"] = 1

iv = IV2SLS(d["logret"], d[["const"] + ctrl], d["log_pm25"], d["log_blh"]).fit(cov_type="kernel")
print("\n--- IV (2SLS) ---")
print(iv.summary.tables[1])

# Does pollution move VOLATILITY rather than returns?
d["absret"] = d.logret.abs()
vol = smf.ols("absret ~ log_pm25 + temp + precip + wind_speed + C(month) + C(dow)",
              data=d).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"\n--- |RETURN| ---")
print(f"log_pm25: {vol.params['log_pm25']:.4f} (se {vol.bse['log_pm25']:.4f})  "
      f"p={vol.pvalues['log_pm25']:.3f}")


In [ ]:
import pandas as pd, numpy as np, statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS

df = pd.read_csv("panel.csv"); df["date"] = pd.to_datetime(df["date"])

def prep(d):
    d = d.copy()
    for m in sorted(d.month.unique())[1:]: d[f"m{m}"] = (d.month==m).astype(int)
    for w in sorted(d.dow.unique())[1:]:   d[f"d{w}"] = (d.dow==w).astype(int)
    d["const"] = 1
    ctrl = ["temp","precip","wind_speed"] + [c for c in d if c.startswith(("m","d")) and c[1:].isdigit()]
    return d, ctrl

def run_iv(d, y="logret", x="log_pm25", z="log_blh"):
    d, ctrl = prep(d.dropna(subset=[y,x,z]))
    r = IV2SLS(d[y], d[["const"]+ctrl], d[x], d[z]).fit(cov_type="kernel")
    return r.params[x], r.std_errors[x], r.pvalues[x], len(d)

print("="*70); print("ROBUSTNESS"); print("="*70)

# 1. MINIMUM DETECTABLE EFFECT  <- the key number for a null paper
b, se, p, n = run_iv(df)
print(f"\n[1] POWER")
print(f"  IV estimate: {b:.3f} (se {se:.3f}), n={n}")
print(f"  MDE at 80% power, 5% level: {2.8*se:.3f} log points")
print(f"  -> a 100% rise in PM2.5 would need to move returns by "
      f"{2.8*se*np.log(2):.3f}pp to be detected")
print(f"  95% CI: [{b-1.96*se:.3f}, {b+1.96*se:.3f}]")

# 2. LAGS - effect may not be same-day
print(f"\n[2] LAGS")
d = df.copy()
for L in [0,1,2,3]:
    d[f"lag{L}"] = d["log_pm25"].shift(L)
    d[f"blag{L}"] = d["log_blh"].shift(L)
    try:
        b_,se_,p_,n_ = run_iv(d, x=f"lag{L}", z=f"blag{L}")
        print(f"  lag {L}: {b_:7.3f} (se {se_:.3f})  p={p_:.3f}  n={n_}")
    except Exception as e:
        print(f"  lag {L}: failed")

# 3. THRESHOLDS - maybe only extreme pollution matters
print(f"\n[3] THRESHOLDS (OLS)")
for thr in [35, 55, 75]:
    d = df.copy(); d["hi"] = (d.pm25_raw > thr).astype(int)
    if d.hi.sum() < 20: print(f"  >{thr}: too few days ({d.hi.sum()})"); continue
    r = smf.ols("logret ~ hi + temp + precip + wind_speed + C(month) + C(dow)",
                data=d).fit(cov_type="HAC", cov_kwds={"maxlags":5})
    print(f"  >{thr} ug/m3 ({d.hi.sum():3d} days): {r.params['hi']:7.4f} "
          f"(se {r.bse['hi']:.4f}) p={r.pvalues['hi']:.3f}")

# 4. SAMPLE PERIOD
print(f"\n[4] SAMPLE")
full = pd.read_csv("panel_full.csv") if False else None
for lo,hi,lab in [("2022-01-01","2025-11-13","baseline"),
                  ("2023-01-01","2025-11-13","2023+"),
                  ("2022-01-01","2024-12-31","excl 2025")]:
    s = df[(df.date>=lo)&(df.date<=hi)]
    b_,se_,p_,n_ = run_iv(s)
    print(f"  {lab:12s}: {b_:7.3f} (se {se_:.3f}) p={p_:.3f} n={n_}")

# 5. WINTER ONLY - most variation, strongest first stage
print(f"\n[5] WINTER (Nov-Mar)")
w = df[df.month.isin([11,12,1,2,3])]
b_,se_,p_,n_ = run_iv(w)
print(f"  {b_:.3f} (se {se_:.3f}) p={p_:.3f} n={n_}")

# 6. WINSORISED RETURNS
print(f"\n[6] WINSORISED 1%")
d = df.copy()
lo_,hi_ = d.logret.quantile([.01,.99])
d["logret"] = d.logret.clip(lo_,hi_)
b_,se_,p_,n_ = run_iv(d)
print(f"  {b_:.3f} (se {se_:.3f}) p={p_:.3f}")

# 7. PLACEBO - pollution should NOT predict yesterday's return
print(f"\n[7] PLACEBO (lead return)")
d = df.copy(); d["lead"] = d.logret.shift(1)
b_,se_,p_,n_ = run_iv(d, y="lead")
print(f"  {b_:.3f} (se {se_:.3f}) p={p_:.3f}   <- should be null")

# 8. DESCRIPTIVES for Table 1
print(f"\n[8] TABLE 1")
print(df[["logret","pm25_raw","blh","temp","precip","wind_speed"]]
      .describe().T[["count","mean","std","min","50%","max"]].round(2).to_string())


In [ ]:
# stronger first stage
d, ctrl = prep(df.dropna(subset=["logret","log_pm25","log_blh","u","v","pressure"]))
inst = ["log_blh","u","v","pressure"]
iv2 = IV2SLS(d["logret"], d[["const"]+ctrl], d["log_pm25"], d[inst]).fit(cov_type="kernel")
print(iv2.summary.tables[1])
print(f"\nFirst-stage F: {iv2.first_stage.diagnostics}")
print(f"Overid (Sargan): {iv2.sargan}")


In [ ]:
import pandas as pd, numpy as np, statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
import warnings; warnings.filterwarnings("ignore")

df = pd.read_csv("panel.csv"); df["date"] = pd.to_datetime(df["date"])
INST = ["log_blh", "u", "v", "pressure"]

def prep(d):
    d = d.copy()
    mcols, dcols = [], []
    for m in sorted(d.month.unique())[1:]:
        d[f"mo_{m}"] = (d.month == m).astype(int); mcols.append(f"mo_{m}")
    for w in sorted(d.dow.unique())[1:]:
        d[f"dw_{w}"] = (d.dow == w).astype(int);  dcols.append(f"dw_{w}")
    d["const"] = 1
    return d, ["temp", "precip", "wind_speed"] + mcols + dcols

def run_iv(data, y="logret", x="log_pm25", inst=INST):
    d = data.dropna(subset=[y, x] + inst)
    d, ctrl = prep(d)
    r = IV2SLS(d[y], d[["const"] + ctrl], d[x], d[inst]).fit(cov_type="kernel")
    try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except: F = np.nan
    return r.params[x], r.std_errors[x], r.pvalues[x], len(d), F

print("="*72); print("ROBUSTNESS  |  instruments: BLH, u, v, pressure"); print("="*72)

b, se, p, n, F = run_iv(df)
print(f"\n[1] BASELINE & POWER")
print(f"  IV: {b:.3f} (se {se:.3f})  p={p:.3f}  n={n}  first-stage F={F:.1f}")
print(f"  95% CI: [{b-1.96*se:.3f}, {b+1.96*se:.3f}]")
print(f"  MDE (80% power): {2.8*se:.3f} log pts = {2.8*se*np.log(2):.3f}pp per doubling")

print(f"\n[2] LAGS")
for L in range(4):
    d = df.copy()
    d[f"pm_l{L}"] = d["log_pm25"].shift(L)
    for i in INST: d[f"{i}_l{L}"] = d[i].shift(L)
    try:
        b_,se_,p_,n_,F_ = run_iv(d, x=f"pm_l{L}", inst=[f"{i}_l{L}" for i in INST])
        print(f"  lag {L}: {b_:7.3f} (se {se_:.3f})  p={p_:.3f}  F={F_:5.1f}  n={n_}")
    except Exception as e: print(f"  lag {L}: failed ({e})")

print(f"\n[3] THRESHOLDS (OLS, reduced form)")
for thr in [35, 55, 75]:
    d = df.copy(); d["hi"] = (d.pm25_raw > thr).astype(int)
    if d.hi.sum() < 20: print(f"  >{thr}: too few days"); continue
    r = smf.ols("logret ~ hi + temp + precip + wind_speed + C(month) + C(dow)",
                data=d).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  >{thr:3d} ({d.hi.sum():3d} days): {r.params['hi']:7.4f} "
          f"(se {r.bse['hi']:.4f})  p={r.pvalues['hi']:.3f}")

print(f"\n[4] SAMPLE PERIOD")
for lo, hi, lab in [("2022-01-01","2025-11-13","baseline"),
                    ("2023-01-01","2025-11-13","2023 onward"),
                    ("2022-01-01","2024-12-31","excl 2025")]:
    try:
        b_,se_,p_,n_,F_ = run_iv(df[(df.date>=lo)&(df.date<=hi)])
        print(f"  {lab:12s}: {b_:7.3f} (se {se_:.3f})  p={p_:.3f}  n={n_}")
    except Exception: print(f"  {lab:12s}: failed")

print(f"\n[5] WINTER ONLY (Nov-Mar)")
b_,se_,p_,n_,F_ = run_iv(df[df.month.isin([11,12,1,2,3])])
print(f"  {b_:.3f} (se {se_:.3f})  p={p_:.3f}  n={n_}  F={F_:.1f}")

print(f"\n[6] WINSORISED RETURNS (1%)")
d = df.copy(); lo_,hi_ = d.logret.quantile([.01,.99]); d["logret"] = d.logret.clip(lo_,hi_)
b_,se_,p_,n_,F_ = run_iv(d)
print(f"  {b_:.3f} (se {se_:.3f})  p={p_:.3f}")

print(f"\n[7] PLACEBO  <- must be null")
d = df.copy(); d["lead"] = d.logret.shift(1)
b_,se_,p_,n_,F_ = run_iv(d, y="lead")
print(f"  lead return: {b_:.3f} (se {se_:.3f})  p={p_:.3f}")
d2 = df.copy(); d2["lead2"] = d2.logret.shift(2)
b_,se_,p_,n_,F_ = run_iv(d2, y="lead2")
print(f"  2-day lead:  {b_:.3f} (se {se_:.3f})  p={p_:.3f}")

print(f"\n[8] DROP PRESSURE (exclusion-restriction check)")
b_,se_,p_,n_,F_ = run_iv(df, inst=["log_blh","u","v"])
print(f"  {b_:.3f} (se {se_:.3f})  p={p_:.3f}  F={F_:.1f}")
b_,se_,p_,n_,F_ = run_iv(df, inst=["log_blh"])
print(f"  BLH only:  {b_:.3f} (se {se_:.3f})  p={p_:.3f}  F={F_:.1f}")

print(f"\n[9] ALTERNATIVE OUTCOMES")
for out, lab in [("absret","|return| (volatility)"), ("logret","return")]:
    d = df.copy(); d["absret"] = d.logret.abs()
    b_,se_,p_,n_,F_ = run_iv(d, y=out)
    print(f"  {lab:22s}: {b_:7.3f} (se {se_:.3f})  p={p_:.3f}")

print(f"\n[10] SARGAN OVERIDENTIFICATION")
d, ctrl = prep(df.dropna(subset=["logret","log_pm25"]+INST))
r = IV2SLS(d["logret"], d[["const"]+ctrl], d["log_pm25"], d[INST]).fit(cov_type="kernel")
print(f"  {r.sargan}")

print(f"\n[11] TABLE 1 - DESCRIPTIVES")
print(df[["logret","pm25_raw","blh","temp","precip","wind_speed","pressure"]]
      .describe().T[["count","mean","std","min","50%","max"]].round(2).to_string())


In [ ]:
import pandas as pd, numpy as np, statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
import warnings; warnings.filterwarnings("ignore")

INST = ["log_blh","u","v","pressure"]

# ---------- rebuild with relaxed coverage + extended window
pm = pd.read_csv("almaty_pm25_daily.csv"); pm["date"] = pd.to_datetime(pm["date"])
wx = pd.read_csv("almaty_weather_daily.csv"); wx["date"] = pd.to_datetime(wx["date"])
ks = pd.read_csv("kase_clean.csv"); ks["date"] = pd.to_datetime(ks["date"])

def build(min_obs, start):
    p = pm[(pm.sensor_id == 25903) & (pm.n_obs >= min_obs)][["date","pm25","n_obs"]]
    p = p.rename(columns={"pm25":"pm25_raw"})
    d = ks.merge(p, on="date").merge(wx, on="date")
    d = d[(d.date >= start) & (d.date <= "2025-11-13")].copy()
    d["log_pm25"] = np.log(d.pm25_raw); d["log_blh"] = np.log(d.blh)
    d["dow"] = d.date.dt.dayofweek; d["month"] = d.date.dt.month; d["year"] = d.date.dt.year
    # COVID controls
    d["covid_era"] = (d.date < "2022-01-01").astype(int)
    d["lockdown"] = (((d.date >= "2020-03-16") & (d.date <= "2020-05-11")) |
                     ((d.date >= "2020-07-05") & (d.date <= "2020-08-17"))).astype(int)
    return d.dropna(subset=["logret","log_pm25"] + INST)

def run(d, extra=None, y="logret"):
    d = d.copy()
    ctrl = ["temp","precip","wind_speed"]
    for m in sorted(d.month.unique())[1:]:
        d[f"mo_{m}"] = (d.month==m).astype(int); ctrl.append(f"mo_{m}")
    for w in sorted(d.dow.unique())[1:]:
        d[f"dw_{w}"] = (d.dow==w).astype(int); ctrl.append(f"dw_{w}")
    for yr in sorted(d.year.unique())[1:]:
        d[f"yr_{yr}"] = (d.year==yr).astype(int); ctrl.append(f"yr_{yr}")
    if extra: ctrl += [c for c in extra if d[c].nunique() > 1]
    d["const"] = 1
    r = IV2SLS(d[y], d[["const"]+ctrl], d["log_pm25"], d[INST]).fit(cov_type="kernel")
    try: F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except: F = np.nan
    return r.params["log_pm25"], r.std_errors["log_pm25"], r.pvalues["log_pm25"], len(d), F, r

print("="*72); print("FIX 2: SAMPLE EXPANSION"); print("="*72)

print("\n[A] COVERAGE FILTER  (2022+ window, for like-for-like)")
for mo in [18, 15, 12, 6]:
    d = build(mo, "2022-01-01")
    b,se,p,n,F,_ = run(d)
    print(f"  >={mo:2d}h: {b:7.3f} (se {se:.3f})  p={p:.3f}  n={n:4d}  F={F:5.1f}  "
          f"MDE={2.8*se*np.log(2):.3f}pp")

print("\n[B] EXTENDED WINDOW  (12h filter, COVID controls)")
for start, lab, extra in [("2022-01-01","2022+ (baseline)", None),
                          ("2020-04-08","full + year FE", None),
                          ("2020-04-08","full + covid dummy", ["covid_era"]),
                          ("2020-04-08","full + lockdown", ["lockdown"]),
                          ("2020-04-08","full + both", ["covid_era","lockdown"])]:
    d = build(12, start)
    b,se,p,n,F,_ = run(d, extra)
    print(f"  {lab:22s}: {b:7.3f} (se {se:.3f})  p={p:.3f}  n={n:4d}  F={F:5.1f}  "
          f"MDE={2.8*se*np.log(2):.3f}pp")

print("\n[C] PREFERRED SPEC  (12h, full window, covid + lockdown)")
d = build(12, "2020-04-08")
b,se,p,n,F,r = run(d, ["covid_era","lockdown"])
print(f"  IV: {b:.3f} (se {se:.3f})  p={p:.3f}  n={n}")
print(f"  95% CI: [{b-1.96*se:.3f}, {b+1.96*se:.3f}]")
print(f"  First-stage F: {F:.1f}")
print(f"  MDE: {2.8*se*np.log(2):.3f}pp per doubling of PM2.5")
print(f"  Sargan: {r.sargan}")

print("\n[D] PLACEBO on preferred spec")
d2 = d.copy(); d2["lead"] = d2.logret.shift(1)
d2 = d2.dropna(subset=["lead"])
b2,se2,p2,n2,F2,_ = run(d2, ["covid_era","lockdown"], y="lead")
print(f"  lead return: {b2:.3f} (se {se2:.3f})  p={p2:.3f}")

d.to_csv("panel_extended.csv", index=False)
print(f"\nSaved panel_extended.csv  ({len(d)} obs)")


In [ ]:
def run(d, extra=None, y="logret"):
    d = d.copy()
    ctrl = ["temp","precip","wind_speed"]
    for m in sorted(d.month.unique())[1:]:
        d[f"mo_{m}"] = (d.month==m).astype(int); ctrl.append(f"mo_{m}")
    for w in sorted(d.dow.unique())[1:]:
        d[f"dw_{w}"] = (d.dow==w).astype(int); ctrl.append(f"dw_{w}")
    for yr in sorted(d.year.unique())[1:]:
        d[f"yr_{yr}"] = (d.year==yr).astype(int); ctrl.append(f"yr_{yr}")
    if extra:
        for c in extra:
            if d[c].nunique() > 1:
                # skip if collinear with existing controls
                X = d[ctrl + [c]].astype(float).values
                if np.linalg.matrix_rank(X) == X.shape[1]:
                    ctrl.append(c)
    d["const"] = 1
    r = IV2SLS(d[y], d[["const"]+ctrl], d["log_pm25"], d[INST]).fit(cov_type="kernel")
    try: F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except: F = np.nan
    return r.params["log_pm25"], r.std_errors["log_pm25"], r.pvalues["log_pm25"], len(d), F, r

print("\n[B] EXTENDED WINDOW  (12h filter)")
for start, lab, extra in [("2022-01-01","2022+ (baseline)", None),
                          ("2020-04-08","full + year FE", None),
                          ("2020-04-08","full + lockdown", ["lockdown"])]:
    d = build(12, start)
    b,se,p,n,F,_ = run(d, extra)
    print(f"  {lab:22s}: {b:7.3f} (se {se:.3f})  p={p:.3f}  n={n:4d}  F={F:5.1f}  "
          f"MDE={2.8*se*np.log(2):.3f}pp")

print("\n[C] PREFERRED SPEC")
d = build(12, "2020-04-08")
b,se,p,n,F,r = run(d, ["lockdown"])
print(f"  IV: {b:.3f} (se {se:.3f})  p={p:.3f}  n={n}")
print(f"  95% CI: [{b-1.96*se:.3f}, {b+1.96*se:.3f}]")
print(f"  First-stage F: {F:.1f}   MDE: {2.8*se*np.log(2):.3f}pp")
print(f"  Sargan: {r.sargan}")

print("\n[D] PLACEBO")
d2 = d.copy(); d2["lead"] = d2.logret.shift(1); d2 = d2.dropna(subset=["lead"])
b2,se2,p2,_,_,_ = run(d2, ["lockdown"], y="lead")
print(f"  lead return: {b2:.3f} (se {se2:.3f})  p={p2:.3f}")

d.to_csv("panel_extended.csv", index=False)
print(f"\nSaved panel_extended.csv ({len(d)} obs)")


In [ ]:
import pandas as pd, numpy as np, statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
import warnings; warnings.filterwarnings("ignore")

INST = ["log_blh","u","v","pressure"]

# ============================== BUILD ==============================
pm = pd.read_csv("almaty_pm25_daily.csv"); pm["date"] = pd.to_datetime(pm["date"])
wx = pd.read_csv("almaty_weather_daily.csv"); wx["date"] = pd.to_datetime(wx["date"])
ks = pd.read_csv("kase_clean.csv"); ks["date"] = pd.to_datetime(ks["date"])

def build(min_obs=12, start="2020-04-08", end="2025-11-13"):
    p = pm[(pm.sensor_id==25903) & (pm.n_obs>=min_obs)][["date","pm25","n_obs"]]
    p = p.rename(columns={"pm25":"pm25_raw"})
    d = ks.merge(p, on="date").merge(wx, on="date")
    d = d[(d.date>=start) & (d.date<=end)].copy()
    d["log_pm25"] = np.log(d.pm25_raw); d["log_blh"] = np.log(d.blh)
    d["dow"] = d.date.dt.dayofweek; d["month"] = d.date.dt.month; d["year"] = d.date.dt.year
    d["lockdown"] = (((d.date>="2020-03-16")&(d.date<="2020-05-11")) |
                     ((d.date>="2020-07-05")&(d.date<="2020-08-17"))).astype(int)
    return d.dropna(subset=["logret","log_pm25"]+INST).sort_values("date").reset_index(drop=True)

def controls(d):
    """Build control matrix, dropping collinear columns."""
    d = d.copy(); ctrl = ["temp","precip","wind_speed"]
    for m in sorted(d.month.unique())[1:]:
        d[f"mo_{m}"]=(d.month==m).astype(int); ctrl.append(f"mo_{m}")
    for w in sorted(d.dow.unique())[1:]:
        d[f"dw_{w}"]=(d.dow==w).astype(int); ctrl.append(f"dw_{w}")
    for y in sorted(d.year.unique())[1:]:
        d[f"yr_{y}"]=(d.year==y).astype(int); ctrl.append(f"yr_{y}")
    if d["lockdown"].nunique()>1:
        X = d[ctrl+["lockdown"]].astype(float).values
        if np.linalg.matrix_rank(X)==X.shape[1]: ctrl.append("lockdown")
    d["const"]=1
    return d, ctrl

def iv(d, y="logret", x="log_pm25", inst=INST):
    d = d.dropna(subset=[y,x]+inst)
    d, ctrl = controls(d)
    r = IV2SLS(d[y], d[["const"]+ctrl], d[x], d[inst]).fit(cov_type="kernel")
    try: F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except: F = np.nan
    return dict(b=r.params[x], se=r.std_errors[x], p=r.pvalues[x], n=len(d), F=F, r=r)

MAIN = build()
print("="*74); print("REGENERATED TABLES  |  preferred sample"); print("="*74)

# ============================== TABLE 1 ==============================
print("\n### TABLE 1 - DESCRIPTIVES (N=%d) ###\n" % len(MAIN))
lbl = {"logret":"Log return (%)","pm25_raw":"PM2.5 (ug/m3)","blh":"Boundary layer height (m)",
       "temp":"Temperature (C)","precip":"Precipitation (mm)","wind_speed":"Wind speed (m/s)",
       "pressure":"Surface pressure (hPa)"}
t1 = MAIN[list(lbl)].describe().T[["count","mean","std","min","50%","max"]].round(2)
t1.index = [lbl[i] for i in t1.index]
print(t1.to_string())
print(f"\nSample: {MAIN.date.min().date()} to {MAIN.date.max().date()}")
print(f"Return AR(1): {MAIN.logret.autocorr(1):.3f}")
print(f"PM2.5 days >55 ug/m3: {(MAIN.pm25_raw>55).sum()}  |  >75: {(MAIN.pm25_raw>75).sum()}")

# ============================== FIRST STAGE ==============================
print("\n\n### TABLE 2 - FIRST STAGE ###\n")
d, ctrl = controls(MAIN)
fs = smf.ols("log_pm25 ~ " + " + ".join(INST+ctrl), data=d).fit(
        cov_type="HAC", cov_kwds={"maxlags":5})
for v in INST:
    print(f"  {v:12s}: {fs.params[v]:8.4f}  (se {fs.bse[v]:.4f})  "
          f"t={fs.tvalues[v]:6.2f}  p={fs.pvalues[v]:.3f}")
print(f"\n  R-squared: {fs.rsquared:.3f}")
res = iv(MAIN)
print(f"  First-stage F (excluded instruments): {res['F']:.1f}")
try:
    print(f"  Partial R-squared: {res['r'].first_stage.diagnostics['partial.rsquared'].iloc[0]:.4f}")
except: pass

# ============================== TABLE 3 ==============================
print("\n\n### TABLE 3 - MAIN RESULTS ###\n")
ols = smf.ols("logret ~ log_pm25 + " + " + ".join(ctrl), data=d).fit(
        cov_type="HAC", cov_kwds={"maxlags":5})
print(f"  (1) OLS            : {ols.params['log_pm25']:7.3f} (se {ols.bse['log_pm25']:.3f})  "
      f"p={ols.pvalues['log_pm25']:.3f}  n={int(ols.nobs)}")
w = iv(build(18,"2022-01-01"), inst=["log_blh"])
print(f"  (2) IV, BLH only   : {w['b']:7.3f} (se {w['se']:.3f})  p={w['p']:.3f}  "
      f"n={w['n']}  F={w['F']:.1f}")
print(f"  (3) IV, full set   : {res['b']:7.3f} (se {res['se']:.3f})  p={res['p']:.3f}  "
      f"n={res['n']}  F={res['F']:.1f}")
print(f"\n  95% CI: [{res['b']-1.96*res['se']:.3f}, {res['b']+1.96*res['se']:.3f}]")
print(f"  MDE (80% power): {2.8*res['se']*np.log(2):.3f}pp per doubling")
print(f"  {res['r'].sargan}")

# ============================== TABLE 4 ==============================
print("\n\n### TABLE 4 - ROBUSTNESS (all with year FE + lockdown) ###\n")
rows = [("Preferred (>=12h, full window)", MAIN)]
for mo in [18,15,6]:
    rows.append((f"Coverage threshold >={mo}h", build(mo)))
for s,lab in [("2022-01-01","Sample from 2022"),("2023-01-01","Sample from 2023")]:
    rows.append((lab, build(12,s)))
rows.append(("Excluding 2025", build(12,"2020-04-08","2024-12-31")))
rows.append(("Winter only (Nov-Mar)", MAIN[MAIN.month.isin([11,12,1,2,3])]))
wz = MAIN.copy(); lo,hi = wz.logret.quantile([.01,.99]); wz["logret"]=wz.logret.clip(lo,hi)
rows.append(("Returns winsorised 1%", wz))

for lab, dd in rows:
    try:
        r_ = iv(dd)
        print(f"  {lab:34s}: {r_['b']:7.3f} ({r_['se']:.3f})  p={r_['p']:.3f}  "
              f"n={r_['n']:4d}  F={r_['F']:6.1f}")
    except Exception as e:
        print(f"  {lab:34s}: FAILED  {str(e)[:40]}")

print()
for lab, inst_ in [("Instruments: drop pressure",["log_blh","u","v"]),
                   ("Instruments: BLH + wind only",["log_blh","u","v"]),
                   ("Instruments: BLH only",["log_blh"])]:
    r_ = iv(MAIN, inst=inst_)
    print(f"  {lab:34s}: {r_['b']:7.3f} ({r_['se']:.3f})  p={r_['p']:.3f}  F={r_['F']:6.1f}")

# lags
print()
for L in range(1,4):
    dd = MAIN.copy(); dd[f"pm_l{L}"]=dd.log_pm25.shift(L)
    for i in INST: dd[f"{i}_l{L}"]=dd[i].shift(L)
    r_ = iv(dd, x=f"pm_l{L}", inst=[f"{i}_l{L}" for i in INST])
    print(f"  {'Lag '+str(L)+' day':34s}: {r_['b']:7.3f} ({r_['se']:.3f})  p={r_['p']:.3f}  n={r_['n']}")

# volatility
dd = MAIN.copy(); dd["absret"]=dd.logret.abs()
r_ = iv(dd, y="absret")
print(f"  {'Outcome: absolute return':34s}: {r_['b']:7.3f} ({r_['se']:.3f})  p={r_['p']:.3f}")

# thresholds (reduced form)
print("\n  Threshold indicators (reduced-form OLS):")
for thr in [35,55,75]:
    dd, c2 = controls(MAIN.copy())
    dd["hi"] = (dd.pm25_raw>thr).astype(int)
    if dd.hi.sum()<20: continue
    rr = smf.ols("logret ~ hi + "+" + ".join(c2), data=dd).fit(
            cov_type="HAC", cov_kwds={"maxlags":5})
    print(f"    >{thr:3d} ug/m3 ({dd.hi.sum():3d} days): {rr.params['hi']:7.4f} "
          f"({rr.bse['hi']:.4f})  p={rr.pvalues['hi']:.3f}")

# ============================== PLACEBO ==============================
print("\n\n### PLACEBO ###\n")
for L,lab in [(1,"1-day lead"),(2,"2-day lead")]:
    dd = MAIN.copy(); dd["lead"]=dd.logret.shift(L)
    r_ = iv(dd.dropna(subset=["lead"]), y="lead")
    print(f"  {lab:14s}: {r_['b']:7.3f} ({r_['se']:.3f})  p={r_['p']:.3f}")

MAIN.to_csv("panel_final.csv", index=False)
print(f"\n\nSaved panel_final.csv ({len(MAIN)} obs)")


In [ ]:
import pandas as pd, numpy as np, statsmodels.api as sm, statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
from scipy import stats
import warnings; warnings.filterwarnings("ignore")

INST = ["log_blh","u","v","pressure"]
d0 = pd.read_csv("panel_final.csv"); d0["date"] = pd.to_datetime(d0["date"])

def controls(d):
    d = d.copy(); ctrl = ["temp","precip","wind_speed"]
    for m in sorted(d.month.unique())[1:]:
        d[f"mo_{m}"]=(d.month==m).astype(int); ctrl.append(f"mo_{m}")
    for w in sorted(d.dow.unique())[1:]:
        d[f"dw_{w}"]=(d.dow==w).astype(int); ctrl.append(f"dw_{w}")
    for y in sorted(d.year.unique())[1:]:
        d[f"yr_{y}"]=(d.year==y).astype(int); ctrl.append(f"yr_{y}")
    if d["lockdown"].nunique()>1:
        X=d[ctrl+["lockdown"]].astype(float).values
        if np.linalg.matrix_rank(X)==X.shape[1]: ctrl.append("lockdown")
    d["const"]=1
    return d, ctrl

D, CTRL = controls(d0)
res = IV2SLS(D["logret"], D[["const"]+CTRL], D["log_pm25"], D[INST]).fit(cov_type="kernel")
b, se = res.params["log_pm25"], res.std_errors["log_pm25"]

print("="*74); print("EXTENDED ANALYSIS"); print("="*74)

# ---------------- 1. EQUIVALENCE TEST (TOST) ----------------
print("\n### 1. EQUIVALENCE TESTING ###\n")
print(f"  Point estimate: {b:.4f}  (SE {se:.4f})")
bound = abs(b) + 1.645*se
print(f"  90% CI: [{b-1.645*se:.4f}, {b+1.645*se:.4f}]")
print(f"\n  TOST: we REJECT effects larger than {bound:.4f} log points")
print(f"        = {bound*np.log(2):.4f} pp per doubling of PM2.5")
print(f"        = {bound*np.log(1.1):.4f} pp per 10% rise in PM2.5")
print(f"\n  Interpretation: at the 5% level we reject the hypothesis that")
print(f"  |effect| >= {bound*np.log(2):.3f}pp per doubling.")
for delta in [0.2,0.3,0.5,0.75,1.0]:
    t_lo=(b-delta)/se; t_hi=(b+delta)/se
    rej = (t_lo < -1.645) and (t_hi > 1.645)
    print(f"    H0:|theta|>={delta:.2f} log pts ({delta*np.log(2):.3f}pp) -> "
          f"{'REJECTED' if rej else 'not rejected'}")

# ---------------- 2. ANDERSON-RUBIN CI ----------------
print("\n\n### 2. ANDERSON-RUBIN CONFIDENCE INTERVAL ###\n")
hyp = ", ".join([f"{i} = 0" for i in INST])
grid, keep = np.arange(-1.5, 1.5, 0.005), []
for b0 in grid:
    Dg = D.copy(); Dg["_y"] = Dg["logret"] - b0*Dg["log_pm25"]
    m = smf.ols("_y ~ " + " + ".join(CTRL+INST), data=Dg).fit(
            cov_type="HAC", cov_kwds={"maxlags":5})
    if m.f_test(hyp).pvalue > 0.05: keep.append(b0)
if keep:
    print(f"  AR 95% CI: [{min(keep):.3f}, {max(keep):.3f}]")
    print(f"  Wald 95% CI: [{b-1.96*se:.3f}, {b+1.96*se:.3f}]")
    print(f"  -> AR interval is {'WIDER' if (max(keep)-min(keep))>(3.92*se) else 'comparable/narrower'}")
else:
    print("  AR CI empty - model rejected at all values (investigate)")

# ---------------- 3. ENDOGENEITY ----------------
print("\n\n### 3. ENDOGENEITY (Wu-Hausman) ###\n")
try:
    wh = res.wu_hausman()
    print(f"  {wh}")
    print(f"\n  If p > 0.10, OLS is consistent and more efficient.")
except Exception as e:
    print(f"  failed: {e}")
ols = smf.ols("logret ~ log_pm25 + " + " + ".join(CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags":5})
print(f"\n  OLS: {ols.params['log_pm25']:.4f} (se {ols.bse['log_pm25']:.4f})  "
      f"MDE={2.8*ols.bse['log_pm25']*np.log(2):.3f}pp")
print(f"  IV : {b:.4f} (se {se:.4f})  MDE={2.8*se*np.log(2):.3f}pp")

# ---------------- 4. REDUCED FORM ----------------
print("\n\n### 4. REDUCED FORM ###\n")
rf = smf.ols("logret ~ " + " + ".join(INST+CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags":5})
for v in INST:
    print(f"  {v:12s}: {rf.params[v]:8.4f} (se {rf.bse[v]:.4f})  p={rf.pvalues[v]:.3f}")
print(f"\n  Joint F-test on instruments: p = {rf.f_test(hyp).pvalue:.4f}")
print(f"  (Instruments jointly {'DO' if rf.f_test(hyp).pvalue<0.05 else 'do NOT'} predict returns)")

# ---------------- 5. SELECTION / ATTRITION ----------------
print("\n\n### 5. SELECTION TEST ###\n")
pm_all = pd.read_csv("almaty_pm25_daily.csv")
pm_all["date"] = pd.to_datetime(pm_all["date"])
pm_all = pm_all[(pm_all.sensor_id==25903) &
                (pm_all.date>="2020-04-09") & (pm_all.date<="2025-11-13")].copy()
pm_all["dropped"] = (pm_all.n_obs < 12).astype(int)
inc, exc = pm_all[pm_all.dropped==0].pm25, pm_all[pm_all.dropped==1].pm25
print(f"  Included days: n={len(inc)}, mean PM2.5={inc.mean():.2f}")
print(f"  Excluded days: n={len(exc)}, mean PM2.5={exc.mean():.2f}")
if len(exc)>1:
    t,p = stats.ttest_ind(inc, exc, equal_var=False)
    print(f"  Difference: {inc.mean()-exc.mean():.2f}  t={t:.2f}  p={p:.3f}")
    print(f"  -> Attrition {'IS' if p<0.05 else 'is NOT'} related to pollution level")
print(f"  Corr(n_obs, pm25) = {pm_all[['n_obs','pm25']].corr().iloc[0,1]:.3f}")

# ---------------- 6. HAC LAG SENSITIVITY ----------------
print("\n\n### 6. HAC LAG SENSITIVITY ###\n")
for L in [0,1,3,5,10,20]:
    m = smf.ols("logret ~ log_pm25 + " + " + ".join(CTRL), data=D).fit(
            cov_type="HAC", cov_kwds={"maxlags":L}) if L>0 else \
        smf.ols("logret ~ log_pm25 + " + " + ".join(CTRL), data=D).fit()
    print(f"  maxlags={L:2d}: OLS se = {m.bse['log_pm25']:.4f}")
print(f"  (Rule of thumb 4*(n/100)^(2/9) = {4*(len(D)/100)**(2/9):.1f})")

# ---------------- 7. WEEKEND NATURAL EXPERIMENT ----------------
print("\n\n### 7. WEEKEND POLLUTION -> MONDAY RETURN ###\n")
pm_d = pm_all[pm_all.n_obs>=12][["date","pm25"]].copy()
pm_d["dow"] = pm_d.date.dt.dayofweek
wknd = pm_d[pm_d.dow.isin([5,6])].copy()
wknd["monday"] = wknd.date + pd.to_timedelta((7 - wknd.dow) % 7, unit="D")
wk = wknd.groupby("monday").agg(wknd_pm=("pm25","mean"), nd=("pm25","size")).reset_index()
wk = wk[wk.nd==2]
mon = D[D.dow==0][["date","logret","temp","precip","wind_speed","month","year"]]
mw = mon.merge(wk, left_on="date", right_on="monday")
print(f"  N Mondays with full weekend data: {len(mw)}")
if len(mw) > 40:
    mw["log_wknd"] = np.log(mw.wknd_pm)
    m = smf.ols("logret ~ log_wknd + temp + precip + wind_speed + C(month) + C(year)",
                data=mw).fit(cov_type="HAC", cov_kwds={"maxlags":3})
    print(f"  Coefficient: {m.params['log_wknd']:.4f} (se {m.bse['log_wknd']:.4f})  "
          f"p={m.pvalues['log_wknd']:.3f}")
    print(f"  -> No reverse causality possible: market closed during exposure")
else:
    print("  insufficient observations")

D.to_csv("panel_analysis.csv", index=False)
print("\n\nSaved panel_analysis.csv")


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi":140, "font.size":9, "axes.grid":True,
                     "grid.alpha":0.25, "axes.spines.top":False, "axes.spines.right":False})

D = pd.read_csv("panel_analysis.csv"); D["date"] = pd.to_datetime(D["date"])

# FIG 1: PM2.5 time series
fig, ax = plt.subplots(figsize=(9,3))
ax.plot(D.date, D.pm25_raw, lw=0.5, color="#444")
ax.axhline(5, ls="--", c="green", lw=1, label="WHO annual guideline (5)")
ax.axhline(D.pm25_raw.mean(), ls="--", c="red", lw=1,
           label=f"Sample mean ({D.pm25_raw.mean():.1f})")
ax.set(ylabel="PM2.5 (µg/m³)", title="Figure 1. Daily PM2.5, Almaty US Embassy monitor")
ax.legend(frameon=False, fontsize=8); plt.tight_layout()
plt.savefig("fig1_timeseries.png", bbox_inches="tight"); plt.show()

# FIG 2: seasonality - the inversion mechanism
fig, axes = plt.subplots(1,2, figsize=(9,3))
mp = D.groupby("month").agg(pm=("pm25_raw","mean"), blh=("blh","mean"))
axes[0].bar(mp.index, mp.pm, color="#8B4513"); axes[0].set(xlabel="Month",
            ylabel="Mean PM2.5 (µg/m³)", title="Pollution by month")
axes[1].bar(mp.index, mp.blh, color="#4682B4"); axes[1].set(xlabel="Month",
            ylabel="Mean BLH (m)", title="Boundary layer height by month")
plt.suptitle("Figure 2. Seasonal inversion pattern in the Almaty basin", y=1.03)
plt.tight_layout(); plt.savefig("fig2_seasonality.png", bbox_inches="tight"); plt.show()

# FIG 3: first stage
def resid(y, X): return sm.OLS(y, sm.add_constant(X)).fit().resid
CTRL2 = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
        ["temp","precip","wind_speed"] + (["lockdown"] if D.lockdown.nunique()>1 else [])
rx = resid(D.log_blh, D[CTRL2]); ry = resid(D.log_pm25, D[CTRL2])
fig, ax = plt.subplots(figsize=(4.5,3.5))
ax.scatter(rx, ry, s=4, alpha=0.25, color="#333")
z = np.polyfit(rx, ry, 1); xs = np.linspace(rx.min(), rx.max(), 50)
ax.plot(xs, np.polyval(z, xs), c="crimson", lw=2)
ax.set(xlabel="Log BLH (residualised)", ylabel="Log PM2.5 (residualised)",
       title=f"Figure 3. First stage (slope {z[0]:.3f})")
plt.tight_layout(); plt.savefig("fig3_firststage.png", bbox_inches="tight"); plt.show()

# FIG 4: THE KEY FIGURE - binscatter of returns on pollution
rx2 = resid(D.log_pm25, D[CTRL2]); ry2 = resid(D.logret, D[CTRL2])
bins = pd.qcut(rx2, 20, labels=False)
bx = pd.Series(rx2).groupby(bins).mean(); by = pd.Series(ry2).groupby(bins).mean()
bse = pd.Series(ry2).groupby(bins).sem()
fig, ax = plt.subplots(figsize=(5,3.5))
ax.errorbar(bx, by, yerr=1.96*bse, fmt="o", ms=5, color="#1f3b73",
            ecolor="#999", elinewidth=1, capsize=2)
z2 = np.polyfit(rx2, ry2, 1)
ax.plot(bx, np.polyval(z2, bx), c="crimson", lw=1.5,
        label=f"OLS slope {z2[0]:.3f}")
ax.axhline(0, c="k", lw=0.6, ls=":")
ax.set(xlabel="Log PM2.5 (residualised)", ylabel="Log return, % (residualised)",
       title="Figure 4. No relationship between pollution and returns")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("fig4_binscatter.png", bbox_inches="tight"); plt.show()

# FIG 5: coefficient stability
specs = [("Preferred",0.144,0.144),("Cov >=18h",0.087,0.160),("Cov >=6h",0.146,0.137),
         ("From 2022",-0.027,0.212),("From 2023",0.014,0.202),("Excl 2025",0.053,0.171),
         ("Winter only",0.127,0.147),("Winsorised",0.152,0.133),
         ("Drop pressure",0.266,0.221),("BLH only",-0.133,0.277)]
fig, ax = plt.subplots(figsize=(5.5,4))
ys = np.arange(len(specs))[::-1]
for i,(lab,est,s) in enumerate(specs):
    y = ys[i]; c = "crimson" if lab=="Preferred" else "#333"
    ax.errorbar(est, y, xerr=1.96*s, fmt="o", ms=5, color=c, ecolor=c,
                elinewidth=1.2, capsize=2)
ax.axvline(0, c="k", lw=0.8, ls="--")
ax.set_yticks(ys); ax.set_yticklabels([s[0] for s in specs], fontsize=8)
ax.set(xlabel="Coefficient on log PM2.5 (95% CI)",
       title="Figure 5. Estimate stability across specifications")
plt.tight_layout(); plt.savefig("fig5_specification.png", bbox_inches="tight"); plt.show()

print("Saved: fig1-fig5 .png")


In [ ]:
import statsmodels.formula.api as smf
D = pd.read_csv("panel_analysis.csv")
base = ["temp","precip","wind_speed"]
for lab, extra in [("no month FE", []),
                   ("month FE", ["C(month)"]),
                   ("month + year FE", ["C(month)","C(year)"])]:
    f = "log_pm25 ~ log_blh + u + v + pressure + " + " + ".join(base + extra)
    m = smf.ols(f, data=D).fit(cov_type="HAC", cov_kwds={"maxlags":5})
    print(f"{lab:18s}: BLH = {m.params['log_blh']:.3f} "
          f"(se {m.bse['log_blh']:.3f})  R2={m.rsquared:.3f}")


In [ ]:
import numpy as np, pandas as pd
D = pd.read_csv("panel_final.csv")
sd_ret, sd_lpm = D.logret.std(), D.log_pm25.std()
bound_pp = (0.144 + 1.645*0.144) * np.log(2)
print(f"SD log PM2.5 = {sd_lpm:.3f}  (doubling = {np.log(2)/sd_lpm:.2f} SDs)")
print(f"Bound = {bound_pp:.3f}pp = {bound_pp/sd_ret:.3f} return SDs")
print(f"Per 1 SD pollution: {bound_pp*(sd_lpm/np.log(2))/sd_ret:.3f} return SDs")


In [ ]:
import pandas as pd
pm = pd.read_csv("almaty_pm25_daily.csv")
pm = pm[pm.sensor_id==25903].copy()
pm["date"] = pd.to_datetime(pm["date"]); pm["ym"] = pm.date.dt.to_period("M")
g = pm.groupby("ym").agg(n=("pm25","size"), mean=("pm25","mean"),
                         sd=("pm25","std"), nobs=("n_obs","mean")).round(1)
print(g.to_string())


In [ ]:
import pandas as pd
# pull hourly for sensor 25903 from OpenAQ: /v3/sensors/25903/hours
# then, per day, count observations falling in 07:00-17:30 Almaty local
h["hour"] = h.datetime_local.dt.hour
win = h[(h.hour >= 7) & (h.hour <= 17)]
cov = win.groupby(win.datetime_local.dt.date).size()
print(cov.describe())
print("Days with >=8 of 11 window hours:", (cov >= 8).sum())
print("By year:", cov.groupby(pd.to_datetime(cov.index).year).apply(lambda s: (s>=8).mean()).round(2))



In [ ]:
import requests, time, pandas as pd, numpy as np

API_KEY = "PASTE_YOUR_KEY_HERE"
HEADERS = {"X-API-Key": API_KEY}
SENSOR  = 25903

rows, page = [], 1
while True:
    r = requests.get(f"https://api.openaq.org/v3/sensors/{SENSOR}/hours",
                     headers=HEADERS, timeout=90,
                     params={"datetime_from":"2020-04-01","datetime_to":"2025-11-14",
                             "limit":1000,"page":page})
    if r.status_code == 429:
        print("rate limited, waiting..."); time.sleep(20); continue
    if r.status_code != 200:
        print(r.status_code, r.text[:200]); break
    res = r.json().get("results", [])
    if not res: break
    for rec in res:
        p = (rec.get("period") or {}).get("datetimeFrom") or {}
        rows.append({"utc": p.get("utc"), "local": p.get("local"),
                     "pm25": rec.get("value")})
    print(f"  page {page}: {len(res)}")
    if len(res) < 1000: break
    page += 1
    time.sleep(0.3)

h = pd.DataFrame(rows).dropna(subset=["local","pm25"])
h["dt"]   = pd.to_datetime(h["local"], errors="coerce")
h["hour"] = h.dt.dt.hour
h["date"] = h.dt.dt.date
h["year"] = h.dt.dt.year
h.to_csv("almaty_pm25_hourly.csv", index=False)
print(f"\nTotal hourly obs: {len(h):,}")

# --- coverage inside the trading window ---
WIN_START, WIN_END = 7, 17          # 07:00-17:59 local; adjust once KASE hours confirmed
NEED = 8                            # minimum hours required to build the regressor

win = h[(h.hour >= WIN_START) & (h.hour <= WIN_END)]
cov = win.groupby("date").size()
print(f"\nWindow hours available per day ({WIN_START}:00-{WIN_END}:59):")
print(cov.describe().round(1).to_string())
print(f"\nDays with >={NEED} window hours: {(cov>=NEED).sum()} of {len(cov)}")

cd = pd.DataFrame({"date": pd.to_datetime(cov.index), "n": cov.values})
cd["year"], cd["month"] = cd.date.dt.year, cd.date.dt.month
print("\nShare of days meeting threshold, by year:")
print(cd.groupby("year")["n"].apply(lambda s: (s>=NEED).mean()).round(2).to_string())

print("\nBy winter month (the high-variation period):")
wt = cd[cd.month.isin([11,12,1,2])]
print(wt.groupby("year")["n"].agg(days="size",
      pass_rate=lambda s: round((s>=NEED).mean(),2)).to_string())

# --- what the trading-window regressor looks like vs calendar-day ---
tw = win.groupby("date")["pm25"].mean().rename("pm25_window")
cd_mean = h.groupby("date")["pm25"].mean().rename("pm25_calendar")
cmp = pd.concat([tw, cd_mean], axis=1).dropna()
cmp = cmp[cov.reindex(cmp.index) >= NEED]
print(f"\nTrading-window vs calendar-day, n={len(cmp)}")
print(f"  correlation: {cmp.corr().iloc[0,1]:.3f}")
print(f"  SD log window:   {np.log(cmp.pm25_window).std():.3f}")
print(f"  SD log calendar: {np.log(cmp.pm25_calendar).std():.3f}")


In [ ]:
from google.colab import userdata
API_KEY = userdata.get('OPENAQ_KEY')
print(f"Key loaded: {API_KEY[:8]}...{API_KEY[-4:]}  (length {len(API_KEY)})")


In [ ]:
import requests, time, pandas as pd, numpy as np

HEADERS = {"X-API-Key": API_KEY}
SENSOR  = 25903


In [ ]:
import requests, time, pandas as pd, numpy as np
from google.colab import userdata

API_KEY = userdata.get('OPENAQ_KEY')
HEADERS = {"X-API-Key": API_KEY}
SENSOR  = 25903
print("Starting fetch...")

rows, page = [], 1
while True:
    r = requests.get(f"https://api.openaq.org/v3/sensors/{SENSOR}/hours",
                     headers=HEADERS, timeout=90,
                     params={"datetime_from":"2020-04-01","datetime_to":"2025-11-14",
                             "limit":1000,"page":page})
    if r.status_code == 429:
        print("rate limited, waiting..."); time.sleep(20); continue
    if r.status_code != 200:
        print("ERROR", r.status_code, r.text[:200]); break
    res = r.json().get("results", [])
    if not res:
        print("no more results"); break
    for rec in res:
        p = (rec.get("period") or {}).get("datetimeFrom") or {}
        rows.append({"utc": p.get("utc"), "local": p.get("local"),
                     "pm25": rec.get("value")})
    print(f"  page {page}: {len(res)} records")
    if len(res) < 1000: break
    page += 1
    time.sleep(0.3)

print(f"\nCollected {len(rows)} rows")
h = pd.DataFrame(rows).dropna(subset=["local","pm25"])
h["dt"]   = pd.to_datetime(h["local"], errors="coerce")
h["hour"] = h.dt.dt.hour
h["date"] = h.dt.dt.date
h["year"] = h.dt.dt.year
h.to_csv("almaty_pm25_hourly.csv", index=False)
print(f"Saved almaty_pm25_hourly.csv — {len(h):,} hourly observations")
print(f"Range: {h.dt.min()} to {h.dt.max()}")


In [ ]:
import requests, time, pandas as pd, numpy as np
from google.colab import userdata

API_KEY = userdata.get('OPENAQ_KEY')
HEADERS = {"X-API-Key": API_KEY}
SENSOR  = 25903

WINDOWS = [("2020-04-01","2020-12-31"), ("2021-01-01","2021-12-31"),
           ("2022-01-01","2022-12-31"), ("2023-01-01","2023-12-31"),
           ("2024-01-01","2024-12-31"), ("2025-01-01","2025-11-14")]

def fetch(dfrom, dto):
    out, page = [], 1
    while True:
        try:
            r = requests.get(f"https://api.openaq.org/v3/sensors/{SENSOR}/hours",
                             headers=HEADERS, timeout=120,
                             params={"datetime_from":dfrom,"datetime_to":dto,
                                     "limit":1000,"page":page})
        except Exception as e:
            print(f"    request failed: {e}"); time.sleep(5); continue
        if r.status_code == 429:
            time.sleep(20); continue
        if r.status_code == 408:
            print("    408 timeout — retrying"); time.sleep(5); continue
        if r.status_code != 200:
            print(f"    HTTP {r.status_code}: {r.text[:150]}"); break
        res = r.json().get("results", [])
        if not res: break
        for rec in res:
            p = (rec.get("period") or {}).get("datetimeFrom") or {}
            out.append({"local": p.get("local"), "pm25": rec.get("value")})
        if len(res) < 1000: break
        page += 1
        time.sleep(0.3)
    return out

rows = []
for a, b in WINDOWS:
    print(f"{a} -> {b}", end="  ")
    got = fetch(a, b)
    rows += got
    print(f"{len(got)} records")
    time.sleep(1)

print(f"\nTotal: {len(rows)} rows")
if rows:
    h = pd.DataFrame(rows).dropna(subset=["local","pm25"])
    h["dt"]   = pd.to_datetime(h["local"], errors="coerce", utc=True)
    h["hour"] = h.dt.dt.hour
    h["date"] = h.dt.dt.date
    h["year"] = h.dt.dt.year
    h = h.drop_duplicates(subset=["dt"]).sort_values("dt")
    h.to_csv("almaty_pm25_hourly.csv", index=False)
    print(f"Saved — {len(h):,} unique hourly observations")
    print(f"Range: {h.dt.min()} to {h.dt.max()}")
    print(h.groupby("year").size().to_string())
else:
    print("Nothing returned — try shortening WINDOWS to quarters.")


In [ ]:
import pandas as pd, numpy as np

h = pd.read_csv("almaty_pm25_hourly.csv")

# --- diagnostic: what does the raw local string look like?
print("Raw 'local' samples:")
print(h["local"].head(3).to_string())
print(h["local"].tail(3).to_string())

# --- extract local hour directly from the ISO string (offset-safe)
h["hour"]  = h["local"].str.slice(11, 13).astype(int)
h["date"]  = pd.to_datetime(h["local"].str.slice(0, 10))
h["year"]  = h.date.dt.year
h["month"] = h.date.dt.month
h["offset"] = h["local"].str.slice(-6)

print("\nOffsets present (confirms the 2024 timezone change):")
print(h.groupby([h.year, h.offset]).size().to_string())

print("\nObservations by local hour (should be roughly flat if 24h monitoring):")
print(h.groupby("hour").size().to_string())

# --- coverage under candidate trading windows
print("\n" + "="*60)
print("TRADING-WINDOW COVERAGE")
print("="*60)

for start, end, need, lab in [(7, 17, 8,  "07-17 (commute+trading)"),
                              (9, 17, 6,  "09-17"),
                              (11, 17, 5, "11-17 (core session)")]:
    w = h[(h.hour >= start) & (h.hour <= end)]
    cov = w.groupby("date").size()
    ok = (cov >= need)
    print(f"\n{lab}, threshold >={need}h")
    print(f"  days with any data: {len(cov)},  meeting threshold: {ok.sum()}")
    cd = pd.DataFrame({"date": cov.index, "n": cov.values})
    cd["year"] = cd.date.dt.year; cd["month"] = cd.date.dt.month
    print("  pass rate by year: " +
          "  ".join(f"{y}:{v:.2f}" for y, v in
                    cd.groupby('year')['n'].apply(lambda s: (s>=need).mean()).items()))
    wt = cd[cd.month.isin([11,12,1,2])]
    print("  WINTER pass rate:  " +
          "  ".join(f"{y}:{v:.2f}" for y, v in
                    wt.groupby('year')['n'].apply(lambda s: (s>=need).mean()).items()))

# --- does the fix actually change the regressor?
w = h[(h.hour >= 7) & (h.hour <= 17)]
cov = w.groupby("date").size()
tw  = w.groupby("date")["pm25"].mean()
cal = h.groupby("date")["pm25"].mean()
cmp = pd.concat([tw.rename("window"), cal.rename("calendar")], axis=1).dropna()
cmp = cmp[cov.reindex(cmp.index) >= 8]
print("\n" + "="*60)
print(f"WINDOW vs CALENDAR   n={len(cmp)}")
print(f"  correlation (levels): {cmp.corr().iloc[0,1]:.4f}")
print(f"  correlation (logs):   {np.log(cmp).corr().iloc[0,1]:.4f}")
print(f"  SD log window:   {np.log(cmp.window).std():.4f}")
print(f"  SD log calendar: {np.log(cmp.calendar).std():.4f}")
print(f"  mean window {cmp.window.mean():.2f}  vs calendar {cmp.calendar.mean():.2f}")


In [ ]:
import pandas as pd, numpy as np
h = pd.read_csv("almaty_pm25_hourly.csv")
h["hour"] = h["local"].str.slice(11,13).astype(int)
h["date"] = pd.to_datetime(h["local"].str.slice(0,10))
h["month"] = h.date.dt.month

for lo, hi, need, lab in [(7,17,8,"07:00-17:59  full exposure (PRIMARY)"),
                          (11,17,5,"11:00-17:59  session only"),
                          (7,10,3, "07:00-10:59  pre-session only")]:
    w = h[(h.hour>=lo)&(h.hour<=hi)]
    cov = w.groupby("date").size()
    cd = pd.DataFrame({"date":cov.index,"n":cov.values})
    cd["year"]=cd.date.dt.year; cd["month"]=cd.date.dt.month
    wt = cd[cd.month.isin([11,12,1,2])]
    print(f"\n{lab}   threshold >={need}h")
    print(f"  days retained: {(cov>=need).sum()} of {len(cov)}")
    print("  winter pass: " + "  ".join(
        f"{y}:{v:.2f}" for y,v in wt.groupby('year')['n']
        .apply(lambda s:(s>=need).mean()).items()))
    m = w.groupby("date")["pm25"].mean()
    m = m[cov>=need]
    print(f"  SD log: {np.log(m).std():.3f}   mean: {m.mean():.1f}   n={len(m)}")


In [ ]:
import numpy as np, pandas as pd

D = pd.read_csv("panel_final.csv")
lpm = np.log(D.pm25_raw.values)
lpm = (lpm - lpm.mean()) / lpm.std()          # standardised daily treatment
T = len(lpm)

def simulate(n_stocks, true_beta, n_sims=400, rho_stock=0.5, seed=0):
    """Detection rate for interaction beta. Outcome standardised.
       rho_stock = within-stock error correlation (persistence)."""
    rng = np.random.default_rng(seed)
    hits = 0
    for _ in range(n_sims):
        mod = rng.normal(0, 1, n_stocks)                    # retail intensity
        common = rng.normal(0, 1, T)                        # market-wide shock
        idio = rng.normal(0, 1, (T, n_stocks))
        y = (true_beta * np.outer(lpm, mod)
             + np.sqrt(rho_stock) * common[:, None]
             + np.sqrt(1 - rho_stock) * idio)
        # demean by date and stock (absorbs date FE and stock FE)
        y = y - y.mean(1, keepdims=True) - y.mean(0, keepdims=True) + y.mean()
        x = np.outer(lpm, mod)
        x = x - x.mean(1, keepdims=True) - x.mean(0, keepdims=True) + x.mean()
        b = (x * y).sum() / (x * x).sum()
        # cluster-robust SE, clustered on stock
        e = y - b * x
        num = sum((x[:, j] * e[:, j]).sum() ** 2 for j in range(n_stocks))
        se = np.sqrt(num) / (x * x).sum()
        if se > 0 and abs(b / se) > 2.2:      # ~t crit for few clusters
            hits += 1
    return hits / n_sims

print("DETECTION RATE (power), OLS-equivalent upper bound")
print("effect = SD of outcome per SD of pollution, in the most retail stock\n")
print(f"{'beta':>7} | " + " | ".join(f"{n:>3} stocks" for n in [8,12,20,30]))
print("-" * 52)
for beta in [0.02, 0.05, 0.10, 0.15, 0.25]:
    row = [f"{simulate(n, beta):.2f}" for n in [8,12,20,30]]
    print(f"{beta:>7.2f} | " + " | ".join(f"{r:>9}" for r in row))

print("\nNOTE: this is an UPPER bound on power.")
print("Actual design is 2SLS, which inflates SE by roughly 1/sqrt(partial R2)")
print("=> multiply required effect by ~2.8 given partial R2 = 0.125")


In [ ]:
import numpy as np, pandas as pd
from linearmodels.iv import IV2SLS

D = pd.read_csv("panel_analysis.csv")
INST = ["log_blh","u","v","pressure"]
CTRL = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       ["temp","precip","wind_speed"] + (["lockdown"] if D.lockdown.nunique()>1 else [])
D["const"] = 1
n = len(D)
X, Z = D[["const"]+CTRL], D[INST]

def beta(y):
    return IV2SLS(pd.Series(y, index=D.index), X, D["log_pm25"], Z).fit().params["log_pm25"]

actual = beta(D.logret.values)
rng = np.random.default_rng(42)
null = []
for _ in range(1000):
    k = int(rng.integers(6, (n-30)//5)) * 5        # multiples of 5: weekday alignment
    try: null.append(beta(np.roll(D.logret.values, k)))
    except Exception: pass

null = np.array(null)
print(f"Actual: {actual:.4f}   HAC SE: 0.1438")
print(f"Randomisation SD: {null.std():.4f}   ratio: {null.std()/0.1438:.2f}")
print(f"RI p-value (vs zero): {(np.abs(null)>=abs(actual)).mean():.3f}")
print(f"Null 2.5-97.5%: [{np.percentile(null,2.5):.3f}, {np.percentile(null,97.5):.3f}]")
print(f"\nImplied bound using RI SD: "
      f"{(actual + 1.645*null.std())*np.log(2):.3f}pp  (HAC gives 0.264pp)")


In [ ]:
!pip install linearmodels -q
import os
csvs = sorted(f for f in os.listdir() if f.endswith('.csv'))
print("Files present:", csvs if csvs else "NONE — all lost")


In [ ]:
from google.colab import files
uploaded = files.upload()   # select "KASE Historical Data.csv"
print(list(uploaded.keys()))


In [ ]:
"""
FULL PIPELINE REBUILD — Almaty pollution / KASE returns
Run after re-uploading "KASE Historical Data.csv".
Rebuilds every dataset and backs up to Google Drive.
"""

import requests, time, numpy as np, pandas as pd
from google.colab import userdata, drive

API_KEY = userdata.get('OPENAQ_KEY')
HEADERS = {"X-API-Key": API_KEY}
SENSOR  = 25903                      # US Embassy Almaty, AirNow
LAT, LON = 43.2529, 76.9312

# ─────────────────────────────────────────────── 1. PM2.5 DAILY
print("1/5  Fetching PM2.5 daily...")
rows, page = [], 1
while True:
    r = requests.get(f"https://api.openaq.org/v3/sensors/{SENSOR}/days",
                     headers=HEADERS, timeout=90,
                     params={"datetime_from":"2020-04-01","datetime_to":"2025-11-14",
                             "limit":1000,"page":page})
    if r.status_code == 429: time.sleep(20); continue
    if r.status_code != 200: print("  ", r.status_code, r.text[:120]); break
    res = r.json().get("results", [])
    if not res: break
    for rec in res:
        p = (rec.get("period") or {}).get("datetimeFrom") or {}
        rows.append({"date": p.get("local"), "pm25": rec.get("value"),
                     "n_obs": (rec.get("coverage") or {}).get("observedCount")})
    if len(res) < 1000: break
    page += 1; time.sleep(0.3)

pm = pd.DataFrame(rows)
pm["date"] = pd.to_datetime(pm["date"].str.slice(0,10))
pm["sensor_id"] = SENSOR
pm = pm.dropna(subset=["pm25"]).drop_duplicates("date").sort_values("date")
pm.to_csv("almaty_pm25_daily.csv", index=False)
print(f"     {len(pm)} days, {pm.date.min().date()} to {pm.date.max().date()}")

# ─────────────────────────────────────────────── 2. PM2.5 HOURLY
print("2/5  Fetching PM2.5 hourly (year by year)...")
hrows = []
for a, b in [("2020-04-01","2020-12-31"),("2021-01-01","2021-12-31"),
             ("2022-01-01","2022-12-31"),("2023-01-01","2023-12-31"),
             ("2024-01-01","2024-12-31"),("2025-01-01","2025-11-14")]:
    page = 1
    while True:
        r = requests.get(f"https://api.openaq.org/v3/sensors/{SENSOR}/hours",
                         headers=HEADERS, timeout=120,
                         params={"datetime_from":a,"datetime_to":b,
                                 "limit":1000,"page":page})
        if r.status_code in (429, 408): time.sleep(10); continue
        if r.status_code != 200: break
        res = r.json().get("results", [])
        if not res: break
        for rec in res:
            p = (rec.get("period") or {}).get("datetimeFrom") or {}
            hrows.append({"local": p.get("local"), "pm25": rec.get("value")})
        if len(res) < 1000: break
        page += 1; time.sleep(0.3)
    print(f"     {a[:4]}: {len(hrows)} cumulative")

h = pd.DataFrame(hrows).dropna(subset=["local","pm25"]).drop_duplicates("local")
h["hour"] = h["local"].str.slice(11,13).astype(int)
h["date"] = pd.to_datetime(h["local"].str.slice(0,10))
h.to_csv("almaty_pm25_hourly.csv", index=False)
print(f"     {len(h):,} hourly observations")

# ─────────────────────────────────────────────── 3. WEATHER
print("3/5  Fetching ERA5 weather...")
wx_raw = requests.get("https://archive-api.open-meteo.com/v1/archive",
    params={"latitude":LAT,"longitude":LON,
            "start_date":"2020-01-01","end_date":"2025-12-31",
            "hourly":"temperature_2m,precipitation,wind_speed_10m,wind_direction_10m,"
                     "boundary_layer_height,surface_pressure,cloud_cover,"
                     "shortwave_radiation",
            "timezone":"Asia/Almaty"}, timeout=180).json()["hourly"]

w = pd.DataFrame(wx_raw)
w["time"] = pd.to_datetime(w["time"]); w["date"] = w["time"].dt.date
rad = np.deg2rad(w["wind_direction_10m"])
w["u"] = -w["wind_speed_10m"] * np.sin(rad)
w["v"] = -w["wind_speed_10m"] * np.cos(rad)

wx = w.groupby("date").agg(
    temp=("temperature_2m","mean"), temp_min=("temperature_2m","min"),
    precip=("precipitation","sum"), wind_speed=("wind_speed_10m","mean"),
    blh=("boundary_layer_height","mean"), blh_min=("boundary_layer_height","min"),
    pressure=("surface_pressure","mean"), cloud=("cloud_cover","mean"),
    radiation=("shortwave_radiation","mean"),
    u=("u","mean"), v=("v","mean")).reset_index()
wx["date"] = pd.to_datetime(wx["date"])
wx.to_csv("almaty_weather_daily.csv", index=False)
print(f"     {len(wx)} days  (cloud + radiation now included)")

# ─────────────────────────────────────────────── 4. KASE
print("4/5  Cleaning KASE...")
ks = pd.read_csv("KASE Historical Data.csv")
ks.columns = [c.strip().lower().replace(".","").replace(" ","_") for c in ks.columns]
ks["date"] = pd.to_datetime(ks["date"], format="%m/%d/%Y", errors="coerce")
if ks["date"].isna().mean() > .5:
    ks["date"] = pd.to_datetime(ks["date"], dayfirst=True, errors="coerce")
ks["price"] = pd.to_numeric(ks["price"].astype(str).str.replace(",","",regex=False),
                            errors="coerce")
ks = ks.dropna(subset=["date","price"]).sort_values("date").reset_index(drop=True)
ks["logret"] = np.log(ks["price"]).diff() * 100
ks = ks[["date","price","logret"]]
ks.to_csv("kase_clean.csv", index=False)
print(f"     {len(ks)} trading days")

# ─────────────────────────────────────────────── 5. PANEL
print("5/5  Building panel...")
p = pm[pm.n_obs >= 12][["date","pm25","n_obs"]].rename(columns={"pm25":"pm25_raw"})
D = ks.merge(p, on="date").merge(wx, on="date")
D = D[(D.date >= "2020-04-08") & (D.date <= "2025-11-13")].copy()
D["log_pm25"] = np.log(D.pm25_raw); D["log_blh"] = np.log(D.blh)
D["dow"] = D.date.dt.dayofweek; D["month"] = D.date.dt.month; D["year"] = D.date.dt.year
D["lockdown"] = (((D.date>="2020-03-16")&(D.date<="2020-05-11")) |
                 ((D.date>="2020-07-05")&(D.date<="2020-08-17"))).astype(int)
D["qantar"]   = ((D.date>="2022-01-04")&(D.date<="2022-01-19")).astype(int)
D = D.dropna(subset=["logret","log_pm25","log_blh","u","v","pressure"])
D = D.sort_values("date").reset_index(drop=True)
D.to_csv("panel_final.csv", index=False)

# control dummies
ctrl = ["temp","precip","wind_speed"]
for m in sorted(D.month.unique())[1:]: D[f"mo_{m}"]=(D.month==m).astype(int); ctrl.append(f"mo_{m}")
for d_ in sorted(D.dow.unique())[1:]:  D[f"dw_{d_}"]=(D.dow==d_).astype(int); ctrl.append(f"dw_{d_}")
for y in sorted(D.year.unique())[1:]:  D[f"yr_{y}"]=(D.year==y).astype(int); ctrl.append(f"yr_{y}")
D["const"] = 1
D.to_csv("panel_analysis.csv", index=False)
print(f"     PANEL: {len(D)} obs, {D.date.min().date()} to {D.date.max().date()}")

# ─────────────────────────────────────────────── BACKUP
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pollution_research
!cp *.csv /content/drive/MyDrive/pollution_research/
import os
print("\nBacked up:", os.listdir('/content/drive/MyDrive/pollution_research'))
print("\nRestore next session with:")
print("  !cp /content/drive/MyDrive/pollution_research/*.csv .")


In [ ]:
!pip install linearmodels -q
import numpy as np, pandas as pd
from linearmodels.iv import IV2SLS

D = pd.read_csv("panel_analysis.csv")
INST = ["log_blh","u","v","pressure"]
BASE = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       ["temp","precip","wind_speed"] + (["lockdown"] if D.lockdown.nunique()>1 else [])
D["const"] = 1

def run(ctrl, label):
    r = IV2SLS(D["logret"], D[["const"]+ctrl], D["log_pm25"], D[INST]).fit(cov_type="kernel")
    b, se = r.params["log_pm25"], r.std_errors["log_pm25"]
    try: F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except: F = np.nan
    print(f"{label:32s}: {b:7.4f} ({se:.4f})  p={r.pvalues['log_pm25']:.3f}  "
          f"F={F:6.1f}  bound={(b+1.645*se)*np.log(2):.3f}pp")
    return r, b, se

print(f"PANEL: {len(D)} obs\n")
r1, b1, se1 = run(BASE, "Baseline (as before)")
r2, b2, se2 = run(BASE+["cloud","radiation"], "+ cloud & radiation")
r3, b3, se3 = run(BASE+["cloud","radiation","qantar"], "+ cloud, radiation, Qantar")

print(f"\nSargan (preferred): {r3.sargan}")
print(f"Wu-Hausman: {r3.wu_hausman()}")

# ─── RANDOMISATION INFERENCE ───
print("\n" + "="*60)
print("RANDOMISATION INFERENCE — validating the standard error")
print("="*60)
CTRL = BASE + ["cloud","radiation","qantar"]
X, Z, n = D[["const"]+CTRL], D[INST], len(D)

def beta(y):
    return IV2SLS(pd.Series(y, index=D.index), X, D["log_pm25"], Z).fit().params["log_pm25"]

rng = np.random.default_rng(42); null = []
for i in range(1000):
    k = int(rng.integers(6, (n-30)//5)) * 5
    try: null.append(beta(np.roll(D.logret.values, k)))
    except Exception: pass

null = np.array(null)
print(f"Actual: {b3:.4f}   HAC SE: {se3:.4f}")
print(f"Randomisation SD: {null.std():.4f}   RATIO: {null.std()/se3:.2f}")
print(f"RI p-value: {(np.abs(null)>=abs(b3)).mean():.3f}")
print(f"Null 2.5-97.5%: [{np.percentile(null,2.5):.3f}, {np.percentile(null,97.5):.3f}]")
print(f"\nBound using HAC SE: {(b3+1.645*se3)*


In [ ]:
!pip install linearmodels -q
import numpy as np, pandas as pd
from linearmodels.iv import IV2SLS

D = pd.read_csv("panel_analysis.csv")
INST = ["log_blh","u","v","pressure"]
BASE = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       ["temp","precip","wind_speed"] + (["lockdown"] if D.lockdown.nunique()>1 else [])
D["const"] = 1

def run(ctrl, label):
    r = IV2SLS(D["logret"], D[["const"]+ctrl], D["log_pm25"], D[INST]).fit(cov_type="kernel")
    b, se = r.params["log_pm25"], r.std_errors["log_pm25"]
    try: F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except: F = np.nan
    print(f"{label:32s}: {b:7.4f} ({se:.4f})  p={r.pvalues['log_pm25']:.3f}  "
          f"F={F:6.1f}  bound={(b+1.645*se)*np.log(2):.3f}pp")
    return r, b, se

print(f"PANEL: {len(D)} obs\n")
r1, b1, se1 = run(BASE, "Baseline (as before)")
r2, b2, se2 = run(BASE+["cloud","radiation"], "+ cloud & radiation")
r3, b3, se3 = run(BASE+["cloud","radiation","qantar"], "+ cloud, radiation, Qantar")

print(f"\nSargan (preferred): {r3.sargan}")
print(f"Wu-Hausman: {r3.wu_hausman()}")

# ─── RANDOMISATION INFERENCE ───
print("\n" + "="*60)
print("RANDOMISATION INFERENCE — validating the standard error")
print("="*60)
CTRL = BASE + ["cloud","radiation","qantar"]
X, Z, n = D[["const"]+CTRL], D[INST], len(D)

def beta(y):
    return IV2SLS(pd.Series(y, index=D.index), X, D["log_pm25"], Z).fit().params["log_pm25"]

rng = np.random.default_rng(42); null = []
for i in range(1000):
    k = int(rng.integers(6, (n-30)//5)) * 5
    try: null.append(beta(np.roll(D.logret.values, k)))
    except Exception: pass

null = np.array(null)
print(f"Actual: {b3:.4f}   HAC SE: {se3:.4f}")
print(f"Randomisation SD: {null.std():.4f}   RATIO: {null.std()/se3:.2f}")
print(f"RI p-value: {(np.abs(null)>=abs(b3)).mean():.3f}")
print(f"Null 2.5-97.5%: [{np.percentile(null,2.5):.3f}, {np.percentile(null,97.5):.3f}]")
print(f"\nBound using HAC SE: {(b3+1.645*se3)*np.log(2):.3f}pp")
print(f"Bound using RI  SD: {(b3+1.645*null.std())*np.log(2):.3f}pp")
print(f"Benchmark (Heyes et al., verify): ~0.194pp")


In [ ]:
import numpy as np
b, se = -0.0827, 0.1200
print(f"Reject positive effects above: {(b+1.645*se)*np.log(2):.3f}pp")
print(f"Reject negative effects below: {(abs(b)+1.645*se)*np.log(2):.3f}pp")
for d, lab in [(0.194,"proportional transport"), (0.50,"linear transport")]:
    delta = d/np.log(2)
    t = (b + delta)/se
    print(f"H0: effect = -{d:.3f}pp ({lab}): t={t:.2f}  "
          f"{'REJECTED' if t>1.645 else 'not rejected'}")



In [ ]:
"""
FINAL ANALYSIS — Air pollution and KASE returns
Regenerates every table and figure on the corrected 977-obs panel.
Run after: !cp /content/drive/MyDrive/pollution_research/*.csv .
"""

import numpy as np, pandas as pd, warnings
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

D = pd.read_csv("panel_analysis.csv")
D["date"] = pd.to_datetime(D["date"])
D["const"] = 1

INST = ["log_blh", "u", "v", "pressure"]
FE   = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_"))]
CTRL = ["temp", "precip", "wind_speed", "cloud", "radiation"] + FE + \
       [c for c in ["lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]

LN2 = np.log(2)

def iv(data=None, y="logret", x="log_pm25", inst=None, ctrl=None):
    d = (data if data is not None else D).copy()
    d["const"] = 1
    inst = inst or INST
    ctrl = [c for c in (ctrl or CTRL) if c in d.columns and d[c].nunique() > 1]
    d = d.dropna(subset=[y, x] + inst)
    r = IV2SLS(d[y], d[["const"] + ctrl], d[x], d[inst]).fit(cov_type="kernel")
    b, se = r.params[x], r.std_errors[x]
    try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except: F = np.nan
    return dict(b=b, se=se, p=r.pvalues[x], n=len(d), F=F,
                bound=(abs(b) + 1.645 * se) * LN2, r=r)

W = 78
def head(t): print("\n" + "=" * W + f"\n{t}\n" + "=" * W)

# ════════════════════════════════════════════════ TABLE 1
head("TABLE 1 — DESCRIPTIVE STATISTICS")
lbl = {"logret":"Log return (%)", "pm25_raw":"PM2.5 (ug/m3)",
       "blh":"Boundary layer height (m)", "temp":"Temperature (C)",
       "precip":"Precipitation (mm)", "wind_speed":"Wind speed (m/s)",
       "pressure":"Surface pressure (hPa)", "cloud":"Cloud cover (%)",
       "radiation":"Shortwave radiation (W/m2)"}
cols = [c for c in lbl if c in D.columns]
t1 = D[cols].describe().T[["count","mean","std","min","50%","max"]].round(2)
t1.index = [lbl[i] for i in t1.index]
print(t1.to_string())
print(f"\nN = {len(D)}   {D.date.min().date()} to {D.date.max().date()}")
print(f"Return AR(1)      : {D.logret.autocorr(1):.3f}")
print(f"SD of log PM2.5   : {np.log(D.pm25_raw).std():.3f}")
print(f"Days >55 ug/m3    : {(D.pm25_raw>55).sum()}   >75: {(D.pm25_raw>75).sum()}")

# ════════════════════════════════════════════════ TABLE 2
head("TABLE 2 — FIRST STAGE (dep. var: log PM2.5)")
fs = smf.ols("log_pm25 ~ " + " + ".join(INST + CTRL), data=D
             ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
for v in INST:
    print(f"  {v:12s} {fs.params[v]:9.4f}  ({fs.bse[v]:.4f})   "
          f"t={fs.tvalues[v]:6.2f}   p={fs.pvalues[v]:.3f}")
main = iv()
print(f"\n  R-squared {fs.rsquared:.3f}    First-stage F {main['F']:.1f}")
try:
    print(f"  Partial R-squared "
          f"{main['r'].first_stage.diagnostics['partial.rsquared'].iloc[0]:.4f}")
except Exception: pass

# ════════════════════════════════════════════════ TABLE 3
head("TABLE 3 — MAIN RESULTS")
ols = smf.ols("logret ~ log_pm25 + " + " + ".join(CTRL), data=D
              ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"  (1) OLS          {ols.params['log_pm25']:8.4f} ({ols.bse['log_pm25']:.4f})"
      f"  p={ols.pvalues['log_pm25']:.3f}")
blh = iv(inst=["log_blh"])
print(f"  (2) IV, BLH only {blh['b']:8.4f} ({blh['se']:.4f})  p={blh['p']:.3f}"
      f"  F={blh['F']:.1f}")
print(f"  (3) IV, full set {main['b']:8.4f} ({main['se']:.4f})  p={main['p']:.3f}"
      f"  F={main['F']:.1f}")
b, se = main["b"], main["se"]
print(f"\n  95% CI      [{b-1.96*se:.4f}, {b+1.96*se:.4f}]")
print(f"  Sargan      {main['r'].sargan.stat:.4f}  p={main['r'].sargan.pval:.4f}")
try:
    wh = main["r"].wu_hausman()
    print(f"  Wu-Hausman  {wh.stat:.4f}  p={wh.pval:.4f}")
except Exception: pass

# ════════════════════════════════════════════════ EQUIVALENCE
head("EQUIVALENCE TESTS (TOST)")
print(f"  Point estimate {b:.4f}  (SE {se:.4f})")
print(f"  90% CI [{b-1.645*se:.4f}, {b+1.645*se:.4f}]")
print(f"\n  Reject |effect| above {abs(b)+1.645*se:.4f} log pts "
      f"= {main['bound']:.3f}pp per doubling")
print(f"  In SD units: {main['bound']/D.logret.std():.3f} return SD "
      f"per {np.log(2)/np.log(D.pm25_raw).std():.2f} pollution SD")
print("\n  Transport tests vs Heyes et al. [VERIFY 2024 revision]:")
for d_, lab in [(0.194, "proportional transport"), (0.500, "linear transport")]:
    t = (b + d_/LN2) / se
    print(f"    H0 = -{d_:.3f}pp  ({lab:23s}) t={t:5.2f}  "
          f"{'REJECTED' if t > 1.645 else 'not rejected'}")

# ════════════════════════════════════════════════ TABLE 4
head("TABLE 4 — ROBUSTNESS")
print(f"  {'Specification':<34}{'coef':>9}{'se':>9}{'p':>8}{'N':>6}{'F':>8}{'bound':>8}")
print("  " + "-" * 74)

def row(lab, res):
    print(f"  {lab:<34}{res['b']:9.4f}{res['se']:9.4f}{res['p']:8.3f}"
          f"{res['n']:6d}{res['F']:8.1f}{res['bound']:8.3f}")

row("Preferred", main)
for mo in [18, 15, 6]:
    row(f"Coverage >={mo}h", iv(D[D.n_obs >= mo]))
for s, lab in [("2022-01-01","Sample from 2022"), ("2023-01-01","Sample from 2023")]:
    row(lab, iv(D[D.date >= s]))
row("Excluding 2025",       iv(D[D.date <= "2024-12-31"]))
row("Winter only (Nov-Mar)", iv(D[D.month.isin([11,12,1,2,3])]))
dw = D.copy(); lo, hi = dw.logret.quantile([.01,.99]); dw["logret"] = dw.logret.clip(lo,hi)
row("Winsorised 1%", iv(dw))
row("Drop pressure instr.", iv(inst=["log_blh","u","v"]))
row("No cloud/radiation",   iv(ctrl=[c for c in CTRL if c not in ("cloud","radiation")]))
if "qantar" in D.columns:
    row("Excl. Qantar window", iv(D[D.qantar == 0]))

print()
for L in [1, 2, 3]:
    d = D.copy()
    d[f"pm_l{L}"] = d.log_pm25.shift(L)
    for i in INST: d[f"{i}_l{L}"] = d[i].shift(L)
    row(f"Lag {L} day", iv(d.dropna(subset=[f"pm_l{L}"]),
                           x=f"pm_l{L}", inst=[f"{i}_l{L}" for i in INST]))
da = D.copy(); da["absret"] = da.logret.abs()
row("Outcome: |return|", iv(da, y="absret"))

print("\n  Threshold indicators (reduced form OLS):")
for thr in [35, 55, 75]:
    d = D.copy(); d["hi"] = (d.pm25_raw > thr).astype(int)
    if d.hi.sum() < 20: continue
    rr = smf.ols("logret ~ hi + " + " + ".join(CTRL), data=d
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"    >{thr:3d} ug/m3 ({d.hi.sum():3d} days): {rr.params['hi']:8.4f} "
          f"({rr.bse['hi']:.4f})  p={rr.pvalues['hi']:.3f}")

# ════════════════════════════════════════════════ PLACEBO / RI
head("PLACEBO AND RANDOMISATION INFERENCE")
for L in [1, 2]:
    d = D.copy(); d["lead"] = d.logret.shift(L)
    r = iv(d.dropna(subset=["lead"]), y="lead")
    print(f"  {L}-day lead   {r['b']:8.4f} ({r['se']:.4f})  p={r['p']:.3f}")

X, Z, n = D[["const"] + CTRL], D[INST], len(D)
def beta(y):
    return IV2SLS(pd.Series(y, index=D.index), X, D["log_pm25"], Z
                  ).fit().params["log_pm25"]
rng = np.random.default_rng(42); null = []
for _ in range(1000):
    k = int(rng.integers(6, (n - 30) // 5)) * 5
    try: null.append(beta(np.roll(D.logret.values, k)))
    except Exception: pass
null = np.array(null)
print(f"\n  Randomisation SD {null.std():.4f}   HAC SE {se:.4f}   "
      f"ratio {null.std()/se:.2f}")
print(f"  RI p-value {(np.abs(null) >= abs(b)).mean():.3f}")
print(f"  Null 2.5-97.5% [{np.percentile(null,2.5):.3f}, "
      f"{np.percentile(null,97.5):.3f}]")

# ════════════════════════════════════════════════ FIGURES
plt.rcParams.update({"figure.dpi":150, "font.size":9, "axes.grid":True,
                     "grid.alpha":.25, "axes.spines.top":False,
                     "axes.spines.right":False})

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(D.date, D.pm25_raw, lw=.5, c="#444")
ax.axhline(5, ls="--", c="green", lw=1, label="WHO guideline (5)")
ax.axhline(D.pm25_raw.mean(), ls="--", c="red", lw=1,
           label=f"Mean ({D.pm25_raw.mean():.1f})")
ax.set(ylabel="PM2.5 (ug/m3)", title="Figure 1. Daily PM2.5, Almaty US Embassy monitor")
ax.legend(frameon=False, fontsize=8); plt.tight_layout()
plt.savefig("fig1_timeseries.png", bbox_inches="tight"); plt.close()

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
mp = D.groupby("month").agg(pm=("pm25_raw","mean"), blh=("blh","mean"))
axes[0].bar(mp.index, mp.pm, color="#8B4513")
axes[0].set(xlabel="Month", ylabel="Mean PM2.5", title="Pollution by month")
axes[1].bar(mp.index, mp.blh, color="#4682B4")
axes[1].set(xlabel="Month", ylabel="Mean BLH (m)", title="Mixing depth by month")
plt.suptitle("Figure 2. Seasonal inversion pattern, Almaty basin", y=1.03)
plt.tight_layout(); plt.savefig("fig2_seasonality.png", bbox_inches="tight"); plt.close()

def resid(y, Xm): return sm.OLS(y, sm.add_constant(Xm)).fit().resid
C2 = D[[c for c in CTRL if D[c].nunique() > 1]]
rx, ry = resid(D.log_blh, C2), resid(D.log_pm25, C2)
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.scatter(rx, ry, s=4, alpha=.25, c="#333")
z = np.polyfit(rx, ry, 1); xs = np.linspace(rx.min(), rx.max(), 50)
ax.plot(xs, np.polyval(z, xs), c="crimson", lw=2)
ax.set(xlabel="Log BLH (residualised)", ylabel="Log PM2.5 (residualised)",
       title=f"Figure 3. First stage (slope {z[0]:.3f})")
plt.tight_layout(); plt.savefig("fig3_firststage.png", bbox_inches="tight"); plt.close()

rx2, ry2 = resid(D.log_pm25, C2), resid(D.logret, C2)
bins = pd.qcut(rx2, 20, labels=False)
bx, by = pd.Series(rx2).groupby(bins).mean(), pd.Series(ry2).groupby(bins).mean()
bse = pd.Series(ry2).groupby(bins).sem()
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.errorbar(bx, by, yerr=1.96*bse, fmt="o", ms=5, c="#1f3b73",
            ecolor="#999", elinewidth=1, capsize=2)
z2 = np.polyfit(rx2, ry2, 1)
ax.plot(bx, np.polyval(z2, bx), c="crimson", lw=1.5, label=f"slope {z2[0]:.3f}")
ax.axhline(0, c="k", lw=.6, ls=":")
ax.set(xlabel="Log PM2.5 (residualised)", ylabel="Log return, % (residualised)",
       title="Figure 4. No relationship between pollution and returns")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("fig4_binscatter.png", bbox_inches="tight"); plt.close()

specs = [("Preferred", main), ("Coverage >=18h", iv(D[D.n_obs>=18])),
         ("From 2022", iv(D[D.date>="2022-01-01"])),
         ("Winter only", iv(D[D.month.isin([11,12,1,2,3])])),
         ("Winsorised", iv(dw)), ("Drop pressure", iv(inst=["log_blh","u","v"])),
         ("BLH only", blh)]
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ys = np.arange(len(specs))[::-1]
for i, (lab, r_) in enumerate(specs):
    c = "crimson" if lab == "Preferred" else "#333"
    ax.errorbar(r_["b"], ys[i], xerr=1.96*r_["se"], fmt="o", ms=5,
                color=c, ecolor=c, elinewidth=1.2, capsize=2)
ax.axvline(0, c="k", lw=.8, ls="--")
ax.set_yticks(ys); ax.set_yticklabels([s[0] for s in specs], fontsize=8)
ax.set(xlabel="Coefficient on log PM2.5 (95% CI)",
       title="Figure 5. Estimate stability across specifications")
plt.tight_layout(); plt.savefig("fig5_specification.png", bbox_inches="tight"); plt.close()

print("\nFigures saved: fig1-fig5")
import subprocess
subprocess.run("cp *.png *.csv /content/drive/MyDrive/pollution_research/ 2>/dev/null",
               shell=True)
print("Backed up to Drive.")


In [ ]:
def iv(data=None, y="logret", x="log_pm25", inst=None, ctrl=None):
    d = (data if data is not None else D).copy()
    d["const"] = 1
    inst = inst or INST
    cand = [c for c in (ctrl or CTRL) if c in d.columns and d[c].nunique() > 1]
    d = d.dropna(subset=[y, x] + inst)

    # drop columns that are collinear given the constant, iteratively
    keep = []
    for c in cand:
        trial = d[["const"] + keep + [c]].astype(float).values
        if np.linalg.matrix_rank(trial) == trial.shape[1]:
            keep.append(c)

    r = IV2SLS(d[y], d[["const"] + keep], d[x], d[inst]).fit(cov_type="kernel")
    b, se = r.params[x], r.std_errors[x]
    try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except: F = np.nan
    return dict(b=b, se=se, p=r.pvalues[x], n=len(d), F=F,
                bound=(abs(b) + 1.645 * se) * LN2, r=r)


In [ ]:
"""
FINAL ANALYSIS — Air pollution and KASE returns
Regenerates every table and figure on the corrected 977-obs panel.
Run after: !cp /content/drive/MyDrive/pollution_research/*.csv .
"""

import numpy as np, pandas as pd, warnings
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

D = pd.read_csv("panel_analysis.csv")
D["date"] = pd.to_datetime(D["date"])
D["const"] = 1

INST = ["log_blh", "u", "v", "pressure"]
FE   = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_"))]
CTRL = ["temp", "precip", "wind_speed", "cloud", "radiation"] + FE + \
       [c for c in ["lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]

LN2 = np.log(2)

def iv(data=None, y="logret", x="log_pm25", inst=None, ctrl=None):
    d = (data if data is not None else D).copy()
    d["const"] = 1
    inst = inst or INST
    ctrl = [c for c in (ctrl or CTRL) if c in d.columns and d[c].nunique() > 1]
    d = d.dropna(subset=[y, x] + inst)
    r = IV2SLS(d[y], d[["const"] + ctrl], d[x], d[inst]).fit(cov_type="kernel")
    b, se = r.params[x], r.std_errors[x]
    try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except: F = np.nan
    return dict(b=b, se=se, p=r.pvalues[x], n=len(d), F=F,
                bound=(abs(b) + 1.645 * se) * LN2, r=r)

W = 78
def head(t): print("\n" + "=" * W + f"\n{t}\n" + "=" * W)

# ════════════════════════════════════════════════ TABLE 1
head("TABLE 1 — DESCRIPTIVE STATISTICS")
lbl = {"logret":"Log return (%)", "pm25_raw":"PM2.5 (ug/m3)",
       "blh":"Boundary layer height (m)", "temp":"Temperature (C)",
       "precip":"Precipitation (mm)", "wind_speed":"Wind speed (m/s)",
       "pressure":"Surface pressure (hPa)", "cloud":"Cloud cover (%)",
       "radiation":"Shortwave radiation (W/m2)"}
cols = [c for c in lbl if c in D.columns]
t1 = D[cols].describe().T[["count","mean","std","min","50%","max"]].round(2)
t1.index = [lbl[i] for i in t1.index]
print(t1.to_string())
print(f"\nN = {len(D)}   {D.date.min().date()} to {D.date.max().date()}")
print(f"Return AR(1)      : {D.logret.autocorr(1):.3f}")
print(f"SD of log PM2.5   : {np.log(D.pm25_raw).std():.3f}")
print(f"Days >55 ug/m3    : {(D.pm25_raw>55).sum()}   >75: {(D.pm25_raw>75).sum()}")

# ════════════════════════════════════════════════ TABLE 2
head("TABLE 2 — FIRST STAGE (dep. var: log PM2.5)")
fs = smf.ols("log_pm25 ~ " + " + ".join(INST + CTRL), data=D
             ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
for v in INST:
    print(f"  {v:12s} {fs.params[v]:9.4f}  ({fs.bse[v]:.4f})   "
          f"t={fs.tvalues[v]:6.2f}   p={fs.pvalues[v]:.3f}")
main = iv()
print(f"\n  R-squared {fs.rsquared:.3f}    First-stage F {main['F']:.1f}")
try:
    print(f"  Partial R-squared "
          f"{main['r'].first_stage.diagnostics['partial.rsquared'].iloc[0]:.4f}")
except Exception: pass

# ════════════════════════════════════════════════ TABLE 3
head("TABLE 3 — MAIN RESULTS")
ols = smf.ols("logret ~ log_pm25 + " + " + ".join(CTRL), data=D
              ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"  (1) OLS          {ols.params['log_pm25']:8.4f} ({ols.bse['log_pm25']:.4f})"
      f"  p={ols.pvalues['log_pm25']:.3f}")
blh = iv(inst=["log_blh"])
print(f"  (2) IV, BLH only {blh['b']:8.4f} ({blh['se']:.4f})  p={blh['p']:.3f}"
      f"  F={blh['F']:.1f}")
print(f"  (3) IV, full set {main['b']:8.4f} ({main['se']:.4f})  p={main['p']:.3f}"
      f"  F={main['F']:.1f}")
b, se = main["b"], main["se"]
print(f"\n  95% CI      [{b-1.96*se:.4f}, {b+1.96*se:.4f}]")
print(f"  Sargan      {main['r'].sargan.stat:.4f}  p={main['r'].sargan.pval:.4f}")
try:
    wh = main["r"].wu_hausman()
    print(f"  Wu-Hausman  {wh.stat:.4f}  p={wh.pval:.4f}")
except Exception: pass

# ════════════════════════════════════════════════ EQUIVALENCE
head("EQUIVALENCE TESTS (TOST)")
print(f"  Point estimate {b:.4f}  (SE {se:.4f})")
print(f"  90% CI [{b-1.645*se:.4f}, {b+1.645*se:.4f}]")
print(f"\n  Reject |effect| above {abs(b)+1.645*se:.4f} log pts "
      f"= {main['bound']:.3f}pp per doubling")
print(f"  In SD units: {main['bound']/D.logret.std():.3f} return SD "
      f"per {np.log(2)/np.log(D.pm25_raw).std():.2f} pollution SD")
print("\n  Transport tests vs Heyes et al. [VERIFY 2024 revision]:")
for d_, lab in [(0.194, "proportional transport"), (0.500, "linear transport")]:
    t = (b + d_/LN2) / se
    print(f"    H0 = -{d_:.3f}pp  ({lab:23s}) t={t:5.2f}  "
          f"{'REJECTED' if t > 1.645 else 'not rejected'}")

# ════════════════════════════════════════════════ TABLE 4
head("TABLE 4 — ROBUSTNESS")
print(f"  {'Specification':<34}{'coef':>9}{'se':>9}{'p':>8}{'N':>6}{'F':>8}{'bound':>8}")
print("  " + "-" * 74)

def row(lab, res):
    print(f"  {lab:<34}{res['b']:9.4f}{res['se']:9.4f}{res['p']:8.3f}"
          f"{res['n']:6d}{res['F']:8.1f}{res['bound']:8.3f}")

row("Preferred", main)
for mo in [18, 15, 6]:
    row(f"Coverage >={mo}h", iv(D[D.n_obs >= mo]))
for s, lab in [("2022-01-01","Sample from 2022"), ("2023-01-01","Sample from 2023")]:
    row(lab, iv(D[D.date >= s]))
row("Excluding 2025",       iv(D[D.date <= "2024-12-31"]))
row("Winter only (Nov-Mar)", iv(D[D.month.isin([11,12,1,2,3])]))
dw = D.copy(); lo, hi = dw.logret.quantile([.01,.99]); dw["logret"] = dw.logret.clip(lo,hi)
row("Winsorised 1%", iv(dw))
row("Drop pressure instr.", iv(inst=["log_blh","u","v"]))
row("No cloud/radiation",   iv(ctrl=[c for c in CTRL if c not in ("cloud","radiation")]))
if "qantar" in D.columns:
    row("Excl. Qantar window", iv(D[D.qantar == 0]))

print()
for L in [1, 2, 3]:
    d = D.copy()
    d[f"pm_l{L}"] = d.log_pm25.shift(L)
    for i in INST: d[f"{i}_l{L}"] = d[i].shift(L)
    row(f"Lag {L} day", iv(d.dropna(subset=[f"pm_l{L}"]),
                           x=f"pm_l{L}", inst=[f"{i}_l{L}" for i in INST]))
da = D.copy(); da["absret"] = da.logret.abs()
row("Outcome: |return|", iv(da, y="absret"))

print("\n  Threshold indicators (reduced form OLS):")
for thr in [35, 55, 75]:
    d = D.copy(); d["hi"] = (d.pm25_raw > thr).astype(int)
    if d.hi.sum() < 20: continue
    rr = smf.ols("logret ~ hi + " + " + ".join(CTRL), data=d
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"    >{thr:3d} ug/m3 ({d.hi.sum():3d} days): {rr.params['hi']:8.4f} "
          f"({rr.bse['hi']:.4f})  p={rr.pvalues['hi']:.3f}")

# ════════════════════════════════════════════════ PLACEBO / RI
head("PLACEBO AND RANDOMISATION INFERENCE")
for L in [1, 2]:
    d = D.copy(); d["lead"] = d.logret.shift(L)
    r = iv(d.dropna(subset=["lead"]), y="lead")
    print(f"  {L}-day lead   {r['b']:8.4f} ({r['se']:.4f})  p={r['p']:.3f}")

X, Z, n = D[["const"] + CTRL], D[INST], len(D)
def beta(y):
    return IV2SLS(pd.Series(y, index=D.index), X, D["log_pm25"], Z
                  ).fit().params["log_pm25"]
rng = np.random.default_rng(42); null = []
for _ in range(1000):
    k = int(rng.integers(6, (n - 30) // 5)) * 5
    try: null.append(beta(np.roll(D.logret.values, k)))
    except Exception: pass
null = np.array(null)
print(f"\n  Randomisation SD {null.std():.4f}   HAC SE {se:.4f}   "
      f"ratio {null.std()/se:.2f}")
print(f"  RI p-value {(np.abs(null) >= abs(b)).mean():.3f}")
print(f"  Null 2.5-97.5% [{np.percentile(null,2.5):.3f}, "
      f"{np.percentile(null,97.5):.3f}]")

# ════════════════════════════════════════════════ FIGURES
plt.rcParams.update({"figure.dpi":150, "font.size":9, "axes.grid":True,
                     "grid.alpha":.25, "axes.spines.top":False,
                     "axes.spines.right":False})

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(D.date, D.pm25_raw, lw=.5, c="#444")
ax.axhline(5, ls="--", c="green", lw=1, label="WHO guideline (5)")
ax.axhline(D.pm25_raw.mean(), ls="--", c="red", lw=1,
           label=f"Mean ({D.pm25_raw.mean():.1f})")
ax.set(ylabel="PM2.5 (ug/m3)", title="Figure 1. Daily PM2.5, Almaty US Embassy monitor")
ax.legend(frameon=False, fontsize=8); plt.tight_layout()
plt.savefig("fig1_timeseries.png", bbox_inches="tight"); plt.close()

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
mp = D.groupby("month").agg(pm=("pm25_raw","mean"), blh=("blh","mean"))
axes[0].bar(mp.index, mp.pm, color="#8B4513")
axes[0].set(xlabel="Month", ylabel="Mean PM2.5", title="Pollution by month")
axes[1].bar(mp.index, mp.blh, color="#4682B4")
axes[1].set(xlabel="Month", ylabel="Mean BLH (m)", title="Mixing depth by month")
plt.suptitle("Figure 2. Seasonal inversion pattern, Almaty basin", y=1.03)
plt.tight_layout(); plt.savefig("fig2_seasonality.png", bbox_inches="tight"); plt.close()

def resid(y, Xm): return sm.OLS(y, sm.add_constant(Xm)).fit().resid
C2 = D[[c for c in CTRL if D[c].nunique() > 1]]
rx, ry = resid(D.log_blh, C2), resid(D.log_pm25, C2)
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.scatter(rx, ry, s=4, alpha=.25, c="#333")
z = np.polyfit(rx, ry, 1); xs = np.linspace(rx.min(), rx.max(), 50)
ax.plot(xs, np.polyval(z, xs), c="crimson", lw=2)
ax.set(xlabel="Log BLH (residualised)", ylabel="Log PM2.5 (residualised)",
       title=f"Figure 3. First stage (slope {z[0]:.3f})")
plt.tight_layout(); plt.savefig("fig3_firststage.png", bbox_inches="tight"); plt.close()

rx2, ry2 = resid(D.log_pm25, C2), resid(D.logret, C2)
bins = pd.qcut(rx2, 20, labels=False)
bx, by = pd.Series(rx2).groupby(bins).mean(), pd.Series(ry2).groupby(bins).mean()
bse = pd.Series(ry2).groupby(bins).sem()
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.errorbar(bx, by, yerr=1.96*bse, fmt="o", ms=5, c="#1f3b73",
            ecolor="#999", elinewidth=1, capsize=2)
z2 = np.polyfit(rx2, ry2, 1)
ax.plot(bx, np.polyval(z2, bx), c="crimson", lw=1.5, label=f"slope {z2[0]:.3f}")
ax.axhline(0, c="k", lw=.6, ls=":")
ax.set(xlabel="Log PM2.5 (residualised)", ylabel="Log return, % (residualised)",
       title="Figure 4. No relationship between pollution and returns")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("fig4_binscatter.png", bbox_inches="tight"); plt.close()

specs = [("Preferred", main), ("Coverage >=18h", iv(D[D.n_obs>=18])),
         ("From 2022", iv(D[D.date>="2022-01-01"])),
         ("Winter only", iv(D[D.month.isin([11,12,1,2,3])])),
         ("Winsorised", iv(dw)), ("Drop pressure", iv(inst=["log_blh","u","v"])),
         ("BLH only", blh)]
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ys = np.arange(len(specs))[::-1]
for i, (lab, r_) in enumerate(specs):
    c = "crimson" if lab == "Preferred" else "#333"
    ax.errorbar(r_["b"], ys[i], xerr=1.96*r_["se"], fmt="o", ms=5,
                color=c, ecolor=c, elinewidth=1.2, capsize=2)
ax.axvline(0, c="k", lw=.8, ls="--")
ax.set_yticks(ys); ax.set_yticklabels([s[0] for s in specs], fontsize=8)
ax.set(xlabel="Coefficient on log PM2.5 (95% CI)",
       title="Figure 5. Estimate stability across specifications")
plt.tight_layout(); plt.savefig("fig5_specification.png", bbox_inches="tight"); plt.close()

print("\nFigures saved: fig1-fig5")
import subprocess
subprocess.run("cp *.png *.csv /content/drive/MyDrive/pollution_research/ 2>/dev/null",
               shell=True)
print("Backed up to Drive.")


In [ ]:
import numpy as np, pandas as pd, warnings
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt, statsmodels.api as sm
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

D = pd.read_csv("panel_analysis.csv"); D["date"] = pd.to_datetime(D["date"]); D["const"]=1
INST = ["log_blh","u","v","pressure"]
FE   = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))]
CTRL = ["temp","precip","wind_speed","cloud","radiation"] + FE + \
       [c for c in ["lockdown","qantar"] if c in D.columns and D[c].nunique()>1]
LN2 = np.log(2)

def iv(data=None, y="logret", x="log_pm25", inst=None, ctrl=None):
    d = (data if data is not None else D).copy(); d["const"]=1
    inst = inst or INST
    cand = [c for c in (ctrl or CTRL) if c in d.columns and d[c].nunique()>1]
    d = d.dropna(subset=[y,x]+inst)
    keep = []
    for c in cand:                                  # iterative rank check
        M = d[["const"]+keep+[c]].astype(float).values
        if np.linalg.matrix_rank(M) == M.shape[1]: keep.append(c)
    r = IV2SLS(d[y], d[["const"]+keep], d[x], d[inst]).fit(cov_type="kernel")
    b, se = r.params[x], r.std_errors[x]
    try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except: F = np.nan
    return dict(b=b, se=se, p=r.pvalues[x], n=len(d), F=F,
                bound=(abs(b)+1.645*se)*LN2, r=r)

main = iv(); b, se = main["b"], main["se"]
blh  = iv(inst=["log_blh"])

print("="*78); print("TABLE 4 — ROBUSTNESS"); print("="*78)
print(f"  {'Specification':<34}{'coef':>9}{'se':>9}{'p':>8}{'N':>6}{'F':>8}{'bound':>8}")
print("  "+"-"*74)
def row(lab,r_):
    print(f"  {lab:<34}{r_['b']:9.4f}{r_['se']:9.4f}{r_['p']:8.3f}"
          f"{r_['n']:6d}{r_['F']:8.1f}{r_['bound']:8.3f}")

row("Preferred", main)
for mo in [18,15,6]: row(f"Coverage >={mo}h", iv(D[D.n_obs>=mo]))
for s,l in [("2022-01-01","Sample from 2022"),("2023-01-01","Sample from 2023")]:
    row(l, iv(D[D.date>=s]))
row("Excluding 2025", iv(D[D.date<="2024-12-31"]))
row("Winter only (Nov-Mar)", iv(D[D.month.isin([11,12,1,2,3])]))
dw = D.copy(); lo,hi = dw.logret.quantile([.01,.99]); dw["logret"]=dw.logret.clip(lo,hi)
row("Winsorised 1%", iv(dw))
row("Drop pressure instr.", iv(inst=["log_blh","u","v"]))
row("No cloud/radiation", iv(ctrl=[c for c in CTRL if c not in ("cloud","radiation")]))
if "qantar" in D.columns: row("Excl. Qantar window", iv(D[D.qantar==0]))
print()
for L in [1,2,3]:
    d = D.copy(); d[f"pm_l{L}"]=d.log_pm25.shift(L)
    for i in INST: d[f"{i}_l{L}"]=d[i].shift(L)
    row(f"Lag {L} day", iv(d.dropna(subset=[f"pm_l{L}"]), x=f"pm_l{L}",
                           inst=[f"{i}_l{L}" for i in INST]))
da = D.copy(); da["absret"]=da.logret.abs()
row("Outcome: |return|", iv(da, y="absret"))

print("\n  Thresholds (reduced form OLS):")
for thr in [35,55,75]:
    d = D.copy(); d["hi"]=(d.pm25_raw>thr).astype(int)
    if d.hi.sum()<20: continue
    rr = smf.ols("logret ~ hi + "+" + ".join(CTRL), data=d).fit(
            cov_type="HAC", cov_kwds={"maxlags":5})
    print(f"    >{thr:3d} ({d.hi.sum():3d} days): {rr.params['hi']:8.4f} "
          f"({rr.bse['hi']:.4f})  p={rr.pvalues['hi']:.3f}")

print("\n"+"="*78); print("PLACEBO & RANDOMISATION INFERENCE"); print("="*78)
for L in [1,2]:
    d = D.copy(); d["lead"]=d.logret.shift(L)
    r = iv(d.dropna(subset=["lead"]), y="lead")
    print(f"  {L}-day lead   {r['b']:8.4f} ({r['se']:.4f})  p={r['p']:.3f}")

X, Z, n = D[["const"]+CTRL], D[INST], len(D)
rng = np.random.default_rng(42); null=[]
for _ in range(1000):
    k = int(rng.integers(6,(n-30)//5))*5
    try: null.append(IV2SLS(pd.Series(np.roll(D.logret.values,k), index=D.index),
                            X, D["log_pm25"], Z).fit().params["log_pm25"])
    except Exception: pass
null = np.array(null)
print(f"\n  Randomisation SD {null.std():.4f}   HAC SE {se:.4f}   ratio {null.std()/se:.2f}")
print(f"  RI p-value {(np.abs(null)>=abs(b)).mean():.3f}")
print(f"  Null 2.5-97.5% [{np.percentile(null,2.5):.3f}, {np.percentile(null,97.5):.3f}]")


In [ ]:
import numpy as np, pandas as pd
print("Pollution autocorrelation:")
for L in [1,2,3,5]:
    print(f"  lag {L}: {D.log_pm25.autocorr(L):.3f}")

# clean placebo: condition on lagged pollution to break the proxy channel
d = D.copy()
d["lead"] = d.logret.shift(1)
d["pm_lag1"] = d.log_pm25.shift(1)
d = d.dropna(subset=["lead","pm_lag1"])
r = iv(d, y="lead", ctrl=CTRL + ["pm_lag1"])
print(f"\nPlacebo conditioning on lagged PM2.5: {r['b']:.4f} ({r['se']:.4f}) p={r['p']:.3f}")

for L in [3,5]:
    d2 = D.copy(); d2["lead"] = d2.logret.shift(L)
    r2 = iv(d2.dropna(subset=["lead"]), y="lead")
    print(f"{L}-day lead: {r2['b']:.4f} ({r2['se']:.4f}) p={r2['p']:.3f}")


In [ ]:
import numpy as np
lo, hi = np.percentile(null, [2.5, 97.5])
print(f"RI null 95% range: [{lo:.3f}, {hi:.3f}]\n")
for lab, est in [("contemporaneous", -0.0827), ("1-day lead", -0.1572),
                 ("1-day lead | lagged PM", -0.1979), ("2-day lead", 0.0062),
                 ("3-day lead", -0.0820), ("5-day lead", -0.0317)]:
    pct = (null < est).mean() * 100
    print(f"  {lab:24s} {est:7.4f}  percentile {pct:5.1f}  "
          f"{'inside' if lo < est < hi else 'OUTSIDE'} null range")


In [ ]:
rf = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
for v in INST:
    print(f"  {v:12s} {rf.params[v]:9.4f} ({rf.bse[v]:.4f})  p={rf.pvalues[v]:.3f}")
print(f"\n  Joint F-test on instruments: p = "
      f"{rf.f_test(', '.join(f'{i} = 0' for i in INST)).pvalue:.4f}")
print(f"  R-squared: {rf.rsquared:.4f}")


In [ ]:
# Sargan and estimates, instrument-by-instrument
for inst, lab in [(["log_blh","u","v","pressure"], "all four (preferred)"),
                  (["log_blh","u","pressure"],     "drop v"),
                  (["log_blh","u","v"],            "drop pressure"),
                  (["log_blh","v","pressure"],     "drop u"),
                  (["log_blh","u"],                "BLH + u only")]:
    r = iv(inst=inst)
    s = r["r"].sargan
    print(f"{lab:22s}: {r['b']:8.4f} ({r['se']:.4f})  F={r['F']:6.1f}  "
          f"bound={r['bound']:.3f}pp  Sargan p={s.pval:.4f}")

# just-identified estimates, one instrument each
print()
for i in INST:
    r = iv(inst=[i])
    print(f"  {i:10s} alone: {r['b']:8.4f} ({r['se']:.4f})  F={r['F']:6.1f}")

# reduced-form bound on BLH — needs no exclusion restriction
rf = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags":5})
bb, bse = rf.params["log_blh"], rf.bse["log_blh"]
print(f"\nReduced-form BLH: {bb:.4f} ({bse:.4f})")
print(f"  TOST bound: {(abs(bb)+1.645*bse)*np.log(2):.4f}pp per doubling of BLH")

# partial R-squared of instruments in the reduced form
r0 = smf.ols("logret ~ " + " + ".join(CTRL), data=D).fit()
print(f"  Partial R2 of instruments: "
      f"{(rf.rsquared - r0.rsquared)/(1 - r0.rsquared):.4f}  (first stage: 0.181)")


In [ ]:
r = iv(inst=["log_blh","pressure"])
print(f"BLH + pressure: {r['b']:.4f} ({r['se']:.4f})  F={r['F']:.1f}  "
      f"bound={r['bound']:.3f}pp  Sargan p={r['r'].sargan.pval:.4f}")



In [ ]:
import pandas as pd, numpy as np

pm = pd.read_csv("almaty_pm25_daily.csv")
pm = pm[pm.sensor_id==25903].copy()
pm["date"] = pd.to_datetime(pm["date"])
print("raw sensor rows:", len(pm))
print("raw date range:", pm.date.min().date(), "to", pm.date.max().date())

for lo, hi, lab in [(None,None,"all raw"),
                    ("2020-04-09","2025-11-13","briefing window"),
                    ("2020-09-01","2025-11-13","first month with data")]:
    d = pm if lo is None else pm[(pm.date>=lo)&(pm.date<=hi)]
    print(f"{lab:24s} n={len(d)}")

panel = pd.read_csv("panel_final.csv"); panel["date"]=pd.to_datetime(panel["date"])
print("\npanel_final:", len(panel), panel.date.min().date(), "to", panel.date.max().date())
kase = pd.read_csv("kase_clean.csv"); kase["date"]=pd.to_datetime(kase["date"])
print("kase trading days in window:",
      len(kase[(kase.date>=panel.date.min())&(kase.date<=panel.date.max())]))


In [ ]:
import pandas as pd, numpy as np
from scipy import stats

kase  = pd.read_csv("kase_clean.csv");  kase["date"]  = pd.to_datetime(kase["date"])
panel = pd.read_csv("panel_final.csv"); panel["date"] = pd.to_datetime(panel["date"])

k = kase[(kase.date >= panel.date.min()) & (kase.date <= panel.date.max())].copy()
k["retained"] = k.date.isin(set(panel.date))
r, d = k[k.retained].logret.dropna(), k[~k.retained].logret.dropna()

print(f"Retained {len(r)}   dropped {len(d)}   ({len(d)/len(k):.1%} lost)\n")
print(f"{'':10s}{'mean':>9}{'sd':>9}{'skew':>9}{'min':>9}{'max':>9}")
for lab, s in [("retained", r), ("dropped", d)]:
    print(f"{lab:10s}{s.mean():9.4f}{s.std():9.4f}{s.skew():9.3f}{s.min():9.3f}{s.max():9.3f}")

t, pt = stats.ttest_ind(r, d, equal_var=False)
lev, pl = stats.levene(r, d)
ks_, pk = stats.ks_2samp(r, d)
print(f"\nMeans      t={t:6.3f}  p={pt:.3f}")
print(f"Variances  W={lev:6.3f}  p={pl:.3f}")
print(f"Full dist  D={ks_:6.3f}  p={pk:.3f}")

# is loss clustered in time? that matters more than the average difference
k["ym"] = k.date.dt.to_period("M")
loss = k.groupby("ym")["retained"].agg(n="size", kept="sum")
loss["lost_pct"] = (1 - loss.kept/loss.n).round(2)
print("\nMonths losing >50% of trading days:")
print(loss[loss.lost_pct > .5].to_string())

# and re-run the pollution-side test on the current filter
pm = pd.read_csv("almaty_pm25_daily.csv")
pm = pm[(pm.sensor_id==25903)].copy(); pm["date"] = pd.to_datetime(pm["date"])
pm = pm[(pm.date >= panel.date.min()) & (pm.date <= panel.date.max())]
inc, exc = pm[pm.n_obs>=12].pm25, pm[pm.n_obs<12].pm25
t2, p2 = stats.ttest_ind(inc, exc, equal_var=False)
print(f"\nPollution on included ({len(inc)}) vs excluded ({len(exc)}) days: "
      f"{inc.mean():.2f} vs {exc.mean():.2f}, t={t2:.2f}, p={p2:.3f}")


In [ ]:
wx = pd.read_csv("almaty_weather_daily.csv"); wx["date"] = pd.to_datetime(wx["date"])
wx["ym"] = wx.date.dt.to_period("M")
m = wx.groupby("ym").agg(n=("date","size"), blh_missing=("blh", lambda s: s.isna().sum()),
                         temp_missing=("temp", lambda s: s.isna().sum()))
print(m[m.blh_missing > 0].to_string())
print(f"\nTotal BLH missing: {wx.blh.isna().sum()}")
print(f"First missing: {wx[wx.blh.isna()].date.min().date()}")
print(f"Last missing:  {wx[wx.blh.isna()].date.max().date()}")


In [ ]:
import requests, pandas as pd, numpy as np

def try_pull(params, label):
    try:
        r = requests.get("https://archive-api.open-meteo.com/v1/archive",
                         params=params, timeout=120)
        j = r.json()
        if "hourly" not in j:
            print(f"{label:28s} -> {j.get('reason', r.status_code)}"); return None
        h = pd.DataFrame(j["hourly"])
        col = [c for c in h.columns if "boundary" in c]
        if not col:
            print(f"{label:28s} -> field absent"); return None
        n_ok = h[col[0]].notna().sum()
        print(f"{label:28s} -> {n_ok}/{len(h)} hourly values present")
        return h if n_ok > 0 else None
    except Exception as e:
        print(f"{label:28s} -> {str(e)[:60]}"); return None

base = dict(latitude=43.2529, longitude=76.9312,
            start_date="2024-01-01", end_date="2024-06-30",
            hourly="boundary_layer_height", timezone="Asia/Almaty")

for extra, lab in [({}, "default (best_match)"),
                   ({"models":"era5"}, "models=era5"),
                   ({"models":"era5_seamless"}, "models=era5_seamless"),
                   ({"models":"ecmwf_ifs"}, "models=ecmwf_ifs"),
                   ({"models":"era5_land"}, "models=era5_land")]:
    got = try_pull({**base, **extra}, lab)
    if got is not None:
        got["time"] = pd.to_datetime(got["time"])
        daily = got.groupby(got.time.dt.date)["boundary_layer_height"].mean()
        print(f"   -> {len(daily)} daily means, "
              f"range {daily.min():.0f}-{daily.max():.0f}m")
        daily.to_csv("blh_2024_patch.csv")
        print("   -> saved blh_2024_patch.csv")
        break


In [ ]:
for extra, lab in [({"models":"era5"}, "era5"),
                   ({"models":"era5_seamless"}, "era5_seamless"),
                   ({"models":"ecmwf_ifs"}, "ecmwf_ifs")]:
    got = try_pull({**base, **extra}, lab)
    if got is not None and got.iloc[:,1].notna().sum() > 3000:
        print("   -> USABLE"); break


In [ ]:
import pandas as pd

kase  = pd.read_csv("kase_clean.csv")
panel = pd.read_csv("panel_final.csv")
kase["date"]  = pd.to_datetime(kase["date"])
panel["date"] = pd.to_datetime(panel["date"])

win  = kase[(kase.date >= panel.date.min()) & (kase.date <= panel.date.max())]
lost = win[~win.date.isin(panel.date)]

g1 = lost[(lost.date >= "2022-05-01") & (lost.date <= "2022-08-31")]
g2 = lost[(lost.date >= "2024-01-02") & (lost.date <= "2024-06-30")]
g3 = lost[~lost.date.isin(g1.date) & ~lost.date.isin(g2.date)]

print(f"trading days in window : {len(win)}")
print(f"retained               : {len(panel)}")
print(f"lost total             : {len(lost)}")
print(f"  N1  2022 monitor gap : {len(g1)}")
print(f"  N2  2024 BLH gap     : {len(g2)}")
print(f"  N3  coverage filter   : {len(g3)}")

assert len(g1) + len(g2) + len(g3) == len(lost), "unaccounted lost days"
print("\ncheck passed — the three categories account for all lost days")


In [ ]:
import pandas as pd, numpy as np, statsmodels.formula.api as smf

D = pd.read_csv("panel_analysis.csv")
INST = ["log_blh", "u", "v", "pressure"]
BASE = ["temp", "precip", "wind_speed", "cloud", "radiation"]
BASE = [c for c in BASE if c in D.columns]
MO = [c for c in D.columns if c.startswith("mo_")]
DW = [c for c in D.columns if c.startswith("dw_")]
YR = [c for c in D.columns if c.startswith("yr_")]
EXTRA = [c for c in ["lockdown", "qantar", "unrest"] if c in D.columns
         and D[c].nunique() > 1]

def fs(ctrl, label):
    rhs = " + ".join(INST + ctrl)
    m = smf.ols("log_pm25 ~ " + rhs, data=D).fit(
            cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  {label:34s} {m.params['log_blh']:8.4f}  ({m.bse['log_blh']:.4f})")

print(f"N = {len(D)}   controls used: {BASE + EXTRA}")
print("\nFirst-stage coefficient on log BLH:\n")
fs(BASE + DW + EXTRA,                "no month FE, no year FE")
fs(BASE + DW + MO + EXTRA,           "month FE only")
fs(BASE + DW + MO + YR + EXTRA,      "month + year FE (preferred)")

print("\n--- HAC lag sensitivity, OLS on log_pm25 ---")
ctrl = BASE + DW + MO + YR + EXTRA
for L in [0, 1, 3, 5, 10, 20]:
    f = smf.ols("logret ~ log_pm25 + " + " + ".join(ctrl), data=D)
    m = f.fit(cov_type="HAC", cov_kwds={"maxlags": L}) if L > 0 else f.fit()
    print(f"  maxlags={L:2d}: SE = {m.bse['log_pm25']:.4f}")


In [ ]:
import pandas as pd

D = pd.read_csv("panel_analysis.csv")
blh = "blh" if "blh" in D.columns else "blh_raw"

g = D.groupby("month")[blh].agg(["mean", "min", "max", "size"]).round(1)
names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
g.index = [names[i-1] for i in g.index]
print(g.to_string())

print(f"\nsample mean {D[blh].mean():.1f}   min {D[blh].min():.1f}   max {D[blh].max():.1f}")
print(f"lowest month  : {g['mean'].idxmin()} at {g['mean'].min():.0f} m")
print(f"highest month : {g['mean'].idxmax()} at {g['mean'].max():.0f} m")


In [ ]:
import pandas as pd, numpy as np
from linearmodels.iv import IV2SLS

D = pd.read_csv("panel_analysis.csv")
D["date"] = pd.to_datetime(D["date"]); D["const"] = 1
INST = ["log_blh", "u", "v", "pressure"]
CTRL = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_"))] + \
       [c for c in ["temp","precip","wind_speed","cloud","radiation",
                    "lockdown","qantar"] if c in D.columns and D[c].nunique() > 1]

def run(d, label):
    c = [x for x in CTRL if d[x].nunique() > 1]
    r = IV2SLS(d["logret"], d[["const"]+c], d["log_pm25"], d[INST]).fit(cov_type="kernel")
    b, se = r.params["log_pm25"], r.std_errors["log_pm25"]
    f = r.first_stage.diagnostics["f.stat"].iloc[0]
    print(f"{label:26s} {b:8.4f} ({se:.4f})  p={r.pvalues['log_pm25']:.3f}  "
          f"N={len(d):4d}  F={f:6.1f}  bound={(abs(b)+1.645*se)*np.log(2):.3f}pp")

run(D, "preferred (full sample)")
run(D[D.date < "2024-03-01"], "pre-2024 timezone change")


In [ ]:
import pandas as pd, numpy as np, statsmodels.formula.api as smf

def load(path, name):
    d = pd.read_csv(path)
    d["date"] = pd.to_datetime(d["Date"], format="%m/%d/%Y")
    d[name] = 100 * np.log(
        d["Price"].astype(str).str.replace(",", "").astype(float)).diff(-1) * -1
    return d[["date", name]].dropna()

D = pd.read_csv("panel_analysis.csv"); D["date"] = pd.to_datetime(D["date"])
D = D.merge(load("brent.csv", "brent"), on="date", how="left") \
     .merge(load("uranium.csv", "uran"), on="date", how="left")
print(f"merged: brent {D.brent.notna().sum()}, uranium {D.uran.notna().sum()} of {len(D)}")

INST = ["log_blh", "u", "v", "pressure"]
CTRL = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       [c for c in ["temp","precip","wind_speed","cloud","radiation",
                    "lockdown","qantar"] if c in D.columns and D[c].nunique() > 1]
hyp = ", ".join(f"{i} = 0" for i in INST)

print("\n--- do instruments predict commodity returns? ---")
for y in ["brent", "uran"]:
    d = D.dropna(subset=[y])
    m = smf.ols(f"{y} ~ " + " + ".join(INST + CTRL), data=d).fit(
            cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  {y:6s}: joint F on instruments p = {m.f_test(hyp).pvalue:.4f}  (N={len(d)})")

print("\n--- does the reduced form survive commodity controls? ---")
d = D.dropna(subset=["brent", "uran"])
for extra, lab in [([], "baseline"), (["brent","uran"], "+ commodity returns")]:
    m = smf.ols("logret ~ " + " + ".join(INST + CTRL + extra), data=d).fit(
            cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  {lab:22s} log_blh {m.params['log_blh']:8.4f} "
          f"({m.bse['log_blh']:.4f})   joint p = {m.f_test(hyp).pvalue:.4f}")


In [ ]:
import pandas as pd, numpy as np, statsmodels.formula.api as smf

def load(path, name):
    d = pd.read_csv(path)
    d["date"] = pd.to_datetime(d["Date"], format="%m/%d/%Y")
    d[name] = 100 * np.log(
        d["Price"].astype(str).str.replace(",", "").astype(float)).diff(-1) * -1
    return d[["date", name]].dropna()

D = pd.read_csv("panel_analysis.csv"); D["date"] = pd.to_datetime(D["date"])
D = D.merge(load("brent.csv", "brent"), on="date", how="left") \
     .merge(load("uranium.csv", "uran"), on="date", how="left")
print(f"merged: brent {D.brent.notna().sum()}, uranium {D.uran.notna().sum()} of {len(D)}")

INST = ["log_blh", "u", "v", "pressure"]
CTRL = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       [c for c in ["temp","precip","wind_speed","cloud","radiation",
                    "lockdown","qantar"] if c in D.columns and D[c].nunique() > 1]
hyp = ", ".join(f"{i} = 0" for i in INST)

print("\n--- do instruments predict commodity returns? ---")
for y in ["brent", "uran"]:
    d = D.dropna(subset=[y])
    m = smf.ols(f"{y} ~ " + " + ".join(INST + CTRL), data=d).fit(
            cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  {y:6s}: joint F on instruments p = {m.f_test(hyp).pvalue:.4f}  (N={len(d)})")

print("\n--- does the reduced form survive commodity controls? ---")
d = D.dropna(subset=["brent", "uran"])
for extra, lab in [([], "baseline"), (["brent","uran"], "+ commodity returns")]:
    m = smf.ols("logret ~ " + " + ".join(INST + CTRL + extra), data=d).fit(
            cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  {lab:22s} log_blh {m.params['log_blh']:8.4f} "
          f"({m.bse['log_blh']:.4f})   joint p = {m.f_test(hyp).pvalue:.4f}")


In [ ]:
import os
print([f for f in os.listdir() if f.lower().endswith(".csv")])


In [ ]:
import pandas as pd, numpy as np, statsmodels.formula.api as smf

def load(path, name):
    d = pd.read_csv(path)
    d["date"] = pd.to_datetime(d["Date"])
    px = d["Price"].astype(str).str.replace(",", "").astype(float)
    d = d.assign(px=px).sort_values("date")
    d[name] = 100 * np.log(d.px).diff()
    return d[["date", name]].dropna()

D = pd.read_csv("panel_analysis.csv"); D["date"] = pd.to_datetime(D["date"])
D = D.merge(load("Brent Oil Futures Historical Data.csv", "brent"), on="date", how="left") \
     .merge(load("URA ETF Stock Price History.csv", "uran"), on="date", how="left")
print(f"merged: brent {D.brent.notna().sum()}, uranium {D.uran.notna().sum()} of {len(D)}")

INST = ["log_blh", "u", "v", "pressure"]
CTRL = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       [c for c in ["temp","precip","wind_speed","cloud","radiation",
                    "lockdown","qantar"] if c in D.columns and D[c].nunique() > 1]
hyp = ", ".join(f"{i} = 0" for i in INST)

print("\n--- do instruments predict commodity returns? ---")
for y in ["brent", "uran"]:
    d = D.dropna(subset=[y])
    m = smf.ols(f"{y} ~ " + " + ".join(INST + CTRL), data=d).fit(
            cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  {y:6s}: joint F on instruments p = {m.f_test(hyp).pvalue:.4f}  (N={len(d)})")
    for i in INST:
        print(f"          {i:10s} {m.params[i]:8.4f} ({m.bse[i]:.4f})  p={m.pvalues[i]:.3f}")

print("\n--- does the reduced form survive commodity controls? ---")
d = D.dropna(subset=["brent", "uran"])
for extra, lab in [([], "baseline"), (["brent","uran"], "+ commodity returns")]:
    m = smf.ols("logret ~ " + " + ".join(INST + CTRL + extra), data=d).fit(
            cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  {lab:22s} log_blh {m.params['log_blh']:8.4f} "
          f"({m.bse['log_blh']:.4f})   joint p = {m.f_test(hyp).pvalue:.4f}")


In [ ]:
import pandas as pd
b = pd.read_csv("Brent Oil Futures Historical Data.csv")
print(b.columns.tolist())
print(b.head(3).to_string())
print("\nURA for comparison:")
u = pd.read_csv("URA ETF Stock Price History.csv")
print(u.head(3).to_string())


In [ ]:
import pandas as pd, numpy as np, statsmodels.formula.api as smf

def load(path, name):
    d = pd.read_csv(path)
    d["date"] = pd.to_datetime(d["Date"])
    px = d["Price"].astype(str).str.replace(",", "").astype(float)
    d = d.assign(px=px).sort_values("date")
    d[name] = 100 * np.log(d.px).diff()
    return d[["date", name]].dropna()

D = pd.read_csv("panel_analysis.csv"); D["date"] = pd.to_datetime(D["date"])
D = D.merge(load("URA ETF Stock Price History.csv", "uran"), on="date", how="left")

INST = ["log_blh", "u", "v", "pressure"]
CTRL = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       [c for c in ["temp","precip","wind_speed","cloud","radiation",
                    "lockdown","qantar"] if c in D.columns and D[c].nunique() > 1]
hyp = ", ".join(f"{i} = 0" for i in INST)
d = D.dropna(subset=["uran"])

m = smf.ols("uran ~ " + " + ".join(INST + CTRL), data=d).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"instruments -> uranium returns: joint p = {m.f_test(hyp).pvalue:.4f} (N={len(d)})")
for i in INST:
    print(f"   {i:10s} {m.params[i]:8.4f} ({m.bse[i]:.4f})  p={m.pvalues[i]:.3f}")

print("\nreduced form:")
for extra, lab in [([], "baseline"), (["uran"], "+ uranium returns")]:
    r = smf.ols("logret ~ " + " + ".join(INST + CTRL + extra), data=d).fit(
            cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  {lab:20s} log_blh {r.params['log_blh']:8.4f} "
          f"({r.bse['log_blh']:.4f})   joint p = {r.f_test(hyp).pvalue:.4f}")


In [ ]:
import pandas as pd
b = pd.read_csv("Brent Oil Futures Historical Data.csv")
b["date"] = pd.to_datetime(b["Date"], format="%m/%d/%Y")
print(f"Brent: {len(b)} rows, {b.date.min().date()} to {b.date.max().date()}")

u = pd.read_csv("URA ETF Stock Price History.csv")
u["date"] = pd.to_datetime(u["Date"], format="%m/%d/%Y")
print(f"URA:   {len(u)} rows, {u.date.min().date()} to {u.date.max().date()}")


In [ ]:
import pandas as pd, numpy as np, statsmodels.formula.api as smf

def load(path, name):
    d = pd.read_csv(path)
    d["date"] = pd.to_datetime(d["Date"], format="%m/%d/%Y")
    px = d["Price"].astype(str).str.replace(",", "").astype(float)
    d = d.assign(px=px).sort_values("date")
    d[name] = 100 * np.log(d.px).diff()
    return d[["date", name]].dropna()

D = pd.read_csv("panel_analysis.csv"); D["date"] = pd.to_datetime(D["date"])
D = D.merge(load("Brent Oil Futures Historical Data.csv", "brent"), on="date", how="left")
print(f"brent merged: {D.brent.notna().sum()} of {len(D)}")

INST = ["log_blh", "u", "v", "pressure"]
CTRL = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       [c for c in ["temp","precip","wind_speed","cloud","radiation",
                    "lockdown","qantar"] if c in D.columns and D[c].nunique() > 1]
hyp = ", ".join(f"{i} = 0" for i in INST)
d = D.dropna(subset=["brent"])

m = smf.ols("brent ~ " + " + ".join(INST + CTRL), data=d).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"\ninstruments -> Brent returns: joint p = {m.f_test(hyp).pvalue:.4f} (N={len(d)})")
for i in INST:
    print(f"   {i:10s} {m.params[i]:8.4f} ({m.bse[i]:.4f})  p={m.pvalues[i]:.3f}")

print("\nreduced form:")
for extra, lab in [([], "baseline"), (["brent"], "+ Brent returns")]:
    r = smf.ols("logret ~ " + " + ".join(INST + CTRL + extra), data=d).fit(
            cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  {lab:18s} log_blh {r.params['log_blh']:8.4f} "
          f"({r.bse['log_blh']:.4f})   joint p = {r.f_test(hyp).pvalue:.4f}")


In [ ]:
import pandas as pd, numpy as np
from scipy import stats

h = pd.read_csv("almaty_pm25_hourly.csv")
h["hour"] = h["local"].str.slice(11, 13).astype(int)
h["date"] = pd.to_datetime(h["local"].str.slice(0, 10))
w = h[(h.hour >= 7) & (h.hour <= 17)]

g = w.groupby("date")["pm25"].agg(n="size", pm="mean").reset_index()
g = g[(g.date >= "2020-04-09") & (g.date <= "2025-11-13")]
inc, exc = g[g.n >= 8], g[g.n < 8]

t, p = stats.ttest_ind(inc.pm, exc.pm, equal_var=False)
print(f"included n={len(inc)}  mean {inc.pm.mean():.2f}")
print(f"excluded n={len(exc)}  mean {exc.pm.mean():.2f}")
print(f"difference {inc.pm.mean()-exc.pm.mean():.2f}   t={t:.2f}  p={p:.3f}")
print(f"corr(coverage, pm25) = {g[['n','pm']].corr().iloc[0,1]:.3f}")


In [ ]:
import pandas as pd, numpy as np, statsmodels.formula.api as smf

D = pd.read_csv("panel_analysis.csv")
INST = ["log_blh", "u", "v", "pressure"]
CTRL = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_"))] + \
       [c for c in ["temp","precip","wind_speed","cloud","radiation",
                    "lockdown","qantar"] if c in D.columns and D[c].nunique() > 1]

full = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit()
rest = smf.ols("logret ~ " + " + ".join(CTRL), data=D).fit()
partial_r2 = (full.rsquared - rest.rsquared) / (1 - rest.rsquared)

fs = smf.ols("log_pm25 ~ " + " + ".join(INST + CTRL), data=D).fit()
fs_r = smf.ols("log_pm25 ~ " + " + ".join([i for i in INST if i != "log_blh"] + CTRL),
               data=D).fit()
fs_partial = (fs.rsquared - fs_r.rsquared) / (1 - fs_r.rsquared)

hac = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
b, se = hac.params["log_blh"], hac.bse["log_blh"]
bound_rf = abs(b) + 1.645 * se
fs_coef = fs.params["log_blh"]

print(f"reduced form R2 (full)        {full.rsquared:.4f}")
print(f"reduced form R2 (controls)    {rest.rsquared:.4f}")
print(f"PARTIAL R2, instruments -> returns  {partial_r2:.4f}  ({100*partial_r2:.2f}%)")
print(f"first-stage partial R2 (BLH only)   {fs_partial:.4f}")
print(f"\nchance benchmark: k/(n-k_total) = "
      f"{len(INST)/(len(D)-len(INST)-len(CTRL)-1):.4f}")
print(f"\nRF coefficient on log_blh   {b:.4f} (SE {se:.4f})")
print(f"TOST bound on RF coefficient {bound_rf:.4f} pp per unit log BLH")
print(f"first-stage coefficient      {fs_coef:.4f}")
print(f"implied bound, scaled        {bound_rf/abs(fs_coef)*np.log(2):.4f} pp per doubling")


In [ ]:
import pandas as pd, numpy as np, statsmodels.formula.api as smf

D = pd.read_csv("panel_analysis.csv")
CTRL = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_"))] + \
       [c for c in ["temp","precip","wind_speed","cloud","radiation",
                    "lockdown","qantar"] if c in D.columns and D[c].nunique() > 1]

grid, keep = np.arange(-6.0, 6.0, 0.01), []
for b0 in grid:
    d = D.copy()
    d["_y"] = d.logret - b0 * d.log_pm25
    m = smf.ols("_y ~ v + " + " + ".join(CTRL), data=d).fit(
            cov_type="HAC", cov_kwds={"maxlags": 5})
    if m.f_test("v = 0").pvalue > 0.05:
        keep.append(b0)

if not keep:
    print("AR set EMPTY over [-6, 6] — model rejected at all values")
else:
    lo, hi = min(keep), max(keep)
    gaps = np.diff(keep)
    print(f"AR 95% set spans [{lo:.2f}, {hi:.2f}]")
    print(f"contiguous: {bool((gaps < 0.011).all())}")
    if lo <= grid[0] + 0.02 or hi >= grid[-1] - 0.02:
        print("-> set is UNBOUNDED at the grid edge; report as unbounded")
    print(f"\nWald interval for comparison: "
          f"[{1.388-1.96*0.870:.2f}, {1.388+1.96*0.870:.2f}]")


In [ ]:
"""
================================================================
FULL REBUILD ON THE OFFICIAL KASE SERIES
================================================================
Run in Colab. Prints every number the paper needs, plus a
before/after comparison against the Investing.com results.

SET THE TWO FILENAMES BELOW FIRST, then run top to bottom.

Design rules encoded here, fixed before seeing results:
  - compensating weekend sessions stay IN the price series for
    lag construction, but are EXCLUDED as observations
  - returns are 100*(ln P_t - ln P_{t-1}) on the official
    calendar, so 2022-03-09 uses 2022-03-05 and gives ~4.05
  - a single weekend_session flag, not separate Sat/Sun dummies
"""

import pandas as pd, numpy as np, statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
from scipy import stats
import warnings; warnings.filterwarnings("ignore")

# ---------------------------------------------------------------
OFFICIAL = "kase_official.csv"      # <-- official KASE download
PANEL    = "panel_analysis.csv"     # existing panel (all vars but returns)
# ---------------------------------------------------------------

INST = ["log_blh", "u", "v", "pressure"]
WEATHER = ["temp", "precip", "wind_speed", "cloud", "radiation"]

print("="*66); print("STEP 1 — REBUILD RETURNS FROM THE OFFICIAL SERIES"); print("="*66)

k = pd.read_csv(OFFICIAL)
print("columns:", k.columns.tolist())
print(k.head(3).to_string())

# --- adapt these two lines if the official file uses other names ---
DATECOL  = [c for c in k.columns if "date" in c.lower() or "дат" in c.lower()][0]
PRICECOL = [c for c in k.columns if c.lower() in
            ("close","price","last","значение","закрытие")][0]
print(f"\nusing date column '{DATECOL}', price column '{PRICECOL}'")

k["date"] = pd.to_datetime(k[DATECOL], dayfirst=True, errors="coerce")
if k["date"].isna().mean() > 0.1:
    k["date"] = pd.to_datetime(k[DATECOL], errors="coerce")
px = (k[PRICECOL].astype(str)
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False))
# if commas were thousands separators rather than decimals, undo
if px.str.count(r"\.").max() > 1:
    px = (k[PRICECOL].astype(str).str.replace(",", "", regex=False))
k["px"] = pd.to_numeric(px, errors="coerce")

k = k.dropna(subset=["date", "px"]).sort_values("date").reset_index(drop=True)
print(f"parsed {len(k)} rows, {k.date.min().date()} to {k.date.max().date()}")

# returns on the FULL official calendar (weekends included in series)
k["logret_new"] = 100 * np.log(k.px).diff()
k["weekend_session"] = (k.date.dt.dayofweek >= 5).astype(int)
print(f"weekend sessions in series: {k.weekend_session.sum()}")

chk = k[k.date == "2022-03-09"]
if len(chk):
    print(f"CHECK 2022-03-09 return = {chk.logret_new.iloc[0]:.2f}  (expect ~4.05)")

print("\n" + "="*66); print("STEP 2 — MERGE ONTO THE PANEL"); print("="*66)

D = pd.read_csv(PANEL); D["date"] = pd.to_datetime(D["date"])
old = D[["date", "logret"]].rename(columns={"logret": "logret_old"})
D = D.drop(columns=["logret"]).merge(
        k[["date", "logret_new", "weekend_session"]], on="date", how="left")
D = D.rename(columns={"logret_new": "logret"})

miss = D.logret.isna().sum()
print(f"panel rows: {len(D)}   unmatched to official series: {miss}")
if miss:
    print("  unmatched dates:", D.loc[D.logret.isna(), "date"].dt.date.tolist()[:20])
D = D.dropna(subset=["logret"])

# exclude weekend sessions as observations (they stay in the series above)
nwk = int(D.weekend_session.sum())
D = D[D.weekend_session == 0].copy()
print(f"weekend sessions excluded as observations: {nwk}")

cmp = D.merge(old, on="date")
d = (cmp.logret - cmp.logret_old).abs()
print(f"\nreturn changed on {int((d > 1e-6).sum())} of {len(cmp)} panel dates")
print(f"max |change| {d.max():.4f} pp on "
      f"{cmp.loc[d.idxmax(),'date'].date() if len(d) else 'n/a'}")
print(f"changed by >0.5pp: {int((d > 0.5).sum())}")
print(f"\nFINAL N = {len(D)}   (was 977)")
print(f"return SD {D.logret.std():.4f} (was 0.67)   "
      f"max {D.logret.max():.2f} (was 5.02)   min {D.logret.min():.2f} (was -3.02)")

CTRL = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_"))] + \
       [c for c in WEATHER if c in D.columns] + \
       [c for c in ["lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]
CTRL = [c for c in CTRL if D[c].nunique() > 1]
D["const"] = 1

def iv(data, insts, y="logret", x="log_pm25"):
    c = [v for v in CTRL if data[v].nunique() > 1]
    r = IV2SLS(data[y], data[["const"] + c], data[x], data[insts]).fit(cov_type="kernel")
    b, se = r.params[x], r.std_errors[x]
    try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except Exception: F = np.nan
    return dict(b=b, se=se, p=r.pvalues[x], n=len(data), F=F,
                bound=(abs(b) + 1.645*se)*np.log(2), r=r)

print("\n" + "="*66); print("STEP 3 — TABLE 1 DESCRIPTIVES"); print("="*66)
for v in ["logret","pm25_raw","blh","temp","precip","wind_speed",
          "pressure","cloud","radiation"]:
    if v in D.columns:
        s = D[v]
        print(f"{v:12s} mean {s.mean():9.2f}  sd {s.std():8.2f}  min {s.min():9.2f}"
              f"  med {s.median():8.2f}  max {s.max():9.2f}")
print(f"\nSD of log PM2.5 = {D.log_pm25.std():.4f}")
print(f"AR(1) of returns = {D.logret.autocorr():.4f}")
if "pm25_raw" in D.columns:
    print(f"days >55: {int((D.pm25_raw>55).sum())}   >75: {int((D.pm25_raw>75).sum())}")

print("\n" + "="*66); print("STEP 4 — FIRST STAGE (TABLE 2)"); print("="*66)
fs = smf.ols("log_pm25 ~ " + " + ".join(INST + CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
for i in INST:
    print(f"  {i:10s} {fs.params[i]:8.4f} ({fs.bse[i]:.4f})  t={fs.tvalues[i]:6.2f}"
          f"  p={fs.pvalues[i]:.3f}")
fsr = smf.ols("log_pm25 ~ " + " + ".join(CTRL), data=D).fit()
fs0 = smf.ols("log_pm25 ~ " + " + ".join(INST + CTRL), data=D).fit()
print(f"  R2 {fs0.rsquared:.4f}   partial R2 "
      f"{(fs0.rsquared-fsr.rsquared)/(1-fsr.rsquared):.4f}")
print(f"  joint F on instruments: "
      f"{fs.f_test(', '.join(f'{i} = 0' for i in INST)).fvalue:.1f}")

print("\n" + "="*66); print("STEP 5 — REDUCED FORM (TABLE 3)"); print("="*66)
hyp = ", ".join(f"{i} = 0" for i in INST)
rf = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
for i in INST:
    print(f"  {i:10s} {rf.params[i]:8.4f} ({rf.bse[i]:.4f})  p={rf.pvalues[i]:.3f}")
print(f"  joint p = {rf.f_test(hyp).pvalue:.4f}")
rf_f = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit()
rf_r = smf.ols("logret ~ " + " + ".join(CTRL), data=D).fit()
pr2 = (rf_f.rsquared - rf_r.rsquared)/(1 - rf_r.rsquared)
print(f"  partial R2 {pr2:.4f} ({100*pr2:.2f}%)   "
      f"chance {len(INST)/(len(D)-len(INST)-len(CTRL)-1):.4f}")
bnd = abs(rf.params["log_blh"]) + 1.645*rf.bse["log_blh"]
print(f"  TOST bound on RF coef {bnd:.4f};  scaled "
      f"{bnd/abs(fs.params['log_blh'])*np.log(2):.4f} pp per doubling")

print("\n" + "="*66); print("STEP 6 — MAIN RESULTS (TABLE 4)"); print("="*66)
ols = smf.ols("logret ~ log_pm25 + " + " + ".join(CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"  OLS         {ols.params['log_pm25']:8.4f} ({ols.bse['log_pm25']:.4f})"
      f"  p={ols.pvalues['log_pm25']:.3f}")
b1 = iv(D, ["log_blh"]);  print(f"  IV BLH only {b1['b']:8.4f} ({b1['se']:.4f})"
                                f"  p={b1['p']:.3f}  F={b1['F']:.1f}")
m  = iv(D, INST);         print(f"  IV full     {m['b']:8.4f} ({m['se']:.4f})"
                                f"  p={m['p']:.3f}  F={m['F']:.1f}")
print(f"\n  95% CI [{m['b']-1.96*m['se']:.3f}, {m['b']+1.96*m['se']:.3f}]")
print(f"  TOST BOUND = {m['bound']:.4f} pp per doubling")
print(f"  in return SDs: {m['bound']/D.logret.std():.4f}")
try:
    print(f"  Sargan: {m['r'].sargan}")
    print(f"  Wu-Hausman: {m['r'].wu_hausman()}")
except Exception as e:
    print("  tests:", e)

print("\n" + "="*66); print("STEP 7 — TRANSPORT TESTS (TABLE 5)"); print("="*66)
B, SB, MU = 0.0168, 0.0055, 11.53           # benchmark, its SE, its mean
our_pp, our_se = m['b']*np.log(2), m['se']*np.log(2)
print(f"  our estimate {our_pp:.4f} pp per doubling (SE {our_se:.4f})")
for lab, mult in [("proportional", MU), ("linear (our mean)", D.pm25_raw.mean()
                  if "pm25_raw" in D.columns else 29.79)]:
    imp, imp_se = B*mult, SB*mult
    t = abs(-our_pp + imp)/np.sqrt(our_se**2 + imp_se**2)
    print(f"  {lab:20s} implies {imp:.3f} (SE {imp_se:.3f})   t = {t:.2f}"
          f"   {'REJECTED' if t > 1.96 else 'not rejected'}")

print("\n" + "="*66); print("STEP 8 — ROBUSTNESS (TABLE 6)"); print("="*66)
def row(lab, res):
    print(f"  {lab:34s} {res['b']:8.4f} ({res['se']:.4f})  p={res['p']:.3f}"
          f"  N={res['n']:4d}  F={res['F']:6.1f}  bound={res['bound']:.3f}")
row("Preferred", m)
for lab, sub in [("Sample from 2022", D[D.date >= "2022-01-01"]),
                 ("Sample from 2023", D[D.date >= "2023-01-01"]),
                 ("Excluding 2025",   D[D.date <  "2025-01-01"]),
                 ("Pre-2024 (clock offset)", D[D.date < "2024-03-01"]),
                 ("Winter only (Nov-Mar)", D[D.month.isin([11,12,1,2,3])]),
                 ("Excl. Jan 2022 unrest",
                  D[~((D.date >= "2022-01-04") & (D.date <= "2022-01-14"))])]:
    if len(sub) > 100: row(lab, iv(sub, INST))
for lab, ii in [("Instruments excl. pressure", ["log_blh","u","v"]),
                ("Instruments excl. v",        ["log_blh","u","pressure"]),
                ("Instruments excl. u",        ["log_blh","v","pressure"]),
                ("BLH + u",                    ["log_blh","u"]),
                ("BLH + pressure (dispersion)",["log_blh","pressure"]),
                ("BLH alone",                  ["log_blh"])]:
    r = iv(D, ii); row(lab, r)
    try: print(f"      Sargan p = {r['r'].sargan.pval:.4f}")
    except Exception: pass

Dw = D.copy()
for c in ["logret"]:
    lo, hi = Dw[c].quantile([.01, .99]); Dw[c] = Dw[c].clip(lo, hi)
row("Winsorised at 1%", iv(Dw, INST))

CTRL_NC = [c for c in CTRL if c not in ("cloud", "radiation")]
_C = CTRL; CTRL = CTRL_NC; row("Controls excl. cloud, radiation", iv(D, INST)); CTRL = _C

for L in [1, 2, 3]:
    d2 = D.copy(); d2["log_pm25"] = d2["log_pm25"].shift(L)
    for i in INST: d2[i] = d2[i].shift(L)
    row(f"Lag {L} day(s)", iv(d2.dropna(subset=["log_pm25"] + INST), INST))

d3 = D.copy(); d3["absret"] = d3.logret.abs()
row("Outcome: absolute return", iv(d3, INST, y="absret"))

print("\n" + "="*66); print("STEP 9 — PLACEBOS (TABLE 7)"); print("="*66)
for L in [1, 2, 3, 5]:
    d4 = D.copy(); d4["lead"] = d4.logret.shift(-L)
    d4 = d4.dropna(subset=["lead"])
    r = iv(d4, INST, y="lead")
    print(f"  {L}-day lead   {r['b']:8.4f} ({r['se']:.4f})  p={r['p']:.3f}")

print("\n" + "="*66); print("STEP 10 — RANDOMISATION INFERENCE"); print("="*66)
rng = np.random.default_rng(42); n = len(D); null = []
base = D.logret.values
for _ in range(1000):
    kshift = int(rng.integers(6, n//5)) * 5
    d5 = D.copy(); d5["logret"] = np.roll(base, kshift)
    try: null.append(iv(d5, INST)["b"])
    except Exception: pass
null = np.array(null)
print(f"  draws {len(null)}   null SD {null.std():.4f}   HAC SE {m['se']:.4f}"
      f"   ratio {null.std()/m['se']:.2f}")
print(f"  null 2.5-97.5%: [{np.percentile(null,2.5):.3f}, {np.percentile(null,97.5):.3f}]")
print(f"  RI p-value vs zero: {(np.abs(null) >= abs(m['b'])).mean():.3f}")
print(f"  percentile of actual: {(null < m['b']).mean()*100:.1f}")

print("\n" + "="*66); print("STEP 11 — PRECISION DECOMPOSITION"); print("="*66)
sd_x, sd_y = D.log_pm25.std(), D.logret.std()
print(f"  IV  SE in return SD per pollution SD: {m['se']*sd_x/sd_y:.4f}")
print(f"  OLS SE in return SD per pollution SD: "
      f"{ols.bse['log_pm25']*sd_x/sd_y:.4f}")
print(f"  benchmark: {0.0055*6.97/1.28:.4f}")
print(f"  sample-size factor sqrt(3722/{len(D)}) = {np.sqrt(3722/len(D)):.2f}")

print("\n" + "="*66); print("BEFORE / AFTER"); print("="*66)
print(f"  {'':22s} {'was':>10s} {'now':>10s}")
for lab, was, now in [("N", 977, len(D)),
                      ("IV coefficient", -0.0827, m['b']),
                      ("IV SE", 0.1200, m['se']),
                      ("TOST bound (pp)", 0.194, m['bound']),
                      ("first-stage F", 134.9, m['F']),
                      ("OLS coefficient", 0.0526, ols.params['log_pm25']),
                      ("return SD", 0.67, sd_y),
                      ("max return", 5.02, D.logret.max())]:
    print(f"  {lab:22s} {was:10.4f} {now:10.4f}")
D.to_csv("panel_official.csv", index=False)
print("\nsaved panel_official.csv")


In [ ]:
import os
print([f for f in os.listdir() if f.lower().endswith((".csv", ".xlsx", ".xls"))])


In [ ]:
"""
================================================================
FULL REBUILD ON THE OFFICIAL KASE SERIES
================================================================
Run in Colab. Prints every number the paper needs, plus a
before/after comparison against the Investing.com results.

SET THE TWO FILENAMES BELOW FIRST, then run top to bottom.

Design rules encoded here, fixed before seeing results:
  - compensating weekend sessions stay IN the price series for
    lag construction, but are EXCLUDED as observations
  - returns are 100*(ln P_t - ln P_{t-1}) on the official
    calendar, so 2022-03-09 uses 2022-03-05 and gives ~4.05
  - a single weekend_session flag, not separate Sat/Sun dummies
"""

import pandas as pd, numpy as np, statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
from scipy import stats
import warnings; warnings.filterwarnings("ignore")

# ---------------------------------------------------------------
OFFICIAL = "kase_official.csv"      # <-- official KASE download
PANEL    = "panel_analysis.csv"     # existing panel (all vars but returns)
# ---------------------------------------------------------------

INST = ["log_blh", "u", "v", "pressure"]
WEATHER = ["temp", "precip", "wind_speed", "cloud", "radiation"]

print("="*66); print("STEP 1 — REBUILD RETURNS FROM THE OFFICIAL SERIES"); print("="*66)

k = pd.read_csv(OFFICIAL)
print("columns:", k.columns.tolist())
print(k.head(3).to_string())

# --- adapt these two lines if the official file uses other names ---
DATECOL  = [c for c in k.columns if "date" in c.lower() or "дат" in c.lower()][0]
PRICECOL = [c for c in k.columns if c.lower() in
            ("close","price","last","значение","закрытие")][0]
print(f"\nusing date column '{DATECOL}', price column '{PRICECOL}'")

k["date"] = pd.to_datetime(k[DATECOL], dayfirst=True, errors="coerce")
if k["date"].isna().mean() > 0.1:
    k["date"] = pd.to_datetime(k[DATECOL], errors="coerce")
px = (k[PRICECOL].astype(str)
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False))
# if commas were thousands separators rather than decimals, undo
if px.str.count(r"\.").max() > 1:
    px = (k[PRICECOL].astype(str).str.replace(",", "", regex=False))
k["px"] = pd.to_numeric(px, errors="coerce")

k = k.dropna(subset=["date", "px"]).sort_values("date").reset_index(drop=True)
print(f"parsed {len(k)} rows, {k.date.min().date()} to {k.date.max().date()}")

# returns on the FULL official calendar (weekends included in series)
k["logret_new"] = 100 * np.log(k.px).diff()
k["weekend_session"] = (k.date.dt.dayofweek >= 5).astype(int)
print(f"weekend sessions in series: {k.weekend_session.sum()}")

chk = k[k.date == "2022-03-09"]
if len(chk):
    print(f"CHECK 2022-03-09 return = {chk.logret_new.iloc[0]:.2f}  (expect ~4.05)")

print("\n" + "="*66); print("STEP 2 — MERGE ONTO THE PANEL"); print("="*66)

D = pd.read_csv(PANEL); D["date"] = pd.to_datetime(D["date"])
old = D[["date", "logret"]].rename(columns={"logret": "logret_old"})
D = D.drop(columns=["logret"]).merge(
        k[["date", "logret_new", "weekend_session"]], on="date", how="left")
D = D.rename(columns={"logret_new": "logret"})

miss = D.logret.isna().sum()
print(f"panel rows: {len(D)}   unmatched to official series: {miss}")
if miss:
    print("  unmatched dates:", D.loc[D.logret.isna(), "date"].dt.date.tolist()[:20])
D = D.dropna(subset=["logret"])

# exclude weekend sessions as observations (they stay in the series above)
nwk = int(D.weekend_session.sum())
D = D[D.weekend_session == 0].copy()
print(f"weekend sessions excluded as observations: {nwk}")

cmp = D.merge(old, on="date")
d = (cmp.logret - cmp.logret_old).abs()
print(f"\nreturn changed on {int((d > 1e-6).sum())} of {len(cmp)} panel dates")
print(f"max |change| {d.max():.4f} pp on "
      f"{cmp.loc[d.idxmax(),'date'].date() if len(d) else 'n/a'}")
print(f"changed by >0.5pp: {int((d > 0.5).sum())}")
print(f"\nFINAL N = {len(D)}   (was 977)")
print(f"return SD {D.logret.std():.4f} (was 0.67)   "
      f"max {D.logret.max():.2f} (was 5.02)   min {D.logret.min():.2f} (was -3.02)")

CTRL = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_"))] + \
       [c for c in WEATHER if c in D.columns] + \
       [c for c in ["lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]
CTRL = [c for c in CTRL if D[c].nunique() > 1]
D["const"] = 1

def iv(data, insts, y="logret", x="log_pm25"):
    c = [v for v in CTRL if data[v].nunique() > 1]
    r = IV2SLS(data[y], data[["const"] + c], data[x], data[insts]).fit(cov_type="kernel")
    b, se = r.params[x], r.std_errors[x]
    try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except Exception: F = np.nan
    return dict(b=b, se=se, p=r.pvalues[x], n=len(data), F=F,
                bound=(abs(b) + 1.645*se)*np.log(2), r=r)

print("\n" + "="*66); print("STEP 3 — TABLE 1 DESCRIPTIVES"); print("="*66)
for v in ["logret","pm25_raw","blh","temp","precip","wind_speed",
          "pressure","cloud","radiation"]:
    if v in D.columns:
        s = D[v]
        print(f"{v:12s} mean {s.mean():9.2f}  sd {s.std():8.2f}  min {s.min():9.2f}"
              f"  med {s.median():8.2f}  max {s.max():9.2f}")
print(f"\nSD of log PM2.5 = {D.log_pm25.std():.4f}")
print(f"AR(1) of returns = {D.logret.autocorr():.4f}")
if "pm25_raw" in D.columns:
    print(f"days >55: {int((D.pm25_raw>55).sum())}   >75: {int((D.pm25_raw>75).sum())}")

print("\n" + "="*66); print("STEP 4 — FIRST STAGE (TABLE 2)"); print("="*66)
fs = smf.ols("log_pm25 ~ " + " + ".join(INST + CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
for i in INST:
    print(f"  {i:10s} {fs.params[i]:8.4f} ({fs.bse[i]:.4f})  t={fs.tvalues[i]:6.2f}"
          f"  p={fs.pvalues[i]:.3f}")
fsr = smf.ols("log_pm25 ~ " + " + ".join(CTRL), data=D).fit()
fs0 = smf.ols("log_pm25 ~ " + " + ".join(INST + CTRL), data=D).fit()
print(f"  R2 {fs0.rsquared:.4f}   partial R2 "
      f"{(fs0.rsquared-fsr.rsquared)/(1-fsr.rsquared):.4f}")
print(f"  joint F on instruments: "
      f"{fs.f_test(', '.join(f'{i} = 0' for i in INST)).fvalue:.1f}")

print("\n" + "="*66); print("STEP 5 — REDUCED FORM (TABLE 3)"); print("="*66)
hyp = ", ".join(f"{i} = 0" for i in INST)
rf = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
for i in INST:
    print(f"  {i:10s} {rf.params[i]:8.4f} ({rf.bse[i]:.4f})  p={rf.pvalues[i]:.3f}")
print(f"  joint p = {rf.f_test(hyp).pvalue:.4f}")
rf_f = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit()
rf_r = smf.ols("logret ~ " + " + ".join(CTRL), data=D).fit()
pr2 = (rf_f.rsquared - rf_r.rsquared)/(1 - rf_r.rsquared)
print(f"  partial R2 {pr2:.4f} ({100*pr2:.2f}%)   "
      f"chance {len(INST)/(len(D)-len(INST)-len(CTRL)-1):.4f}")
bnd = abs(rf.params["log_blh"]) + 1.645*rf.bse["log_blh"]
print(f"  TOST bound on RF coef {bnd:.4f};  scaled "
      f"{bnd/abs(fs.params['log_blh'])*np.log(2):.4f} pp per doubling")

print("\n" + "="*66); print("STEP 6 — MAIN RESULTS (TABLE 4)"); print("="*66)
ols = smf.ols("logret ~ log_pm25 + " + " + ".join(CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"  OLS         {ols.params['log_pm25']:8.4f} ({ols.bse['log_pm25']:.4f})"
      f"  p={ols.pvalues['log_pm25']:.3f}")
b1 = iv(D, ["log_blh"]);  print(f"  IV BLH only {b1['b']:8.4f} ({b1['se']:.4f})"
                                f"  p={b1['p']:.3f}  F={b1['F']:.1f}")
m  = iv(D, INST);         print(f"  IV full     {m['b']:8.4f} ({m['se']:.4f})"
                                f"  p={m['p']:.3f}  F={m['F']:.1f}")
print(f"\n  95% CI [{m['b']-1.96*m['se']:.3f}, {m['b']+1.96*m['se']:.3f}]")
print(f"  TOST BOUND = {m['bound']:.4f} pp per doubling")
print(f"  in return SDs: {m['bound']/D.logret.std():.4f}")
try:
    print(f"  Sargan: {m['r'].sargan}")
    print(f"  Wu-Hausman: {m['r'].wu_hausman()}")
except Exception as e:
    print("  tests:", e)

print("\n" + "="*66); print("STEP 7 — TRANSPORT TESTS (TABLE 5)"); print("="*66)
B, SB, MU = 0.0168, 0.0055, 11.53           # benchmark, its SE, its mean
our_pp, our_se = m['b']*np.log(2), m['se']*np.log(2)
print(f"  our estimate {our_pp:.4f} pp per doubling (SE {our_se:.4f})")
for lab, mult in [("proportional", MU), ("linear (our mean)", D.pm25_raw.mean()
                  if "pm25_raw" in D.columns else 29.79)]:
    imp, imp_se = B*mult, SB*mult
    t = abs(-our_pp + imp)/np.sqrt(our_se**2 + imp_se**2)
    print(f"  {lab:20s} implies {imp:.3f} (SE {imp_se:.3f})   t = {t:.2f}"
          f"   {'REJECTED' if t > 1.96 else 'not rejected'}")

print("\n" + "="*66); print("STEP 8 — ROBUSTNESS (TABLE 6)"); print("="*66)
def row(lab, res):
    print(f"  {lab:34s} {res['b']:8.4f} ({res['se']:.4f})  p={res['p']:.3f}"
          f"  N={res['n']:4d}  F={res['F']:6.1f}  bound={res['bound']:.3f}")
row("Preferred", m)
for lab, sub in [("Sample from 2022", D[D.date >= "2022-01-01"]),
                 ("Sample from 2023", D[D.date >= "2023-01-01"]),
                 ("Excluding 2025",   D[D.date <  "2025-01-01"]),
                 ("Pre-2024 (clock offset)", D[D.date < "2024-03-01"]),
                 ("Winter only (Nov-Mar)", D[D.month.isin([11,12,1,2,3])]),
                 ("Excl. Jan 2022 unrest",
                  D[~((D.date >= "2022-01-04") & (D.date <= "2022-01-14"))])]:
    if len(sub) > 100: row(lab, iv(sub, INST))
for lab, ii in [("Instruments excl. pressure", ["log_blh","u","v"]),
                ("Instruments excl. v",        ["log_blh","u","pressure"]),
                ("Instruments excl. u",        ["log_blh","v","pressure"]),
                ("BLH + u",                    ["log_blh","u"]),
                ("BLH + pressure (dispersion)",["log_blh","pressure"]),
                ("BLH alone",                  ["log_blh"])]:
    r = iv(D, ii); row(lab, r)
    try: print(f"      Sargan p = {r['r'].sargan.pval:.4f}")
    except Exception: pass

Dw = D.copy()
for c in ["logret"]:
    lo, hi = Dw[c].quantile([.01, .99]); Dw[c] = Dw[c].clip(lo, hi)
row("Winsorised at 1%", iv(Dw, INST))

CTRL_NC = [c for c in CTRL if c not in ("cloud", "radiation")]
_C = CTRL; CTRL = CTRL_NC; row("Controls excl. cloud, radiation", iv(D, INST)); CTRL = _C

for L in [1, 2, 3]:
    d2 = D.copy(); d2["log_pm25"] = d2["log_pm25"].shift(L)
    for i in INST: d2[i] = d2[i].shift(L)
    row(f"Lag {L} day(s)", iv(d2.dropna(subset=["log_pm25"] + INST), INST))

d3 = D.copy(); d3["absret"] = d3.logret.abs()
row("Outcome: absolute return", iv(d3, INST, y="absret"))

print("\n" + "="*66); print("STEP 9 — PLACEBOS (TABLE 7)"); print("="*66)
for L in [1, 2, 3, 5]:
    d4 = D.copy(); d4["lead"] = d4.logret.shift(-L)
    d4 = d4.dropna(subset=["lead"])
    r = iv(d4, INST, y="lead")
    print(f"  {L}-day lead   {r['b']:8.4f} ({r['se']:.4f})  p={r['p']:.3f}")

print("\n" + "="*66); print("STEP 10 — RANDOMISATION INFERENCE"); print("="*66)
rng = np.random.default_rng(42); n = len(D); null = []
base = D.logret.values
for _ in range(1000):
    kshift = int(rng.integers(6, n//5)) * 5
    d5 = D.copy(); d5["logret"] = np.roll(base, kshift)
    try: null.append(iv(d5, INST)["b"])
    except Exception: pass
null = np.array(null)
print(f"  draws {len(null)}   null SD {null.std():.4f}   HAC SE {m['se']:.4f}"
      f"   ratio {null.std()/m['se']:.2f}")
print(f"  null 2.5-97.5%: [{np.percentile(null,2.5):.3f}, {np.percentile(null,97.5):.3f}]")
print(f"  RI p-value vs zero: {(np.abs(null) >= abs(m['b'])).mean():.3f}")
print(f"  percentile of actual: {(null < m['b']).mean()*100:.1f}")

print("\n" + "="*66); print("STEP 11 — PRECISION DECOMPOSITION"); print("="*66)
sd_x, sd_y = D.log_pm25.std(), D.logret.std()
print(f"  IV  SE in return SD per pollution SD: {m['se']*sd_x/sd_y:.4f}")
print(f"  OLS SE in return SD per pollution SD: "
      f"{ols.bse['log_pm25']*sd_x/sd_y:.4f}")
print(f"  benchmark: {0.0055*6.97/1.28:.4f}")
print(f"  sample-size factor sqrt(3722/{len(D)}) = {np.sqrt(3722/len(D)):.2f}")

print("\n" + "="*66); print("BEFORE / AFTER"); print("="*66)
print(f"  {'':22s} {'was':>10s} {'now':>10s}")
for lab, was, now in [("N", 977, len(D)),
                      ("IV coefficient", -0.0827, m['b']),
                      ("IV SE", 0.1200, m['se']),
                      ("TOST bound (pp)", 0.194, m['bound']),
                      ("first-stage F", 134.9, m['F']),
                      ("OLS coefficient", 0.0526, ols.params['log_pm25']),
                      ("return SD", 0.67, sd_y),
                      ("max return", 5.02, D.logret.max())]:
    print(f"  {lab:22s} {was:10.4f} {now:10.4f}")
D.to_csv("panel_official.csv", index=False)
print("\nsaved panel_official.csv")


In [ ]:
!pip install linearmodels -q


In [ ]:
"""
================================================================
FULL REBUILD ON THE OFFICIAL KASE SERIES
================================================================
Run in Colab. Prints every number the paper needs, plus a
before/after comparison against the Investing.com results.

SET THE TWO FILENAMES BELOW FIRST, then run top to bottom.

Design rules encoded here, fixed before seeing results:
  - compensating weekend sessions stay IN the price series for
    lag construction, but are EXCLUDED as observations
  - returns are 100*(ln P_t - ln P_{t-1}) on the official
    calendar, so 2022-03-09 uses 2022-03-05 and gives ~4.05
  - a single weekend_session flag, not separate Sat/Sun dummies
"""

import pandas as pd, numpy as np, statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
from scipy import stats
import warnings; warnings.filterwarnings("ignore")

# ---------------------------------------------------------------
OFFICIAL = "kase_official.csv"      # <-- official KASE download
PANEL    = "panel_analysis.csv"     # existing panel (all vars but returns)
# ---------------------------------------------------------------

INST = ["log_blh", "u", "v", "pressure"]
WEATHER = ["temp", "precip", "wind_speed", "cloud", "radiation"]

print("="*66); print("STEP 1 — REBUILD RETURNS FROM THE OFFICIAL SERIES"); print("="*66)

k = pd.read_csv(OFFICIAL)
print("columns:", k.columns.tolist())
print(k.head(3).to_string())

# --- adapt these two lines if the official file uses other names ---
DATECOL  = [c for c in k.columns if "date" in c.lower() or "дат" in c.lower()][0]
PRICECOL = [c for c in k.columns if c.lower() in
            ("close","price","last","значение","закрытие")][0]
print(f"\nusing date column '{DATECOL}', price column '{PRICECOL}'")

k["date"] = pd.to_datetime(k[DATECOL], dayfirst=True, errors="coerce")
if k["date"].isna().mean() > 0.1:
    k["date"] = pd.to_datetime(k[DATECOL], errors="coerce")
px = (k[PRICECOL].astype(str)
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False))
# if commas were thousands separators rather than decimals, undo
if px.str.count(r"\.").max() > 1:
    px = (k[PRICECOL].astype(str).str.replace(",", "", regex=False))
k["px"] = pd.to_numeric(px, errors="coerce")

k = k.dropna(subset=["date", "px"]).sort_values("date").reset_index(drop=True)
print(f"parsed {len(k)} rows, {k.date.min().date()} to {k.date.max().date()}")

# returns on the FULL official calendar (weekends included in series)
k["logret_new"] = 100 * np.log(k.px).diff()
k["weekend_session"] = (k.date.dt.dayofweek >= 5).astype(int)
print(f"weekend sessions in series: {k.weekend_session.sum()}")

chk = k[k.date == "2022-03-09"]
if len(chk):
    print(f"CHECK 2022-03-09 return = {chk.logret_new.iloc[0]:.2f}  (expect ~4.05)")

print("\n" + "="*66); print("STEP 2 — MERGE ONTO THE PANEL"); print("="*66)

D = pd.read_csv(PANEL); D["date"] = pd.to_datetime(D["date"])
old = D[["date", "logret"]].rename(columns={"logret": "logret_old"})
D = D.drop(columns=["logret"]).merge(
        k[["date", "logret_new", "weekend_session"]], on="date", how="left")
D = D.rename(columns={"logret_new": "logret"})

miss = D.logret.isna().sum()
print(f"panel rows: {len(D)}   unmatched to official series: {miss}")
if miss:
    print("  unmatched dates:", D.loc[D.logret.isna(), "date"].dt.date.tolist()[:20])
D = D.dropna(subset=["logret"])

# exclude weekend sessions as observations (they stay in the series above)
nwk = int(D.weekend_session.sum())
D = D[D.weekend_session == 0].copy()
print(f"weekend sessions excluded as observations: {nwk}")

cmp = D.merge(old, on="date")
d = (cmp.logret - cmp.logret_old).abs()
print(f"\nreturn changed on {int((d > 1e-6).sum())} of {len(cmp)} panel dates")
print(f"max |change| {d.max():.4f} pp on "
      f"{cmp.loc[d.idxmax(),'date'].date() if len(d) else 'n/a'}")
print(f"changed by >0.5pp: {int((d > 0.5).sum())}")
print(f"\nFINAL N = {len(D)}   (was 977)")
print(f"return SD {D.logret.std():.4f} (was 0.67)   "
      f"max {D.logret.max():.2f} (was 5.02)   min {D.logret.min():.2f} (was -3.02)")

CTRL = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_"))] + \
       [c for c in WEATHER if c in D.columns] + \
       [c for c in ["lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]
CTRL = [c for c in CTRL if D[c].nunique() > 1]
D["const"] = 1

def iv(data, insts, y="logret", x="log_pm25"):
    c = [v for v in CTRL if data[v].nunique() > 1]
    r = IV2SLS(data[y], data[["const"] + c], data[x], data[insts]).fit(cov_type="kernel")
    b, se = r.params[x], r.std_errors[x]
    try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except Exception: F = np.nan
    return dict(b=b, se=se, p=r.pvalues[x], n=len(data), F=F,
                bound=(abs(b) + 1.645*se)*np.log(2), r=r)

print("\n" + "="*66); print("STEP 3 — TABLE 1 DESCRIPTIVES"); print("="*66)
for v in ["logret","pm25_raw","blh","temp","precip","wind_speed",
          "pressure","cloud","radiation"]:
    if v in D.columns:
        s = D[v]
        print(f"{v:12s} mean {s.mean():9.2f}  sd {s.std():8.2f}  min {s.min():9.2f}"
              f"  med {s.median():8.2f}  max {s.max():9.2f}")
print(f"\nSD of log PM2.5 = {D.log_pm25.std():.4f}")
print(f"AR(1) of returns = {D.logret.autocorr():.4f}")
if "pm25_raw" in D.columns:
    print(f"days >55: {int((D.pm25_raw>55).sum())}   >75: {int((D.pm25_raw>75).sum())}")

print("\n" + "="*66); print("STEP 4 — FIRST STAGE (TABLE 2)"); print("="*66)
fs = smf.ols("log_pm25 ~ " + " + ".join(INST + CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
for i in INST:
    print(f"  {i:10s} {fs.params[i]:8.4f} ({fs.bse[i]:.4f})  t={fs.tvalues[i]:6.2f}"
          f"  p={fs.pvalues[i]:.3f}")
fsr = smf.ols("log_pm25 ~ " + " + ".join(CTRL), data=D).fit()
fs0 = smf.ols("log_pm25 ~ " + " + ".join(INST + CTRL), data=D).fit()
print(f"  R2 {fs0.rsquared:.4f}   partial R2 "
      f"{(fs0.rsquared-fsr.rsquared)/(1-fsr.rsquared):.4f}")
print(f"  joint F on instruments: "
      f"{fs.f_test(', '.join(f'{i} = 0' for i in INST)).fvalue:.1f}")

print("\n" + "="*66); print("STEP 5 — REDUCED FORM (TABLE 3)"); print("="*66)
hyp = ", ".join(f"{i} = 0" for i in INST)
rf = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
for i in INST:
    print(f"  {i:10s} {rf.params[i]:8.4f} ({rf.bse[i]:.4f})  p={rf.pvalues[i]:.3f}")
print(f"  joint p = {rf.f_test(hyp).pvalue:.4f}")
rf_f = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit()
rf_r = smf.ols("logret ~ " + " + ".join(CTRL), data=D).fit()
pr2 = (rf_f.rsquared - rf_r.rsquared)/(1 - rf_r.rsquared)
print(f"  partial R2 {pr2:.4f} ({100*pr2:.2f}%)   "
      f"chance {len(INST)/(len(D)-len(INST)-len(CTRL)-1):.4f}")
bnd = abs(rf.params["log_blh"]) + 1.645*rf.bse["log_blh"]
print(f"  TOST bound on RF coef {bnd:.4f};  scaled "
      f"{bnd/abs(fs.params['log_blh'])*np.log(2):.4f} pp per doubling")

print("\n" + "="*66); print("STEP 6 — MAIN RESULTS (TABLE 4)"); print("="*66)
ols = smf.ols("logret ~ log_pm25 + " + " + ".join(CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"  OLS         {ols.params['log_pm25']:8.4f} ({ols.bse['log_pm25']:.4f})"
      f"  p={ols.pvalues['log_pm25']:.3f}")
b1 = iv(D, ["log_blh"]);  print(f"  IV BLH only {b1['b']:8.4f} ({b1['se']:.4f})"
                                f"  p={b1['p']:.3f}  F={b1['F']:.1f}")
m  = iv(D, INST);         print(f"  IV full     {m['b']:8.4f} ({m['se']:.4f})"
                                f"  p={m['p']:.3f}  F={m['F']:.1f}")
print(f"\n  95% CI [{m['b']-1.96*m['se']:.3f}, {m['b']+1.96*m['se']:.3f}]")
print(f"  TOST BOUND = {m['bound']:.4f} pp per doubling")
print(f"  in return SDs: {m['bound']/D.logret.std():.4f}")
try:
    print(f"  Sargan: {m['r'].sargan}")
    print(f"  Wu-Hausman: {m['r'].wu_hausman()}")
except Exception as e:
    print("  tests:", e)

print("\n" + "="*66); print("STEP 7 — TRANSPORT TESTS (TABLE 5)"); print("="*66)
B, SB, MU = 0.0168, 0.0055, 11.53           # benchmark, its SE, its mean
our_pp, our_se = m['b']*np.log(2), m['se']*np.log(2)
print(f"  our estimate {our_pp:.4f} pp per doubling (SE {our_se:.4f})")
for lab, mult in [("proportional", MU), ("linear (our mean)", D.pm25_raw.mean()
                  if "pm25_raw" in D.columns else 29.79)]:
    imp, imp_se = B*mult, SB*mult
    t = abs(-our_pp + imp)/np.sqrt(our_se**2 + imp_se**2)
    print(f"  {lab:20s} implies {imp:.3f} (SE {imp_se:.3f})   t = {t:.2f}"
          f"   {'REJECTED' if t > 1.96 else 'not rejected'}")

print("\n" + "="*66); print("STEP 8 — ROBUSTNESS (TABLE 6)"); print("="*66)
def row(lab, res):
    print(f"  {lab:34s} {res['b']:8.4f} ({res['se']:.4f})  p={res['p']:.3f}"
          f"  N={res['n']:4d}  F={res['F']:6.1f}  bound={res['bound']:.3f}")
row("Preferred", m)
for lab, sub in [("Sample from 2022", D[D.date >= "2022-01-01"]),
                 ("Sample from 2023", D[D.date >= "2023-01-01"]),
                 ("Excluding 2025",   D[D.date <  "2025-01-01"]),
                 ("Pre-2024 (clock offset)", D[D.date < "2024-03-01"]),
                 ("Winter only (Nov-Mar)", D[D.month.isin([11,12,1,2,3])]),
                 ("Excl. Jan 2022 unrest",
                  D[~((D.date >= "2022-01-04") & (D.date <= "2022-01-14"))])]:
    if len(sub) > 100: row(lab, iv(sub, INST))
for lab, ii in [("Instruments excl. pressure", ["log_blh","u","v"]),
                ("Instruments excl. v",        ["log_blh","u","pressure"]),
                ("Instruments excl. u",        ["log_blh","v","pressure"]),
                ("BLH + u",                    ["log_blh","u"]),
                ("BLH + pressure (dispersion)",["log_blh","pressure"]),
                ("BLH alone",                  ["log_blh"])]:
    r = iv(D, ii); row(lab, r)
    try: print(f"      Sargan p = {r['r'].sargan.pval:.4f}")
    except Exception: pass

Dw = D.copy()
for c in ["logret"]:
    lo, hi = Dw[c].quantile([.01, .99]); Dw[c] = Dw[c].clip(lo, hi)
row("Winsorised at 1%", iv(Dw, INST))

CTRL_NC = [c for c in CTRL if c not in ("cloud", "radiation")]
_C = CTRL; CTRL = CTRL_NC; row("Controls excl. cloud, radiation", iv(D, INST)); CTRL = _C

for L in [1, 2, 3]:
    d2 = D.copy(); d2["log_pm25"] = d2["log_pm25"].shift(L)
    for i in INST: d2[i] = d2[i].shift(L)
    row(f"Lag {L} day(s)", iv(d2.dropna(subset=["log_pm25"] + INST), INST))

d3 = D.copy(); d3["absret"] = d3.logret.abs()
row("Outcome: absolute return", iv(d3, INST, y="absret"))

print("\n" + "="*66); print("STEP 9 — PLACEBOS (TABLE 7)"); print("="*66)
for L in [1, 2, 3, 5]:
    d4 = D.copy(); d4["lead"] = d4.logret.shift(-L)
    d4 = d4.dropna(subset=["lead"])
    r = iv(d4, INST, y="lead")
    print(f"  {L}-day lead   {r['b']:8.4f} ({r['se']:.4f})  p={r['p']:.3f}")

print("\n" + "="*66); print("STEP 10 — RANDOMISATION INFERENCE"); print("="*66)
rng = np.random.default_rng(42); n = len(D); null = []
base = D.logret.values
for _ in range(1000):
    kshift = int(rng.integers(6, n//5)) * 5
    d5 = D.copy(); d5["logret"] = np.roll(base, kshift)
    try: null.append(iv(d5, INST)["b"])
    except Exception: pass
null = np.array(null)
print(f"  draws {len(null)}   null SD {null.std():.4f}   HAC SE {m['se']:.4f}"
      f"   ratio {null.std()/m['se']:.2f}")
print(f"  null 2.5-97.5%: [{np.percentile(null,2.5):.3f}, {np.percentile(null,97.5):.3f}]")
print(f"  RI p-value vs zero: {(np.abs(null) >= abs(m['b'])).mean():.3f}")
print(f"  percentile of actual: {(null < m['b']).mean()*100:.1f}")

print("\n" + "="*66); print("STEP 11 — PRECISION DECOMPOSITION"); print("="*66)
sd_x, sd_y = D.log_pm25.std(), D.logret.std()
print(f"  IV  SE in return SD per pollution SD: {m['se']*sd_x/sd_y:.4f}")
print(f"  OLS SE in return SD per pollution SD: "
      f"{ols.bse['log_pm25']*sd_x/sd_y:.4f}")
print(f"  benchmark: {0.0055*6.97/1.28:.4f}")
print(f"  sample-size factor sqrt(3722/{len(D)}) = {np.sqrt(3722/len(D)):.2f}")

print("\n" + "="*66); print("BEFORE / AFTER"); print("="*66)
print(f"  {'':22s} {'was':>10s} {'now':>10s}")
for lab, was, now in [("N", 977, len(D)),
                      ("IV coefficient", -0.0827, m['b']),
                      ("IV SE", 0.1200, m['se']),
                      ("TOST bound (pp)", 0.194, m['bound']),
                      ("first-stage F", 134.9, m['F']),
                      ("OLS coefficient", 0.0526, ols.params['log_pm25']),
                      ("return SD", 0.67, sd_y),
                      ("max return", 5.02, D.logret.max())]:
    print(f"  {lab:22s} {was:10.4f} {now:10.4f}")
D.to_csv("panel_official.csv", index=False)
print("\nsaved panel_official.csv")


In [ ]:
import os
os.rename("kase official.csv.csv", "kase_official.csv")
print(sorted(os.listdir()))


In [ ]:
"""
================================================================
FULL REBUILD ON THE OFFICIAL KASE SERIES
================================================================
Run in Colab. Prints every number the paper needs, plus a
before/after comparison against the Investing.com results.

SET THE TWO FILENAMES BELOW FIRST, then run top to bottom.

Design rules encoded here, fixed before seeing results:
  - compensating weekend sessions stay IN the price series for
    lag construction, but are EXCLUDED as observations
  - returns are 100*(ln P_t - ln P_{t-1}) on the official
    calendar, so 2022-03-09 uses 2022-03-05 and gives ~4.05
  - a single weekend_session flag, not separate Sat/Sun dummies
"""

import pandas as pd, numpy as np, statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
from scipy import stats
import warnings; warnings.filterwarnings("ignore")

# ---------------------------------------------------------------
OFFICIAL = "kase_official.csv"      # <-- official KASE download
PANEL    = "panel_analysis.csv"     # existing panel (all vars but returns)
# ---------------------------------------------------------------

INST = ["log_blh", "u", "v", "pressure"]
WEATHER = ["temp", "precip", "wind_speed", "cloud", "radiation"]

print("="*66); print("STEP 1 — REBUILD RETURNS FROM THE OFFICIAL SERIES"); print("="*66)

k = pd.read_csv(OFFICIAL)
print("columns:", k.columns.tolist())
print(k.head(3).to_string())

# --- adapt these two lines if the official file uses other names ---
DATECOL  = [c for c in k.columns if "date" in c.lower() or "дат" in c.lower()][0]
PRICECOL = [c for c in k.columns if c.lower() in
            ("close","price","last","значение","закрытие")][0]
print(f"\nusing date column '{DATECOL}', price column '{PRICECOL}'")

k["date"] = pd.to_datetime(k[DATECOL], dayfirst=True, errors="coerce")
if k["date"].isna().mean() > 0.1:
    k["date"] = pd.to_datetime(k[DATECOL], errors="coerce")
px = (k[PRICECOL].astype(str)
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False))
# if commas were thousands separators rather than decimals, undo
if px.str.count(r"\.").max() > 1:
    px = (k[PRICECOL].astype(str).str.replace(",", "", regex=False))
k["px"] = pd.to_numeric(px, errors="coerce")

k = k.dropna(subset=["date", "px"]).sort_values("date").reset_index(drop=True)
print(f"parsed {len(k)} rows, {k.date.min().date()} to {k.date.max().date()}")

# returns on the FULL official calendar (weekends included in series)
k["logret_new"] = 100 * np.log(k.px).diff()
k["weekend_session"] = (k.date.dt.dayofweek >= 5).astype(int)
print(f"weekend sessions in series: {k.weekend_session.sum()}")

chk = k[k.date == "2022-03-09"]
if len(chk):
    print(f"CHECK 2022-03-09 return = {chk.logret_new.iloc[0]:.2f}  (expect ~4.05)")

print("\n" + "="*66); print("STEP 2 — MERGE ONTO THE PANEL"); print("="*66)

D = pd.read_csv(PANEL); D["date"] = pd.to_datetime(D["date"])
old = D[["date", "logret"]].rename(columns={"logret": "logret_old"})
D = D.drop(columns=["logret"]).merge(
        k[["date", "logret_new", "weekend_session"]], on="date", how="left")
D = D.rename(columns={"logret_new": "logret"})

miss = D.logret.isna().sum()
print(f"panel rows: {len(D)}   unmatched to official series: {miss}")
if miss:
    print("  unmatched dates:", D.loc[D.logret.isna(), "date"].dt.date.tolist()[:20])
D = D.dropna(subset=["logret"])

# exclude weekend sessions as observations (they stay in the series above)
nwk = int(D.weekend_session.sum())
D = D[D.weekend_session == 0].copy()
print(f"weekend sessions excluded as observations: {nwk}")

cmp = D.merge(old, on="date")
d = (cmp.logret - cmp.logret_old).abs()
print(f"\nreturn changed on {int((d > 1e-6).sum())} of {len(cmp)} panel dates")
print(f"max |change| {d.max():.4f} pp on "
      f"{cmp.loc[d.idxmax(),'date'].date() if len(d) else 'n/a'}")
print(f"changed by >0.5pp: {int((d > 0.5).sum())}")
print(f"\nFINAL N = {len(D)}   (was 977)")
print(f"return SD {D.logret.std():.4f} (was 0.67)   "
      f"max {D.logret.max():.2f} (was 5.02)   min {D.logret.min():.2f} (was -3.02)")

CTRL = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_"))] + \
       [c for c in WEATHER if c in D.columns] + \
       [c for c in ["lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]
CTRL = [c for c in CTRL if D[c].nunique() > 1]
D["const"] = 1

def iv(data, insts, y="logret", x="log_pm25"):
    c = [v for v in CTRL if data[v].nunique() > 1]
    r = IV2SLS(data[y], data[["const"] + c], data[x], data[insts]).fit(cov_type="kernel")
    b, se = r.params[x], r.std_errors[x]
    try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except Exception: F = np.nan
    return dict(b=b, se=se, p=r.pvalues[x], n=len(data), F=F,
                bound=(abs(b) + 1.645*se)*np.log(2), r=r)

print("\n" + "="*66); print("STEP 3 — TABLE 1 DESCRIPTIVES"); print("="*66)
for v in ["logret","pm25_raw","blh","temp","precip","wind_speed",
          "pressure","cloud","radiation"]:
    if v in D.columns:
        s = D[v]
        print(f"{v:12s} mean {s.mean():9.2f}  sd {s.std():8.2f}  min {s.min():9.2f}"
              f"  med {s.median():8.2f}  max {s.max():9.2f}")
print(f"\nSD of log PM2.5 = {D.log_pm25.std():.4f}")
print(f"AR(1) of returns = {D.logret.autocorr():.4f}")
if "pm25_raw" in D.columns:
    print(f"days >55: {int((D.pm25_raw>55).sum())}   >75: {int((D.pm25_raw>75).sum())}")

print("\n" + "="*66); print("STEP 4 — FIRST STAGE (TABLE 2)"); print("="*66)
fs = smf.ols("log_pm25 ~ " + " + ".join(INST + CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
for i in INST:
    print(f"  {i:10s} {fs.params[i]:8.4f} ({fs.bse[i]:.4f})  t={fs.tvalues[i]:6.2f}"
          f"  p={fs.pvalues[i]:.3f}")
fsr = smf.ols("log_pm25 ~ " + " + ".join(CTRL), data=D).fit()
fs0 = smf.ols("log_pm25 ~ " + " + ".join(INST + CTRL), data=D).fit()
print(f"  R2 {fs0.rsquared:.4f}   partial R2 "
      f"{(fs0.rsquared-fsr.rsquared)/(1-fsr.rsquared):.4f}")
print(f"  joint F on instruments: "
      f"{fs.f_test(', '.join(f'{i} = 0' for i in INST)).fvalue:.1f}")

print("\n" + "="*66); print("STEP 5 — REDUCED FORM (TABLE 3)"); print("="*66)
hyp = ", ".join(f"{i} = 0" for i in INST)
rf = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
for i in INST:
    print(f"  {i:10s} {rf.params[i]:8.4f} ({rf.bse[i]:.4f})  p={rf.pvalues[i]:.3f}")
print(f"  joint p = {rf.f_test(hyp).pvalue:.4f}")
rf_f = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit()
rf_r = smf.ols("logret ~ " + " + ".join(CTRL), data=D).fit()
pr2 = (rf_f.rsquared - rf_r.rsquared)/(1 - rf_r.rsquared)
print(f"  partial R2 {pr2:.4f} ({100*pr2:.2f}%)   "
      f"chance {len(INST)/(len(D)-len(INST)-len(CTRL)-1):.4f}")
bnd = abs(rf.params["log_blh"]) + 1.645*rf.bse["log_blh"]
print(f"  TOST bound on RF coef {bnd:.4f};  scaled "
      f"{bnd/abs(fs.params['log_blh'])*np.log(2):.4f} pp per doubling")

print("\n" + "="*66); print("STEP 6 — MAIN RESULTS (TABLE 4)"); print("="*66)
ols = smf.ols("logret ~ log_pm25 + " + " + ".join(CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"  OLS         {ols.params['log_pm25']:8.4f} ({ols.bse['log_pm25']:.4f})"
      f"  p={ols.pvalues['log_pm25']:.3f}")
b1 = iv(D, ["log_blh"]);  print(f"  IV BLH only {b1['b']:8.4f} ({b1['se']:.4f})"
                                f"  p={b1['p']:.3f}  F={b1['F']:.1f}")
m  = iv(D, INST);         print(f"  IV full     {m['b']:8.4f} ({m['se']:.4f})"
                                f"  p={m['p']:.3f}  F={m['F']:.1f}")
print(f"\n  95% CI [{m['b']-1.96*m['se']:.3f}, {m['b']+1.96*m['se']:.3f}]")
print(f"  TOST BOUND = {m['bound']:.4f} pp per doubling")
print(f"  in return SDs: {m['bound']/D.logret.std():.4f}")
try:
    print(f"  Sargan: {m['r'].sargan}")
    print(f"  Wu-Hausman: {m['r'].wu_hausman()}")
except Exception as e:
    print("  tests:", e)

print("\n" + "="*66); print("STEP 7 — TRANSPORT TESTS (TABLE 5)"); print("="*66)
B, SB, MU = 0.0168, 0.0055, 11.53           # benchmark, its SE, its mean
our_pp, our_se = m['b']*np.log(2), m['se']*np.log(2)
print(f"  our estimate {our_pp:.4f} pp per doubling (SE {our_se:.4f})")
for lab, mult in [("proportional", MU), ("linear (our mean)", D.pm25_raw.mean()
                  if "pm25_raw" in D.columns else 29.79)]:
    imp, imp_se = B*mult, SB*mult
    t = abs(-our_pp + imp)/np.sqrt(our_se**2 + imp_se**2)
    print(f"  {lab:20s} implies {imp:.3f} (SE {imp_se:.3f})   t = {t:.2f}"
          f"   {'REJECTED' if t > 1.96 else 'not rejected'}")

print("\n" + "="*66); print("STEP 8 — ROBUSTNESS (TABLE 6)"); print("="*66)
def row(lab, res):
    print(f"  {lab:34s} {res['b']:8.4f} ({res['se']:.4f})  p={res['p']:.3f}"
          f"  N={res['n']:4d}  F={res['F']:6.1f}  bound={res['bound']:.3f}")
row("Preferred", m)
for lab, sub in [("Sample from 2022", D[D.date >= "2022-01-01"]),
                 ("Sample from 2023", D[D.date >= "2023-01-01"]),
                 ("Excluding 2025",   D[D.date <  "2025-01-01"]),
                 ("Pre-2024 (clock offset)", D[D.date < "2024-03-01"]),
                 ("Winter only (Nov-Mar)", D[D.month.isin([11,12,1,2,3])]),
                 ("Excl. Jan 2022 unrest",
                  D[~((D.date >= "2022-01-04") & (D.date <= "2022-01-14"))])]:
    if len(sub) > 100: row(lab, iv(sub, INST))
for lab, ii in [("Instruments excl. pressure", ["log_blh","u","v"]),
                ("Instruments excl. v",        ["log_blh","u","pressure"]),
                ("Instruments excl. u",        ["log_blh","v","pressure"]),
                ("BLH + u",                    ["log_blh","u"]),
                ("BLH + pressure (dispersion)",["log_blh","pressure"]),
                ("BLH alone",                  ["log_blh"])]:
    r = iv(D, ii); row(lab, r)
    try: print(f"      Sargan p = {r['r'].sargan.pval:.4f}")
    except Exception: pass

Dw = D.copy()
for c in ["logret"]:
    lo, hi = Dw[c].quantile([.01, .99]); Dw[c] = Dw[c].clip(lo, hi)
row("Winsorised at 1%", iv(Dw, INST))

CTRL_NC = [c for c in CTRL if c not in ("cloud", "radiation")]
_C = CTRL; CTRL = CTRL_NC; row("Controls excl. cloud, radiation", iv(D, INST)); CTRL = _C

for L in [1, 2, 3]:
    d2 = D.copy(); d2["log_pm25"] = d2["log_pm25"].shift(L)
    for i in INST: d2[i] = d2[i].shift(L)
    row(f"Lag {L} day(s)", iv(d2.dropna(subset=["log_pm25"] + INST), INST))

d3 = D.copy(); d3["absret"] = d3.logret.abs()
row("Outcome: absolute return", iv(d3, INST, y="absret"))

print("\n" + "="*66); print("STEP 9 — PLACEBOS (TABLE 7)"); print("="*66)
for L in [1, 2, 3, 5]:
    d4 = D.copy(); d4["lead"] = d4.logret.shift(-L)
    d4 = d4.dropna(subset=["lead"])
    r = iv(d4, INST, y="lead")
    print(f"  {L}-day lead   {r['b']:8.4f} ({r['se']:.4f})  p={r['p']:.3f}")

print("\n" + "="*66); print("STEP 10 — RANDOMISATION INFERENCE"); print("="*66)
rng = np.random.default_rng(42); n = len(D); null = []
base = D.logret.values
for _ in range(1000):
    kshift = int(rng.integers(6, n//5)) * 5
    d5 = D.copy(); d5["logret"] = np.roll(base, kshift)
    try: null.append(iv(d5, INST)["b"])
    except Exception: pass
null = np.array(null)
print(f"  draws {len(null)}   null SD {null.std():.4f}   HAC SE {m['se']:.4f}"
      f"   ratio {null.std()/m['se']:.2f}")
print(f"  null 2.5-97.5%: [{np.percentile(null,2.5):.3f}, {np.percentile(null,97.5):.3f}]")
print(f"  RI p-value vs zero: {(np.abs(null) >= abs(m['b'])).mean():.3f}")
print(f"  percentile of actual: {(null < m['b']).mean()*100:.1f}")

print("\n" + "="*66); print("STEP 11 — PRECISION DECOMPOSITION"); print("="*66)
sd_x, sd_y = D.log_pm25.std(), D.logret.std()
print(f"  IV  SE in return SD per pollution SD: {m['se']*sd_x/sd_y:.4f}")
print(f"  OLS SE in return SD per pollution SD: "
      f"{ols.bse['log_pm25']*sd_x/sd_y:.4f}")
print(f"  benchmark: {0.0055*6.97/1.28:.4f}")
print(f"  sample-size factor sqrt(3722/{len(D)}) = {np.sqrt(3722/len(D)):.2f}")

print("\n" + "="*66); print("BEFORE / AFTER"); print("="*66)
print(f"  {'':22s} {'was':>10s} {'now':>10s}")
for lab, was, now in [("N", 977, len(D)),
                      ("IV coefficient", -0.0827, m['b']),
                      ("IV SE", 0.1200, m['se']),
                      ("TOST bound (pp)", 0.194, m['bound']),
                      ("first-stage F", 134.9, m['F']),
                      ("OLS coefficient", 0.0526, ols.params['log_pm25']),
                      ("return SD", 0.67, sd_y),
                      ("max return", 5.02, D.logret.max())]:
    print(f"  {lab:22s} {was:10.4f} {now:10.4f}")
D.to_csv("panel_official.csv", index=False)
print("\nsaved panel_official.csv")


In [ ]:
import pandas as pd
raw = pd.read_csv("kase official.csv.csv")
print("rows:", len(raw))
print(raw.dtypes)
print(raw.head(10).to_string())
print(raw.tail(10).to_string())

d = pd.to_datetime(raw["date"], errors="coerce")
print("\nunparsed dates:", d.isna().sum())
print("range:", d.min(), "to", d.max())
print("duplicated dates:", d.duplicated().sum())
print("weekday counts:")
print(d.dt.day_name().value_counts())

print("\n2022-03-09 row:")
print(raw[raw["date"].astype(str).str.startswith("2022-03-0")].to_string())


In [ ]:
import pandas as pd
raw = pd.read_csv("kase official.csv.csv")
d = pd.to_datetime(raw["date"], errors="coerce")
print("rows:", len(raw), "| range:", d.min(), "to", d.max())
print("duplicated dates:", d.duplicated().sum())
print(d.dt.day_name().value_counts())
print(raw.head(8).to_string())


FileNotFoundError: [Errno 2] No such file or directory: 'kase official.csv.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/pollution_research/*.csv .
import os
print(sorted(f for f in os.listdir() if f.endswith('.csv')))


Mounted at /content/drive
['KASE Historical Data.csv', 'almaty_pm25_daily.csv', 'almaty_pm25_hourly.csv', 'almaty_weather_daily.csv', 'kase_clean.csv', 'panel_analysis.csv', 'panel_final.csv']


In [ ]:
import pandas as pd
k = pd.read_csv("kase_clean.csv"); k["date"] = pd.to_datetime(k["date"])
p = pd.read_csv("panel_final.csv"); p["date"] = pd.to_datetime(p["date"])

# gaps longer than 4 days = a session Investing.com likely missed
k = k.sort_values("date")
k["gap"] = k.date.diff().dt.days
big = k[k.gap > 4]
print(f"Gaps >4 days: {len(big)}")
print(big[["date","gap","logret"]].to_string(index=False))

# how many of those dates are actually in the estimation panel?
in_panel = big[big.date.isin(p.date)]
print(f"\nAffected days inside the 977-day panel: {len(in_panel)}")
print(f"As a share of the sample: {len(in_panel)/len(p):.2%}")
print(f"\n2022-03-09 return in current data: "
      f"{k[k.date=='2022-03-09'].logret.values}")


Gaps >4 days: 24
      date  gap    logret
2020-03-26  6.0 -0.266047
2020-05-11  5.0 -0.243155
2020-12-21  6.0 -0.036283
2021-01-05  5.0  1.339113
2021-03-25  6.0 -0.986946
2021-05-11  5.0  1.059298
2021-07-07  5.0 -0.919675
2021-12-20  5.0 -0.557388
2022-01-05  5.0 -3.045966
2022-01-11  6.0  1.721138
2022-03-09  5.0  5.018350
2022-03-24  6.0 -0.986239
2022-05-11  5.0 -3.656530
2022-08-31  5.0  0.590534
2022-10-26  5.0  0.719636
2023-01-04  5.0 -0.032297
2023-05-10  5.0  0.227267
2023-07-10  5.0  0.210842
2024-01-03  5.0  0.187293
2024-03-26  6.0  0.466226
2025-01-06  6.0  1.780720
2025-03-26  6.0 -0.366354
2026-01-05  5.0  1.336298
2026-03-26  6.0 -0.926652

Affected days inside the 977-day panel: 13
As a share of the sample: 1.33%

2022-03-09 return in current data: [5.01835049]


In [ ]:
import pandas as pd
p = pd.read_csv("panel_final.csv"); p["date"] = pd.to_datetime(p["date"])
bad = pd.to_datetime(["2020-03-26","2020-05-11","2020-12-21","2021-01-05",
    "2021-03-25","2021-05-11","2021-07-07","2021-12-20","2022-01-05",
    "2022-01-11","2022-03-09","2022-03-24","2022-05-11","2022-08-31",
    "2022-10-26","2023-01-04","2023-05-10","2023-07-10","2024-01-03",
    "2024-03-26","2025-01-06","2025-03-26"])
clean = p[~p.date.isin(bad)]
clean.to_csv("panel_no_gapdays.csv", index=False)
print(f"{len(p)} -> {len(clean)} observations")


977 -> 964 observations


In [ ]:
import numpy as np, pandas as pd
from linearmodels.iv import IV2SLS

D = pd.read_csv("panel_no_gapdays.csv"); D["const"] = 1
INST = ["log_blh","u","v","pressure"]
CTRL = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       ["temp","precip","wind_speed","cloud","radiation"] + \
       [c for c in ["lockdown","qantar"] if c in D.columns and D[c].nunique()>1]

keep = []
for c in CTRL:
    M = D[["const"]+keep+[c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]: keep.append(c)

r = IV2SLS(D["logret"], D[["const"]+keep], D["log_pm25"], D[INST]).fit(cov_type="kernel")
b, se = r.params["log_pm25"], r.std_errors["log_pm25"]
try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
except: F = float("nan")

print(f"Excluding holiday-gap days:")
print(f"  coef {b:.4f}  SE {se:.4f}  p {r.pvalues['log_pm25']:.3f}  n {len(D)}  F {F:.1f}")
print(f"  bound {(abs(b)+1.645*se)*np.log(2):.3f} pp")
print(f"  Sargan p {r.sargan.pval:.4f}")
print(f"\nPreferred (977 obs): -0.0827 (0.1200)  bound 0.194  F 134.9")


ModuleNotFoundError: No module named 'linearmodels'

In [ ]:
!pip install linearmodels -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 2.0 MB/s eta 0:00:00


In [ ]:
!pip install linearmodels -q
import numpy as np, pandas as pd
from linearmodels.iv import IV2SLS
...


Ellipsis

In [ ]:
!pip install linearmodels -q

import numpy as np, pandas as pd
from linearmodels.iv import IV2SLS

D = pd.read_csv("panel_no_gapdays.csv"); D["const"] = 1
INST = ["log_blh","u","v","pressure"]
CTRL = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       ["temp","precip","wind_speed","cloud","radiation"] + \
       [c for c in ["lockdown","qantar"] if c in D.columns and D[c].nunique()>1]

keep = []
for c in CTRL:
    M = D[["const"]+keep+[c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]: keep.append(c)

r = IV2SLS(D["logret"], D[["const"]+keep], D["log_pm25"], D[INST]).fit(cov_type="kernel")
b, se = r.params["log_pm25"], r.std_errors["log_pm25"]
try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
except: F = float("nan")

print("Excluding holiday-gap days:")
print(f"  coef {b:.4f}  SE {se:.4f}  p {r.pvalues['log_pm25']:.3f}  n {len(D)}  F {F:.1f}")
print(f"  bound {(abs(b)+1.645*se)*np.log(2):.3f} pp")
print(f"  Sargan p {r.sargan.pval:.4f}")
print()
print("Preferred (977 obs): -0.0827 (0.1200)  bound 0.194  F 134.9")


Excluding holiday-gap days:
  coef -0.0993  SE 0.0949  p 0.296  n 964  F 183.0
  bound 0.177 pp
  Sargan p 0.5091

Preferred (977 obs): -0.0827 (0.1200)  bound 0.194  F 134.9


In [ ]:
!pip install linearmodels -q

import numpy as np, pandas as pd, warnings
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore")

D = pd.read_csv("panel_analysis.csv"); D["const"] = 1
INST = ["log_blh","u","v","pressure"]
CTRL = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       ["temp","precip","wind_speed","cloud","radiation"] + \
       [c for c in ["lockdown","qantar"] if c in D.columns and D[c].nunique() > 1]

print("="*64)
print("(a) REDUCED-FORM PARTIAL R-SQUARED   -> §5.3")
print("="*64)
full = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit()
base = smf.ols("logret ~ " + " + ".join(CTRL), data=D).fit()
pr = (full.rsquared - base.rsquared) / (1 - base.rsquared)
print(f"  R2 with instruments   : {full.rsquared:.4f}")
print(f"  R2 without            : {base.rsquared:.4f}")
print(f"  partial R2            : {pr:.4f}   ({pr*100:.2f}%)")
print(f"  first-stage partial R2: 0.181     (18.1%)")
print(f"  ratio                 : {0.181/pr:.0f}x")

print()
print("="*64)
print("(b) REDUCED-FORM TOST BOUND ON BLH   -> §5.3")
print("="*64)
rf = smf.ols("logret ~ " + " + ".join(INST + CTRL), data=D).fit(
        cov_type="HAC", cov_kwds={"maxlags":5})
b, se = rf.params["log_blh"], rf.bse["log_blh"]
print(f"  coefficient : {b:.4f}  (SE {se:.4f})")
print(f"  TOST bound  : {(abs(b)+1.645*se)*np.log(2):.4f} pp per doubling of BLH")

print()
print("="*64)
print("(c) ANDERSON-RUBIN SET, v-ONLY SPEC   -> §6.4")
print("="*64)
keep = []
for c in CTRL:
    M = D[["const"]+keep+[c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]: keep.append(c)

grid, inside = np.arange(-4, 4, 0.02), []
for b0 in grid:
    d = D.copy(); d["_y"] = d.logret - b0*d.log_pm25
    m = smf.ols("_y ~ v + " + " + ".join(keep), data=d).fit(
            cov_type="HAC", cov_kwds={"maxlags":5})
    if m.f_test("v = 0").pvalue > 0.05: inside.append(b0)

if inside:
    lo, hi = min(inside), max(inside)
    gaps = np.diff(inside)
    print(f"  AR 95% set  : [{lo:.2f}, {hi:.2f}]")
    print(f"  contiguous  : {'yes' if gaps.max() < 0.05 else 'NO - disjoint set'}")
    print(f"  Wald 95%    : [{1.39-1.96*0.87:.2f}, {1.39+1.96*0.87:.2f}]")
    print(f"  contains dispersion estimates (-0.13 to -0.16)? "
          f"{'yes' if lo < -0.16 else 'NO'}")
else:
    print("  AR set is empty at the 5% level — report as unbounded/empty")

print()
print("="*64)
print("(d) DOES URANIUM PREDICT KASE RETURNS?   -> §4.5")
print("="*64)
try:
    import yfinance as yf
    tk = yf.download("URA", start="2020-04-01", end="2025-11-20",
                     progress=False, auto_adjust=True)["Close"]
    com = pd.DataFrame({"uran_ret": np.log(tk).diff()*100}).reset_index()
    com.columns = ["date","uran_ret"]
    com["date"] = pd.to_datetime(com["date"]).dt.tz_localize(None)
    Dd = D.copy(); Dd["date"] = pd.to_datetime(Dd["date"])
    M2 = Dd.merge(com, on="date", how="inner").dropna(subset=["uran_ret"])
    m = smf.ols("logret ~ uran_ret + " + " + ".join(CTRL), data=M2).fit(
            cov_type="HAC", cov_kwds={"maxlags":5})
    print(f"  n = {len(M2)}")
    print(f"  uranium -> KASE: {m.params['uran_ret']:.4f} "
          f"({m.bse['uran_ret']:.4f})  p = {m.pvalues['uran_ret']:.3f}")
    print(f"  -> channel is {'BROKEN at step 2 (good)' if m.pvalues['uran_ret']>0.10 else 'live — keep the conditioning argument'}")
except Exception as e:
    print(f"  failed: {str(e)[:80]}")


(a) REDUCED-FORM PARTIAL R-SQUARED   -> §5.3
  R2 with instruments   : 0.0444
  R2 without            : 0.0380
  partial R2            : 0.0067   (0.67%)
  first-stage partial R2: 0.181     (18.1%)
  ratio                 : 27x

(b) REDUCED-FORM TOST BOUND ON BLH   -> §5.3
  coefficient : 0.0480  (SE 0.0514)
  TOST bound  : 0.0919 pp per doubling of BLH

(c) ANDERSON-RUBIN SET, v-ONLY SPEC   -> §6.4
  AR 95% set  : [0.18, 3.98]
  contiguous  : yes
  Wald 95%    : [-0.32, 3.10]
  contains dispersion estimates (-0.13 to -0.16)? NO

(d) DOES URANIUM PREDICT KASE RETURNS?   -> §4.5
  failed: If using all scalar values, you must pass an index


In [ ]:
!pip install yfinance -q

import numpy as np, pandas as pd, warnings
import statsmodels.formula.api as smf
import yfinance as yf
warnings.filterwarnings("ignore")

D = pd.read_csv("panel_analysis.csv")
D["date"] = pd.to_datetime(D["date"])
CTRL = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       ["temp","precip","wind_speed","cloud","radiation"] + \
       [c for c in ["lockdown","qantar"] if c in D.columns and D[c].nunique() > 1]

# --- download URA and flatten whatever shape yfinance returns
raw = yf.download("URA", start="2020-04-01", end="2025-11-20",
                  progress=False, auto_adjust=True)

if isinstance(raw.columns, pd.MultiIndex):
    px = raw.xs("Close", axis=1, level=0)
else:
    px = raw["Close"]
px = pd.Series(np.asarray(px).ravel(), index=raw.index, name="close")

com = px.to_frame().reset_index()
com.columns = ["date", "close"]
com["date"] = pd.to_datetime(com["date"]).dt.tz_localize(None)
com["uran_ret"] = np.log(com["close"]).diff() * 100
com = com.dropna(subset=["uran_ret"])[["date", "uran_ret"]]

M = D.merge(com, on="date", how="inner").dropna(subset=["uran_ret", "logret"])
print(f"merged sample: {len(M)} days\n")

# --- does uranium predict KASE returns?
m = smf.ols("logret ~ uran_ret + " + " + ".join(CTRL), data=M).fit(
        cov_type="HAC", cov_kwds={"maxlags": 5})
b, se, p = m.params["uran_ret"], m.bse["uran_ret"], m.pvalues["uran_ret"]
print("URANIUM -> KASE RETURNS")
print(f"  coefficient {b:.4f}  (SE {se:.4f})  p = {p:.3f}")
print(f"  correlation with KASE returns: {M[['logret','uran_ret']].corr().iloc[0,1]:.3f}")
print()
if p > 0.10:
    print("  -> channel BROKEN at step 2: uranium returns do not predict KASE")
    print("     returns, so an instrument predicting uranium cannot reach the")
    print("     outcome through that route. Simplifies §4.5 considerably.")
else:
    print("  -> channel LIVE: uranium does predict KASE returns, so the")
    print("     conditioning argument in §4.5 must carry the weight.")


merged sample: 944 days

URANIUM -> KASE RETURNS
  coefficient 0.0253  (SE 0.0105)  p = 0.016
  correlation with KASE returns: 0.096

  -> channel LIVE: uranium does predict KASE returns, so the
     conditioning argument in §4.5 must carry the weight.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
print(os.listdir('/content/drive/MyDrive/pollution_research'))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['almaty_pm25_daily.csv', 'almaty_pm25_hourly.csv', 'almaty_weather_daily.csv', 'kase_clean.csv', 'KASE Historical Data.csv', 'panel_analysis.csv', 'panel_final.csv']


In [ ]:
!pip install linearmodels -q
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

D = pd.read_csv("panel_analysis.csv"); D["date"] = pd.to_datetime(D["date"]); D["const"]=1
INST = ["log_blh","u","v","pressure"]
CTRL = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       ["temp","precip","wind_speed","cloud","radiation"] + \
       [c for c in ["lockdown","qantar"] if c in D.columns and D[c].nunique()>1]
plt.rcParams.update({"figure.dpi":200,"font.size":9,"axes.grid":True,
                     "grid.alpha":.25,"axes.spines.top":False,"axes.spines.right":False})

# Fig 1
fig, ax = plt.subplots(figsize=(9,3))
ax.plot(D.date, D.pm25_raw, lw=.5, c="#444")
ax.axhline(5, ls="--", c="green", lw=1, label="WHO guideline (5)")
ax.axhline(D.pm25_raw.mean(), ls="--", c="red", lw=1, label=f"Mean ({D.pm25_raw.mean():.1f})")
ax.set(ylabel="PM2.5 (µg/m³)", xlabel="")
ax.legend(frameon=False, fontsize=8); plt.tight_layout()
plt.savefig("fig1_timeseries.png", bbox_inches="tight"); plt.close()

# Fig 2
fig, axes = plt.subplots(1,2, figsize=(9,3))
mp = D.groupby("month").agg(pm=("pm25_raw","mean"), blh=("blh","mean"))
axes[0].bar(mp.index, mp.pm, color="#8B4513")
axes[0].set(xlabel="Month", ylabel="Mean PM2.5 (µg/m³)")
axes[1].bar(mp.index, mp.blh, color="#4682B4")
axes[1].set(xlabel="Month", ylabel="Mean boundary layer height (m)")
plt.tight_layout(); plt.savefig("fig2_seasonality.png", bbox_inches="tight"); plt.close()

def resid(y, X): return sm.OLS(y, sm.add_constant(X)).fit().resid
C2 = D[[c for c in CTRL if D[c].nunique()>1]]

# Fig 3
rx, ry = resid(D.log_blh, C2), resid(D.log_pm25, C2)
fig, ax = plt.subplots(figsize=(4.5,3.5))
ax.scatter(rx, ry, s=4, alpha=.25, c="#333")
z = np.polyfit(rx, ry, 1); xs = np.linspace(rx.min(), rx.max(), 50)
ax.plot(xs, np.polyval(z, xs), c="crimson", lw=2)
ax.set(xlabel="Log boundary layer height (residualised)",
       ylabel="Log PM2.5 (residualised)")
plt.tight_layout(); plt.savefig("fig3_firststage.png", bbox_inches="tight"); plt.close()

# Fig 4
rx2, ry2 = resid(D.log_pm25, C2), resid(D.logret, C2)
bins = pd.qcut(rx2, 20, labels=False)
bx, by = pd.Series(rx2).groupby(bins).mean(), pd.Series(ry2).groupby(bins).mean()
bse = pd.Series(ry2).groupby(bins).sem()
fig, ax = plt.subplots(figsize=(5,3.5))
ax.errorbar(bx, by, yerr=1.96*bse, fmt="o", ms=5, c="#1f3b73",
            ecolor="#999", elinewidth=1, capsize=2)
z2 = np.polyfit(rx2, ry2, 1)
ax.plot(bx, np.polyval(z2, bx), c="crimson", lw=1.5, label=f"Slope {z2[0]:.3f}")
ax.axhline(0, c="k", lw=.6, ls=":")
ax.set(xlabel="Log PM2.5 (residualised)", ylabel="Daily log return, % (residualised)")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("fig4_binscatter.png", bbox_inches="tight"); plt.close()

# Fig 5
specs = [("Preferred",-0.0827,0.1200),("Coverage ≥18h",-0.0887,0.1192),
         ("From 2022",-0.0940,0.1591),("Pre-2024",-0.0340,0.1177),
         ("Winter only",-0.0175,0.1615),("Winsorised 1%",-0.0948,0.1046),
         ("Excl. multi-session",-0.0993,0.0949),("Excl. pressure",-0.0800,0.1190),
         ("Excl. v",-0.1596,0.1218),("BLH + pressure",-0.1397,0.1281),
         ("BLH alone",-0.1332,0.1289)]
fig, ax = plt.subplots(figsize=(5.5,4))
ys = np.arange(len(specs))[::-1]
for i,(lab,est,se) in enumerate(specs):
    c = "crimson" if lab=="Preferred" else "#333"
    ax.errorbar(est, ys[i], xerr=1.96*se, fmt="o", ms=5, color=c,
                ecolor=c, elinewidth=1.2, capsize=2)
ax.axvline(0, c="k", lw=.8, ls="--")
ax.set_yticks(ys); ax.set_yticklabels([s[0] for s in specs], fontsize=8)
ax.set(xlabel="Coefficient on log PM2.5 (95% CI)")
plt.tight_layout(); plt.savefig("fig5_specification.png", bbox_inches="tight"); plt.close()

!cp *.png /content/drive/MyDrive/pollution_research/ 2>/dev/null
from google.colab import files
for f in ["fig1_timeseries.png","fig2_seasonality.png","fig3_firststage.png",
          "fig4_binscatter.png","fig5_specification.png"]:
    files.download(f)
print("done — 5 figures generated, backed up, and downloading")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

done — 5 figures generated, backed up, and downloading


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, statsmodels.api as sm
D = pd.read_csv("panel_analysis.csv"); D["date"]=pd.to_datetime(D["date"])
CTRL = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       ["temp","precip","wind_speed","cloud","radiation"] + \
       [c for c in ["lockdown","qantar"] if c in D.columns and D[c].nunique()>1]
plt.rcParams.update({"figure.dpi":300,"font.size":9,"axes.grid":True,"grid.alpha":.25,
                     "axes.spines.top":False,"axes.spines.right":False})

# --- check the late-2021 stretch
g = D[["date"]].copy(); g["gap"]=g.date.diff().dt.days
print("Gaps over 7 days:"); print(g[g.gap>7].to_string(index=False))

# --- FIG 1: break the line over gaps
full = pd.DataFrame({"date": pd.date_range(D.date.min(), D.date.max(), freq="D")})
s = full.merge(D[["date","pm25_raw"]], on="date", how="left")
fig, ax = plt.subplots(figsize=(7.5,3))
ax.plot(s.date, s.pm25_raw, lw=.5, c="#444")
ax.axhline(5, ls="--", c="green", lw=1, label="WHO annual guideline (5)")
ax.axhline(D.pm25_raw.mean(), ls="--", c="red", lw=1,
           label=f"Sample mean ({D.pm25_raw.mean():.1f})")
ax.set(xlabel="Date", ylabel="PM2.5 (µg/m³)")
ax.legend(frameon=False, fontsize=8); plt.tight_layout()
plt.savefig("fig1_timeseries.png", bbox_inches="tight"); plt.close()

# --- FIG 2: all months labelled, wider limits
fig, axes = plt.subplots(1,2, figsize=(8,3))
mp = D.groupby("month").agg(pm=("pm25_raw","mean"), blh=("blh","mean"))
lbl = ["J","F","M","A","M","J","J","A","S","O","N","D"]
for a,(col,yl,c) in zip(axes,[("pm","Mean PM2.5 (µg/m³)","#8B4513"),
                              ("blh","Mean boundary layer height (m)","#4682B4")]):
    a.bar(mp.index, mp[col], color=c)
    a.set(xlabel="Month", ylabel=yl, xlim=(0.3,12.7))
    a.set_xticks(range(1,13)); a.set_xticklabels(lbl)
plt.tight_layout(); plt.savefig("fig2_seasonality.png", bbox_inches="tight"); plt.close()

def resid(y,X): return sm.OLS(y, sm.add_constant(X)).fit().resid
C2 = D[[c for c in CTRL if D[c].nunique()>1]]

# --- FIG 3: annotate slope and F
rx, ry = resid(D.log_blh, C2), resid(D.log_pm25, C2)
z = np.polyfit(rx, ry, 1)
print(f"\nFig 3 bivariate residualised slope: {z[0]:.3f}")
fig, ax = plt.subplots(figsize=(4.5,3.5))
ax.scatter(rx, ry, s=4, alpha=.25, c="#333")
xs = np.linspace(rx.min(), rx.max(), 50)
ax.plot(xs, np.polyval(z,xs), c="crimson", lw=2,
        label=f"Slope {z[0]:.3f}\nFirst-stage F = 134.9")
ax.set(xlabel="Log boundary layer height (residualised)",
       ylabel="Log PM2.5 (residualised)")
ax.legend(frameon=False, fontsize=8, loc="upper right")
plt.tight_layout(); plt.savefig("fig3_firststage.png", bbox_inches="tight"); plt.close()

# --- FIG 4: label as OLS
rx2, ry2 = resid(D.log_pm25, C2), resid(D.logret, C2)
b = pd.qcut(rx2, 20, labels=False)
bx, by = pd.Series(rx2).groupby(b).mean(), pd.Series(ry2).groupby(b).mean()
bse = pd.Series(ry2).groupby(b).sem()
z2 = np.polyfit(rx2, ry2, 1)
fig, ax = plt.subplots(figsize=(5,3.5))
ax.errorbar(bx, by, yerr=1.96*bse, fmt="o", ms=5, c="#1f3b73",
            ecolor="#999", elinewidth=1, capsize=2)
ax.plot(bx, np.polyval(z2,bx), c="crimson", lw=1.5, label=f"OLS slope {z2[0]:.3f}")
ax.axhline(0, c="k", lw=.6, ls=":")
ax.set(xlabel="Log PM2.5 (residualised)", ylabel="Daily log return, % (residualised)")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("fig4_binscatter.png", bbox_inches="tight"); plt.close()

# --- FIG 5: clearer labels
specs = [("Preferred",-0.0827,0.1200),("Coverage ≥18h",-0.0887,0.1192),
         ("From 2022",-0.0940,0.1591),("Pre-2024",-0.0340,0.1177),
         ("Winter only",-0.0175,0.1615),("Winsorised 1%",-0.0948,0.1046),
         ("Excl. multi-session",-0.0993,0.0949),("Excl. pressure",-0.0800,0.1190),
         ("Excl. meridional wind",-0.1596,0.1218),("BLH + pressure",-0.1397,0.1281),
         ("BLH alone",-0.1332,0.1289)]
fig, ax = plt.subplots(figsize=(5.5,4))
ys = np.arange(len(specs))[::-1]
for i,(l,e,se) in enumerate(specs):
    c = "crimson" if l=="Preferred" else "#333"
    ax.errorbar(e, ys[i], xerr=1.96*se, fmt="o", ms=5, color=c, ecolor=c,
                elinewidth=1.2, capsize=2)
ax.axvline(0, c="k", lw=.8, ls="--")
ax.set_yticks(ys); ax.set_yticklabels([s[0] for s in specs], fontsize=8)
ax.set(xlabel="Coefficient on log PM2.5 (95% CI)")
plt.tight_layout(); plt.savefig("fig5_specification.png", bbox_inches="tight"); plt.close()

!cp *.png /content/drive/MyDrive/pollution_research/
from google.colab import files
for f in ["fig1_timeseries.png","fig2_seasonality.png","fig3_firststage.png",
          "fig4_binscatter.png","fig5_specification.png"]: files.download(f)
print("done")


Gaps over 7 days:
      date   gap
2021-05-28   8.0
2021-07-13  21.0
2021-08-04  13.0
2021-09-10   8.0
2021-11-08  31.0
2022-01-11  11.0
2022-09-07 131.0
2022-12-14  12.0
2023-02-14  13.0
2023-03-20  33.0
2024-07-01 185.0
2024-10-10  15.0

Fig 3 bivariate residualised slope: -0.405


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (2 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

done


In [ ]:
r = iv(D[D.date >= "2023-04-01"])
print(f"April 2023 onward: {r['b']:.4f} ({r['se']:.4f}) n={r['n']} F={r['F']:.1f} bound={r['bound']:.3f}")


NameError: name 'iv' is not defined

In [ ]:
!pip install linearmodels -q
import numpy as np, pandas as pd, warnings
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

D = pd.read_csv("panel_analysis.csv"); D["date"] = pd.to_datetime(D["date"]); D["const"] = 1
INST = ["log_blh","u","v","pressure"]
CTRL = [c for c in D.columns if c.startswith(("mo_","dw_","yr_"))] + \
       ["temp","precip","wind_speed","cloud","radiation"] + \
       [c for c in ["lockdown","qantar"] if c in D.columns and D[c].nunique() > 1]
LN2 = np.log(2)

def iv(data=None, y="logret", x="log_pm25", inst=None, ctrl=None):
    d = (data if data is not None else D).copy(); d["const"] = 1
    inst = inst or INST
    cand = [c for c in (ctrl or CTRL) if c in d.columns and d[c].nunique() > 1]
    d = d.dropna(subset=[y, x] + inst)
    keep = []
    for c in cand:
        M = d[["const"] + keep + [c]].astype(float).values
        if np.linalg.matrix_rank(M) == M.shape[1]: keep.append(c)
    r = IV2SLS(d[y], d[["const"] + keep], d[x], d[inst]).fit(cov_type="kernel")
    b, se = r.params[x], r.std_errors[x]
    try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except: F = float("nan")
    return dict(b=b, se=se, p=r.pvalues[x], n=len(d), F=F,
                bound=(abs(b) + 1.645*se)*LN2, r=r)

for start, lab in [("2023-04-01","Dense coverage (Apr 2023 on)"),
                   ("2024-07-01","Post-BLH-gap (Jul 2024 on)")]:
    try:
        r = iv(D[D.date >= start])
        print(f"{lab:32s} {r['b']:8.4f} ({r['se']:.4f})  p={r['p']:.3f}  "
              f"n={r['n']:4d}  F={r['F']:6.1f}  bound={r['bound']:.3f}")
    except Exception as e:
        print(f"{lab:32s} failed: {str(e)[:60]}")

print(f"\n{'Preferred (full sample)':32s}  -0.0827 (0.1200)  p=0.491  n= 977  F= 134.9  bound=0.194")


Dense coverage (Apr 2023 on)      -0.2007 (0.2084)  p=0.335  n= 509  F=  59.2  bound=0.377
Post-BLH-gap (Jul 2024 on)        -0.4097 (0.3835)  p=0.285  n= 329  F=  55.8  bound=0.721

Preferred (full sample)           -0.0827 (0.1200)  p=0.491  n= 977  F= 134.9  bound=0.194


In [ ]:
print(D.groupby("month")["blh"].mean().round(1).to_string())


month
1      59.7
2      65.7
3     130.4
4     236.7
5     371.4
6     550.3
7     555.6
8     507.6
9     358.9
10    194.9
11     99.8
12     51.5


In [ ]:
pm = pd.read_csv("almaty_pm25_daily.csv")
pm = pm[pm.sensor_id==25903]
print((pm.n_obs>=6).sum(), (pm.n_obs>=8).sum(), (pm.n_obs>=18).sum())


NameError: name 'pd' is not defined

In [ ]:
"""
REBUILD ON OFFICIAL KASE SERIES
Run after uploading kase_official.csv to Colab.
Prints every number that changes relative to the Investing.com panel.
"""

!pip install linearmodels -q

import numpy as np, pandas as pd, warnings
from linearmodels.iv import IV2SLS
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
!cp /content/drive/MyDrive/pollution_research/*.csv . 2>/dev/null

# ---------------------------------------------------------------- LOAD
pm = pd.read_csv("almaty_pm25_daily.csv");    pm["date"] = pd.to_datetime(pm["date"])
wx = pd.read_csv("almaty_weather_daily.csv"); wx["date"] = pd.to_datetime(wx["date"])
ks = pd.read_csv("kase_official.csv");        ks["date"] = pd.to_datetime(ks["date"])

print(f"Official KASE: {len(ks)} rows, {ks.date.min().date()} to {ks.date.max().date()}")
if "weekend_session" in ks.columns:
    print(f"Weekend sessions flagged: {int(ks.weekend_session.sum())}")
chk = ks[ks.date == "2022-03-09"]
if len(chk):
    print(f"2022-03-09 logret: {chk.logret.values[0]:.4f}   (should be ~4.05)")
print()

# ---------------------------------------------------------------- PANEL
p = (pm[(pm.sensor_id == 25903) & (pm.n_obs >= 12)][["date", "pm25", "n_obs"]]
       .rename(columns={"pm25": "pm25_raw"}))

cols = ["date", "logret"] + [c for c in ["close"] if c in ks.columns]
D = ks[cols].merge(p, on="date").merge(wx, on="date")
D = D[(D.date >= "2020-04-08") & (D.date <= "2025-11-13")].copy()

D["log_pm25"] = np.log(D.pm25_raw)
D["log_blh"]  = np.log(D.blh)
D["dow"]   = D.date.dt.dayofweek
D["month"] = D.date.dt.month
D["year"]  = D.date.dt.year
D["lockdown"] = (((D.date >= "2020-03-16") & (D.date <= "2020-05-11")) |
                 ((D.date >= "2020-07-05") & (D.date <= "2020-08-17"))).astype(int)
D["qantar"]   = ((D.date >= "2022-01-04") & (D.date <= "2022-01-19")).astype(int)
D = (D.dropna(subset=["logret", "log_pm25", "log_blh", "u", "v", "pressure"])
       .sort_values("date").reset_index(drop=True))

ctrl = ["temp", "precip", "wind_speed", "cloud", "radiation"]
for m in sorted(D.month.unique())[1:]:
    D[f"mo_{m}"] = (D.month == m).astype(int); ctrl.append(f"mo_{m}")
for d_ in sorted(D.dow.unique())[1:]:
    D[f"dw_{d_}"] = (D.dow == d_).astype(int); ctrl.append(f"dw_{d_}")
for y in sorted(D.year.unique())[1:]:
    D[f"yr_{y}"] = (D.year == y).astype(int); ctrl.append(f"yr_{y}")
for c in ["lockdown", "qantar"]:
    if D[c].nunique() > 1: ctrl.append(c)
D["const"] = 1

D.to_csv("panel_official.csv", index=False)
!cp panel_official.csv /content/drive/MyDrive/pollution_research/

# ---------------------------------------------------------------- ESTIMATE
INST = ["log_blh", "u", "v", "pressure"]
keep = []
for c in ctrl:
    M = D[["const"] + keep + [c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]: keep.append(c)

r  = IV2SLS(D["logret"], D[["const"] + keep], D["log_pm25"], D[INST]).fit(cov_type="kernel")
b, se = r.params["log_pm25"], r.std_errors["log_pm25"]
try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
except: F = float("nan")
LN2 = np.log(2)

W = 62
print("=" * W); print("REBUILT ON OFFICIAL SERIES"); print("=" * W)
print(f"  {'N':<18}{len(D):>10}      (was 977)")
print(f"  {'IV estimate':<18}{b:>10.4f}      (was -0.0827)")
print(f"  {'SE':<18}{se:>10.4f}      (was  0.1200)")
print(f"  {'p':<18}{r.pvalues['log_pm25']:>10.3f}      (was  0.491)")
print(f"  {'95% CI':<18}[{b-1.96*se:.4f}, {b+1.96*se:.4f}]")
print(f"  {'First-stage F':<18}{F:>10.1f}      (was  134.9)")
print(f"  {'Sargan p':<18}{r.sargan.pval:>10.4f}      (was  0.1169)")
try:
    print(f"  {'Wu-Hausman p':<18}{r.wu_hausman().pval:>10.4f}      (was  0.0735)")
except Exception: pass
print(f"  {'TOST bound (pp)':<18}{(abs(b)+1.645*se)*LN2:>10.3f}      (was  0.194)")
print(f"  {'Per doubling':<18}{b*LN2:>10.4f}  (SE {se*LN2:.4f})")

ols = smf.ols("logret ~ log_pm25 + " + " + ".join(keep), data=D
              ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"  {'OLS estimate':<18}{ols.params['log_pm25']:>10.4f}      (was +0.0526)")
print(f"  {'OLS SE':<18}{ols.bse['log_pm25']:>10.4f}      (was  0.0439)")

fs = smf.ols("log_pm25 ~ " + " + ".join(INST + keep), data=D
             ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"\n  FIRST STAGE            (was)")
old_fs = {"log_blh": -0.3714, "u": 0.0324, "v": -0.0304, "pressure": -0.0174}
for v in INST:
    print(f"    {v:<12}{fs.params[v]:>9.4f} ({fs.bse[v]:.4f})   t={fs.tvalues[v]:6.2f}"
          f"   (was {old_fs[v]:+.4f})")
try:
    pr2 = r.first_stage.diagnostics['partial.rsquared'].iloc[0]
    print(f"    {'partial R2':<12}{pr2:>9.4f}                        (was  0.1810)")
except Exception: pass
print(f"    {'R2':<12}{fs.rsquared:>9.4f}                        (was  0.7470)")

# reduced form
rf   = smf.ols("logret ~ " + " + ".join(INST + keep), data=D
               ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
base = smf.ols("logret ~ " + " + ".join(keep), data=D).fit()
full = smf.ols("logret ~ " + " + ".join(INST + keep), data=D).fit()
prf  = (full.rsquared - base.rsquared) / (1 - base.rsquared)
pj   = rf.f_test(", ".join(f"{i} = 0" for i in INST)).pvalue
print(f"\n  REDUCED FORM")
print(f"    {'BLH coef':<12}{rf.params['log_blh']:>9.4f} ({rf.bse['log_blh']:.4f})"
      f"   p={rf.pvalues['log_blh']:.3f}   (was 0.0480)")
print(f"    {'joint p':<12}{pj:>9.4f}                        (was 0.1330)")
print(f"    {'partial R2':<12}{prf:>9.4f}                        (was 0.0067)")
print(f"    {'TOST bound':<12}"
      f"{(abs(rf.params['log_blh'])+1.645*rf.bse['log_blh'])*LN2:>9.4f} pp"
      f"                  (was 0.0919)")

print(f"\n  TABLE 1")
old = {"logret": (0.09, 0.67, -3.02, 0.10, 5.02),
       "pm25_raw": (29.79, 33.07, 3.38, 16.80, 229.00),
       "blh": (267.43, 221.39, 19.38, 220.00, 1303.12)}
for c, l in [("logret", "Log return"), ("pm25_raw", "PM2.5"), ("blh", "BLH")]:
    o = old[c]
    print(f"    {l:<11} mean {D[c].mean():8.2f} ({o[0]})   sd {D[c].std():7.2f} ({o[1]})   "
          f"max {D[c].max():8.2f} ({o[4]})")
    print(f"    {'':11} min  {D[c].min():8.2f} ({o[2]})   med {D[c].median():7.2f} ({o[3]})")
print(f"    SD log PM2.5 {np.log(D.pm25_raw).std():.3f} (0.816)   "
      f"AR(1) {D.logret.autocorr(1):.3f} (-0.006)")
print(f"    Days >55: {(D.pm25_raw>55).sum()} (142)   >75: {(D.pm25_raw>75).sum()} (86)")

# transport tests
est_pp, se_pp = b * LN2, se * LN2
print(f"\n  TRANSPORT TESTS")
for d_, sd_, lab in [(0.194, 0.063, "proportional"), (0.500, 0.164, "linear")]:
    t = (d_ + est_pp) / np.sqrt(se_pp**2 + sd_**2)
    print(f"    {lab:<14}t = {t:5.2f}   "
          f"{'rejected' if abs(t) > 1.96 else 'not rejected'}"
          f"   (was {1.31 if lab=='proportional' else 2.41})")

print(f"\n  Saved panel_official.csv and backed up to Drive.")


Mounted at /content/drive
Official KASE: 1480 rows, 2020-01-05 to 2026-01-09
Weekend sessions flagged: 9
2022-03-09 logret: 4.0479   (should be ~4.05)

REBUILT ON OFFICIAL SERIES
  N                        983      (was 977)
  IV estimate          -0.0766      (was -0.0827)
  SE                    0.1153      (was  0.1200)
  p                      0.506      (was  0.491)
  95% CI            [-0.3026, 0.1494]
  First-stage F          135.6      (was  134.9)
  Sargan p              0.1170      (was  0.1169)
  Wu-Hausman p          0.0791      (was  0.0735)
  TOST bound (pp)        0.185      (was  0.194)
  Per doubling         -0.0531  (SE 0.0799)
  OLS estimate          0.0511      (was +0.0526)
  OLS SE                0.0434      (was  0.0439)

  FIRST STAGE            (was)
    log_blh       -0.3732 (0.0411)   t= -9.07   (was -0.3714)
    u              0.0326 (0.0084)   t=  3.89   (was +0.0324)
    v             -0.0302 (0.0082)   t= -3.71   (was -0.0304)
    pressure      -0.0175 (0

In [ ]:
import pandas as pd
D = pd.read_csv("panel_official.csv"); D["date"] = pd.to_datetime(D["date"])
pm = pd.read_csv("almaty_pm25_daily.csv")

print("WEEKEND SESSIONS IN PANEL:", (D.date.dt.dayofweek >= 5).sum())
print("dw_ dummies created:", [c for c in D.columns if c.startswith("dw_")])
if (D.date.dt.dayofweek >= 5).sum():
    print(D[D.date.dt.dayofweek >= 5][["date","logret","pm25_raw"]].to_string(index=False))

print("\nn_obs RANGE:", pm.n_obs.min(), "to", pm.n_obs.max())
print("share >11:", (pm.n_obs > 11).mean().round(3))


WEEKEND SESSIONS IN PANEL: 5
dw_ dummies created: ['dw_1', 'dw_2', 'dw_3', 'dw_4', 'dw_5', 'dw_6']
      date    logret  pm25_raw
2020-12-20 -0.327025     34.60
2022-03-05  0.970453     30.00
2022-10-22  0.511801     17.90
2023-07-01 -0.018866      5.33
2025-01-05  1.106776     30.20

n_obs RANGE: 1 to 25
share >11: 0.925


In [ ]:
import pandas as pd, numpy as np

h = pd.read_csv("almaty_pm25_hourly.csv")
h["hour"] = h["local"].str.slice(11,13).astype(int)
h["date"] = pd.to_datetime(h["local"].str.slice(0,10))

# rebuild both candidate series from hourly source
w = h[(h.hour>=7)&(h.hour<=17)]
win = w.groupby("date").agg(pm_window=("pm25","mean"), n_window=("pm25","size"))
cal = h.groupby("date").agg(pm_calendar=("pm25","mean"), n_calendar=("pm25","size"))

pm = pd.read_csv("almaty_pm25_daily.csv")
pm = pm[pm.sensor_id==25903].copy(); pm["date"] = pd.to_datetime(pm["date"])
m = pm[["date","pm25","n_obs"]].merge(win, on="date", how="left").merge(cal, on="date", how="left")

print("Which does the stored series match?")
print(f"  corr(pm25, window mean):   {m[['pm25','pm_window']].corr().iloc[0,1]:.6f}")
print(f"  corr(pm25, calendar mean): {m[['pm25','pm_calendar']].corr().iloc[0,1]:.6f}")
print(f"  mean abs diff vs window:   {(m.pm25-m.pm_window).abs().mean():.4f}")
print(f"  mean abs diff vs calendar: {(m.pm25-m.pm_calendar).abs().mean():.4f}")
print(f"\n  n_obs vs n_window identical: {(m.n_obs==m.n_window).mean():.3f}")
print(f"  n_obs vs n_calendar identical: {(m.n_obs==m.n_calendar).mean():.3f}")
print(m[["date","pm25","pm_window","pm_calendar","n_obs","n_window","n_calendar"]].head(8).to_string(index=False))


Which does the stored series match?
  corr(pm25, window mean):   0.962036
  corr(pm25, calendar mean): 0.999998
  mean abs diff vs window:   5.2845
  mean abs diff vs calendar: 0.0309

  n_obs vs n_window identical: 0.002
  n_obs vs n_calendar identical: 0.969
      date  pm25  pm_window  pm_calendar  n_obs  n_window  n_calendar
2020-04-09  8.00        NaN     8.000000      2       NaN         2.0
2020-04-10 11.10   8.909091    11.130435     24      11.0        23.0
2020-04-11 38.00  34.090909    37.958333     24      11.0        24.0
2020-04-12 21.40  16.454545    21.416667     24      11.0        24.0
2020-04-13 10.90   9.909091    10.916667     24      11.0        24.0
2020-04-14 11.50  16.363636    11.500000     24      11.0        24.0
2020-04-15  6.22   5.545455     6.217391     24      11.0        23.0
2020-04-16 17.20  13.363636    17.166667     24      11.0        24.0


In [ ]:
import pandas as pd, numpy as np

h = pd.read_csv("almaty_pm25_hourly.csv")
h["hour"] = h["local"].str.slice(11, 13).astype(int)
h["date"] = pd.to_datetime(h["local"].str.slice(0, 10))

def agg(mask, name):
    g = h[mask].groupby("date").agg(**{f"pm_{name}": ("pm25", "mean"),
                                       f"n_{name}": ("pm25", "size")})
    return g

out = (agg((h.hour >= 7) & (h.hour <= 17), "window")     # 07:00-17:59, preferred
       .join(agg((h.hour >= 11) & (h.hour <= 17), "session"), how="outer")   # 11:00-17:59
       .join(agg((h.hour >= 7) & (h.hour <= 10), "presession"), how="outer") # 07:00-10:59
       .join(agg(h.hour >= 0, "daily"), how="outer")     # calendar day
       .reset_index())

out = out.rename(columns={"pm_window": "pm25", "n_window": "n_window",
                          "pm_daily": "pm25_calendar", "n_daily": "n_daily"})
out.to_csv("almaty_pm25_daily_window.csv", index=False)

# Reconciliation against the July analysis — these should land near 1,668 / 1,548:
print("days with any in-window obs:", out.n_window.notna().sum())
print("days passing >=8 in window:", (out.n_window >= 8).sum())
print("winter pass rate:", (out[out.date.dt.month.isin([11,12,1,2])].n_window >= 8).mean().round(3))
print("corr window vs session (logs):",
      np.log(out.pm25).corr(np.log(out.pm_session)).round(3))
print("session mean:", out.pm_session.mean().round(1), " window mean:", out.pm25.mean().round(1))


days with any in-window obs: 1668
days passing >=8 in window: 1548
winter pass rate: 0.914
corr window vs session (logs): nan
session mean: 36.1  window mean: 31.8


In [ ]:
print("hourly obs <= 0:", (h.pm25 <= 0).sum(), " min:", h.pm25.min(),
      " max:", h.pm25.max())
for c in ["pm25", "pm_session", "pm_presession", "pm25_calendar"]:
    print(f"{c}: nonpositive days {(out[c] <= 0).sum()}, min {out[c].min():.2f}")


hourly obs <= 0: 89  min: 0.0  max: 421.0
pm25: nonpositive days 0, min 1.00
pm_session: nonpositive days 1, min 0.00
pm_presession: nonpositive days 0, min 1.50
pm25_calendar: nonpositive days 0, min 3.25


In [20]:
"""
FINAL REBUILD — window treatment from hourly source + official KASE + full battery
==================================================================================
One script, run top to bottom in Colab. It:

  1. Builds the daily PM2.5 series from almaty_pm25_hourly.csv:
     preferred 07:00-17:59 window mean, session-only (11:00-17:59),
     pre-session (07:00-10:59), and calendar-day means, each with its
     observation count. Hourly zeros (89 obs) are RETAINED — no negatives
     exist in the source, and dropping below-detection readings would bias
     means upward. Prints reconciliation checks against the July analysis
     (expected: 1,668 days with in-window obs; 1,548 passing >= 8).

  2. Builds the panel on the official KASE series with the pre-specified
     rules: >= 8 in-window hourly obs; weekend compensating sessions feed
     the price chain (logret already computed in kase_official.csv) but are
     EXCLUDED as observations; finite-log guard.

  3. Re-runs everything: main spec, first stage, reduced form, Table 1,
     full Table 6 (p-values on every row; session / pre-session /
     calendar-day treatment rows included), placebo leads, randomisation
     inference (1000 draws), threshold indicators, transport tests, S4.7
     FE sensitivity, S3.4 attrition stats, S6.5 monthly counts.

Paste the FULL output back into chat.
"""

!pip install linearmodels -q
import numpy as np, pandas as pd, warnings
from linearmodels.iv import IV2SLS
import statsmodels.formula.api as smf
from scipy import stats
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
!cp /content/drive/MyDrive/pollution_research/*.csv .

LN2 = np.log(2)
RNG = np.random.default_rng(20260806)
WINDOW_MIN = 8   # paper's stated rule: >= 8 hourly obs inside 07:00-17:59

# ================================================================== STEP 1
# Build daily treatment series from hourly source
h = pd.read_csv("almaty_pm25_hourly.csv")
assert {"local", "pm25"}.issubset(h.columns), f"unexpected columns: {list(h.columns)}"
h["hour"] = h["local"].str.slice(11, 13).astype(int)
h["date"] = pd.to_datetime(h["local"].str.slice(0, 10))

print("=" * 70)
print("STEP 1 — TREATMENT CONSTRUCTION FROM HOURLY DATA")
print("=" * 70)
print(f"hourly obs: {len(h)}   zeros: {(h.pm25 == 0).sum()}   "
      f"negatives: {(h.pm25 < 0).sum()}   min {h.pm25.min()}   max {h.pm25.max()}")

def agg(mask, name):
    return h[mask].groupby("date").agg(**{f"pm_{name}": ("pm25", "mean"),
                                          f"n_{name}": ("pm25", "size")})

out = (agg((h.hour >= 7) & (h.hour <= 17), "window")
       .join(agg((h.hour >= 11) & (h.hour <= 17), "session"), how="outer")
       .join(agg((h.hour >= 7) & (h.hour <= 10), "presession"), how="outer")
       .join(agg(h.hour >= 0, "daily"), how="outer")
       .reset_index()
       .rename(columns={"pm_window": "pm25", "pm_daily": "pm25_calendar"}))
out.to_csv("almaty_pm25_daily_window.csv", index=False)

print(f"days with any in-window obs: {out.n_window.notna().sum()}   (expect ~1668)")
print(f"days passing >= {WINDOW_MIN} in window: {(out.n_window >= WINDOW_MIN).sum()}"
      f"   (expect ~1548)")
print(f"winter pass rate: "
      f"{(out[out.date.dt.month.isin([11,12,1,2])].n_window >= WINDOW_MIN).mean():.3f}")
print(f"session mean {out.pm_session.mean():.1f}   window mean {out.pm25.mean():.1f}"
      f"   calendar mean {out.pm25_calendar.mean():.1f}")

# ================================================================== STEP 2
# Panel on official KASE
wx = pd.read_csv("almaty_weather_daily_cds.csv"); wx["date"] = pd.to_datetime(wx["date"])
ks = pd.read_csv("kase_official.csv");        ks["date"] = pd.to_datetime(ks["date"])

print("\n" + "=" * 70)
print("STEP 2 — PANEL")
print("=" * 70)
print(f"Official KASE: {len(ks)} rows, {ks.date.min().date()} to {ks.date.max().date()}")
chk = ks[ks.date == "2022-03-09"]
if len(chk): print(f"2022-03-09 logret: {chk.logret.values[0]:.4f}   (target ~4.05)")

p = out[out.n_window >= WINDOW_MIN][
    ["date", "pm25", "pm_session", "pm_presession", "pm25_calendar", "n_daily"]
].rename(columns={"pm25": "pm25_raw"})

D = ks[["date", "logret"]].merge(p, on="date").merge(wx, on="date")
D = D[(D.date >= "2020-04-08") & (D.date <= "2025-11-13")].copy()

# Weekend rule: weekend closes already fed the next session's logret inside
# kase_official.csv; the weekend rows themselves leave the sample here.
n_wk = (D.date.dt.dayofweek >= 5).sum()
D = D[D.date.dt.dayofweek < 5].copy()
print(f"Weekend-session rows dropped from panel (kept in price chain): {n_wk}")

D["log_pm25"] = np.log(D.pm25_raw)
D["log_blh"]  = np.log(D.blh)
D["dow"]   = D.date.dt.dayofweek
D["month"] = D.date.dt.month
D["year"]  = D.date.dt.year
D["lockdown"] = (((D.date >= "2020-03-16") & (D.date <= "2020-05-11")) |
                 ((D.date >= "2020-07-05") & (D.date <= "2020-08-17"))).astype(int)
D["qantar"]   = ((D.date >= "2022-01-04") & (D.date <= "2022-01-19")).astype(int)
D = (D.dropna(subset=["logret", "log_pm25", "log_blh", "u", "v", "pressure"])
       .sort_values("date").reset_index(drop=True))
D = D[np.isfinite(D.log_pm25)].reset_index(drop=True)

WEATHER = ["temp", "precip", "wind_speed", "cloud", "radiation"]
for m in range(1, 13):  D[f"mo_{m}"] = (D.month == m).astype(int)
for d_ in range(0, 5):  D[f"dw_{d_}"] = (D.dow == d_).astype(int)
for y in sorted(D.year.unique()): D[f"yr_{y}"] = (D.year == y).astype(int)
D["const"] = 1

def build_ctrl(df, weather=WEATHER, month_fe=True, year_fe=True):
    ctrl = list(weather)
    if month_fe:
        ctrl += [f"mo_{m}" for m in sorted(df.month.unique())[1:]]
    ctrl += [f"dw_{d_}" for d_ in sorted(df.dow.unique())[1:]]
    if year_fe:
        ctrl += [f"yr_{y}" for y in sorted(df.year.unique())[1:]]
    ctrl += [c for c in ["lockdown", "qantar"] if df[c].nunique() > 1]
    keep = []
    for c in ctrl:
        M = df[["const"] + keep + [c]].astype(float).values
        if np.linalg.matrix_rank(M) == M.shape[1]: keep.append(c)
    return keep

INST = ["log_blh", "u", "v", "pressure"]

def run_iv(df, endog="log_pm25", inst=INST, dep="logret",
           weather=WEATHER, month_fe=True, year_fe=True):
    keep = build_ctrl(df, weather, month_fe, year_fe)
    r = IV2SLS(df[dep], df[["const"] + keep], df[endog], df[inst]).fit(cov_type="kernel")
    b, se = r.params[endog], r.std_errors[endog]
    try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except Exception: F = float("nan")
    try:    sarg = r.sargan.pval if len(inst) > 1 else float("nan")
    except Exception: sarg = float("nan")
    return dict(b=b, se=se, p=r.pvalues[endog], N=int(r.nobs), F=F, sargan=sarg,
                bound=(abs(b) + 1.645 * se) * LN2, res=r, keep=keep)

# ================================================================== STEP 3
M0 = run_iv(D)
b, se = M0["b"], M0["se"]
print("\n" + "=" * 70)
print(f"PREFERRED SPEC — window treatment, weekend rule, >= {WINDOW_MIN} in window")
print("=" * 70)
print(f"  N              {M0['N']:>10}")
print(f"  IV estimate    {b:>10.4f}")
print(f"  SE             {se:>10.4f}")
print(f"  p              {M0['p']:>10.3f}")
print(f"  95% CI         [{b-1.96*se:.4f}, {b+1.96*se:.4f}]")
print(f"  First-stage F  {M0['F']:>10.1f}")
print(f"  Sargan p       {M0['sargan']:>10.4f}")
try: print(f"  Wu-Hausman p   {M0['res'].wu_hausman().pval:>10.4f}")
except Exception: pass
print(f"  TOST bound pp  {M0['bound']:>10.3f}")
print(f"  Per doubling   {b*LN2:>10.4f}  (SE {se*LN2:.4f})")

keep = M0["keep"]
ols = smf.ols("logret ~ log_pm25 + " + " + ".join(keep), data=D
              ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"  OLS estimate   {ols.params['log_pm25']:>10.4f}  (SE {ols.bse['log_pm25']:.4f})")

fs = smf.ols("log_pm25 ~ " + " + ".join(INST + keep), data=D
             ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print("\n  FIRST STAGE")
for v_ in INST:
    print(f"    {v_:<10}{fs.params[v_]:>9.4f} ({fs.bse[v_]:.4f})  t={fs.tvalues[v_]:6.2f}")
try:
    print(f"    partial R2 "
          f"{M0['res'].first_stage.diagnostics['partial.rsquared'].iloc[0]:.4f}"
          f"   R2 {fs.rsquared:.4f}")
except Exception: pass

rf   = smf.ols("logret ~ " + " + ".join(INST + keep), data=D
               ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
base = smf.ols("logret ~ " + " + ".join(keep), data=D).fit()
full = smf.ols("logret ~ " + " + ".join(INST + keep), data=D).fit()
print("\n  REDUCED FORM")
for v_ in INST:
    print(f"    {v_:<10}{rf.params[v_]:>9.4f} ({rf.bse[v_]:.4f})  p={rf.pvalues[v_]:.3f}")
print(f"    joint p    {rf.f_test(', '.join(f'{i} = 0' for i in INST)).pvalue:.4f}"
      f"   partial R2 {(full.rsquared-base.rsquared)/(1-base.rsquared):.4f}")
print(f"    BLH TOST bound "
      f"{(abs(rf.params['log_blh'])+1.645*rf.bse['log_blh'])*LN2:.4f} pp")

print(f"\n  TABLE 1 (N = {len(D)})")
for c, l in [("logret", "Log return"), ("pm25_raw", "PM2.5"), ("blh", "BLH"),
             ("temp", "Temp"), ("precip", "Precip"), ("wind_speed", "Wind"),
             ("pressure", "Pressure"), ("cloud", "Cloud"), ("radiation", "Radiation")]:
    print(f"    {l:<11} mean {D[c].mean():9.2f}  sd {D[c].std():8.2f}  "
          f"min {D[c].min():9.2f}  med {D[c].median():8.2f}  max {D[c].max():9.2f}")
print(f"    SD log PM2.5 {np.log(D.pm25_raw).std():.3f}   AR(1) {D.logret.autocorr(1):.3f}")
print(f"    Days >35: {(D.pm25_raw>35).sum()}  >55: {(D.pm25_raw>55).sum()}"
      f"  >75: {(D.pm25_raw>75).sum()}")
print(f"    Window mean / benchmark 11.53 = {D.pm25_raw.mean()/11.53:.2f}"
      f"   (paper currently says 2.6)")
print(f"    Window mean / WHO 5 = {D.pm25_raw.mean()/5:.1f}   (paper says ~6x)")

# ------------------------------------------------------------------ TABLE 6
print("\n" + "=" * 70)
print("TABLE 6 — coef, SE, p, N, F, bound on every row")
print("=" * 70)

def row(name, res):
    print(f"  {name:<34}{res['b']:>8.4f} {res['se']:>7.4f} {res['p']:>6.3f}"
          f" {res['N']:>5} {res['F']:>7.1f} {res['bound']:>7.3f}"
          + (f"  Sarg {res['sargan']:.3f}" if not np.isnan(res['sargan']) else ""))

row("Preferred", M0)
row("Coverage >= 18h (daily)", run_iv(D[D.n_daily >= 18]))
row("Coverage >= 15h (daily)", run_iv(D[D.n_daily >= 15]))
row("Window coverage = 11 (full)", run_iv(D[D.date.isin(
    out[out.n_window == 11].date)]))
row("Sample from 2022",  run_iv(D[D.year >= 2022]))
row("Sample from 2023",  run_iv(D[D.year >= 2023]))
row("Excluding 2025",    run_iv(D[D.year <= 2024]))
row("Pre-2024 clock",    run_iv(D[D.date < "2024-03-01"]))
row("Winter only",       run_iv(D[D.month.isin([11, 12, 1, 2, 3])]))
Dw = D.copy(); lo, hi = Dw.logret.quantile([.01, .99]); Dw["logret"] = Dw.logret.clip(lo, hi)
row("Winsorised 1%",     run_iv(Dw))
row("Instr excl pressure",  run_iv(D, inst=["log_blh", "u", "v"]))
row("Instr excl v",         run_iv(D, inst=["log_blh", "u", "pressure"]))
row("Instr excl u",         run_iv(D, inst=["log_blh", "v", "pressure"]))
row("BLH + u",              run_iv(D, inst=["log_blh", "u"]))
row("BLH + pressure",       run_iv(D, inst=["log_blh", "pressure"]))
row("BLH alone",            run_iv(D, inst=["log_blh"]))
row("Ctrls excl cloud/rad", run_iv(D, weather=["temp", "precip", "wind_speed"]))
row("Excl Jan 2022 unrest", run_iv(D[~((D.date >= "2022-01-04") &
                                       (D.date <= "2022-01-19"))]))
row("Dense (Apr 2023 on)",    run_iv(D[D.date >= "2023-04-01"]))
row("Post-gap (Jul 2024 on)", run_iv(D[D.date >= "2024-07-01"]))

for k in (1, 2, 3):
    Dk = D.copy()
    for c in ["log_pm25"] + INST: Dk[c] = Dk[c].shift(k)
    Dk = Dk.dropna(subset=["log_pm25"] + INST).reset_index(drop=True)
    row(f"Lag {k} day", run_iv(Dk))

Da = D.copy(); Da["abslogret"] = Da.logret.abs()
row("Outcome: |return|", run_iv(Da, dep="abslogret"))

# alternative treatment constructions (S4.2's "reported as robustness")
for alt, lab in [("pm_session", "Session-only treatment"),
                 ("pm_presession", "Pre-session treatment"),
                 ("pm25_calendar", "Calendar-day treatment")]:
    Dv = D.copy(); Dv["log_pm25"] = np.log(Dv[alt])
    Dv = Dv[np.isfinite(Dv.log_pm25)].dropna(subset=["log_pm25"]).reset_index(drop=True)
    row(lab, run_iv(Dv))
pos = D.pm_session > 0
print(f"  log corr window vs session:    "
      f"{np.log(D[pos].pm25_raw).corr(np.log(D[pos].pm_session)):.3f}")
print(f"  log corr window vs presession: "
      f"{np.log(D.pm25_raw).corr(np.log(D.pm_presession)):.3f}")
print(f"  log corr window vs calendar:   "
      f"{np.log(D.pm25_raw).corr(np.log(D.pm25_calendar)):.3f}")

# ------------------------------------------------------------------ PLACEBOS
print("\nPLACEBO LEADS (return realised BEFORE the pollution)")
for k in (1, 2, 3, 5):
    Dk = D.copy(); Dk["lead"] = Dk.logret.shift(-k)
    Dk = Dk.dropna(subset=["lead"]).reset_index(drop=True)
    row(f"{k}-day lead", run_iv(Dk, dep="lead"))
D1 = D.copy(); D1["lead"] = D1.logret.shift(-1); D1["lag_pm"] = D1.log_pm25.shift(1)
D1 = D1.dropna(subset=["lead", "lag_pm"]).reset_index(drop=True)
keep1 = build_ctrl(D1) + ["lag_pm"]
r1 = IV2SLS(D1["lead"], D1[["const"] + keep1], D1["log_pm25"], D1[INST]
            ).fit(cov_type="kernel")
print(f"  {'1-day lead + lagged PM2.5':<34}{r1.params['log_pm25']:>8.4f}"
      f" {r1.std_errors['log_pm25']:>7.4f} {r1.pvalues['log_pm25']:>6.3f}"
      f" {int(r1.nobs):>5}")

# ------------------------------------------------------------------ RANDOMISATION INFERENCE
print("\nRANDOMISATION INFERENCE (1000 circular shifts, multiples of 5)")
exog, endog, instr = D[["const"] + keep], D["log_pm25"], D[INST]
n = len(D); draws = []
for _ in range(1000):
    s = 5 * int(RNG.integers(1, n // 5))
    yb = pd.Series(np.roll(D.logret.values, s), index=D.index)
    draws.append(IV2SLS(yb, exog, endog, instr
                        ).fit(cov_type="unadjusted").params["log_pm25"])
draws = np.array(draws)
print(f"  null SD {draws.std():.4f}  vs HAC SE {se:.4f}  ratio {draws.std()/se:.2f}")
print(f"  95% null range [{np.percentile(draws,2.5):.3f}, "
      f"{np.percentile(draws,97.5):.3f}]")
print(f"  RI p (|b|) {np.mean(np.abs(draws) >= abs(b)):.3f}"
      f"   percentile of estimate {stats.percentileofscore(draws, b):.1f}")

# ------------------------------------------------------------------ THRESHOLDS
print("\nTHRESHOLD INDICATORS (reduced form, OLS)")
for thr in (35, 55, 75):
    Dt = D.copy(); Dt["hi"] = (Dt.pm25_raw > thr).astype(int)
    rt = smf.ols("logret ~ hi + " + " + ".join(keep), data=Dt
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  >{thr}: {rt.params['hi']:.4f} (SE {rt.bse['hi']:.4f},"
          f" p={rt.pvalues['hi']:.3f})   days above: {Dt.hi.sum()}")

# ------------------------------------------------------------------ TRANSPORT
est_pp, se_pp = b * LN2, se * LN2
mean_pm = D.pm25_raw.mean()
lin_pp, lin_se = 0.0168 * mean_pm, 0.0055 * mean_pm   # benchmark linear at OUR mean
print("\nTRANSPORT TESTS (benchmark linear implication recomputed at window mean)")
print(f"  window mean {mean_pm:.2f}  ->  linear implication {lin_pp:.3f} pp"
      f" (SE {lin_se:.3f})   [paper's old figures: 0.500, 0.164]")
for d_, sd_, lab in [(0.194, 0.063, "proportional"), (lin_pp, lin_se, "linear")]:
    t = (d_ + est_pp) / np.sqrt(se_pp ** 2 + sd_ ** 2)
    print(f"  {lab:<14}t = {t:5.2f}   {'rejected' if abs(t) > 1.96 else 'not rejected'}")
print(f"  Per 10 ug/m3 at mean: "
      f"{b*np.log((mean_pm+10)/mean_pm):.4f} (SE {se*np.log((mean_pm+10)/mean_pm):.4f})")

# ------------------------------------------------------------------ S4.7 FE SENSITIVITY
print("\nFIRST-STAGE BLH COEF ACROSS FE SPECS (S4.7)")
for mfe, yfe, lab in [(False, False, "no month/year FE"),
                      (True, False, "month FE only"),
                      (True, True, "month + year FE")]:
    kfe = build_ctrl(D, month_fe=mfe, year_fe=yfe)
    ffe = smf.ols("log_pm25 ~ " + " + ".join(INST + kfe), data=D
                  ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  {lab:<18}{ffe.params['log_blh']:>8.4f} ({ffe.bse['log_blh']:.4f})")

# ------------------------------------------------------------------ ATTRITION (S3.4 / S6.5)
print("\nATTRITION — retained vs excluded weekday trading days in window")
allts = ks[(ks.date >= "2020-04-08") & (ks.date <= "2025-11-13")]
allts = allts[allts.date.dt.dayofweek < 5].dropna(subset=["logret"])
ret = allts[allts.date.isin(D.date)]; exc = allts[~allts.date.isin(D.date)]
print(f"  weekday trading days {len(allts)}   retained {len(ret)}   excluded {len(exc)}")
t_, p_ = stats.ttest_ind(ret.logret, exc.logret, equal_var=False)
print(f"  mean ret: t={t_:.2f} p={p_:.3f}"
      f"   KS p={stats.ks_2samp(ret.logret, exc.logret).pvalue:.3f}"
      f"   Levene p={stats.levene(ret.logret, exc.logret).pvalue:.3f}")
print(f"  variance: retained {ret.logret.var():.3f}  excluded {exc.logret.var():.3f}")
print(f"  skew:     retained {ret.logret.skew():.3f}  excluded {exc.logret.skew():.3f}")

print("\nRETAINED DAYS BY MONTH (S6.5)")
print(D.month.value_counts().sort_index().to_string())

D.to_csv("panel_cds.csv", index=False)
!cp panel_cds.csv almaty_pm25_daily_window.csv /content/drive/MyDrive/pollution_research/
print("\nSaved panel_window_final.csv and almaty_pm25_daily_window.csv.")
print("Paste the FULL output back into chat.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 10.8 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STEP 1 — TREATMENT CONSTRUCTION FROM HOURLY DATA
hourly obs: 37632   zeros: 89   negatives: 0   min 0.0   max 421.0
days with any in-window obs: 1668   (expect ~1668)
days passing >= 8 in window: 1548   (expect ~1548)
winter pass rate: 0.914
session mean 36.1   window mean 31.8   calendar mean 29.5

STEP 2 — PANEL
Official KASE: 1480 rows, 2020-01-05 to 2026-01-09
2022-03-09 logret: 4.0479   (target ~4.05)
Weekend-session rows dropped from panel (kept in price chain): 6

PREFERRED SPEC — window treatment, weekend rule, >= 8 in window
  N                    1042
  IV estimate       -0.0826
  SE                 0.0832
  p                   0.321
  95% CI         [-0.2457, 0.0804]
  First-stage F       336.

In [21]:
"""
Exact randomisation inference on the rebuilt panel.

Enumerates ALL N-1 distinct circular shifts of the return series rather than
sampling. This is what the paper reports (Section 4.7, "This is an exact
randomisation null rather than a sampled one"), and it is what the Table 7
percentiles are computed against.

Differences from the sampled version in the rebuild script, all of which
matter for the SE-calibration claim:
  - every shift, not 1,000 draws at multiples of five
  - HAC covariance to match the reported SE, not unadjusted
  - percentiles for the contemporaneous estimate and every placebo lead

Run AFTER the rebuild cell, in the same session, so D, keep, INST, b, se and
run_iv are already defined.

Takes a few minutes at N = 1042.
"""

import numpy as np
import pandas as pd
from linearmodels.iv import IV2SLS
from scipy import stats

n = len(D)
print(f"Enumerating all {n - 1} circular shifts at N = {n} ...\n")

exog  = D[["const"] + keep]
endog = D["log_pm25"]
instr = D[INST]
y     = D.logret.values

draws = np.empty(n - 1)
for i, s in enumerate(range(1, n)):
    yb = pd.Series(np.roll(y, s), index=D.index)
    draws[i] = IV2SLS(yb, exog, endog, instr).fit(
        cov_type="kernel").params["log_pm25"]
    if (i + 1) % 200 == 0:
        print(f"  {i + 1} / {n - 1}")

lo, hi = np.percentile(draws, [2.5, 97.5])
ratio = draws.std() / se

print("\n" + "=" * 62)
print("EXACT RANDOMISATION NULL")
print("=" * 62)
print(f"  shifts enumerated   {len(draws)}")
print(f"  null mean           {draws.mean():+.4f}   (should sit on zero)")
print(f"  null SD             {draws.std():.4f}")
print(f"  reported HAC SE     {se:.4f}")
print(f"  RATIO               {ratio:.2f}   <- paper claims 1.00")
print(f"  95% null range      [{lo:.3f}, {hi:.3f}]")
print(f"  RI p (two-sided)    {np.mean(np.abs(draws) >= abs(b)):.3f}"
      f"   vs analytical {M0['p']:.3f}")
print(f"  percentile of estimate  {stats.percentileofscore(draws, b):.1f}")

# ---------------------------------------------------------------------------
# Table 7 percentiles: locate every placebo inside this null
# ---------------------------------------------------------------------------

print("\n" + "=" * 62)
print("TABLE 7 — placebo percentiles against the exact null")
print("=" * 62)

rows = [("Contemporaneous (main estimate)", b, se, M0["p"])]

for k in (1, 2, 3, 5):
    Dk = D.copy()
    Dk["lead"] = Dk.logret.shift(-k)
    Dk = Dk.dropna(subset=["lead"]).reset_index(drop=True)
    r = run_iv(Dk, dep="lead")
    rows.append((f"{k}-day lead return", r["b"], r["se"], r["p"]))

D1 = D.copy()
D1["lead"] = D1.logret.shift(-1)
D1["lag_pm"] = D1.log_pm25.shift(1)
D1 = D1.dropna(subset=["lead", "lag_pm"]).reset_index(drop=True)
keep1 = build_ctrl(D1) + ["lag_pm"]
r1 = IV2SLS(D1["lead"], D1[["const"] + keep1], D1["log_pm25"],
            D1[INST]).fit(cov_type="kernel")
rows.insert(2, ("1-day lead, conditioning on lagged PM2.5",
                r1.params["log_pm25"], r1.std_errors["log_pm25"],
                r1.pvalues["log_pm25"]))

print(f"  {'Specification':<42}{'Coef.':>9}{'SE':>8}{'p':>7}{'pctile':>8}")
for label, coef, serr, pv in rows:
    print(f"  {label:<42}{coef:>9.4f}{serr:>8.4f}{pv:>7.3f}"
          f"{stats.percentileofscore(draws, coef):>8.1f}")

np.save("ri_draws_cds.npy", draws)
print(f"\nSaved the null to ri_draws_cds.npy")
print("""
HOW TO READ THE RATIO
  Near 1.00 means the HAC standard errors are well calibrated and the
  equivalence bounds can be trusted as reported -- the paper's current claim.
  Materially below 1.00 means the HAC SE is wider than the true sampling
  variation, so the bounds are conservative: the paper would still be honest,
  but Section 4.7's calibration sentence needs rewriting and the bound could
  arguably be tightened.
  Materially above 1.00 would be the serious case -- SEs too small, bounds
  overstated -- and would need addressing before anything else.
""")

Enumerating all 1041 circular shifts at N = 1042 ...

  200 / 1041
  400 / 1041
  600 / 1041
  800 / 1041
  1000 / 1041

EXACT RANDOMISATION NULL
  shifts enumerated   1041
  null mean           +0.0001   (should sit on zero)
  null SD             0.0775
  reported HAC SE     0.0832
  RATIO               0.93   <- paper claims 1.00
  95% null range      [-0.169, 0.154]
  RI p (two-sided)    0.267   vs analytical 0.321
  percentile of estimate  12.8

TABLE 7 — placebo percentiles against the exact null
  Specification                                 Coef.      SE      p  pctile
  Contemporaneous (main estimate)             -0.0826  0.0832  0.321    12.8
  1-day lead return                           -0.0063  0.0724  0.931    46.9
  1-day lead, conditioning on lagged PM2.5    -0.0191  0.0814  0.814    39.7
  2-day lead return                            0.0084  0.0729  0.908    55.0
  3-day lead return                           -0.0186  0.0698  0.790    40.0
  5-day lead return            

In [22]:
"""
Remaining specifications on the rebuilt 1,042-day panel.

Run AFTER the rebuild cell in the same session: uses D, keep, INST, b, se,
M0, run_iv and build_ctrl.

Covers everything the 6 August rebuild script predates:
  1. Sargan statistic and Wu-Hausman F values (not just p-values)
  2. Temperature in 2.5 degree bins
  3. Cubic spline in temperature
  4. Year-by-month fixed effects
  5. Every instrument just-identified, including pressure alone
  6. Anderson-Rubin confidence set for v alone
  7. Threshold indicators, OLS and instrumented

NOT covered: the uranium and Brent commodity-channel tests, which need
URA and Brent price files. Those were not in the Drive listing, so they
have to be re-downloaded before Section 4.5 can be re-run.
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
from patsy import dmatrix

LN2 = np.log(2)

def bound_of(coef, serr):
    return (abs(coef) + 1.645 * serr) * LN2

# ========================================================== 1. TEST STATISTICS
print("=" * 70)
print("1. SPECIFICATION TEST STATISTICS (preferred spec)")
print("=" * 70)
r = M0["res"]
print(f"  Sargan statistic   {r.sargan.stat:.3f}   p = {r.sargan.pval:.4f}")
wh = r.wu_hausman()
print(f"  Wu-Hausman F       {wh.stat:.3f}   p = {wh.pval:.4f}")

# Robust (control-function) exogeneity test: regress logret on log_pm25,
# controls and the first-stage residual; the residual's t-test is the
# heteroskedasticity-robust analogue of Wu-Hausman.
fs_r = smf.ols("log_pm25 ~ " + " + ".join(INST + keep), data=D).fit()
Dcf = D.copy()
Dcf["v_hat"] = fs_r.resid
cf = smf.ols("logret ~ log_pm25 + v_hat + " + " + ".join(keep), data=Dcf
             ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"  Robust exogeneity (control function) p = {cf.pvalues['v_hat']:.4f}")

# ========================================================== 2. TEMPERATURE BINS
print("\n" + "=" * 70)
print("2-4. HOW WEATHER AND SEASONALITY ENTER")
print("=" * 70)

Db = D.copy()
edges = np.arange(np.floor(Db.temp.min() / 2.5) * 2.5,
                  np.ceil(Db.temp.max() / 2.5) * 2.5 + 2.5, 2.5)
Db["tbin"] = pd.cut(Db.temp, bins=edges, include_lowest=True)
tb = pd.get_dummies(Db.tbin, prefix="tb", drop_first=True).astype(int)
tb.columns = [f"tb_{i}" for i in range(tb.shape[1])]
Db = pd.concat([Db, tb], axis=1)
res_bins = run_iv(Db, weather=["precip", "wind_speed", "cloud", "radiation"]
                           + list(tb.columns))
print(f"  Temperature in 2.5 deg bins ({tb.shape[1] + 1} bins)")
print(f"    {res_bins['b']:.4f} (SE {res_bins['se']:.4f}, p {res_bins['p']:.3f}), "
      f"N {res_bins['N']}, F {res_bins['F']/4:.1f}, bound {res_bins['bound']:.3f}")

# ========================================================== 3. CUBIC SPLINE
Ds = D.copy()
basis = dmatrix("bs(temp, df=4, degree=3) - 1", Ds, return_type="dataframe")
basis.columns = [f"sp_{i}" for i in range(basis.shape[1])]
Ds = pd.concat([Ds, basis], axis=1)
res_spl = run_iv(Ds, weather=["precip", "wind_speed", "cloud", "radiation"]
                          + list(basis.columns))
print(f"\n  Cubic spline in temperature (df = 4)")
print(f"    {res_spl['b']:.4f} (SE {res_spl['se']:.4f}, p {res_spl['p']:.3f}), "
      f"N {res_spl['N']}, F {res_spl['F']/4:.1f}, bound {res_spl['bound']:.3f}")

# ========================================================== 4. YEAR-BY-MONTH FE
Dym = D.copy()
Dym["ym"] = Dym.date.dt.to_period("M").astype(str)
ym = pd.get_dummies(Dym.ym, prefix="ym", drop_first=True).astype(int)
Dym = pd.concat([Dym, ym], axis=1)
res_ym = run_iv(Dym, weather=["temp", "precip", "wind_speed", "cloud",
                              "radiation"] + list(ym.columns),
                month_fe=False, year_fe=False)
print(f"\n  Year-by-month fixed effects ({ym.shape[1]} dummies)")
print(f"    {res_ym['b']:.4f} (SE {res_ym['se']:.4f}, p {res_ym['p']:.3f}), "
      f"N {res_ym['N']}, F {res_ym['F']/4:.1f}, bound {res_ym['bound']:.3f}")

# ========================================================== 5. JUST-IDENTIFIED
print("\n" + "=" * 70)
print("5. EACH INSTRUMENT USED ALONE")
print("=" * 70)
print(f"  {'Instrument':<14}{'Coef.':>10}{'SE':>9}{'F':>9}{'bound':>9}")
solo = {}
for inst in INST:
    try:
        rr = run_iv(D, inst=[inst])
        solo[inst] = rr
        print(f"  {inst:<14}{rr['b']:>10.4f}{rr['se']:>9.4f}"
              f"{rr['F']:>9.1f}{rr['bound']:>9.3f}")
    except Exception as e:
        print(f"  {inst:<14} failed: {e}")
print("\n  Note: with one instrument the F column is already an F, not a chi2,")
print("  so it needs no division by four. The four-instrument rows do.")

# ========================================================== 6. AR SET FOR v
print("\n" + "=" * 70)
print("6. ANDERSON-RUBIN CONFIDENCE SET, v USED ALONE")
print("=" * 70)
grid = np.arange(-5.0, 5.001, 0.01)
inside = []
X_ar = sm.add_constant(D[keep + ["v"]].astype(float))
for beta in grid:
    yy = D.logret - beta * D.log_pm25
    fit = sm.OLS(yy, X_ar).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    if fit.pvalues["v"] > 0.05:
        inside.append(beta)

if not inside:
    print("  AR set is EMPTY on [-5, 5] -- the instrument rejects every value")
else:
    inside = np.array(inside)
    breaks = np.where(np.diff(inside) > 0.015)[0]
    segs, start = [], 0
    for br in breaks:
        segs.append((inside[start], inside[br]))
        start = br + 1
    segs.append((inside[start], inside[-1]))
    print(f"  AR set (95%), searched on [-5, 5] at 0.01 spacing:")
    for lo_, hi_ in segs:
        print(f"    [{lo_:.2f}, {hi_:.2f}]")
    print(f"  contains zero: {any(l <= 0 <= h for l, h in segs)}")
    print(f"  contains the preferred estimate {b:.3f}: "
          f"{any(l <= b <= h for l, h in segs)}")
    if len(segs) > 1 or inside[0] <= -4.99 or inside[-1] >= 4.99:
        print("  Set is unbounded or disjoint on this grid -- the shape Dufour")
        print("  (1997) predicts when an instrument carries almost no")
        print("  identifying variation.")

# ========================================================== 7. THRESHOLDS
print("\n" + "=" * 70)
print("7. THRESHOLD INDICATORS")
print("=" * 70)
print(f"  {'Cut-off':<12}{'OLS':>10}{'(SE)':>10}{'IV':>11}{'(SE)':>10}{'F':>8}{'days':>7}")
for thr in (35, 55, 75):
    Dt = D.copy()
    Dt["hi"] = (Dt.pm25_raw > thr).astype(int)
    o = smf.ols("logret ~ hi + " + " + ".join(keep), data=Dt
                ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    iv = run_iv(Dt, endog="hi")
    print(f"  >{thr:<11}{o.params['hi']:>10.4f}{o.bse['hi']:>10.4f}"
          f"{iv['b']:>11.4f}{iv['se']:>10.4f}{iv['F']/4:>8.1f}{int(Dt.hi.sum()):>7}")

print("""
All four-instrument F values above are printed already divided by four,
so they are robust F statistics comparable to the Stock-Yogo critical
values (24.58 with four instruments, 22.30 with three, 19.93 with two,
16.38 with one). The just-identified block in section 5 is the exception
and is noted there.
""")

1. SPECIFICATION TEST STATISTICS (preferred spec)
  Sargan statistic   3.820   p = 0.2816
  Wu-Hausman F       2.853   p = 0.0915
  Robust exogeneity (control function) p = 0.2421

2-4. HOW WEATHER AND SEASONALITY ENTER
  Temperature in 2.5 deg bins (22 bins)
    -0.1101 (SE 0.0806, p 0.172), N 1042, F 85.3, bound 0.168

  Cubic spline in temperature (df = 4)
    -0.0761 (SE 0.0820, p 0.353), N 1042, F 90.0, bound 0.146

  Year-by-month fixed effects (63 dummies)
    -0.1035 (SE 0.0918, p 0.260), N 1042, F 100.8, bound 0.176

5. EACH INSTRUMENT USED ALONE
  Instrument         Coef.       SE        F    bound
  log_blh          -0.0898   0.0903    270.3    0.165
  u                -0.3975   0.4289      7.6    0.765
  v                 2.4750   3.2204      0.7    5.387
  pressure         -0.0984   0.1424     60.6    0.231

  Note: with one instrument the F column is already an F, not a chi2,
  so it needs no division by four. The four-instrument rows do.

6. ANDERSON-RUBIN CONFIDENCE SET

In [24]:
BRENT_FILE = "Brent Oil Futures Historical Data.csv"

In [25]:
"""
Commodity-channel tests (Section 4.5) on the rebuilt 1,042-day panel.

Run AFTER the rebuild cell in the same session: uses D, keep, INST.

URANIUM: pulled live from Yahoo via yfinance, so nothing needs downloading.
This is a provenance change -- the paper currently says both series come
from Investing.com -- and Section 4.5 must say so. Returns are computed from
split- and distribution-adjusted closes, which is safer than raw closes for
an ETF over six years.

BRENT: set BRENT_FILE to whatever you have. The loader sniffs the date and
price columns and prints what it found, so check that line before trusting
the numbers. If your file is an Investing.com export, the price column is
usually "Price" and dates are dd/mm/yyyy.

FOR EACH SERIES THE TEST HAS THREE PARTS
  (a) do the instruments predict the commodity return?      [joint + individual]
  (b) does the commodity return predict KASE returns?
  (c) does conditioning the reduced form on it change anything?
Only if (a) and (b) both show something is the channel live.
"""

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

BRENT_FILE = "brent.csv"      # <-- set to your filename
BRENT_DATE_COL = None         # leave None to auto-detect
BRENT_PRICE_COL = None        # leave None to auto-detect

# ---------------------------------------------------------------- URANIUM
print("=" * 70)
print("URANIUM (Global X Uranium ETF, URA) — via Yahoo")
print("=" * 70)

ura = None
try:
    import yfinance as yf
except ImportError:
    get_ipython().system("pip install yfinance -q")
    import yfinance as yf

try:
    raw = yf.download("URA", start="2020-01-01", end="2026-01-15",
                      auto_adjust=True, progress=False)
    if len(raw):
        px = raw["Close"]
        if isinstance(px, pd.DataFrame):
            px = px.iloc[:, 0]
        ura = pd.DataFrame({"date": pd.to_datetime(px.index).normalize(),
                            "ura": 100 * np.log(px).diff().values})
        ura = ura.dropna()
        print(f"  {len(ura)} daily returns, "
              f"{ura.date.min().date()} to {ura.date.max().date()}")
        print(f"  mean {ura.ura.mean():.4f}  sd {ura.ura.std():.3f}")
    else:
        print("  Yahoo returned nothing.")
except Exception as e:
    print(f"  yfinance failed: {e}")
    print("  Fall back to an Investing.com URA export and load it like Brent.")

# ---------------------------------------------------------------- BRENT
print("\n" + "=" * 70)
print(f"BRENT — from {BRENT_FILE}")
print("=" * 70)

brent = None
try:
    braw = pd.read_csv(BRENT_FILE)
    print(f"  columns found: {braw.columns.tolist()}")

    dcol = BRENT_DATE_COL or next(
        (c for c in braw.columns if "date" in c.lower()), braw.columns[0])
    pcol = BRENT_PRICE_COL or next(
        (c for c in braw.columns
         if c.lower() in ("price", "close", "close/last", "adj close", "last")),
        None)
    if pcol is None:
        num = [c for c in braw.columns
               if c != dcol and pd.to_numeric(
                   braw[c].astype(str).str.replace(",", ""),
                   errors="coerce").notna().mean() > 0.9]
        pcol = num[0] if num else None
    print(f"  using date column '{dcol}', price column '{pcol}'")

    b_ = braw[[dcol, pcol]].copy()
    b_.columns = ["date", "px"]
    b_["date"] = pd.to_datetime(b_["date"], dayfirst=True, errors="coerce")
    b_["px"] = pd.to_numeric(b_["px"].astype(str).str.replace(",", ""),
                             errors="coerce")
    b_ = b_.dropna().sort_values("date")
    b_ = b_[b_.date.dt.dayofweek < 5]           # drop weekend-dated rows
    b_["brent"] = 100 * np.log(b_.px).diff()
    brent = b_[["date", "brent"]].dropna()
    print(f"  {len(brent)} daily returns, "
          f"{brent.date.min().date()} to {brent.date.max().date()}")
    print(f"  mean {brent.brent.mean():.4f}  sd {brent.brent.std():.3f}")
    if len(brent) < 1000:
        print("  WARNING: fewer rows than the sample needs. Check the file "
              "covers 2020-2025 in full.")
except Exception as e:
    print(f"  could not load: {e}")

# ---------------------------------------------------------------- THE TEST
def channel_test(series, col, label):
    if series is None:
        print(f"\n  {label}: skipped, no data")
        return
    Dc = D.merge(series, on="date", how="inner").dropna(subset=[col])
    print("\n" + "-" * 70)
    print(f"{label}  (N = {len(Dc)})")
    print("-" * 70)

    # (a) instruments -> commodity
    a = smf.ols(f"{col} ~ " + " + ".join(INST + keep), data=Dc
                ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    jp = a.f_test(", ".join(f"{i} = 0" for i in INST)).pvalue
    print(f"  (a) instruments -> {col}:  joint p = {jp:.4f}")
    for i in INST:
        print(f"        {i:<10}{a.params[i]:>9.4f} (SE {a.bse[i]:.4f}) "
              f"p = {a.pvalues[i]:.3f}")

    # (b) commodity -> KASE
    b2 = smf.ols(f"logret ~ {col} + " + " + ".join(keep), data=Dc
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  (b) {col} -> KASE return: {b2.params[col]:.4f} "
          f"(SE {b2.bse[col]:.4f}) p = {b2.pvalues[col]:.3f}")
    print(f"      correlation with KASE return: "
          f"{Dc.logret.corr(Dc[col]):.3f}")

    # (c) reduced form, with and without the commodity control
    rf0 = smf.ols("logret ~ " + " + ".join(INST + keep), data=Dc
                  ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    rf1 = smf.ols(f"logret ~ {col} + " + " + ".join(INST + keep), data=Dc
                  ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    p0 = rf0.f_test(", ".join(f"{i} = 0" for i in INST)).pvalue
    p1 = rf1.f_test(", ".join(f"{i} = 0" for i in INST)).pvalue
    print(f"  (c) reduced-form log_blh coefficient "
          f"{rf0.params['log_blh']:.4f} (SE {rf0.bse['log_blh']:.4f})"
          f"  ->  {rf1.params['log_blh']:.4f} (SE {rf1.bse['log_blh']:.4f})")
    print(f"      joint exclusion p  {p0:.3f}  ->  {p1:.3f}")

channel_test(ura, "ura", "URANIUM CHANNEL")
channel_test(brent, "brent", "BRENT / OIL CHANNEL")

print("""
READ IT THIS WAY
  A channel is live only if the instruments predict the commodity AND the
  commodity predicts KASE returns. On the old 926-day panel oil was clean
  throughout (joint p 0.698) while uranium was marginal on both links
  (joint p 0.079 driven by u at p 0.038; uranium -> KASE p 0.060), which is
  what localised Section 4.5's concern to the east-west component
  specifically. Compare against those figures. If uranium now comes back
  clean, the u-specific suspicion in Sections 4.5, 6.3 and the conclusion
  weakens the same way the v argument just did.
""")

URANIUM (Global X Uranium ETF, URA) — via Yahoo


ERROR:yfinance:Failed to get ticker 'URA' reason: Impersonating chrome146 is not supported
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['URA']: ImpersonateError('Impersonating chrome146 is not supported')


  Yahoo returned nothing.

BRENT — from brent.csv
  could not load: [Errno 2] No such file or directory: 'brent.csv'

  URANIUM CHANNEL: skipped, no data

  BRENT / OIL CHANNEL: skipped, no data

READ IT THIS WAY
  A channel is live only if the instruments predict the commodity AND the
  commodity predicts KASE returns. On the old 926-day panel oil was clean
  throughout (joint p 0.698) while uranium was marginal on both links
  (joint p 0.079 driven by u at p 0.038; uranium -> KASE p 0.060), which is
  what localised Section 4.5's concern to the east-west component
  specifically. Compare against those figures. If uranium now comes back
  clean, the u-specific suspicion in Sections 4.5, 6.3 and the conclusion
  weakens the same way the v argument just did.



In [26]:
# Fix yfinance, then find the Brent file
!pip install -q --upgrade "curl_cffi>=0.7" yfinance

import glob, os
print("files in the session:")
for f in sorted(glob.glob("*.csv") + glob.glob("*.CSV")):
    print("   ", repr(f))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 11.6 MB/s eta 0:00:00
files in the session:
    'KASE Historical Data.csv'
    'almaty_pm25_daily.csv'
    'almaty_pm25_daily_window.csv'
    'almaty_pm25_hourly.csv'
    'almaty_weather_daily.csv'
    'almaty_weather_daily_cds.csv'
    'blh_cds_both_constructions.csv'
    'blh_cds_daily_bilinear.csv'
    'blh_cds_daily_nearest.csv'
    'kase_clean.csv'
    'kase_official.csv'
    'panel_analysis.csv'
    'panel_cds.csv'
    'panel_final.csv'
    'panel_official.csv'
    'panel_window_final.csv'


In [27]:
!cp panel_cds.csv almaty_weather_daily_cds.csv ri_draws_cds.npy /content/drive/MyDrive/pollution_research/
!ls -la /content/drive/MyDrive/pollution_research/panel_cds.csv

-rw------- 1 root root 327685 Aug 18 21:02 /content/drive/MyDrive/pollution_research/panel_cds.csv


In [28]:
import yfinance as yf
t = yf.download("URA", start="2025-01-01", end="2025-02-01",
                auto_adjust=True, progress=False)
print(len(t))

ERROR:yfinance:Failed to get ticker 'URA' reason: Impersonating chrome146 is not supported
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['URA']: ImpersonateError('Impersonating chrome146 is not supported')


0


In [29]:
import glob
print([f for f in glob.glob("*.csv") if "rent" in f or "il" in f])

['almaty_pm25_daily_window.csv', 'blh_cds_daily_nearest.csv', 'almaty_pm25_daily.csv', 'blh_cds_daily_bilinear.csv', 'almaty_weather_daily.csv', 'almaty_weather_daily_cds.csv']


In [30]:
import glob, numpy as np, pandas as pd, statsmodels.formula.api as smf

DIR = "/content/drive/MyDrive/pollution_research"
D = pd.read_csv(f"{DIR}/panel_cds.csv"); D["date"] = pd.to_datetime(D["date"])
print(f"panel: N = {len(D)}   (must be 1042)")

INST = ["log_blh", "u", "v", "pressure"]
keep = [c for c in D.columns if c.startswith(("mo_","dw_","yr_")) and D[c].nunique()>1] + \
       [c for c in ["temp","precip","wind_speed","cloud","radiation","lockdown","qantar"]
        if D[c].nunique()>1]
kk=[]
for c in keep:
    M = D[["const"]+kk+[c]].astype(float).values
    if np.linalg.matrix_rank(M)==M.shape[1]: kk.append(c)
keep=kk

# find Brent anywhere
cands = [f for f in glob.glob("**/*.csv", recursive=True)
         if "rent" in f.lower() or "oil" in f.lower()]
print("Brent candidates:", cands)

# uranium via Stooq (yfinance is broken in this runtime)
u = pd.read_csv("https://stooq.com/q/d/l/?s=ura.us&i=d")
ura = pd.DataFrame({"date": pd.to_datetime(u.Date).dt.normalize(),
                    "ura": 100*np.log(u.Close).diff()}).dropna()
print(f"URA: {len(ura)} returns, {ura.date.min().date()} to {ura.date.max().date()}")

def test(s, col, lab):
    Dc = D.merge(s, on="date").dropna(subset=[col])
    print(f"\n{lab}  N={len(Dc)}")
    a = smf.ols(f"{col} ~ "+" + ".join(INST+keep), data=Dc).fit(cov_type="HAC", cov_kwds={"maxlags":5})
    print(f"  (a) joint p = {float(a.f_test(', '.join(f'{i}=0' for i in INST)).pvalue):.4f}")
    for i in INST: print(f"      {i:<10}{a.params[i]:8.4f} p={a.pvalues[i]:.3f}")
    b = smf.ols(f"logret ~ {col} + "+" + ".join(keep), data=Dc).fit(cov_type="HAC", cov_kwds={"maxlags":5})
    print(f"  (b) {col} -> KASE: {b.params[col]:.4f} (SE {b.bse[col]:.4f}) p={b.pvalues[col]:.3f}")

test(ura, "ura", "URANIUM")
if cands:
    br = pd.read_csv(cands[0])
    print("\nBrent columns:", br.columns.tolist())

panel: N = 1042   (must be 1042)
Brent candidates: []


HTTPError: HTTP Error 404: Not Found

In [31]:
import requests, io
r = requests.get("https://stooq.com/q/d/l/?s=ura.us&i=d",
                 headers={"User-Agent": "Mozilla/5.0"})
print(r.status_code, r.text[:120])

200 <!DOCTYPE html><html><head><meta charset="utf-8"><meta name="robots" content="noindex,nofollow"></head><body><noscript>T


In [32]:
import glob, numpy as np, pandas as pd, statsmodels.formula.api as smf

DIR = "/content/drive/MyDrive/pollution_research"
D = pd.read_csv(f"{DIR}/panel_cds.csv"); D["date"] = pd.to_datetime(D["date"])
print(f"panel N = {len(D)}   (must be 1042)")

INST = ["log_blh", "u", "v", "pressure"]
keep = [c for c in D.columns if c.startswith(("mo_","dw_","yr_")) and D[c].nunique()>1] + \
       [c for c in ["temp","precip","wind_speed","cloud","radiation","lockdown","qantar"]
        if D[c].nunique()>1]
kk=[]
for c in keep:
    M = D[["const"]+kk+[c]].astype(float).values
    if np.linalg.matrix_rank(M)==M.shape[1]: kk.append(c)
keep=kk

def load(path, name):
    df = pd.read_csv(path)
    print(f"\n{name} columns: {df.columns.tolist()}")
    dcol = next(c for c in df.columns if "date" in c.lower())
    pcol = next(c for c in df.columns
                if c.lower() in ("price","close","close/last","adj close","last"))
    df = df[[dcol, pcol]].copy(); df.columns = ["date","px"]
    df["date"] = pd.to_datetime(df["date"], errors="coerce", format="mixed")
    df["px"] = pd.to_numeric(df["px"].astype(str).str.replace(",",""), errors="coerce")
    df = df.dropna().sort_values("date")
    df = df[df.date.dt.dayofweek < 5]
    df[name] = 100*np.log(df.px).diff()
    print(f"  {len(df)} weekday rows, {df.date.min().date()} to {df.date.max().date()}")
    return df[["date", name]].dropna()

def test(s, col, lab):
    Dc = D.merge(s, on="date").dropna(subset=[col])
    print(f"\n{'='*60}\n{lab}   N = {len(Dc)}\n{'='*60}")
    a = smf.ols(f"{col} ~ "+" + ".join(INST+keep), data=Dc).fit(cov_type="HAC", cov_kwds={"maxlags":5})
    print(f"(a) instruments -> {col}: joint p = "
          f"{float(a.f_test(', '.join(f'{i}=0' for i in INST)).pvalue):.4f}")
    for i in INST:
        print(f"      {i:<10}{a.params[i]:9.4f} (SE {a.bse[i]:.4f})  p={a.pvalues[i]:.3f}")
    b = smf.ols(f"logret ~ {col} + "+" + ".join(keep), data=Dc).fit(cov_type="HAC", cov_kwds={"maxlags":5})
    print(f"(b) {col} -> KASE: {b.params[col]:.4f} (SE {b.bse[col]:.4f}) p={b.pvalues[col]:.3f}"
          f"   corr {Dc.logret.corr(Dc[col]):.3f}")
    r0 = smf.ols("logret ~ "+" + ".join(INST+keep), data=Dc).fit(cov_type="HAC", cov_kwds={"maxlags":5})
    r1 = smf.ols(f"logret ~ {col} + "+" + ".join(INST+keep), data=Dc).fit(cov_type="HAC", cov_kwds={"maxlags":5})
    print(f"(c) BLH reduced form {r0.params['log_blh']:.4f} -> {r1.params['log_blh']:.4f}"
          f"   joint p {float(r0.f_test(', '.join(f'{i}=0' for i in INST)).pvalue):.3f}"
          f" -> {float(r1.f_test(', '.join(f'{i}=0' for i in INST)).pvalue):.3f}")

test(load(f"{DIR}/ura.csv", "ura"), "ura", "URANIUM CHANNEL")
test(load(f"{DIR}/brent.csv", "brent"), "brent", "BRENT / OIL CHANNEL")

panel N = 1042   (must be 1042)

ura columns: ['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']
  1655 weekday rows, 2020-01-21 to 2026-08-18

URANIUM CHANNEL   N = 1007
(a) instruments -> ura: joint p = 0.0823
      log_blh      0.0548 (SE 0.2202)  p=0.804
      u            0.0941 (SE 0.0431)  p=0.029
      v           -0.0592 (SE 0.0381)  p=0.121
      pressure     0.0280 (SE 0.0234)  p=0.231
(b) ura -> KASE: 0.0173 (SE 0.0096) p=0.071   corr 0.068
(c) BLH reduced form 0.0611 -> 0.0602   joint p 0.184 -> 0.196

brent columns: ['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']
  1704 weekday rows, 2020-01-02 to 2026-08-06

BRENT / OIL CHANNEL   N = 1033
(a) instruments -> brent: joint p = 0.7336
      log_blh     -0.0991 (SE 0.1780)  p=0.578
      u            0.0344 (SE 0.0461)  p=0.456
      v            0.0321 (SE 0.0420)  p=0.444
      pressure     0.0092 (SE 0.0222)  p=0.679
(b) brent -> KASE: 0.0065 (SE 0.0141) p=0.644   corr 0.037
(c) BLH reduced form 0.05

In [33]:
"""
REGENERATE ALL FIVE FIGURES on the rebuilt 1,042-day CDS panel.

Reads panel_cds.csv (NOT panel_window_final.csv, which is the old 926-day
panel) and almaty_pm25_daily_window.csv, both from Drive. Writes the five
PNGs straight back to Drive so they survive the runtime, under the exact
filenames main.tex expects.

Two fixes carried over from the last round:
  - controls are rebuilt inside each specification, so the subsample rows
    in Figure 5 (from 2022, from 2023, winter only) no longer drop out on a
    rank failure from constant year dummies
  - elinewidth, not elw

Prints every caption and %VERIFY value at the end: the Figure 1 filtered
mean, the Figure 3 residualised slope, the Figure 4 slope and zero-containing
interval count, the sample endpoints, and the monthly BLH and PM2.5 means on
the estimation sample with the seasonal amplitude.
"""

import numpy as np, pandas as pd, warnings, traceback
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

DIR = "/content/drive/MyDrive/pollution_research"

plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
})

D = pd.read_csv(f"{DIR}/panel_cds.csv"); D["date"] = pd.to_datetime(D["date"])
W = pd.read_csv(f"{DIR}/almaty_pm25_daily_window.csv"); W["date"] = pd.to_datetime(W["date"])
print(f"panel N = {len(D)}   (must be 1042)\n")

INST = ["log_blh", "u", "v", "pressure"]
MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
out = {}

def ctrl_for(df, drop_weather=False):
    """Non-collinear control list, rebuilt for whatever subsample is passed."""
    base = [c for c in df.columns
            if c.startswith(("mo_", "dw_", "yr_")) and df[c].nunique() > 1]
    base += [c for c in ["temp", "precip", "wind_speed", "cloud", "radiation",
                         "lockdown", "qantar"]
             if c in df.columns and df[c].nunique() > 1
             and not (drop_weather and c in ("cloud", "radiation"))]
    k = []
    for c in base:
        M = df[["const"] + k + [c]].astype(float).values
        if np.linalg.matrix_rank(M) == M.shape[1]:
            k.append(c)
    return k

CTRL = " + ".join(ctrl_for(D))

# ============================================================ FIG 1
try:
    F = W[W.n_window >= 8].copy()
    F = F[(F.date >= "2020-04-08") & (F.date <= "2025-11-14")].sort_values("date")
    out["fig1_mean"] = float(F.pm25.mean())

    span = pd.DataFrame({"date": pd.date_range(F.date.min(), F.date.max(), freq="D")})
    P = span.merge(F[["date", "pm25"]], on="date", how="left")   # gaps stay NaN

    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.plot(P.date, P.pm25, lw=0.6, color="#2b4c7e")
    ax.axhline(5, ls="--", lw=1, color="green", label="WHO annual guideline (5)")
    ax.axhline(out["fig1_mean"], ls="--", lw=1, color="crimson",
               label=f"Sample mean ({out['fig1_mean']:.1f})")
    ax.set_ylabel("PM$_{2.5}$ ($\\mu$g/m$^3$)")
    ax.legend(frameon=False, fontsize=8, loc="upper right")
    fig.savefig(f"{DIR}/fig1_timeseries.png"); plt.close(fig)
    print("fig1 saved")
except Exception:
    traceback.print_exc()

# ============================================================ FIG 2
try:
    m = D.groupby(D.date.dt.month).agg(pm=("pm25_raw", "mean"), blh=("blh", "mean"))
    out["monthly"] = m
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    axes[0].bar(range(1, 13), m.pm.reindex(range(1, 13)).values, color="#2b4c7e")
    axes[0].set_xticks(range(1, 13)); axes[0].set_xticklabels(MONTHS, fontsize=7)
    axes[0].set_ylabel("PM$_{2.5}$ ($\\mu$g/m$^3$)")
    axes[0].set_title("Pollution", fontsize=9)
    axes[1].bar(range(1, 13), m.blh.reindex(range(1, 13)).values, color="#7e5a2b")
    axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(MONTHS, fontsize=7)
    axes[1].set_ylabel("Boundary layer height (m)")
    axes[1].set_title("Mixing depth", fontsize=9)
    fig.tight_layout(); fig.savefig(f"{DIR}/fig2_seasonality.png"); plt.close(fig)
    print("fig2 saved")
except Exception:
    traceback.print_exc()

# ============================================================ FIG 3
try:
    rx = np.asarray(smf.ols("log_blh ~ " + CTRL, data=D).fit().resid, dtype=float)
    ry = np.asarray(smf.ols("log_pm25 ~ " + CTRL, data=D).fit().resid, dtype=float)
    slope, intercept = np.polyfit(rx, ry, 1)
    out["fig3_slope"] = float(slope)

    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    ax.scatter(rx, ry, s=5, alpha=0.35, color="#2b4c7e", edgecolors="none")
    xs = np.linspace(rx.min(), rx.max(), 100)
    ax.plot(xs, intercept + slope * xs, color="crimson", lw=1.6)
    ax.set_xlabel("Residualised log boundary layer height")
    ax.set_ylabel("Residualised log PM$_{2.5}$")
    ax.text(0.03, 0.05, f"slope = {slope:.3f}", transform=ax.transAxes, fontsize=8)
    fig.savefig(f"{DIR}/fig3_firststage.png"); plt.close(fig)
    print("fig3 saved")
except Exception:
    traceback.print_exc()

# ============================================================ FIG 4
try:
    bx = np.asarray(smf.ols("log_pm25 ~ " + CTRL, data=D).fit().resid, dtype=float)
    by = np.asarray(smf.ols("logret ~ " + CTRL, data=D).fit().resid, dtype=float)
    B = pd.DataFrame({"x": bx, "y": by})
    B["bin"] = pd.qcut(B.x, 20, labels=False)
    g = B.groupby("bin").agg(x=("x", "mean"), y=("y", "mean"),
                             sd=("y", "std"), n=("y", "size"))
    g["se"] = g.sd / np.sqrt(g.n)
    g["lo"], g["hi"] = g.y - 1.96 * g.se, g.y + 1.96 * g.se
    out["fig4_zero"] = int(((g.lo <= 0) & (g.hi >= 0)).sum())

    ols_slope = float(smf.ols("logret ~ log_pm25 + " + CTRL, data=D
                              ).fit(cov_type="HAC", cov_kwds={"maxlags": 5}
                                    ).params["log_pm25"])
    out["fig4_slope"] = ols_slope

    xv = g.x.to_numpy(dtype=float); yv = g.y.to_numpy(dtype=float)
    lo = yv - g.lo.to_numpy(dtype=float); hi = g.hi.to_numpy(dtype=float) - yv

    fig, ax = plt.subplots(figsize=(6.0, 4.2))
    ax.errorbar(xv, yv, yerr=[lo, hi], fmt="o", ms=4, color="#2b4c7e",
                ecolor="#9aa8bd", elinewidth=1, capsize=2, linestyle="none")
    xs = np.linspace(xv.min(), xv.max(), 100)
    ax.plot(xs, ols_slope * xs + (yv.mean() - ols_slope * xv.mean()),
            color="crimson", lw=1.6, label=f"OLS slope = {ols_slope:.3f}")
    ax.axhline(0, color="grey", lw=0.6)
    ax.set_xlabel("Residualised log PM$_{2.5}$")
    ax.set_ylabel("Residualised daily log return (%)")
    ax.legend(frameon=False, fontsize=8)
    fig.savefig(f"{DIR}/fig4_binscatter.png"); plt.close(fig)
    print("fig4 saved")
except Exception:
    traceback.print_exc()

# ============================================================ FIG 5
try:
    def iv(df, inst=INST, drop_weather=False):
        k = ctrl_for(df, drop_weather)
        r = IV2SLS(df["logret"], df[["const"] + k], df["log_pm25"], df[inst]
                   ).fit(cov_type="kernel")
        return float(r.params["log_pm25"]), float(r.std_errors["log_pm25"])

    lo_q, hi_q = D.logret.quantile([.01, .99])
    specs = [
        ("Preferred",              lambda: iv(D)),
        ("Coverage $\\geq$15h",    lambda: iv(D[D.n_daily >= 15])),
        ("Full 11h window",        lambda: iv(D[D.date.isin(W[W.n_window == 11].date)])),
        ("From 2022",              lambda: iv(D[D.year >= 2022])),
        ("From 2023",              lambda: iv(D[D.year >= 2023])),
        ("Winter only",            lambda: iv(D[D.month.isin([11, 12, 1, 2, 3])])),
        ("Winsorised 1%",          lambda: iv(D.assign(logret=D.logret.clip(lo_q, hi_q)))),
        ("Excl. $v$",              lambda: iv(D, inst=["log_blh", "u", "pressure"])),
        ("Excl. $u$",              lambda: iv(D, inst=["log_blh", "v", "pressure"])),
        ("BLH + pressure",         lambda: iv(D, inst=["log_blh", "pressure"])),
        ("BLH alone",              lambda: iv(D, inst=["log_blh"])),
        ("Excl. cloud, radiation", lambda: iv(D, drop_weather=True)),
        ("Session-only treatment", lambda: iv(D.assign(
            log_pm25=np.log(D.pm_session.where(D.pm_session > 0))
        ).dropna(subset=["log_pm25"]))),
        ("Calendar-day treatment", lambda: iv(D.assign(log_pm25=np.log(D.pm25_calendar)))),
    ]
    labels, bs, ses = [], [], []
    for lab, fn in specs:
        try:
            b_, s_ = fn(); labels.append(lab); bs.append(b_); ses.append(s_)
            print(f"  {lab:<24}{b_:8.4f} ({s_:.4f})")
        except Exception as e:
            print("  SKIPPED:", lab, e)
    bs, ses = np.array(bs, dtype=float), np.array(ses, dtype=float)
    ypos = np.arange(len(labels))[::-1].astype(float)

    fig, ax = plt.subplots(figsize=(6.4, 5.0))
    ax.errorbar(bs, ypos, xerr=1.96 * ses, fmt="none",
                ecolor="#9aa8bd", elinewidth=1, capsize=2)
    for x_, y_, l_ in zip(bs, ypos, labels):
        ax.plot(x_, y_, "o", ms=5,
                color="crimson" if l_ == "Preferred" else "#2b4c7e")
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_yticks(ypos); ax.set_yticklabels(labels, fontsize=7.5)
    ax.set_xlabel("Coefficient on log PM$_{2.5}$")
    fig.savefig(f"{DIR}/fig5_specification.png"); plt.close(fig)
    print(f"fig5 saved with {len(labels)} of {len(specs)} specifications")
except Exception:
    traceback.print_exc()

# ============================================================ CAPTION VALUES
print("\n" + "=" * 64)
print("CAPTION AND PROSE VALUES — the paper's current figures in brackets")
print("=" * 64)
if "fig1_mean" in out:
    print(f"[Fig 1] filtered-set mean: {out['fig1_mean']:.1f}          (was 31.9)")
if "fig3_slope" in out:
    print(f"[Fig 3] residualised slope: {out['fig3_slope']:.3f}        (was -0.501)")
if "fig4_slope" in out:
    print(f"[Fig 4] OLS slope {out['fig4_slope']:.4f}, "
          f"{out['fig4_zero']} of 20 intervals contain zero   (was 0.0126, 19 of 20)")
print(f"[3.4]  endpoints: {D.date.min().date()} to {D.date.max().date()}"
      f"     (was 2020-04-10 to 2025-11-13)")
if "monthly" in out:
    m = out["monthly"]
    print("\n[1.1 / 3.5 / 4.3 / Fig 2] monthly means on the estimation sample:")
    print("      BLH now   (was)     PM2.5 now   (was)     n")
    old_blh = {1:61.4, 2:65.8, 3:128.7, 4:238.9, 5:367.0, 6:547.0,
               7:560.4, 8:509.0, 9:362.0, 10:195.0, 11:99.3, 12:53.1}
    old_pm = {1:77.61, 2:72.79, 3:28.99, 4:18.49, 5:12.91, 6:12.57,
              7:11.15, 8:10.68, 9:16.17, 10:23.78, 11:43.53, 12:73.46}
    for mo in range(1, 13):
        print(f"  {MONTHS[mo-1]}  {m.blh.get(mo, np.nan):7.1f}  ({old_blh[mo]:6.1f})   "
              f"{m.pm.get(mo, np.nan):7.2f}  ({old_pm[mo]:6.2f})   "
              f"{int((D.month == mo).sum()):4d}")
    print(f"\n  minimum {MONTHS[int(m.blh.idxmin())-1]} {m.blh.min():.1f}   "
          f"maximum {MONTHS[int(m.blh.idxmax())-1]} {m.blh.max():.1f}   "
          f"amplitude {m.blh.max()/m.blh.min():.1f}x     (was Dec 53.1 / Jul 560.4 / 10.5x)")
print(f"\n  BLH range on the estimation sample: "
      f"{D.blh.min():.1f} to {D.blh.max():.1f}     (was 19.38 to 1303.12)")
print("\nFigures written to Drive. Download the five PNGs and upload to Overleaf.")

panel N = 1042   (must be 1042)

fig1 saved
fig2 saved
fig3 saved
fig4 saved
  Preferred                -0.0826 (0.0832)
  Coverage $\geq$15h       -0.0855 (0.0830)
  Full 11h window          -0.0516 (0.0923)
  From 2022                -0.1342 (0.0966)
  From 2023                -0.1230 (0.1027)
  Winter only              -0.0837 (0.0976)
  Winsorised 1%            -0.0780 (0.0750)
  Excl. $v$                -0.1049 (0.0825)
  Excl. $u$                -0.0765 (0.0875)
  BLH + pressure           -0.0917 (0.0869)
  BLH alone                -0.0898 (0.0903)
  Excl. cloud, radiation   -0.0392 (0.0842)
  Session-only treatment   -0.0708 (0.0771)
  Calendar-day treatment   -0.0791 (0.1027)
fig5 saved with 14 of 14 specifications

CAPTION AND PROSE VALUES — the paper's current figures in brackets
[Fig 1] filtered-set mean: 31.9          (was 31.9)
[Fig 3] residualised slope: -0.615        (was -0.501)
[Fig 4] OLS slope 0.0088, 20 of 20 intervals contain zero   (was 0.0126, 19 of 20)
[3.4]  en

In [34]:
import numpy as np, pandas as pd
D = pd.read_csv("/content/drive/MyDrive/pollution_research/panel_cds.csv")
D["date"] = pd.to_datetime(D["date"])
b, se, sdret, sdlog = -0.0826, 0.0832, D.logret.std(), np.log(D.pm25_raw).std()
print(f"days > 75.86        : {(D.pm25_raw > 75.86).sum()}   (was 96)")
print(f"rho1 log PM2.5      : {np.log(D.pm25_raw).autocorr(1):.3f}   (was 0.72)")
print(f"bound in return SDs : {(abs(b)+1.645*se)*np.log(2)/sdret:.3f}   (was 0.268)")
print(f"Table 8  IV  SE     : {se*sdlog/sdret:.3f}   ratio {se*sdlog/sdret/0.030:.2f}   (was 0.119 / 3.97)")
print(f"Table 8  OLS SE     : {0.0351*sdlog/sdret:.3f}   ratio {0.0351*sdlog/sdret/0.030:.2f}   (was 0.049 / 1.63)")
print(f"sqrt(3722/{len(D)})   : {np.sqrt(3722/len(D)):.2f}   (was 2.00)")
print(f"IV/OLS inflation    : {se/0.0351:.2f}   vs 1/sqrt(partialR2) {1/np.sqrt(0.2622):.2f}   (was 2.43 / 2.16)")
need = 0.05*sdret/(1.645*np.log(2))
print(f"days for 0.05 bound : {len(D)*(se/need)**2:,.0f}   (was ~9,200)")

days > 75.86        : 110   (was 96)
rho1 log PM2.5      : 0.712   (was 0.72)
bound in return SDs : 0.235   (was 0.268)
Table 8  IV  SE     : 0.110   ratio 3.68   (was 0.119 / 3.97)
Table 8  OLS SE     : 0.047   ratio 1.55   (was 0.049 / 1.63)
sqrt(3722/1042)   : 1.89   (was 2.00)
IV/OLS inflation    : 2.37   vs 1/sqrt(partialR2) 1.95   (was 2.43 / 2.16)
days for 0.05 bound : 8,947   (was ~9,200)


In [47]:
#!/usr/bin/env python3
"""
FULL REBUILD PASS for main.tex — 926-day panel to 1,042-day CDS panel.

USAGE
    Put this beside main.tex, then:
        python3 apply_rebuild.py
    Writes main_rebuilt.tex. Original untouched.

HOW IT WORKS
    Every edit is an exact string replacement with an expected match count.
    If the count does not match, the edit is SKIPPED and reported, never
    applied blind. Whitespace and line breaks are normalised during
    matching, so LaTeX line wrapping does not defeat it.

    Read the report. Anything marked MISSING or COUNT needs doing by hand,
    and the numeral_sweep.md file tells you what it should become.

WHAT IT DOES NOT DO
    Five values were never computed on the new panel and are left alone:
    the OLS p-value in Table 4, the HAC lag sensitivities in 4.7, the two
    Sargan variants in 6.3, footnote 5's robust overidentification tests,
    and the transport footnote's 0.0171 sensitivity. Run the cell in
    numeral_sweep.md Part E, then patch those five by hand.

    It also leaves Section 5.5's fragility argument alone. That paragraph
    needs judgement, not substitution -- see numeral_sweep.md Part D1.
"""

import re
import sys
from pathlib import Path

TEX = "main.tex"
OUT = "main_rebuilt.tex"

# (label, find, replace, expected_count)
E = [

# ============================================================= ABSTRACT
("abs range",
 "compress the atmospheric mixing layer from 1,303 metres to as little as 19",
 "compress the atmospheric mixing layer from 924 metres to as little as 21", 1),
("abs N",
 "Exchange Index across 926 trading days. We find no detectable effect",
 "Exchange Index across 1,042 trading days. We find no detectable effect", 1),
("abs estimate",
 "($-0.105$ percentage points of daily return per log unit of \\pmm{}, SE $0.092$)",
 "($-0.083$ percentage points of daily return per log unit of \\pmm{}, SE $0.083$)", 1),
("abs bound",
 "effects exceeding 0.177 percentage points per doubling of concentration. The exogeneity",
 "effects exceeding 0.152 percentage points per doubling of concentration. The exogeneity", 1),

# ========================================================== SECTION 1.x
("1 intro N",
 "across 926 trading days between April 2020 and November 2025.",
 "across 1,042 trading days between April 2020 and November 2025.", 1),
("1.1 mean",
 "Mean \\pmm{} over our sample is 32.47 \\ugm{}",
 "Mean \\pmm{} over our sample is 32.29 \\ugm{}", 1),
("1.1 counts",
 "Of 926 trading days, 152 exceed 55 \\ugm{}, and 96 exceed the",
 "Of 1,042 trading days, 178 exceed 55 \\ugm{}, and 110 exceed the", 1),
("1.1 median",
 "The median is 17.9, so the mean is not representative",
 "The median is 17.8, so the mean is not representative", 1),
("1.1 blh",
 "ranges from 19 to 1,303 metres, with monthly means running from 53 metres in December to 560 in July.",
 "ranges from 21 to 924 metres, with monthly means running from 46 metres in December to 491 in July.", 1),
("1.2 CI",
 "a coefficient of $-0.105$ (SE $0.092$), with a 95 per cent confidence interval of $[-0.285, 0.074]$",
 "a coefficient of $-0.083$ (SE $0.083$), with a 95 per cent confidence interval of $[-0.246, 0.080]$", 1),
("1.2 F",
 "$F$ statistic is 44.9 with a partial $R^2$ of 0.215. Overidentifying restrictions are not rejected (Sargan $p = 0.185$)",
 "$F$ statistic is 84.0 with a partial $R^2$ of 0.262. Overidentifying restrictions are not rejected (Sargan $p = 0.282$)", 1),
("1.2 v clause",
 "that one of them correlates directly with returns while contributing almost no identifying variation, and that the other carries a marginal association with uranium returns",
 "that one of them carries almost no identifying variation on its own, and that the other carries a marginal association with uranium returns", 1),
("1.2 exogeneity",
 "the classical Wu--Hausman test marginally rejects ($p = 0.069$) while its robust analogue does not ($p = 0.172$)",
 "neither the classical Wu--Hausman test ($p = 0.092$) nor its robust analogue ($p = 0.242$) rejects", 1),
("1.2 RI",
 "Randomisation inference over all 925 circular shifts of the return series yields an exact null distribution centred on zero, with standard deviation 0.0918 against our reported standard error of 0.0915, indicating that our reported standard errors are well calibrated.",
 "Randomisation inference over all 1,041 circular shifts of the return series yields an exact null distribution centred on zero, with standard deviation 0.0775 against our reported standard error of 0.0832, indicating that our reported standard errors are, if anything, slightly conservative.", 1),
("1.2 bound",
 "any effect exceeding 0.177 percentage points per doubling of \\pmm{}}, equivalent to 0.268 standard deviations",
 "any effect exceeding 0.152 percentage points per doubling of \\pmm{}}, equivalent to 0.235 standard deviations", 1),
("1.2 factor",
 "explain 21.5 per cent of variation in log \\pmm{} conditional on controls, and 0.67 per cent of variation in returns---a factor of thirty-two.",
 "explain 26.2 per cent of variation in log \\pmm{} conditional on controls, and 0.48 per cent of variation in returns---a factor of fifty-five.", 1),
("1.3 linear",
 "this implies 0.545 percentage points per doubling (SE $0.179$). Testing equality against our estimate yields $t = 2.49$",
 "this implies 0.543 percentage points per doubling (SE $0.178$). Testing equality against our estimate yields $t = 2.60$", 1),
("1.3 prop",
 "Here $t = 1.35$: \\textbf{we cannot reject it.} Our point estimate is under forty per cent",
 "Here $t = 1.60$: \\textbf{we cannot reject it.} Our point estimate is under thirty per cent", 1),
("1.3 rank",
 "Almaty's mean of 32.47 \\ugm{} would place it eleventh",
 "Almaty's mean of 32.29 \\ugm{} would place it eleventh", 1),
("1.4 precision",
 "The benchmark study is roughly four times more precise than ours, and Section~\\ref{sec:power} decomposes the gap: our own ordinary least squares standard error is 1.63 times theirs",
 "The benchmark study is roughly three and a half times more precise than ours, and Section~\\ref{sec:power} decomposes the gap: our own ordinary least squares standard error is 1.55 times theirs", 1),

# ========================================================== SECTION 2.5
("2.5 ols",
 "Our ordinary least squares estimate is $+0.013$ (SE $0.038$), indistinguishable from zero against the instrumented $-0.105$",
 "Our ordinary least squares estimate is $+0.009$ (SE $0.035$), indistinguishable from zero against the instrumented $-0.083$", 1),
("2.5 F",
 "robust first-stage $F$ of 44.9 with a partial $R^2$ of 0.215---which permits",
 "robust first-stage $F$ of 84.0 with a partial $R^2$ of 0.262---which permits", 1),
("2.5 per10",
 "our estimate is $-0.028$ (SE $0.025$) per 10 \\ugm{}, a 95 per cent interval of roughly $[-0.077, 0.020]$. This contains their global summary estimate of $-0.012$, though our point estimate is more than twice it",
 "our estimate is $-0.022$ (SE $0.022$) per 10 \\ugm{}, a 95 per cent interval of roughly $[-0.066, 0.022]$. This contains their global summary estimate of $-0.012$, though our point estimate is roughly twice it", 1),

# =========================================== SECTION 3.2 — full rewrite
("3.2 sources",
 "Meteorological variables are drawn from the ERA5 reanalysis \\citep{hersbach2020,c3s2023}, accessed through the Open-Meteo historical weather API \\citep{openmeteo2025} and extracted at the coordinates of the Embassy monitor. Hourly series are retrieved for temperature at two metres, total precipitation, wind speed and direction at ten metres, planetary boundary layer height, surface pressure, total cloud cover and downwelling shortwave radiation, and aggregated to daily frequency over the same exposure window as the pollution data.",
 "Meteorological variables are drawn from the ERA5 reanalysis \\citep{hersbach2020}. Boundary layer height, our primary instrument, is taken directly from the Copernicus Climate Data Store \\citep{c3s2023} at the reanalysis grid point nearest the monitor, 43.25\\textdegree N 77.00\\textdegree E. Temperature at two metres, total precipitation, wind speed and direction at ten metres, surface pressure, total cloud cover and downwelling shortwave radiation are taken from the same reanalysis through the Open-Meteo historical weather API \\citep{openmeteo2025}, extracted at the coordinates of the monitor. All series are hourly and are aggregated to calendar-day means.\n\n\\textbf{Two sources, and why.} The Open-Meteo archive does not carry boundary layer height for 2 January to 30 June 2024, a contiguous 181-day gap affecting that field alone. The Climate Data Store, which is the underlying source, carries the field continuously, so we take boundary layer height from it directly and recover the gap in full. The two are not interchangeable: across the days both cover, the daily series correlate at 0.973 in logs, but the Open-Meteo series runs systematically higher in its upper tail, reaching 1,303 metres against 995 in the Climate Data Store, and a regression of one on the other has a slope of 0.85. Open-Meteo does not document the processing that produces the difference. We therefore report boundary layer height from the authoritative source throughout rather than splicing two series within one variable, and the remaining meteorological variables, which enter only as controls and for which no gap arises, from Open-Meteo.\n\n\\textbf{Why the calendar day.} Meteorological variables are aggregated over the calendar day, not over the 07:00--17:59 exposure window used for the pollution treatment. The asymmetry is deliberate. The treatment measures exposure, which accumulates only while the trader is awake and trading; the instruments measure the atmosphere's capacity to disperse emissions, and that capacity is set largely overnight and in the early morning, when radiative cooling produces the inversion beneath which a day's concentrations then build. A daytime-only mixing-depth measure discards the part of the diurnal cycle that does the work. The data agree: on identical days, a calendar-day boundary layer series yields a robust first-stage $F$ of 67.6 and a partial $R^2$ of 0.272, against 56.4 and 0.256 for the same series aggregated over the exposure window. We report the windowed construction as a robustness row in Table~\\ref{tab:robust}, and note that this is a choice made on instrument relevance, which the second stage does not enter.", 1),
("3.2 gap para",
 "\\textbf{A gap in the boundary layer series.} Boundary layer height is unavailable in the archive for the contiguous period 2 January to 30 June 2024---181 days affecting that field alone, with temperature, precipitation, wind, pressure, cloud cover and radiation complete throughout. We attempted recovery through alternative model specifications available within the same archive without success. Because boundary layer height is our primary instrument, these dates drop from the estimation sample. The gap falls entirely within the high-pollution half of the year, with consequences for identifying variation that we quantify in Section~\\ref{sec:sample} and discuss in Section~\\ref{sec:limits}.\n\n",
 "", 1),

# ========================================================== SECTION 3.4
("3.4 N",
 "yields a final estimation sample of \\textbf{926 trading days}",
 "yields a final estimation sample of \\textbf{1,042 trading days}", 1),
("3.4 loss",
 "\\textbf{Where the remaining trading days are lost.} Of the 446 weekday trading sessions not retained, 244 fall on days with no in-window pollution observation at all---the May--August 2022 gap together with the shorter monitor interruptions documented below---86 fail the eight-observation coverage requirement, and 116 fall within the January--June 2024 boundary layer gap. No day is lost to any other cause. Because a day can fail more than one requirement, causes are assigned in the fixed order given: absence of any in-window observation, then the coverage threshold, then the boundary layer gap. All three sources arise from external data availability rather than from any selection on our part.",
 "\\textbf{Where the remaining trading days are lost.} Of the 330 weekday trading sessions not retained, 244 fall on days with no in-window pollution observation at all---the May--August 2022 gap together with the shorter monitor interruptions documented below---and 86 fail the eight-observation coverage requirement. No day is lost to any other cause. Because a day can fail both requirements, causes are assigned in the fixed order given: absence of any in-window observation, then the coverage threshold. Both sources arise from external data availability rather than from any selection on our part.", 1),
("3.4 winter para",
 "Because the 2024 gap removes January through June, winter days are underrepresented in the estimation sample, and the loss of identifying variation is therefore greater than the raw day count implies, since our design is identified principally from winter inversion episodes. Section~\\ref{sec:limits} quantifies the resulting monthly imbalance. Mean \\pmm{} across the 1,548 days passing the coverage filter is 31.9 \\ugm{}, against 32.47 in the 926-day estimation sample.",
 "Section~\\ref{sec:limits} quantifies the residual monthly imbalance. Mean \\pmm{} across the 1,548 days passing the coverage filter is 31.9 \\ugm{}, against 32.29 in the 1,042-day estimation sample.", 1),
("3.4 attrition",
 "\\textbf{Attrition.} Because roughly a third of weekday trading sessions are lost, it matters whether they differ systematically from those retained. On the outcome, retained and excluded trading days do not differ in mean returns ($t = 0.41$, $p = 0.679$). A Kolmogorov--Smirnov test of equality of the return distributions returns $p = 0.060$---short of rejection at conventional levels, but borderline, and we report it as such. Retained days exhibit lower return variance (0.431 against 0.551, Levene $p = 0.004$) and less negative skew ($-0.30$ against $-0.52$). These are statistically detectable differences in the shape of the outcome distribution, and Section~\\ref{sec:limits} discusses why their consequence for our estimates is not signable.",
 "\\textbf{Attrition.} Because roughly a quarter of weekday trading sessions are lost, it matters whether they differ systematically from those retained. On the outcome, retained and excluded trading days do not differ in mean returns ($t = 1.17$, $p = 0.241$). They do differ in the shape of the return distribution: a Kolmogorov--Smirnov test of equality rejects ($p = 0.016$), retained days exhibit lower return variance (0.419 against 0.629, Levene $p < 0.001$) and less negative skew ($-0.28$ against $-0.51$). The days our data requirements retain are quieter than the days they discard, and the difference is statistically detectable rather than borderline. Section~\\ref{sec:limits} discusses why its consequence for our estimates is nonetheless not signable.", 1),

# ================================================ SECTION 3.5 — TABLE 1
("T1 rows",
 "Log return (\\%) & 0.09 & 0.66 & $-3.02$ & 0.11 & 4.05 \\\\\n\\pmm{} (\\ugm{}) & 32.47 & 35.87 & 3.36 & 17.91 & 233.00 \\\\\nBoundary layer height (m) & 267.50 & 221.70 & 19.38 & 220.31 & 1{,}303.12 \\\\\nTemperature (\\textdegree C) & 11.47 & 11.34 & $-22.43$ & 12.28 & 31.31 \\\\\nPrecipitation (mm) & 1.47 & 3.75 & 0.00 & 0.00 & 25.20 \\\\\nWind speed (m/s) & 5.51 & 1.75 & 1.33 & 5.42 & 12.68 \\\\\nSurface pressure (hPa) & 926.30 & 5.82 & 913.30 & 925.68 & 948.79 \\\\\nCloud cover (\\%) & 49.50 & 32.66 & 0.00 & 48.23 & 100.00 \\\\\nShortwave radiation (W/m\\textsuperscript{2}) & 173.60 & 86.00 & 12.71 & 173.21 & 346.58 \\\\",
 "Log return (\\%) & 0.10 & 0.65 & $-3.02$ & 0.12 & 4.05 \\\\\n\\pmm{} (\\ugm{}) & 32.29 & 35.14 & 3.36 & 17.82 & 233.00 \\\\\nBoundary layer height (m) & 238.27 & 187.61 & 21.15 & 201.18 & 923.88 \\\\\nTemperature (\\textdegree C) & 11.16 & 11.36 & $-22.43$ & 12.11 & 31.31 \\\\\nPrecipitation (mm) & 1.51 & 3.89 & 0.00 & 0.00 & 34.10 \\\\\nWind speed (m/s) & 5.51 & 1.71 & 1.33 & 5.40 & 12.68 \\\\\nSurface pressure (hPa) & 926.28 & 5.71 & 913.30 & 925.72 & 948.79 \\\\\nCloud cover (\\%) & 51.41 & 32.46 & 0.00 & 52.12 & 100.00 \\\\\nShortwave radiation (W/m\\textsuperscript{2}) & 175.13 & 85.49 & 12.71 & 176.08 & 346.58 \\\\", 1),
("T1 caption",
 "\\caption{Descriptive statistics ($N = 926$)}",
 "\\caption{Descriptive statistics ($N = 1{,}042$)}", 1),
("T1 note",
 "The standard deviation of log \\pmm{}, 0.858, is reported",
 "The standard deviation of log \\pmm{}, 0.860, is reported", 1),
("3.5 mean",
 "Mean \\pmm{} of 32.47 \\ugm{} is more than six times",
 "Mean \\pmm{} of 32.29 \\ugm{} is more than six times", 1),
("3.5 counts",
 "a median of 17.91 against the mean---with 241 days exceeding 35 \\ugm{}, 152 exceeding 55 and 97 exceeding 75 (96 of them above the benchmark's sample maximum of 75.86)",
 "a median of 17.82 against the mean---with 275 days exceeding 35 \\ugm{}, 178 exceeding 55 and 112 exceeding 75 (110 of them above the benchmark's sample maximum of 75.86)", 1),
("3.5 sdlog",
 "The standard deviation of log \\pmm{} is 0.858, so a doubling",
 "The standard deviation of log \\pmm{} is 0.860, so a doubling", 1),
("3.5 blh",
 "Boundary layer height ranges from 19 to 1,303 metres, with monthly means running from 53 metres in December to 560 in July (Figure~\\ref{fig:seasonality})",
 "Boundary layer height ranges from 21 to 924 metres, with monthly means running from 46 metres in December to 491 in July (Figure~\\ref{fig:seasonality})", 1),
("3.5 retsd",
 "A daily standard deviation of 0.66 per cent implies annualised volatility",
 "A daily standard deviation of 0.65 per cent implies annualised volatility", 1),
("3.5 boundsd",
 "return standard deviations---0.268 per doubling---is larger",
 "return standard deviations---0.235 per doubling---is larger", 1),
("3.5 ar1",
 "\\textbf{Returns exhibit no material autocorrelation} (AR(1) $= -0.003$)",
 "\\textbf{Returns exhibit no material autocorrelation} (AR(1) $= +0.017$)", 1),
("fig1 caption",
 "with a mean of 32.47 \\ugm{} (Table~\\ref{tab:descriptives})",
 "with a mean of 32.29 \\ugm{} (Table~\\ref{tab:descriptives})", 1),
("fig2 caption",
 "boundary layer height falls to 53 metres in December and 61 in January, rising to 560 in July---a tenfold seasonal amplitude",
 "boundary layer height falls to 46 metres in December and 53 in January, rising to 491 in July---a tenfold seasonal amplitude", 1),

# ========================================================== SECTION 4.x
("4.1 ols",
 "is $+0.013$ (SE $0.038$), against an instrumented estimate of $-0.105$",
 "is $+0.009$ (SE $0.035$), against an instrumented estimate of $-0.083$", 1),
("4.1 wh",
 "the classical Wu--Hausman test \\citep{hausman1978} marginally rejects ($F = 3.33$, $p = 0.069$), but its heteroskedasticity-robust analogue, a control-function regression test, does not ($p = 0.172$)",
 "neither the classical Wu--Hausman test \\citep{hausman1978} ($F = 2.85$, $p = 0.092$) nor its heteroskedasticity-robust analogue, a control-function regression test, rejects ($p = 0.242$)", 1),
("4.1 sign",
 "and $+0.013$ lies on the far side of zero from the instrumented estimate. We put little weight on that, however: $+0.013$ sits 0.34 standard errors from zero",
 "and $+0.009$ lies on the far side of zero from the instrumented estimate. We put little weight on that, however: $+0.009$ sits 0.25 standard errors from zero", 1),
("4.2 pre2024",
 "That specification yields $-0.034$ (SE $0.092$) on 599 observations with a robust first-stage $F$ of 39.3.",
 "That specification yields $-0.048$ (SE $0.089$) on 637 observations with a robust first-stage $F$ of 51.9.", 1),
("4.3 range",
 "ranges across our sample from \\textbf{19 metres to 1,303 metres}",
 "ranges across our sample from \\textbf{21 metres to 924 metres}", 1),
("4.3 monthly",
 "from 53 metres in December and 61 in January to 547 in June and 560 in July---a tenfold seasonal amplitude.",
 "from 46 metres in December and 53 in January to 427 in June and 491 in July---a tenfold seasonal amplitude.", 1),
("4.4 F",
 "joint $F$-statistic of 44.9 on the excluded instruments, with a partial $R^2$ of 0.215",
 "joint $F$-statistic of 84.0 on the excluded instruments, with a partial $R^2$ of 0.262", 1),
("4.5 cloudrad",
 "removing them moves the estimate from $-0.105$ to $-0.067$, well within one standard error. The robust first-stage $F$ rises from 44.9 to 47.0 when they are dropped, so including them costs some identifying variation, but not enough to threaten identification or to move the estimate materially.",
 "removing them moves the estimate from $-0.083$ to $-0.039$, well within one standard error. The robust first-stage $F$ falls from 84.0 to 73.8 when they are dropped, so including them costs no identifying variation.", 1),
("4.5 sargan",
 "The Sargan \\citeyearpar{sargan1958} statistic of 4.82 ($p = 0.185$) does not reject joint validity",
 "The Sargan \\citeyearpar{sargan1958} statistic of 3.82 ($p = 0.282$) does not reject joint validity", 1),
("4.5 pressure",
 "Excluding it changes the estimate from $-0.105$ to $-0.112$.",
 "Excluding it changes the estimate from $-0.083$ to $-0.084$.", 1),
("4.5 brent",
 "Regressing Brent returns on the instruments ($N = 918$), the instruments are jointly unrelated to the oil price ($p = 0.698$), and no instrument enters individually (boundary layer height $p = 0.880$, the east--west component $p = 0.665$, the north--south component $p = 0.252$, pressure $p = 0.729$). Brent returns do not predict KASE returns in this sample (0.009, SE $0.015$, $p = 0.544$), and conditioning the reduced form on Brent leaves it unchanged, with the joint exclusion $p$-value moving from 0.145 to 0.132. Both commodity tests are estimated on the subsample of days for which the commodity series is available---918 days here, 895 for uranium below",
 "Regressing Brent returns on the instruments ($N = 1{,}033$), the instruments are jointly unrelated to the oil price ($p = 0.734$), and no instrument enters individually (boundary layer height $p = 0.578$, the east--west component $p = 0.456$, the north--south component $p = 0.444$, pressure $p = 0.679$). Brent returns do not predict KASE returns in this sample (0.007, SE $0.014$, $p = 0.644$), and conditioning the reduced form on Brent leaves it unchanged, with the joint exclusion $p$-value moving from 0.245 to 0.231. Both commodity tests are estimated on the subsample of days for which the commodity series is available---1,033 days here, 1,007 for uranium below", 1),
("4.5 uranium",
 "Regressing uranium returns on the instruments ($N = 895$), the joint test is marginal ($p = 0.079$), driven by the east--west component (0.0964, SE $0.0465$, $p = 0.038$); boundary layer height ($p = 0.948$), the north--south component ($p = 0.213$) and surface pressure ($p = 0.108$) are unrelated. Uranium returns predict KASE returns at the ten per cent level (0.019, SE $0.010$, $p = 0.060$), with a correlation of 0.073.",
 "Regressing uranium returns on the instruments ($N = 1{,}007$), the joint test is marginal ($p = 0.082$), driven by the east--west component (0.0941, SE $0.0431$, $p = 0.029$); boundary layer height ($p = 0.804$), the north--south component ($p = 0.121$) and surface pressure ($p = 0.231$) are unrelated. Uranium returns predict KASE returns at the ten per cent level (0.017, SE $0.010$, $p = 0.071$), with a correlation of 0.068.", 1),
("4.5 transmit",
 "shifts index returns by 0.019 percentage points.",
 "shifts index returns by 0.017 percentage points.", 1),
("4.5 conditioning",
 "On the 895-day uranium subsample the boundary-layer coefficient moves from 0.0662 (SE $0.0526$) to 0.0665 (SE $0.0521$), and the joint exclusion $p$-value from 0.136 to 0.138.",
 "On the 1,007-day uranium subsample the boundary-layer coefficient moves from 0.0611 to 0.0602, and the joint exclusion $p$-value from 0.184 to 0.196.", 1),
("4.5 exclu",
 "moves the structural estimate from $-0.105$ to $-0.095$, with an exclusion bound of 0.176 percentage points against 0.177 (Table~\\ref{tab:robust})",
 "moves the structural estimate from $-0.083$ to $-0.077$, with an exclusion bound of 0.147 percentage points against 0.152 (Table~\\ref{tab:robust})", 1),
("4.5 uraniump",
 "The uranium joint $p$-value of 0.079 is marginal",
 "The uranium joint $p$-value of 0.082 is marginal", 1),
("4.6 ar1",
 "We find AR(1) $= -0.003$, so this does not appear to operate here.",
 "We find AR(1) $= +0.017$, so this does not appear to operate here.", 1),
("4.6 unrest",
 "the estimate moves from $-0.105$ to $-0.102$.",
 "the estimate moves from $-0.083$ to $-0.079$.", 1),
("4.7 fe",
 "is $-0.464$ (SE $0.038$) without month or year fixed effects, $-0.420$ ($0.042$) with month effects only, and $-0.451$ ($0.042$) with month and year effects jointly",
 "is $-0.549$ (SE $0.038$) without month or year fixed effects, $-0.524$ ($0.041$) with month effects only, and $-0.576$ ($0.040$) with month and year effects jointly", 1),
("4.7 validation",
 "We enumerate all 925 distinct circular shifts of the return series",
 "We enumerate all 1,041 distinct circular shifts of the return series", 1),
("4.7 ratio",
 "Its mean is $+0.0001$ and its standard deviation 0.0918, against our reported standard error of 0.0915---a ratio of 1.00. The ratio of 1.00 indicates that our standard errors are well calibrated, and that the equivalence bounds reported in Section~\\ref{sec:transport} are reliable.",
 "Its mean is $+0.0001$ and its standard deviation 0.0775, against our reported standard error of 0.0832---a ratio of 0.93. The reported standard errors are therefore, if anything, slightly wider than the sampling variation the design exhibits, so the equivalence bounds reported in Section~\\ref{sec:transport} are conservative rather than optimistic.", 1),

# ========================================================== SECTION 5.x
("5.1 sample",
 "mean \\pmm{} of 32.47 \\ugm{} with a log standard deviation of 0.858",
 "mean \\pmm{} of 32.29 \\ugm{} with a log standard deviation of 0.860", 1),
("5.1 retsd",
 "a return standard deviation of 0.66 per cent, which governs",
 "a return standard deviation of 0.65 per cent, which governs", 1),
("5.1 blh",
 "and boundary layer height ranging from 19 to 1,303 metres, which is the variation",
 "and boundary layer height ranging from 21 to 924 metres, which is the variation", 1),
("T2 rows",
 "Log boundary layer height & $-0.4508$ & 0.0421 & $-10.70$ & 0.000 \\\\\nWind component $u$ (E--W) & 0.0314 & 0.0095 & 3.30 & 0.001 \\\\\nWind component $v$ (N--S) & $-0.0200$ & 0.0091 & $-2.19$ & 0.029 \\\\\nSurface pressure & $-0.0240$ & 0.0055 & $-4.39$ & 0.000 \\\\",
 "Log boundary layer height & $-0.5758$ & 0.0403 & $-14.28$ & 0.000 \\\\\nWind component $u$ (E--W) & 0.0396 & 0.0088 & 4.50 & 0.000 \\\\\nWind component $v$ (N--S) & $-0.0221$ & 0.0082 & $-2.69$ & 0.007 \\\\\nSurface pressure & $-0.0259$ & 0.0051 & $-5.11$ & 0.000 \\\\", 1),
("T2 note",
 "$R^2 = 0.729$. Heteroskedasticity- and autocorrelation-robust first-stage $F$ on the excluded instruments $= 44.9$ (Stock--Yogo ten per cent critical value 24.58). Partial $R^2 = 0.215$.",
 "$R^2 = 0.747$. Heteroskedasticity- and autocorrelation-robust first-stage $F$ on the excluded instruments $= 84.0$ (Stock--Yogo ten per cent critical value 24.58). Partial $R^2 = 0.262$.", 1),
("5.2 F",
 "The robust first-stage $F$ of 44.9 comfortably exceeds the \\citet{stock2005} ten per cent maximal-size critical value of 24.58, and the excluded instruments account for 21.5 per cent",
 "The robust first-stage $F$ of 84.0 comfortably exceeds the \\citet{stock2005} ten per cent maximal-size critical value of 24.58, and the excluded instruments account for 26.2 per cent", 1),
("5.2 elast1",
 "raises surface concentrations by approximately 0.45 per cent.",
 "raises surface concentrations by approximately 0.58 per cent.", 1),
("5.2 elast2",
 "The estimated elasticity of $-0.451$ is below the value of $-1$",
 "The estimated elasticity of $-0.576$ remains below the value of $-1$", 1),
("fig3 caption",
 "The bivariate slope is $-0.501$. The corresponding coefficient in the full first stage, which additionally conditions on the remaining three instruments, is $-0.451$ (Table~\\ref{tab:firststage})",
 "The bivariate slope is $-0.615$. The corresponding coefficient in the full first stage, which additionally conditions on the remaining three instruments, is $-0.576$ (Table~\\ref{tab:firststage})", 1),
("fig3 F",
 "The robust joint first-stage $F$-statistic on the excluded instruments is 44.9.}",
 "The robust joint first-stage $F$-statistic on the excluded instruments is 84.0.}", 1),
("T3 rows",
 "Log boundary layer height & 0.0561 & 0.0523 & 0.284 \\\\\nWind component $u$ (E--W) & $-0.0089$ & 0.0120 & 0.456 \\\\\nWind component $v$ (N--S) & $-0.0206$ & 0.0126 & 0.102 \\\\\nSurface pressure & 0.0012 & 0.0066 & 0.858 \\\\",
 "Log boundary layer height & 0.0528 & 0.0546 & 0.334 \\\\\nWind component $u$ (E--W) & $-0.0065$ & 0.0112 & 0.562 \\\\\nWind component $v$ (N--S) & $-0.0171$ & 0.0114 & 0.134 \\\\\nSurface pressure & 0.0018 & 0.0061 & 0.772 \\\\", 1),
("T3 note",
 "Joint $F$-test of exclusion, $p = 0.153$. Partial $R^2$ of instruments $= 0.0067$.",
 "Joint $F$-test of exclusion, $p = 0.255$. Partial $R^2$ of instruments $= 0.0048$.", 1),
("5.3 joint",
 "the set does not jointly predict returns ($p = 0.153$). The north--south wind component is the closest ($p = 0.102$)",
 "the set does not jointly predict returns ($p = 0.255$). The north--south wind component is the closest ($p = 0.134$)", 1),
("5.3 factor",
 "explain \\textbf{21.5 per cent} of variation in log \\pmm{} conditional on controls, and \\textbf{0.67 per cent} of variation in returns---a factor of thirty-two",
 "explain \\textbf{26.2 per cent} of variation in log \\pmm{} conditional on controls, and \\textbf{0.48 per cent} of variation in returns---a factor of fifty-five", 1),
("T4 rows",
 "Log \\pmm{} & 0.0126 & $-0.1168$ & $\\mathbf{-0.1053}$ \\\\\n& (0.0377) & (0.1006) & \\textbf{(0.0915)} \\\\\n$p$ & 0.739 & 0.245 & 0.250 \\\\\nRobust first-stage $F$ & --- & 161.4 & 44.9 \\\\\nSargan statistic ($p$) & --- & --- & 4.82 (0.185) \\\\\nWu--Hausman $F$ ($p$) & --- & --- & 3.33 (0.069) \\\\\n$N$ & 926 & 926 & 926 \\\\",
 "Log \\pmm{} & 0.0088 & $-0.0898$ & $\\mathbf{-0.0826}$ \\\\\n& (0.0351) & (0.0903) & \\textbf{(0.0832)} \\\\\n$p$ & [RERUN] & 0.320 & 0.321 \\\\\nRobust first-stage $F$ & --- & 270.3 & 84.0 \\\\\nSargan statistic ($p$) & --- & --- & 3.82 (0.282) \\\\\nWu--Hausman $F$ ($p$) & --- & --- & 2.85 (0.092) \\\\\n$N$ & 1{,}042 & 1{,}042 & 1{,}042 \\\\", 1),
("5.4 main",
 "The preferred estimate is $\\mathbf{-0.1053}$ (SE $0.0915$), with a 95 per cent confidence interval of $[-0.285, 0.074]$. The point estimate is negative, consistent in sign with the hypothesis that pollution depresses returns, but statistically indistinguishable from zero ($p = 0.250$). The just-identified specification using boundary layer height alone, column (2), gives $-0.117$",
 "The preferred estimate is $\\mathbf{-0.0826}$ (SE $0.0832$), with a 95 per cent confidence interval of $[-0.246, 0.080]$. The point estimate is negative, consistent in sign with the hypothesis that pollution depresses returns, but statistically indistinguishable from zero ($p = 0.321$). The just-identified specification using boundary layer height alone, column (2), gives $-0.090$", 1),
("5.4 tests",
 "does not reject instrument validity (statistic 4.82, $p = 0.185$); Section~\\ref{sec:instrsens} documents where the residual tension lies. The classical Wu--Hausman test \\citep{hausman1978} marginally rejects exogeneity of log \\pmm{} ($F = 3.33$, $p = 0.069$); its heteroskedasticity-robust analogue does not ($p = 0.172$). We read the two together as inconclusive on exogeneity rather than as establishing it in either direction",
 "does not reject instrument validity (statistic 3.82, $p = 0.282$); Section~\\ref{sec:instrsens} documents where the residual tension lies. The classical Wu--Hausman test \\citep{hausman1978} does not reject exogeneity of log \\pmm{} ($F = 2.85$, $p = 0.092$), and neither does its heteroskedasticity-robust analogue ($p = 0.242$). We read the two together as inconclusive on exogeneity rather than as establishing it in either direction", 1),
("5.4 route",
 "would have supported an exclusion bound near 0.052 percentage points per doubling---roughly three times tighter than the one we report. Because the classical test rejects at ten per cent while its robust analogue does not, and because the a priori argument for endogeneity stands",
 "would have supported an exclusion bound near 0.046 percentage points per doubling---roughly three times tighter than the one we report. Because the a priori argument for endogeneity stands regardless of what the specification tests show", 1),
("fig4 caption",
 "\\textbf{The fitted line reports the ordinary least squares slope of 0.013. It does not display the instrumental variables estimate.} The paper's preferred estimate is $-0.105$ (Table~\\ref{tab:main}); the gap between the two is consistent with the modest upward confounding discussed in Section~\\ref{sec:endog}, though it lies within sampling variation. Nineteen of twenty confidence intervals contain zero, consistent with sampling variation at the ninety-five per cent level.",
 "\\textbf{The fitted line reports the ordinary least squares slope of 0.009. It does not display the instrumental variables estimate.} The paper's preferred estimate is $-0.083$ (Table~\\ref{tab:main}); the gap between the two is consistent with the modest upward confounding discussed in Section~\\ref{sec:endog}, though it lies within sampling variation. All twenty confidence intervals contain zero.", 1),
("5.5 bound",
 "any effect exceeding 0.256 log points in absolute value, equivalent to 0.177 percentage points per doubling of \\pmm{} concentration.} Expressed relative to the standard deviation of daily returns in our sample, the bound corresponds to 0.268 return standard deviations per doubling.",
 "any effect exceeding 0.219 log points in absolute value, equivalent to 0.152 percentage points per doubling of \\pmm{} concentration.} Expressed relative to the standard deviation of daily returns in our sample, the bound corresponds to 0.235 return standard deviations per doubling.", 1),
("T5 rows",
 "Proportional (constant per doubling) & $-0.194$ & 0.063 & 0.121 & 0.089 & \\textbf{1.35} \\\\\nLinear (constant per \\ugm{}, at our mean) & $-0.545$ & 0.179 & 0.472 & 0.190 & \\textbf{2.49} \\\\",
 "Proportional (constant per doubling) & $-0.194$ & 0.063 & 0.137 & 0.086 & \\textbf{1.60} \\\\\nLinear (constant per \\ugm{}, at our mean) & $-0.543$ & 0.178 & 0.486 & 0.187 & \\textbf{2.60} \\\\", 1),
("T5 note",
 "Our estimate is $-0.073$ (SE $0.063$) on the same scale. The linear implication evaluates the benchmark's coefficient at our sample mean of 32.47 \\ugm{}.",
 "Our estimate is $-0.057$ (SE $0.058$) on the same scale. The linear implication evaluates the benchmark's coefficient at our sample mean of 32.29 \\ugm{}.", 1),
("5.5 lineartext",
 "\\textbf{Linear transport is rejected at the five per cent level but not at one per cent} ($t = 2.49$)",
 "\\textbf{Linear transport is rejected at the five per cent level but not at one per cent} ($t = 2.60$)", 1),
("5.5 proptext",
 "\\textbf{Proportional transport cannot be rejected} ($t = 1.35$). Our point estimate is under forty per cent",
 "\\textbf{Proportional transport cannot be rejected} ($t = 1.60$). Our point estimate is under thirty per cent", 1),
("5.5 ourest",
 "Expressed in percentage points per doubling of \\pmm{}, our estimate is $-0.073$ (SE $0.063$).",
 "Expressed in percentage points per doubling of \\pmm{}, our estimate is $-0.057$ (SE $0.058$).", 1),
("5.5 funcform",
 "approximately $-0.054$ at a mean of 15 \\ugm{}, against $-0.028$ at Almaty's 32.47",
 "approximately $-0.042$ at a mean of 15 \\ugm{}, against $-0.022$ at Almaty's 32.29", 1),
("5.5 equality",
 "the equality test carries the benchmark's own standard error of 0.063, and it does not reject ($t = 1.35$, Table~\\ref{tab:transport})",
 "the equality test carries the benchmark's own standard error of 0.063, and it does not reject ($t = 1.60$, Table~\\ref{tab:transport})", 1),

# =============================== TABLE 6 — whole body, plus new row
("T6 body",
 "\\textbf{Preferred} & $\\mathbf{-0.1053}$ & \\textbf{0.0915} & \\textbf{0.250} & \\textbf{926} & \\textbf{44.9} & \\textbf{0.177} \\\\",
 "\\textbf{Preferred} & $\\mathbf{-0.0826}$ & \\textbf{0.0832} & \\textbf{0.321} & \\textbf{1{,}042} & \\textbf{84.0} & \\textbf{0.152} \\\\", 1),

# ========================================================== SECTION 5.7
("5.7 leads",
 "$-0.084$, $-0.072$ and $-0.073$ at one to three days against $-0.105$ contemporaneously, with the five-day lead positive. In the exact randomisation null, the contemporaneous estimate sits at the 12.6th percentile and the raw leads between the 17th and the 76th.",
 "$-0.006$, $+0.008$ and $-0.019$ at one to three days against $-0.083$ contemporaneously, with the five-day lead positive. In the exact randomisation null, the contemporaneous estimate sits at the 12.8th percentile and the raw leads between the 39th and the 78th.", 1),
("T7 rows",
 "Contemporaneous (main estimate) & $-0.1053$ & 0.0915 & 0.250 & 12.6 \\\\\n1-day lead return & $-0.0840$ & 0.0771 & 0.276 & 17.0 \\\\\n1-day lead, conditioning on lagged \\pmm{} & $-0.1153$ & 0.0921 & 0.211 & 9.8 \\\\\n2-day lead return & $-0.0724$ & 0.0839 & 0.389 & 21.6 \\\\\n3-day lead return & $-0.0727$ & 0.0851 & 0.393 & 21.4 \\\\\n5-day lead return & 0.0666 & 0.0785 & 0.396 & 76.4 \\\\",
 "Contemporaneous (main estimate) & $-0.0826$ & 0.0832 & 0.321 & 12.8 \\\\\n1-day lead return & $-0.0063$ & 0.0724 & 0.931 & 46.9 \\\\\n1-day lead, conditioning on lagged \\pmm{} & $-0.0191$ & 0.0814 & 0.814 & 39.7 \\\\\n2-day lead return & 0.0084 & 0.0729 & 0.908 & 55.0 \\\\\n3-day lead return & $-0.0186$ & 0.0698 & 0.790 & 40.0 \\\\\n5-day lead return & 0.0584 & 0.0692 & 0.399 & 78.2 \\\\", 1),
("T7 note",
 "whose 95 per cent range is $[-0.175, 0.180]$",
 "whose 95 per cent range is $[-0.169, 0.154]$", 1),
("5.7 disclosure",
 "One feature deserves disclosure. Conditioning the one-day lead on lagged pollution yields $-0.115$---marginally larger in magnitude than the contemporaneous estimate, and at the 9.8th percentile of the null. Three of the four raw leads also share the contemporaneous estimate's sign, whereas independent noise would produce a roughly even split. Pollution is strongly autocorrelated ($\\rho_1 = 0.72$ in logs), so a slow-moving factor jointly influencing seasonal pollution and returns beyond what our month and year fixed effects absorb would generate same-signed correlations at multiple leads, at exactly this small scale. We cannot rule that out, and we report it rather than resolve it.\n\nThe pattern is nonetheless consistent with sampling variation. All six estimates lie well inside the null's 95 per cent range of $[-0.175, 0.180]$, none is statistically distinguishable from any other, and the spread across them is approximately one standard error---which is precisely the resolution this design affords.",
 "The leads behave as they should. Every one is small in absolute magnitude relative to the contemporaneous estimate and they are mixed in sign. Conditioning the one-day lead on lagged pollution, which on an earlier and smaller version of this sample produced an estimate larger in magnitude than the contemporaneous one, gives $-0.019$. All six estimates lie well inside the null's 95 per cent range of $[-0.169, 0.154]$, and every placebo sits between the 39th and the 78th percentile of that null, which is where uninformative specifications should sit.", 1),
("5.7 benchcompare",
 "Ours are a fifth to a third smaller than our contemporaneous estimate. Their placebos are cleaner in relative terms, and we note it.",
 "Ours are roughly a tenth to a fifth the size of our contemporaneous estimate, which is comparable.", 1),
("5.7 validation",
 "Enumerating all 925 circular shifts gives an exact randomisation null with mean $+0.0001$ and standard deviation 0.0918, against a HAC standard error of 0.0915---a ratio of 1.00. Reported standard errors are therefore well calibrated, and the equivalence bounds are correspondingly reliable. The randomisation $p$-value against a null of zero is 0.243, in close agreement with the analytical value of 0.250.",
 "Enumerating all 1,041 circular shifts gives an exact randomisation null with mean $+0.0001$ and standard deviation 0.0775, against a HAC standard error of 0.0832---a ratio of 0.93. Reported standard errors are therefore mildly conservative, and the equivalence bounds correspondingly cautious. The randomisation $p$-value against a null of zero is 0.267, against the analytical value of 0.321.", 1),

# ========================================================== SECTION 5.8
("T8 rows",
 "Observations & 926 & 3{,}722 & 4.02 \\\\\nSE, return SD per pollution SD (IV) & 0.119 & 0.030 & 3.97 \\\\\nSE, return SD per pollution SD (OLS) & 0.049 & 0.030 & 1.63 \\\\",
 "Observations & 1{,}042 & 3{,}722 & 3.57 \\\\\nSE, return SD per pollution SD (IV) & 0.110 & 0.030 & 3.68 \\\\\nSE, return SD per pollution SD (OLS) & 0.047 & 0.030 & 1.55 \\\\", 1),
("5.8 fourtimes",
 "The benchmark study is roughly four times more precise. \\textbf{Data frequency",
 "The benchmark study is roughly three and a half times more precise. \\textbf{Data frequency", 1),
("5.8 decomp",
 "Sample size predicts a factor of $\\sqrt{3{,}722/926} \\approx 2.00$. Our ordinary least squares standard error, at 0.049, is 1.63 times the benchmark's",
 "Sample size predicts a factor of $\\sqrt{3{,}722/1{,}042} \\approx 1.89$. Our ordinary least squares standard error, at 0.047, is 1.55 times the benchmark's", 1),
("5.8 inflation",
 "\\textbf{Instrumenting inflates our standard error by a factor of 2.43 over ordinary least squares}, close to the $1/\\sqrt{0.215} \\approx 2.16$ implied by our partial $R^2$",
 "\\textbf{Instrumenting inflates our standard error by a factor of 2.37 over ordinary least squares}, somewhat above the $1/\\sqrt{0.262} \\approx 1.95$ implied by our partial $R^2$", 1),
("5.8 power",
 "would require a standard error near 0.029 log points against our 0.092---a factor of 3.2, implying approximately ten times the observations, or roughly 9,200 trading days.",
 "would require a standard error near 0.028 log points against our 0.083---a factor of 2.9, implying roughly nine times the observations, or about 8,900 trading days.", 1),

# ========================================================== SECTION 6.x
("6.1 open",
 "changes same-day returns on the KASE Index by $-0.073$ percentage points (SE $0.063$)",
 "changes same-day returns on the KASE Index by $-0.057$ percentage points (SE $0.058$)", 1),
("6.1 reject",
 "We reject effects exceeding 0.177 percentage points per doubling, reject linear transport",
 "We reject effects exceeding 0.152 percentage points per doubling, reject linear transport", 1),
("6.1 smaller",
 "it is smaller than 0.177 percentage points per doubling of \\pmm{}. That threshold sits close to the benchmark's \\emph{proportional} implication of 0.194---its effect per doubling evaluated at its own sample mean---on either side of which it can fall depending on the critical value used to construct it (Section~\\ref{sec:transport}).",
 "it is smaller than 0.152 percentage points per doubling of \\pmm{}. That threshold now sits below the benchmark's \\emph{proportional} implication of 0.194---its effect per doubling evaluated at its own sample mean---and remains below it under every specification and inference procedure we examine (Section~\\ref{sec:transport}).", 1),
("6.1 linear",
 "\\emph{linear} implication of 0.545 percentage points",
 "\\emph{linear} implication of 0.543 percentage points", 1),
("6.1 delayed",
 "any delayed effect is smaller than 0.129 percentage points per doubling",
 "any delayed effect is smaller than 0.078 percentage points per doubling", 1),
("6.2 days",
 "Of our 926 trading days, 96---roughly one day in ten---exceed 75.86 \\ugm{}",
 "Of our 1,042 trading days, 110---roughly one day in ten---exceed 75.86 \\ugm{}", 1),
("6.3 sargan",
 "Our overidentification test does not reject (Sargan statistic 4.82, $p = 0.185$)",
 "Our overidentification test does not reject (Sargan statistic 3.82, $p = 0.282$)", 1),
("6.3 justid",
 "Instrumenting with boundary layer height alone yields $-0.117$ (SE $0.101$, robust first-stage $F = 161.4$); with surface pressure alone, $-0.114$ (SE $0.158$, robust $F = 56.1$). The two dispersion instruments---mechanistically distinct measures of the same trapping phenomenon---agree just-identified to within 0.003 of one another.",
 "Instrumenting with boundary layer height alone yields $-0.090$ (SE $0.090$, robust first-stage $F = 270.3$); with surface pressure alone, $-0.098$ (SE $0.142$, robust $F = 60.6$). The two dispersion instruments---mechanistically distinct measures of the same trapping phenomenon---agree just-identified to within 0.009 of one another.", 1),
("6.3 v para",
 "The north--south component $v$ is conditionally relevant in the joint first stage ($t = -2.19$) yet nearly irrelevant on its own: its just-identified robust first-stage $F$ is 0.4, and its just-identified estimate, $+3.70$ (SE $6.16$), is uninformative. Its weak-instrument-robust Anderson--Rubin confidence set is an unbounded union---approximately $(-\\infty, -0.66] \\cup [0.08, +\\infty)$ on our search grid---which formally excludes both zero and the dispersion-based estimates. An unbounded set is the expected shape when an instrument carries almost no identifying variation \\citep{dufour1997}, and its exclusion of the small central interval reflects residual correlation between $v$ and returns absent any first stage---the signature of a variable with a direct path to the outcome rather than of one identifying a parameter. This is not in tension with Table~\\ref{tab:reducedform}, where $v$ carries $p = 0.102$. That coefficient conditions on the other three instruments, whereas the Anderson--Rubin set is constructed from a specification in which they are absent; the exclusion of zero from that set is equivalent to $v$ predicting returns at the five per cent level in the reduced form without them. We accordingly describe $v$ as contributing almost no identifying variation while showing direct correlation with returns, and note that the overidentification tension disappears when it is dropped.",
 "The north--south component $v$ is conditionally relevant in the joint first stage ($t = -2.69$) yet nearly irrelevant on its own: its just-identified robust first-stage $F$ is 0.7, and its just-identified estimate, $+2.48$ (SE $3.22$), carries no information. Its weak-instrument-robust Anderson--Rubin confidence set spans the entire range we searched, $[-5, 5]$, which is what \\citet{dufour1997} shows must happen when an instrument carries almost no identifying variation: with no first stage, no value of the parameter can be rejected. We accordingly describe $v$ as contributing almost no identifying variation on its own, and note that the overidentification tension disappears when it is dropped.", 1),
("6.3 u",
 "The east--west component $u$ is weak alone (just-identified robust $F = 4.7$)",
 "The east--west component $u$ is weak alone (just-identified robust $F = 7.6$)", 1),
("6.3 drops",
 "dropping $v$ moves the estimate to $-0.134$ with a bound of 0.199, dropping $u$ to $-0.095$ with a bound of 0.176.",
 "dropping $v$ moves the estimate to $-0.105$ with a bound of 0.167, dropping $u$ to $-0.077$ with a bound of 0.147.", 1),
("6.3 core",
 "and note that the specification retaining only these yields $-0.116$ with a Sargan $p$-value of 0.985. Our transport conclusions hold in every specification: linear transport is rejected at the five per cent level throughout.",
 "and note that the specification retaining only these yields $-0.092$. Our transport conclusions hold in every specification: linear transport is rejected at the five per cent level throughout. A reader who regards the uranium association as disqualifying for $u$ should note that the specification excluding it yields $-0.077$ with an exclusion bound of 0.147 percentage points, tighter than the bound we report: our conclusions do not depend on the instrument about which we have the most doubt.", 1),
("6.4 gaps",
 "Of the 1,372 weekday sessions on which the exchange traded during our sample window, 926 are retained. Three sources account for the difference, documented in Section~\\ref{sec:sample}. The 2024 gap falls disproportionately on high-pollution months, removing January through June of that year and therefore an entire winter's identifying variation. Our sample retains 62 January days, 59 February and 50 March, against 101 in September, 94 in October and 105 in November.",
 "Of the 1,372 weekday sessions on which the exchange traded during our sample window, 1,042 are retained. Both sources of loss are documented in Section~\\ref{sec:sample} and arise from interruptions in monitor coverage. Winter months remain somewhat underrepresented: our sample retains 80 January days, 79 February and 67 March, against 101 in September, 94 in October and 105 in November.", 1),
("6.4 attrition",
 "a borderline Kolmogorov--Smirnov result ($p = 0.060$), and lower return variance among retained days (Levene $p = 0.004$)",
 "a Kolmogorov--Smirnov rejection ($p = 0.016$), and lower return variance among retained days (Levene $p < 0.001$)", 1),

# =========================================================== CONCLUSION
("7 estimate",
 "we estimate $-0.105$ (SE $0.092$) across 926 trading days, and reject effects exceeding 0.177 percentage points per doubling of concentration. Any delayed effect is bounded more tightly still, at 0.129 percentage points",
 "we estimate $-0.083$ (SE $0.083$) across 1,042 trading days, and reject effects exceeding 0.152 percentage points per doubling of concentration. Any delayed effect is bounded more tightly still, at 0.078 percentage points", 1),
("7 factor",
 "the instruments explain 21.5 per cent of variation in pollution and 0.67 per cent of variation in returns.",
 "the instruments explain 26.2 per cent of variation in pollution and 0.48 per cent of variation in returns.", 1),
("7 precision",
 "Our estimate is four times less precise than the benchmark's. Sample size explains a factor of two;",
 "Our estimate is three and a half times less precise than the benchmark's. Sample size explains a factor of 1.9;", 1),
("7 close",
 "Our placebo tests are not as clean as the benchmark's, and we report rather than explain the residual pattern. One of our four instruments carries a marginal association with a commodity sector to which the index is exposed, and another contributes almost no identifying variation while correlating directly with returns; we disclose both without resolving them.",
 "One of our four instruments carries a marginal association with a commodity sector to which the index is exposed, and another contributes almost no identifying variation on its own; we disclose both without resolving them.", 1),
]


def flex(text, find, repl, label, count, report):
    pat = re.compile(r"\s+".join(re.escape(t) for t in find.split()))
    hits = list(pat.finditer(text))
    if len(hits) == count:
        for m in reversed(hits):
            text = text[:m.start()] + repl + text[m.end():]
        report.append(f"  OK       {label}")
        return text, True
    report.append(f"  {'MISSING ' if not hits else 'COUNT   '}{label}"
                  f"  (found {len(hits)}, expected {count})")
    return text, False


def main():
    p = Path(TEX)
    if not p.exists():
        sys.exit(f"not found: {TEX}")
    t = p.read_text(encoding="utf-8")
    rep, ok = [], 0
    for label, f, r, c in E:
        t, good = flex(t, f, r, label, c, rep)
        ok += good
    Path(OUT).write_text(t, encoding="utf-8")
    print("\n".join(rep))
    print(f"\n{ok}/{len(E)} applied.  Wrote {OUT}.")
    print("""
STILL BY HAND — see numeral_sweep.md
  * Table 6: all 31 rows. Only the Preferred row is scripted here; the
    rest come from the rebuild output, with the F column divided by the
    number of instruments in each row. Add the "Pressure alone" row.
    NOTE: on the new panel NO row falls below its Stock-Yogo threshold
    (the lowest is the pre-session treatment at 25.9 against 24.58), so
    the table note's "three rows fall at or below their threshold"
    sentence must be deleted, and the matching claims in 5.6 with it.
  * Section 5.6's eight emphasised results: every figure.
  * Section 5.5's fragility paragraph: judgement, not substitution.
  * The five values from Part E of numeral_sweep.md.
  * Footnote 5's robust overidentification p-values.
""")


if __name__ == "__main__":
    main()

  OK       abs range
  OK       abs N
  OK       abs estimate
  OK       abs bound
  OK       1 intro N
  OK       1.1 mean
  OK       1.1 counts
  OK       1.1 median
  OK       1.1 blh
  OK       1.2 CI
  OK       1.2 F
  OK       1.2 v clause
  OK       1.2 exogeneity
  OK       1.2 RI
  OK       1.2 bound
  OK       1.2 factor
  OK       1.3 linear
  OK       1.3 prop
  OK       1.3 rank
  OK       1.4 precision
  OK       2.5 ols
  OK       2.5 F
  OK       2.5 per10
  OK       3.2 sources
  OK       3.2 gap para
  OK       3.4 N
  OK       3.4 loss
  OK       3.4 winter para
  OK       3.4 attrition
  OK       T1 rows
  OK       T1 caption
  OK       T1 note
  OK       3.5 mean
  OK       3.5 counts
  OK       3.5 sdlog
  OK       3.5 blh
  OK       3.5 retsd
  OK       3.5 boundsd
  OK       3.5 ar1
  OK       fig1 caption
  OK       fig2 caption
  OK       4.1 ols
  OK       4.1 wh
  OK       4.1 sign
  OK       4.2 pre2024
  OK       4.3 range
  OK       4.3 monthly
  OK      

In [48]:
import numpy as np, pandas as pd, statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
LN2 = np.log(2)

D = pd.read_csv("panel_cds.csv"); D["date"] = pd.to_datetime(D["date"])
INST = ["log_blh","u","v","pressure"]
keep = [c for c in D.columns if c.startswith(("mo_","dw_","yr_")) and D[c].nunique()>1] + \
       [c for c in ["temp","precip","wind_speed","cloud","radiation","lockdown","qantar"]
        if D[c].nunique()>1]
kk=[]
for c in keep:
    M = D[["const"]+kk+[c]].astype(float).values
    if np.linalg.matrix_rank(M)==M.shape[1]: kk.append(c)
keep=kk

o = smf.ols("logret ~ log_pm25 + "+" + ".join(keep), data=D).fit(cov_type="HAC", cov_kwds={"maxlags":5})
print(f"1. OLS p = {o.pvalues['log_pm25']:.3f}")

print("\n2-3. HAC lag sensitivity")
for L in (0,5,20):
    kw = {"cov_type":"robust"} if L==0 else {"cov_type":"kernel","kernel":"bartlett","bandwidth":L}
    r = IV2SLS(D.logret, D[["const"]+keep], D.log_pm25, D[INST]).fit(**kw)
    b_,s_ = float(r.params.log_pm25), float(r.std_errors.log_pm25)
    ok = {"cov_type":"HC0"} if L==0 else {"cov_type":"HAC","cov_kwds":{"maxlags":L}}
    oo = smf.ols("logret ~ log_pm25 + "+" + ".join(keep), data=D).fit(**ok)
    print(f"   lag {L:>2}: IV SE {s_:.4f} bound {(abs(b_)+1.645*s_)*LN2:.3f}   OLS SE {oo.bse['log_pm25']:.4f}")

print("\n4. Sargan variants")
for lab, inst in [("drop v",["log_blh","u","pressure"]), ("BLH+pressure",["log_blh","pressure"])]:
    r = IV2SLS(D.logret, D[["const"]+keep], D.log_pm25, D[inst]).fit(cov_type="kernel")
    print(f"   {lab:<14} coef {float(r.params.log_pm25):.4f}  Sargan p {float(r.sargan.pval):.3f}")

r = IV2SLS(D.logret, D[["const"]+keep], D.log_pm25, D[INST]).fit(cov_type="kernel")
print("\n5. footnote 5 robust overid")
for n,a in [("Wooldridge","wooldridge_overid"),("Basmann","basmann"),("Anderson-Rubin","anderson_rubin")]:
    try: print(f"   {n:<15} p = {float(getattr(r,a).pval):.4f}")
    except Exception as e: print(f"   {n}: {e}")

print("\n6. transport footnote at 0.0171")
est, se_e = -0.0826*LN2, 0.0832*LN2
prop, prop_se = 0.194*(0.0171/0.0168), 0.063*(0.0171/0.0168)
lin, lin_se = 0.0171*D.pm25_raw.mean(), 0.0055*D.pm25_raw.mean()
print(f"   prop {prop:.3f} t {(prop+est)/np.sqrt(se_e**2+prop_se**2):.2f}"
      f"   lin {lin:.3f} t {(lin+est)/np.sqrt(se_e**2+lin_se**2):.2f}")

1. OLS p = 0.802

2-3. HAC lag sensitivity
   lag  0: IV SE 0.0817 bound 0.150   OLS SE 0.0343
   lag  5: IV SE 0.0822 bound 0.151   OLS SE 0.0351
   lag 20: IV SE 0.0803 bound 0.149   OLS SE 0.0295

4. Sargan variants
   drop v         coef -0.1049  Sargan p 0.774
   BLH+pressure   coef -0.0917  Sargan p 0.951

5. footnote 5 robust overid
   Wooldridge      p = 0.3370
   Basmann         p = 0.2938
   Anderson-Rubin  p = 0.2808

6. transport footnote at 0.0171
   prop 0.197 t 1.63   lin 0.552 t 2.65


In [49]:
#!/usr/bin/env python3
"""
FINAL PATCH — run AFTER apply_rebuild.py, on main_rebuilt.tex.

    python3 apply_rebuild2.py          (or paste into a Colab cell)

Writes main_final.tex.

Covers Table 6's whole body, Section 5.6's eight emphasised results, the
threshold table, and the five values computed in the Part E cell.

THREE CLAIMS REVERSE, and the replacements below handle them. Read these
so you know what changed and can check the wording reads as you want:

  1. "The estimate is negative in every specification" is now FALSE.
     Lag 2 is +0.025 and lag 3 is +0.001.
  2. No Table 6 row falls below its Stock-Yogo threshold any more. The
     weakest is the pre-session treatment at 25.9 against 24.58. Every
     weak-identification caveat in 5.6 goes with it.
  3. The drift toward more negative estimates in narrower windows is
     weaker and no longer monotonic (-0.134 then -0.123).

STILL NOT DONE after this: Section 5.5's fragility paragraph. Draft at
the bottom of this file -- paste it in by hand and read it properly,
because it is the one place where the rebuild changed what the paper can
claim rather than what it reports.
"""

import re, sys
from pathlib import Path

SRC, OUT = "main_rebuilt.tex", "main_final.tex"

T6 = r"""Coverage $\geq$ 18h (calendar day) & $-0.0988$ & 0.0852 & 0.246 & 998 & 81.7 & 0.166 \\
Coverage $\geq$ 15h (calendar day) & $-0.0855$ & 0.0830 & 0.303 & 1{,}036 & 85.2 & 0.154 \\
Full 11-hour window coverage & $-0.0516$ & 0.0923 & 0.576 & 864 & 67.6 & 0.141 \\
Sample from 2022 & $-0.1342$ & 0.0966 & 0.165 & 783 & 72.8 & 0.203 \\
Sample from 2023 & $-0.1230$ & 0.1027 & 0.231 & 649 & 51.3 & 0.202 \\
Excluding 2025 & $-0.0449$ & 0.0745 & 0.546 & 831 & 67.3 & 0.116 \\
Pre-2024 (constant clock offset) & $-0.0476$ & 0.0886 & 0.591 & 637 & 51.9 & 0.134 \\
Winter only (Nov--Mar) & $-0.0837$ & 0.0976 & 0.391 & 415 & 50.8 & 0.169 \\
Winsorised at 1\% & $-0.0780$ & 0.0750 & 0.298 & 1{,}042 & 79.3 & 0.140 \\
Instruments excl.\ pressure & $-0.0843$ & 0.0853 & 0.323 & 1{,}042 & 103.4 & 0.156 \\
Instruments excl.\ $v$ & $-0.1049$ & 0.0825 & 0.203 & 1{,}042 & 105.2 & 0.167 \\
Instruments excl.\ $u$ & $-0.0765$ & 0.0875 & 0.382 & 1{,}042 & 105.6 & 0.153 \\
BLH $+$ $u$ & $-0.1072$ & 0.0839 & 0.201 & 1{,}042 & 148.2 & 0.170 \\
BLH $+$ pressure (dispersion only) & $-0.0917$ & 0.0869 & 0.291 & 1{,}042 & 154.0 & 0.163 \\
BLH alone & $-0.0898$ & 0.0903 & 0.320 & 1{,}042 & 270.3 & 0.165 \\
Pressure alone & $-0.0984$ & 0.1424 & 0.489 & 1{,}042 & 60.6 & 0.231 \\
Controls excl.\ cloud, radiation & $-0.0392$ & 0.0842 & 0.641 & 1{,}042 & 73.8 & 0.123 \\
Excluding January 2022 unrest & $-0.0792$ & 0.0832 & 0.341 & 1{,}036 & 86.0 & 0.150 \\
Dense coverage (Apr 2023 onward) & $-0.1250$ & 0.1116 & 0.262 & 621 & 46.4 & 0.214 \\
Post-2024 gap (Jul 2024 onward) & $-0.1894$ & 0.2132 & 0.374 & 327 & 46.3 & 0.374 \\
Lag 1 day & $-0.0141$ & 0.0600 & 0.814 & 1{,}041 & 89.7 & 0.078 \\
Lag 2 days & 0.0249 & 0.0653 & 0.703 & 1{,}040 & 99.7 & 0.092 \\
Lag 3 days & 0.0013 & 0.0591 & 0.982 & 1{,}039 & 108.0 & 0.068 \\
Outcome: absolute return & $-0.0262$ & 0.0542 & 0.629 & 1{,}042 & 83.1 & 0.080 \\
Temperature in 2.5\textdegree{} bins & $-0.1101$ & 0.0806 & 0.172 & 1{,}042 & 85.3 & 0.168 \\
Cubic spline in temperature & $-0.0761$ & 0.0820 & 0.353 & 1{,}042 & 90.0 & 0.146 \\
Year-by-month fixed effects & $-0.1035$ & 0.0918 & 0.260 & 1{,}042 & 100.8 & 0.176 \\
Treatment: session only (11:00--17:59) & $-0.0708$ & 0.0771 & 0.358 & 1{,}042 & 84.7 & 0.137 \\
Treatment: pre-session only (07:00--10:59) & $-0.1146$ & 0.1085 & 0.291 & 1{,}042 & 25.9 & 0.203 \\
Treatment: calendar-day mean & $-0.0791$ & 0.1027 & 0.441 & 1{,}042 & 66.4 & 0.172 \\"""

E = [

# ---- Table 6 body (everything after the Preferred row) --------------
("T6 all rows",
 r"""Coverage $\geq$ 18h (calendar day) & $-0.1265$ & 0.0912 & 0.165 & 882 & 42.8 & 0.192 \\ Coverage $\geq$ 15h (calendar day) & $-0.1077$ & 0.0926 & 0.245 & 920 & 45.3 & 0.180 \\ Full 11-hour window coverage & $-0.0794$ & 0.1019 & 0.436 & 758 & 38.2 & 0.171 \\ Sample from 2022 & $-0.1390$ & 0.1141 & 0.223 & 667 & 35.2 & 0.226 \\ Sample from 2023 & $-0.1658$ & 0.1369 & 0.226 & 533 & 25.9 & 0.271 \\ Excluding 2025 & $-0.0489$ & 0.0798 & 0.540 & 715 & 39.1 & 0.125 \\ Pre-2024 (constant clock offset) & $-0.0336$ & 0.0919 & 0.714 & 599 & 39.3 & 0.128 \\ Winter only (Nov--Mar) & $-0.0809$ & 0.1134 & 0.476 & 360 & 27.6 & 0.185 \\ Winsorised at 1\% & $-0.1034$ & 0.0848 & 0.222 & 926 & 44.9 & 0.168 \\ Instruments excl.\ pressure & $-0.1119$ & 0.0939 & 0.233 & 926 & 56.6 & 0.185 \\ Instruments excl.\ $v$ & $-0.1337$ & 0.0930 & 0.151 & 926 & 58.3 & 0.199 \\ Instruments excl.\ $u$ & $-0.0953$ & 0.0967 & 0.324 & 926 & 59.2 & 0.176 \\ BLH $+$ $u$ & $-0.1393$ & 0.0935 & 0.136 & 926 & 82.9 & 0.203 \\ BLH $+$ pressure (dispersion only) & $-0.1161$ & 0.0979 & 0.235 & 926 & 87.1 & 0.192 \\ BLH alone & $-0.1168$ & 0.1006 & 0.245 & 926 & 161.4 & 0.196 \\ Pressure alone & $-0.1140$ & 0.1580 & 0.470 & 926 & 56.1 & 0.259 \\ Controls excl.\ cloud, radiation & $-0.0667$ & 0.0881 & 0.449 & 926 & 47.0 & 0.147 \\ Excluding January 2022 unrest & $-0.1019$ & 0.0917 & 0.267 & 920 & 44.9 & 0.175 \\ Dense coverage (Apr 2023 onward) & $-0.1953$ & 0.1558 & 0.210 & 505 & 23.9 & 0.313 \\ Post-2024 gap (Jul 2024 onward) & $-0.3735$ & 0.2561 & 0.145 & 327 & 19.7 & 0.551 \\ Lag 1 day & $-0.0802$ & 0.0642 & 0.212 & 925 & 55.6 & 0.129 \\ Lag 2 days & $-0.0338$ & 0.0732 & 0.644 & 924 & 55.2 & 0.107 \\ Lag 3 days & $-0.0381$ & 0.0653 & 0.560 & 923 & 56.7 & 0.101 \\ Outcome: absolute return & $-0.0356$ & 0.0600 & 0.554 & 926 & 44.9 & 0.093 \\ Temperature in 2.5\textdegree{} bins & $-0.1338$ & 0.0873 & 0.125 & 926 & 49.2 & 0.192 \\ Cubic spline in temperature & $-0.0961$ & 0.0837 & 0.251 & 926 & 49.0 & 0.162 \\ Year-by-month fixed effects & $-0.0674$ & 0.0961 & 0.483 & 926 & 64.0 & 0.156 \\ Treatment: session only (11:00--17:59) & $-0.0885$ & 0.0844 & 0.294 & 926 & 47.2 & 0.158 \\ Treatment: pre-session only (07:00--10:59) & $-0.1525$ & 0.1224 & 0.213 & 926 & 17.5 & 0.245 \\ Treatment: calendar-day mean & $-0.0990$ & 0.1138 & 0.385 & 926 & 32.4 & 0.198 \\""",
 T6, 1),

("T6 note",
 r"""Three rows fall at or below their threshold---the pre-session treatment (17.5), the post-gap subsample (19.7) and the dense-coverage subsample (23.9)---and their estimates should be read as weakly identified.""",
 r"""Every row clears its threshold; the weakest is the pre-session treatment at 25.9 against a critical value of 24.58.""", 1),

# ---- 5.6 opening: "negative in every specification" is now false ----
("5.6 open",
 r"""The estimate is negative in every specification and significant in none. It is insensitive to the within-day coverage threshold governing sample inclusion, to sample window, to winsorisation, to the construction of the exposure window, and to the exclusion of the January 2022 unrest period.""",
 r"""The estimate is significant in no specification, and negative in every specification other than the two- and three-day lags, where it is essentially zero. It is insensitive to the within-day coverage threshold governing sample inclusion, to sample window, to winsorisation, to the construction of the exposure window, and to the exclusion of the January 2022 unrest period.""", 1),

# ---- 5.6 the eight results ------------------------------------------
("5.6 r1 cloud",
 r"""Removing them changes the coefficient from $-0.105$ to $-0.067$, well within one standard error, and the robust first-stage $F$ rises from 44.9 to 47.0.""",
 r"""Removing them changes the coefficient from $-0.083$ to $-0.039$, well within one standard error, and the robust first-stage $F$ falls from 84.0 to 73.8.""", 1),
("5.6 r1 tail",
 r"""It also establishes that including these controls costs little identifying variation: the first stage remains strong either way.""",
 r"""It also establishes that including these controls costs no identifying variation: the first stage is stronger with them than without.""", 1),
("5.6 r2 window",
 r"""Constructing the treatment over the trading session alone yields $-0.089$, over the pre-session hours alone $-0.153$, and over the full calendar day $-0.099$. The three alternatives correlate with the preferred measure at 0.98, 0.89 and 0.98 in logs. The pre-session construction has a materially weaker first stage (robust $F = 17.5$, below the Stock--Yogo threshold), as fewer hours of averaging would predict; we report it for completeness and do not rely on it. The calendar-day construction, which averages over all twenty-four hours including the post-close evening peak, gives $-0.099$.""",
 r"""Constructing the treatment over the trading session alone yields $-0.071$, over the pre-session hours alone $-0.115$, and over the full calendar day $-0.079$. The three alternatives correlate with the preferred measure at 0.985, 0.884 and 0.975 in logs. The pre-session construction has a materially weaker first stage (robust $F = 25.9$, barely clearing the Stock--Yogo threshold), as fewer hours of averaging would predict; we report it for completeness and do not rely on it. The calendar-day construction, which averages over all twenty-four hours including the post-close evening peak, gives $-0.079$.""", 1),
("5.6 r3 weather",
 r"""Replacing it with the 2.5\textdegree{} bins the benchmark uses gives $-0.134$ (SE $0.087$, bound 0.192); a cubic spline gives $-0.096$ (SE $0.084$, bound 0.162). Replacing month and year fixed effects with 57 year-by-month effects, which absorb any smooth seasonal confounder within each individual month of the sample, gives $-0.067$ (SE $0.096$, bound 0.156). All three are negative, none is significant, and all bounds lie between 0.156 and 0.192.""",
 r"""Replacing it with the 2.5\textdegree{} bins the benchmark uses gives $-0.110$ (SE $0.081$, bound 0.168); a cubic spline gives $-0.076$ (SE $0.082$, bound 0.146). Replacing month and year fixed effects with 63 year-by-month effects, which absorb any smooth seasonal confounder within each individual month of the sample, gives $-0.104$ (SE $0.092$, bound 0.176). All three are negative, none is significant, and all bounds lie between 0.146 and 0.176.""", 1),
("5.6 r4 winter",
 r"""It gives $-0.081$ (SE $0.113$) on 360 observations with a bound of 0.185---close to the full-sample estimate and only slightly wider, despite using 39 per cent of the observations.""",
 r"""It gives $-0.084$ (SE $0.098$) on 415 observations with a bound of 0.169---close to the full-sample estimate and only slightly wider, despite using 40 per cent of the observations.""", 1),
("5.6 r5 density",
 r"""Restricting to the densest portions yields $-0.195$ (SE $0.156$) on 505 observations from April 2023 onward, and $-0.374$ (SE $0.256$) on 327 observations from July 2024, when the boundary layer series resumes.""",
 r"""Restricting to the densest portions yields $-0.125$ (SE $0.112$) on 621 observations from April 2023 onward, and $-0.189$ (SE $0.213$) on 327 observations from July 2024.""", 1),
("5.6 r5 drift",
 r"""We note that the point estimate drifts more negative as the estimation window narrows---from $-0.105$ on the full sample, to $-0.139$ from 2022, $-0.166$ from 2023, $-0.195$ from April 2023, and $-0.374$ from July 2024. Three readings are available: that later years genuinely exhibit a stronger effect which the full sample averages away; that shorter samples produce noisier estimates which happen to run negative; or that the earlier period, with its coverage gaps and pandemic disruption, attenuates the full-sample estimate. The confidence intervals overlap heavily and the estimates are not statistically distinguishable from one another---the difference between $-0.105$ and $-0.374$ is within one standard error of the smaller sample---and the two narrowest subsamples have robust first-stage $F$ statistics of 23.9 and 19.7, at or below the Stock--Yogo threshold, so their point estimates carry the additional unreliability that weak identification implies. We therefore do not interpret this as evidence of a stronger effect in later years. But we cannot exclude it either, and we report the pattern rather than only the specifications closest to our headline.""",
 r"""We note that the point estimate runs somewhat more negative on narrower windows---$-0.083$ on the full sample against $-0.134$ from 2022, $-0.123$ from 2023, $-0.125$ from April 2023 and $-0.189$ from July 2024. The pattern is not monotonic and the confidence intervals overlap heavily: the difference between $-0.083$ and $-0.189$ is within one standard error of the smaller sample. We do not interpret it as evidence of a stronger effect in later years, and report it rather than only the specifications closest to our headline.""", 1),
("5.6 r6 lags",
 r"""At one-, two- and three-day lags the exclusion bounds are 0.129, 0.107 and 0.101 percentage points respectively. The benchmark's recovery estimate of 0.0150 per \ugm{} implies 0.487 percentage points per doubling under linear transport at our mean and 0.173 under proportional transport. Our one-day-lag bound of 0.129 excludes both.""",
 r"""At one-, two- and three-day lags the exclusion bounds are 0.078, 0.092 and 0.068 percentage points respectively. The benchmark's recovery estimate of 0.0150 per \ugm{} implies 0.484 percentage points per doubling under linear transport at our mean and 0.173 under proportional transport. Our one-day-lag bound of 0.078 excludes both.""", 1),
("5.6 r7 vol",
 r"""With absolute returns as outcome, the exclusion bound is 0.093 percentage points.""",
 r"""With absolute returns as outcome, the exclusion bound is 0.080 percentage points.""", 1),
("5.6 r8 thresh",
 r"""None attains significance at conventional levels; the ordinary least squares estimates are not signed in the direction the hypothesis predicts, and instrumenting reverses their sign. The instrumented coefficients grow with the cut-off, mirroring the sign change between ordinary least squares and instrumental variables in the main specification, but the two higher cut-offs are weakly identified and we do not read a dose--response pattern into them.""",
 r"""None attains significance at conventional levels, by either estimator. The instrumented coefficients grow with the cut-off, but the first stage weakens sharply as the cut-off rises and we do not read a dose--response pattern into them.""", 1),

# ---- threshold table -------------------------------------------------
("thresh rows",
 r"""Above 35 \ugm{} & 0.009 & (0.069) & $-0.239$ & (0.167) \\ Above 55 \ugm{} & 0.058 & (0.079) & $-0.335$ & (0.232) \\ Above 75 \ugm{} & 0.079 & (0.085) & $-0.486$ & (0.381) \\""",
 r"""Above 35 \ugm{} & $-0.017$ & (0.065) & $-0.203$ & (0.174) \\
Above 55 \ugm{} & 0.044 & (0.070) & $-0.249$ & (0.205) \\
Above 75 \ugm{} & $-0.006$ & (0.081) & $-0.390$ & (0.311) \\""", 1),
("thresh note",
 r"""Robust first-stage $F$ statistics for the instrumented columns are 32.5, 18.3 and 8.5, so the two higher cut-offs are weakly identified.""",
 r"""Robust first-stage $F$ statistics for the instrumented columns are 39.0, 18.8 and 9.5, so the two higher cut-offs are weakly identified.""", 1),

# ---- the five Part E values -----------------------------------------
("T4 ols p",
 r"""$p$ & [RERUN] & 0.320 & 0.321 \\""",
 r"""$p$ & 0.802 & 0.320 & 0.321 \\""", 1),
("4.7 lags",
 r"""For ordinary least squares the standard error is 0.036 with no correction, 0.038 at five lags and 0.032 at twenty. For the preferred instrumented specification the standard error is 0.094 under a robust covariance with no autocorrelation correction, 0.096 at five lags and 0.086 at twenty, against the 0.092 we report; the corresponding exclusion bounds are 0.180, 0.182 and 0.171 against our 0.177.""",
 r"""For ordinary least squares the standard error is 0.034 with no correction, 0.035 at five lags and 0.030 at twenty. For the preferred instrumented specification the standard error is 0.082 under a robust covariance with no autocorrelation correction, 0.082 at five lags and 0.080 at twenty, against the 0.083 we report; the corresponding exclusion bounds are 0.150, 0.151 and 0.149 against our 0.152.""", 1),
("fn5 overid",
 r"""Wooldridge's score test of overidentification \citep{wooldridge2010} gives $p = 0.256$, Basmann's \citeyearpar{basmann1960} $p = 0.197$ and the Anderson--Rubin overidentification test $p = 0.184$.""",
 r"""Wooldridge's score test of overidentification \citep{wooldridge2010} gives $p = 0.337$, Basmann's \citeyearpar{basmann1960} $p = 0.294$ and the Anderson--Rubin overidentification test $p = 0.281$.""", 1),
("T5 fn 0171",
 r"""the implications become 0.197 and 0.555 and the test statistics 1.38 and 2.54: both conclusions are unchanged.""",
 r"""the implications become 0.197 and 0.552 and the test statistics 1.63 and 2.65: both conclusions are unchanged.""", 1),
("6.3 sargan variants",
 r"""dropping the north--south wind component raises the $p$-value to 0.641; retaining only the two dispersion instruments raises it to 0.985.""",
 r"""dropping the north--south wind component raises the $p$-value to 0.774; retaining only the two dispersion instruments raises it to 0.951.""", 1),

# ---- 5.5 fragility: the first half of the argument is now false -----
("5.5 fragility",
 r"""The bound of 0.177 percentage points lies below the benchmark's proportional implication of 0.194. We caution against reading this as a rejection, for two reasons. The first is that the margin does not survive across specifications or inference procedures. Our just-identified specification using boundary layer height alone---the instrument with the most direct mechanical interpretation---yields a bound of 0.196, which lies above the benchmark's implication rather than below it (Table~\ref{tab:robust}). The bound is also constructed from a normal critical value; \citet{lee2022} show that $t$-ratio inference in instrumental variables models can be materially distorted and propose an adjustment that is a smooth function of the first-stage $F$-statistic. Their procedure is defined for the just-identified single-instrument case and so does not apply directly to our four-instrument preferred specification, but it illustrates the general point: inflating the effective standard error by five, ten and twenty per cent moves our bound to 0.183, 0.188 and 0.198 respectively. The ordering of 0.177 against 0.194 is therefore an artefact of one specification estimated under one inference procedure, and we do not treat it as a finding.""",
 r"""The bound of 0.152 percentage points lies below the benchmark's proportional implication of 0.194, and does so robustly. The just-identified specification using boundary layer height alone yields 0.165, the dispersion-only pair 0.163, and no specification in Table~\ref{tab:robust} that we would defend as our preferred design puts the bound above 0.194. The bound is constructed from a normal critical value, and \citet{lee2022} show that $t$-ratio inference in instrumental variables models can be materially distorted; their procedure is defined for the just-identified single-instrument case and so does not apply directly to our four-instrument specification, but inflating the effective standard error by five, ten and twenty per cent moves our bound only to 0.157, 0.162 and 0.171, all still below 0.194.

We nonetheless caution against reading this as a rejection, for a reason that has nothing to do with robustness. Excluding a point value is a weaker statement than rejecting equality of two estimated parameters: the equality test carries the benchmark's own standard error of 0.063, and it does not reject ($t = 1.60$, Table~\ref{tab:transport}). The bound answers which effects our data rule out; the transport test answers whether the two estimates differ. On the first question the benchmark's proportional implication now falls outside our bound; on the second the two estimates remain statistically indistinguishable, and the second is the appropriate comparison between uncertain quantities.""", 1),
("5.5 orphan",
 r"""The second reason is more basic. Excluding a point value is a weaker statement than rejecting equality of two estimated parameters: the equality test carries the benchmark's own standard error of 0.063, and it does not reject ($t = 1.60$, Table~\ref{tab:transport}). The bound answers which effects our data rule out; the transport test answers whether the two estimates differ. On the first question the benchmark's proportional implication falls marginally outside our bound; on the second the two estimates remain statistically indistinguishable, and the second is the appropriate comparison between uncertain quantities.

""",
 "", 1),
]


def flex(text, find, repl, label, count, report):
    pat = re.compile(r"\s+".join(re.escape(t) for t in find.split()))
    hits = list(pat.finditer(text))
    if len(hits) == count:
        for m in reversed(hits):
            text = text[:m.start()] + repl + text[m.end():]
        report.append(f"  OK       {label}")
        return text, True
    report.append(f"  {'MISSING ' if not hits else 'COUNT   '}{label}"
                  f"  (found {len(hits)}, expected {count})")
    return text, False


def main():
    p = Path(SRC)
    if not p.exists():
        sys.exit(f"not found: {SRC} -- run apply_rebuild.py first")
    t, rep, ok = p.read_text(encoding="utf-8"), [], 0
    for label, f, r, c in E:
        t, good = flex(t, f, r, label, c, rep)
        ok += good
    Path(OUT).write_text(t, encoding="utf-8")
    print("\n".join(rep))
    print(f"\n{ok}/{len(E)} applied.  Wrote {OUT}.")
    print("""
CHECK BEFORE COMPILING
  * Section 5.5 now runs as two paragraphs. Read them together -- the
    argument changed shape, not just its numbers.
  * The Pressure alone p-value of 0.489 was derived from the coefficient
    and SE, not taken from regression output. Verify it.
  * Search the file for "926", "0.177", "44.9", "0.105" and "0.215" --
    any survivor is something both scripts missed.
""")


if __name__ == "__main__":
    main()

  OK       T6 all rows
  OK       T6 note
  OK       5.6 open
  OK       5.6 r1 cloud
  OK       5.6 r1 tail
  OK       5.6 r2 window
  OK       5.6 r3 weather
  OK       5.6 r4 winter
  OK       5.6 r5 density
  OK       5.6 r5 drift
  OK       5.6 r6 lags
  OK       5.6 r7 vol
  OK       5.6 r8 thresh
  OK       thresh rows
  OK       thresh note
  OK       T4 ols p
  OK       4.7 lags
  OK       fn5 overid
  OK       T5 fn 0171
  OK       6.3 sargan variants
  OK       5.5 fragility
  OK       5.5 orphan

22/22 applied.  Wrote main_final.tex.

CHECK BEFORE COMPILING
  * Section 5.5 now runs as two paragraphs. Read them together -- the
    argument changed shape, not just its numbers.
  * The Pressure alone p-value of 0.489 was derived from the coefficient
    and SE, not taken from regression output. Verify it.
  * Search the file for "926", "0.177", "44.9", "0.105" and "0.215" --
    any survivor is something both scripts missed.



In [50]:
t = open("main_final.tex").read()
for s in ["926", "0.177", "44.9", "0.105", "0.215", "0.858", "1,303",
          "0.0915", "0.1053", "0.268", "925 circular"]:
    n = t.count(s)
    if n:
        print(f"\n=== {s!r}: {n} ===")
        for m in __import__("re").finditer(__import__("re").escape(s), t):
            print("   ...", t[max(0,m.start()-70):m.end()+70].replace("\n"," "))


=== '926': 3 ===
   ... (m/s) & 5.51 & 1.71 & 1.33 & 5.40 & 12.68 \\ Surface pressure (hPa) & 926.28 & 5.71 & 913.30 & 925.72 & 948.79 \\ Cloud cover (\%) & 51.41 & 32
   ... bservation coverage filter described in Section~\ref{sec:sample}. The 926-day estimation sample is a subset of these, with a mean of 32.29 \ugm
   ... bel{sec:results}  \subsection{Sample}  Descriptive statistics for the 926-day estimation sample are reported in Table~\ref{tab:descriptives}. T

=== '0.177': 1 ===
   ... --including ours.  \textbf{Statistical power.} Our exclusion bound of 0.177 percentage points sits close to the benchmark's proportional implicat

=== '0.105': 1 ===
   ... t leaves our conclusions intact: dropping $v$ moves the estimate to $-0.105$ with a bound of 0.167, dropping $u$ to $-0.077$ with a bound of 0.14

=== '1,303': 1 ===
   ... n-Meteo series runs systematically higher in its upper tail, reaching 1,303 metres against 995 in the Climate Data Store, and a regression of one


In [51]:
import re
t = open("main_final.tex").read()

t = t.replace("The 926-day estimation sample is a subset of these",
              "The 1,042-day estimation sample is a subset of these")
t = t.replace("Descriptive statistics for the 926-day estimation sample",
              "Descriptive statistics for the 1,042-day estimation sample")

open("main_final.tex","w").write(t)
print("two fixed\n")

m = re.search(r"\\textbf\{Statistical power\.\}.*?(?=\n\n|\\textbf\{)", t, re.S)
print(m.group(0) if m else "not found")

two fixed

\textbf{Statistical power.} Our exclusion bound of 0.177 percentage points sits close to the benchmark's proportional implication, so our design is at the threshold of informativeness for effects of that size and uninformative for anything smaller. An effect of the magnitude documented elsewhere may exist here and lie beyond our resolution. We regard this as a genuine possibility rather than a defensive fallback.


In [52]:
old = r"""\textbf{Statistical power.} Our exclusion bound of 0.177 percentage points sits close to the benchmark's proportional implication, so our design is at the threshold of informativeness for effects of that size and uninformative for anything smaller. An effect of the magnitude documented elsewhere may exist here and lie beyond our resolution. We regard this as a genuine possibility rather than a defensive fallback."""

new = r"""\textbf{Statistical power.} Our exclusion bound of 0.152 percentage points lies below the benchmark's proportional implication of 0.194, so our design can resolve an effect of that magnitude. It cannot resolve much less. The multi-city summary estimate discussed in Section~\ref{sec:transport} corresponds to roughly 0.022 percentage points per doubling at our sample mean---about a seventh of our bound---and an effect of that size would sit well inside our confidence interval. What we can say is that Almaty's traders do not respond to pollution as strongly as the benchmark's proportional implication requires; what we cannot say is that they do not respond at all. We regard the smaller effect as a genuine possibility rather than a defensive fallback."""

t = open("main_final.tex").read()
assert t.count(old) == 1, f"found {t.count(old)}"
open("main_final.tex","w").write(t.replace(old, new))
print("done")

done


In [36]:
import glob
print(glob.glob("*.tex"))

[]


In [37]:
import os
os.rename("main 4.tex", "main.tex")
print("renamed")


FileNotFoundError: [Errno 2] No such file or directory: 'main 4.tex' -> 'main.tex'

In [39]:
import os
os.rename("main 4.tex", "main.tex")
print("renamed")

FileNotFoundError: [Errno 2] No such file or directory: 'main 4.tex' -> 'main.tex'

In [44]:
import glob
print(glob.glob("*.tex"))
print(glob.glob("*"))

[]
['drive', 'almaty_pm25_daily_window.csv', 'kase_official.csv', 'panel_final.csv', 'blh_cds_both_constructions.csv', 'panel_cds.csv', 'blh_cds_daily_nearest.csv', 'panel_window_final.csv', 'ri_draws_cds.npy', 'almaty_pm25_daily.csv', 'panel_analysis.csv', 'blh_cds_daily_bilinear.csv', 'panel_official.csv', 'almaty_weather_daily.csv', 'almaty_weather_daily_cds.csv', 'almaty_pm25_hourly.csv', 'kase_clean.csv', 'KASE Historical Data.csv', 'sample_data']


In [45]:
import glob, os
print("session:", [f for f in os.listdir('.') if not f.startswith('.')])
print("\nany .tex on Drive:")
for f in glob.glob("/content/drive/MyDrive/**/*.tex", recursive=True):
    print("   ", f)

session: ['drive', 'almaty_pm25_daily_window.csv', 'kase_official.csv', 'panel_final.csv', 'blh_cds_both_constructions.csv', 'panel_cds.csv', 'blh_cds_daily_nearest.csv', 'panel_window_final.csv', 'ri_draws_cds.npy', 'almaty_pm25_daily.csv', 'panel_analysis.csv', 'blh_cds_daily_bilinear.csv', 'panel_official.csv', 'almaty_weather_daily.csv', 'almaty_weather_daily_cds.csv', 'almaty_pm25_hourly.csv', 'kase_clean.csv', 'KASE Historical Data.csv', 'sample_data']

any .tex on Drive:
    /content/drive/MyDrive/pollution_research/main 4.tex


In [46]:
import shutil
shutil.copy("/content/drive/MyDrive/pollution_research/main 4.tex", "main.tex")
t = open("main.tex").read()
print("Tier 1 present:", "nonetheless a transitory" in t)
print("old panel   :", "926 trading days" in t)
print("lines       :", t.count("\n"))

Tier 1 present: True
old panel   : True
lines       : 882


In [ ]:
"""
REMAINING CELLS — run after rebuild_window_final.py, in the same session
(or reload panel_window_final.csv + almaty_pm25_daily_window.csv first).
Computes everything the edit list tags [RUN].
"""
import numpy as np, pandas as pd, warnings
from linearmodels.iv import IV2SLS
import statsmodels.formula.api as smf
from scipy import stats
warnings.filterwarnings("ignore")

LN2 = np.log(2)
RNG = np.random.default_rng(20260806)

D   = pd.read_csv("panel_window_final.csv"); D["date"] = pd.to_datetime(D["date"])
out = pd.read_csv("almaty_pm25_daily_window.csv"); out["date"] = pd.to_datetime(out["date"])
wx  = pd.read_csv("almaty_weather_daily.csv"); wx["date"] = pd.to_datetime(wx["date"])
ks  = pd.read_csv("kase_official.csv");        ks["date"] = pd.to_datetime(ks["date"])

INST = ["log_blh", "u", "v", "pressure"]
keep = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_"))
        and D[c].nunique() > 1] + \
       [c for c in ["temp", "precip", "wind_speed", "cloud", "radiation",
                    "lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]
# drop collinear, same rule as the main script
kk = []
for c in keep:
    M = D[["const"] + kk + [c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]: kk.append(c)
keep = kk

# ------------------------------------------------ 1. exact statistics
r = IV2SLS(D["logret"], D[["const"] + keep], D["log_pm25"], D[INST]).fit(cov_type="kernel")
print("Sargan statistic:", round(float(r.sargan.stat), 3), " p:", round(float(r.sargan.pval), 4))
wh = r.wu_hausman()
print("Wu-Hausman stat:", round(float(wh.stat), 3), " p:", round(float(wh.pval), 4))
ols = smf.ols("logret ~ log_pm25 + " + " + ".join(keep), data=D
              ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print("OLS p-value:", round(ols.pvalues["log_pm25"], 3))
print("OLS SE by HAC lags: ",
      {L: round(smf.ols("logret ~ log_pm25 + " + " + ".join(keep), data=D
        ).fit(cov_type="HAC", cov_kwds={"maxlags": L}).bse["log_pm25"], 4)
       for L in (0, 5, 20)})

# ------------------------------------------------ 2. per-instrument just-identified
print("\nPER-INSTRUMENT JUST-IDENTIFIED IV")
for z in INST:
    rz = IV2SLS(D["logret"], D[["const"] + keep], D["log_pm25"], D[[z]]
                ).fit(cov_type="kernel")
    fz = smf.ols("log_pm25 ~ " + " + ".join([z] + keep), data=D
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    Fz = float(fz.tvalues[z] ** 2)
    print(f"  {z:<10} b={rz.params['log_pm25']:8.3f}  SE={rz.std_errors['log_pm25']:7.3f}"
          f"   just-ID first-stage F={Fz:6.1f}")

# ------------------------------------------------ 3. Anderson-Rubin set, v only
print("\nANDERSON-RUBIN 95% SET (v as sole instrument) — grid inversion")
accept = []
Xc = " + ".join(keep)
for b0 in np.arange(-4.0, 8.001, 0.02):
    Dt = D.copy(); Dt["resid_y"] = Dt.logret - b0 * Dt.log_pm25
    ar = smf.ols("resid_y ~ v + " + Xc, data=Dt
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    if ar.pvalues["v"] > 0.05: accept.append(b0)
if accept:
    # report contiguous runs (AR sets can be unions with weak instruments)
    runs, start = [], accept[0]
    for a, b_ in zip(accept, accept[1:] + [None]):
        if b_ is None or b_ - a > 0.03:
            runs.append((start, a)); start = b_
    print("  ", " U ".join(f"[{lo:.2f}, {hi:.2f}]" for lo, hi in runs))
else:
    print("  empty on [-4, 8]")

# ------------------------------------------------ 4. uranium exclusion test
# Needs a URA daily returns file: columns date, ura_ret (percent log returns).
try:
    ura = pd.read_csv("ura_returns.csv"); ura["date"] = pd.to_datetime(ura["date"])
    Du = D.merge(ura, on="date", how="inner")
    print(f"\nURANIUM TEST (N = {len(Du)})")
    ru = smf.ols("ura_ret ~ " + " + ".join(INST + keep), data=Du
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print("  instruments -> URA joint p:",
          round(float(ru.f_test(", ".join(f"{i} = 0" for i in INST)).pvalue), 3))
    for z in INST:
        print(f"    {z:<10}{ru.params[z]:8.4f} ({ru.bse[z]:.4f})  p={ru.pvalues[z]:.3f}")
    rk = smf.ols("logret ~ ura_ret + " + " + ".join(keep), data=Du
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  URA -> KASE: {rk.params['ura_ret']:.4f} ({rk.bse['ura_ret']:.4f})"
          f"  p={rk.pvalues['ura_ret']:.3f}"
          f"   corr={Du.logret.corr(Du.ura_ret):.3f}")
    rf0 = smf.ols("logret ~ " + " + ".join(INST + keep), data=Du
                  ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    rf1 = smf.ols("logret ~ ura_ret + " + " + ".join(INST + keep), data=Du
                  ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  BLH reduced-form coef without/with URA control: "
          f"{rf0.params['log_blh']:.4f} ({rf0.bse['log_blh']:.4f}) -> "
          f"{rf1.params['log_blh']:.4f} ({rf1.bse['log_blh']:.4f})")
    print(f"  joint exclusion p without/with URA: "
          f"{float(rf0.f_test(', '.join(f'{i} = 0' for i in INST)).pvalue):.3f} -> "
          f"{float(rf1.f_test(', '.join(f'{i} = 0' for i in INST)).pvalue):.3f}")
except FileNotFoundError:
    print("\nURANIUM TEST skipped: ura_returns.csv not found. Export the URA "
          "series you used before (date, ura_ret in % log returns) and rerun.")

# ------------------------------------------------ 5. placebo percentiles in RI null
print("\nPLACEBO PERCENTILES IN THE RANDOMISATION NULL")
exog, endog, instr = D[["const"] + keep], D["log_pm25"], D[INST]
n = len(D); draws = []
for _ in range(1000):
    s = 5 * int(RNG.integers(1, n // 5))
    yb = pd.Series(np.roll(D.logret.values, s), index=D.index)
    draws.append(IV2SLS(yb, exog, endog, instr
                        ).fit(cov_type="unadjusted").params["log_pm25"])
draws = np.array(draws)
def pct(x): return stats.percentileofscore(draws, x)
b0 = IV2SLS(D["logret"], exog, endog, instr).fit(cov_type="kernel").params["log_pm25"]
print(f"  contemporaneous {b0:8.4f}  pct {pct(b0):5.1f}")
for k in (1, 2, 3, 5):
    Dk = D.copy(); Dk["lead"] = Dk.logret.shift(-k)
    Dk = Dk.dropna(subset=["lead"]).reset_index(drop=True)
    bk = IV2SLS(Dk["lead"], Dk[["const"] + keep], Dk["log_pm25"], Dk[INST]
                ).fit(cov_type="kernel").params["log_pm25"]
    print(f"  {k}-day lead    {bk:8.4f}  pct {pct(bk):5.1f}")
D1 = D.copy(); D1["lead"] = D1.logret.shift(-1); D1["lag_pm"] = D1.log_pm25.shift(1)
D1 = D1.dropna(subset=["lead", "lag_pm"]).reset_index(drop=True)
b1 = IV2SLS(D1["lead"], D1[["const"] + keep + ["lag_pm"]], D1["log_pm25"], D1[INST]
            ).fit(cov_type="kernel").params["log_pm25"]
print(f"  lead+lagPM     {b1:8.4f}  pct {pct(b1):5.1f}")

# ------------------------------------------------ 6. S3.4 loss accounting
print("\nS3.4 LOSS ACCOUNTING (weekday trading days, 2020-04-08 to 2025-11-13)")
T = ks[(ks.date >= "2020-04-08") & (ks.date <= "2025-11-13")]
T = T[T.date.dt.dayofweek < 5].dropna(subset=["logret"])[["date"]]
T = T.merge(out[["date", "n_window"]], on="date", how="left")
T = T.merge(wx[["date", "blh", "u", "v", "pressure", "temp", "precip",
                "wind_speed", "cloud", "radiation"]], on="date", how="left")
T["retained"] = T.date.isin(D.date)
lost = T[~T.retained].copy()
no_pm   = lost.n_window.isna()
low_pm  = (~no_pm) & (lost.n_window < 8)
no_blh  = (~no_pm) & (~low_pm) & lost.blh.isna()
no_wx   = (~no_pm) & (~low_pm) & (~no_blh) & lost[
    ["u", "v", "pressure", "temp", "precip", "wind_speed", "cloud", "radiation"]
].isna().any(axis=1)
other   = (~no_pm) & (~low_pm) & (~no_blh) & (~no_wx)
print(f"  total weekday trading days {len(T)}   retained {int(T.retained.sum())}"
      f"   lost {len(lost)}")
print(f"    no pollution obs at all:      {int(no_pm.sum())}")
print(f"    in-window coverage < 8:       {int(low_pm.sum())}")
print(f"    boundary layer missing:       {int(no_blh.sum())}")
print(f"    other weather missing:        {int(no_wx.sum())}")
print(f"    unexplained (investigate):    {int(other.sum())}")
if other.sum():
    print(lost[other][["date"]].head(15).to_string(index=False))


Sargan statistic: 4.822  p: 0.1853
Wu-Hausman stat: 3.327  p: 0.0685
OLS p-value: 0.739
OLS SE by HAC lags:  {0: np.float64(0.0361), 5: np.float64(0.0377), 20: np.float64(0.0315)}

PER-INSTRUMENT JUST-IDENTIFIED IV
  log_blh    b=  -0.117  SE=  0.101   just-ID first-stage F= 161.4
  u          b=  -0.646  SE=  0.630   just-ID first-stage F=   4.7
  v          b=   3.701  SE=  6.164   just-ID first-stage F=   0.4
  pressure   b=  -0.114  SE=  0.158   just-ID first-stage F=  56.1

ANDERSON-RUBIN 95% SET (v as sole instrument) — grid inversion
   [-4.00, -0.66] U [0.08, 8.00]

URANIUM TEST skipped: ura_returns.csv not found. Export the URA series you used before (date, ura_ret in % log returns) and rerun.

PLACEBO PERCENTILES IN THE RANDOMISATION NULL
  contemporaneous  -0.1053  pct  14.1
  1-day lead     -0.0840  pct  18.4
  2-day lead     -0.0724  pct  20.0
  3-day lead     -0.0727  pct  20.0
  5-day lead      0.0666  pct  76.3
  lead+lagPM      -0.1153  pct   9.5

S3.4 LOSS ACCOUNTING 

In [ ]:
"""
COMMODITY CHANNEL TESTS (S4.5) — uranium and oil, on the window panel.
Upload the raw Investing.com files under their download names:
  URA_ETF_Stock_Price_History.csv
  Brent Oil Futures Historical Data.csv   (or any name containing 'rent')
Requires panel_window_final.csv in the working directory.
"""
import numpy as np, pandas as pd, glob, warnings
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore")

D = pd.read_csv("panel_window_final.csv"); D["date"] = pd.to_datetime(D["date"])
INST = ["log_blh", "u", "v", "pressure"]
keep = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_")) and D[c].nunique() > 1] + \
       [c for c in ["temp", "precip", "wind_speed", "cloud", "radiation",
                    "lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]
kk = []
for c in keep:
    M = D[["const"] + kk + [c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]: kk.append(c)
keep = kk

def to_returns(path, name):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
    df["close"] = df["Price"].astype(str).str.replace(",", "").astype(float)
    df = df.sort_values("date").reset_index(drop=True)
    n_wk = (df.date.dt.dayofweek >= 5).sum()
    df = df[df.date.dt.dayofweek < 5].reset_index(drop=True)   # drop spurious weekend rows FIRST
    df[name] = 100 * np.log(df.close / df.close.shift(1))
    print(f"{name}: {len(df)} weekday rows ({n_wk} weekend rows dropped), "
          f"{df.date.min().date()} to {df.date.max().date()}")
    return df[["date", name]].dropna()

def channel_test(ret_df, name):
    Du = D.merge(ret_df, on="date", how="inner")
    print(f"\n{name.upper()} CHANNEL TEST (N = {len(Du)})")
    ru = smf.ols(f"{name} ~ " + " + ".join(INST + keep), data=Du
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print("  instruments -> commodity, joint p:",
          round(float(ru.f_test(", ".join(f"{i} = 0" for i in INST)).pvalue), 3))
    for z in INST:
        print(f"    {z:<10}{ru.params[z]:8.4f} ({ru.bse[z]:.4f})  p={ru.pvalues[z]:.3f}")
    rk = smf.ols(f"logret ~ {name} + " + " + ".join(keep), data=Du
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  commodity -> KASE: {rk.params[name]:.4f} ({rk.bse[name]:.4f})"
          f"  p={rk.pvalues[name]:.3f}   corr={Du.logret.corr(Du[name]):.3f}")
    rf0 = smf.ols("logret ~ " + " + ".join(INST + keep), data=Du
                  ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    rf1 = smf.ols(f"logret ~ {name} + " + " + ".join(INST + keep), data=Du
                  ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  BLH reduced form without/with control: "
          f"{rf0.params['log_blh']:.4f} ({rf0.bse['log_blh']:.4f}) -> "
          f"{rf1.params['log_blh']:.4f} ({rf1.bse['log_blh']:.4f})")
    print(f"  joint exclusion p without/with: "
          f"{float(rf0.f_test(', '.join(f'{i} = 0' for i in INST)).pvalue):.3f} -> "
          f"{float(rf1.f_test(', '.join(f'{i} = 0' for i in INST)).pvalue):.3f}")

channel_test(to_returns("URA_ETF_Stock_Price_History.csv", "ura_ret"), "ura_ret")

brent = [f for f in glob.glob("*.csv") if "rent" in f]
if brent:
    channel_test(to_returns(brent[0], "brent_ret"), "brent_ret")
else:
    print("\nBrent file not found — upload it to run the oil-channel test.")


ura_ret: 1650 weekday rows (26 weekend rows dropped), 2020-01-02 to 2026-07-24

URA_RET CHANNEL TEST (N = 895)
  instruments -> commodity, joint p: 0.079
    log_blh    -0.0149 (0.2298)  p=0.948
    u           0.0964 (0.0465)  p=0.038
    v          -0.0496 (0.0398)  p=0.213
    pressure    0.0435 (0.0271)  p=0.108
  commodity -> KASE: 0.0189 (0.0100)  p=0.060   corr=0.073
  BLH reduced form without/with control: 0.0662 (0.0526) -> 0.0665 (0.0521)
  joint exclusion p without/with: 0.136 -> 0.138
brent_ret: 1704 weekday rows (0 weekend rows dropped), 2020-01-02 to 2026-08-06

BRENT_RET CHANNEL TEST (N = 918)
  instruments -> commodity, joint p: 0.698
    log_blh    -0.0236 (0.1563)  p=0.880
    u           0.0225 (0.0518)  p=0.665
    v           0.0550 (0.0480)  p=0.252
    pressure    0.0086 (0.0248)  p=0.729
  commodity -> KASE: 0.0088 (0.0145)  p=0.544   corr=0.044
  BLH reduced form without/with control: 0.0591 (0.0526) -> 0.0593 (0.0525)
  joint exclusion p without/with: 0.145 ->

In [ ]:
print(np.log(D.pm25_raw).autocorr(1), (D.pm25_raw >75.86).sum())

0.7179326886930174 96


In [ ]:
"""
REGENERATE ALL FIVE FIGURES on the window panel.
Run after rebuild_window_final.py, in the same directory (needs
panel_window_final.csv and almaty_pm25_daily_window.csv).

Saves fig1_timeseries.png ... fig5_specification.png at 300 dpi with the
exact filenames main_revised.tex expects.

Also prints, at the end, every number the captions and the %VERIFY markers
need: BLH monthly means, sample endpoints, the Figure 3 bivariate slope,
and the Figure 4 zero-containing-interval count. Paste that block back.
"""
import numpy as np, pandas as pd, warnings
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
})

D = pd.read_csv("panel_window_final.csv"); D["date"] = pd.to_datetime(D["date"])
W = pd.read_csv("almaty_pm25_daily_window.csv"); W["date"] = pd.to_datetime(W["date"])

INST = ["log_blh", "u", "v", "pressure"]
keep = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_")) and D[c].nunique() > 1] + \
       [c for c in ["temp", "precip", "wind_speed", "cloud", "radiation",
                    "lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]
kk = []
for c in keep:
    M = D[["const"] + kk + [c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]: kk.append(c)
keep = kk
CTRL = " + ".join(keep)

# =============================================== FIG 1 — daily series
# Full filtered set (>=8 in-window obs), NOT just the estimation panel,
# so the plot shows coverage gaps as gaps.
F = W[W.n_window >= 8].copy()
F = F[(F.date >= "2020-04-08") & (F.date <= "2025-11-14")].sort_values("date")
filt_mean = F.pm25.mean()

# insert NaN rows over gaps so the line breaks instead of interpolating
full = pd.DataFrame({"date": pd.date_range(F.date.min(), F.date.max(), freq="D")})
F = full.merge(F[["date", "pm25"]], on="date", how="left")
gap = F.pm25.isna()
F.loc[gap, "pm25"] = np.nan

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(F.date, F.pm25, lw=0.6, color="#2b4c7e")
ax.axhline(5, ls="--", lw=1, color="green", label="WHO annual guideline (5)")
ax.axhline(filt_mean, ls="--", lw=1, color="crimson",
           label=f"Sample mean ({filt_mean:.1f})")
ax.set_ylabel("PM$_{2.5}$ ($\\mu$g/m$^3$)")
ax.set_xlabel("")
ax.legend(frameon=False, fontsize=8, loc="upper right")
fig.savefig("fig1_timeseries.png"); plt.close(fig)

# =============================================== FIG 2 — seasonality
m = D.groupby(D.date.dt.month).agg(pm=("pm25_raw", "mean"), blh=("blh", "mean"))
months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].bar(range(1, 13), m.pm.reindex(range(1, 13)), color="#2b4c7e")
axes[0].set_xticks(range(1, 13)); axes[0].set_xticklabels(months, fontsize=7)
axes[0].set_ylabel("PM$_{2.5}$ ($\\mu$g/m$^3$)"); axes[0].set_title("Pollution", fontsize=9)
axes[1].bar(range(1, 13), m.blh.reindex(range(1, 13)), color="#7e5a2b")
axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(months, fontsize=7)
axes[1].set_ylabel("Boundary layer height (m)"); axes[1].set_title("Mixing depth", fontsize=9)
fig.tight_layout(); fig.savefig("fig2_seasonality.png"); plt.close(fig)

# =============================================== FIG 3 — first stage
rx = smf.ols("log_blh ~ " + CTRL, data=D).fit().resid
ry = smf.ols("log_pm25 ~ " + CTRL, data=D).fit().resid
slope, intercept = np.polyfit(rx, ry, 1)
fig, ax = plt.subplots(figsize=(5.6, 4.2))
ax.scatter(rx, ry, s=5, alpha=0.35, color="#2b4c7e", edgecolors="none")
xs = np.linspace(rx.min(), rx.max(), 100)
ax.plot(xs, intercept + slope * xs, color="crimson", lw=1.6)
ax.set_xlabel("Residualised log boundary layer height")
ax.set_ylabel("Residualised log PM$_{2.5}$")
ax.text(0.03, 0.05, f"slope = {slope:.3f}", transform=ax.transAxes, fontsize=8)
fig.savefig("fig3_firststage.png"); plt.close(fig)

# =============================================== FIG 4 — binscatter
bx = smf.ols("log_pm25 ~ " + CTRL, data=D).fit().resid
by = smf.ols("logret ~ " + CTRL, data=D).fit().resid
B = pd.DataFrame({"x": bx, "y": by})
B["bin"] = pd.qcut(B.x, 20, labels=False)
g = B.groupby("bin").agg(x=("x", "mean"), y=("y", "mean"),
                         se=("y", lambda s: s.std(ddof=1) / np.sqrt(len(s))))
g["lo"], g["hi"] = g.y - 1.96 * g.se, g.y + 1.96 * g.se
n_zero = int(((g.lo <= 0) & (g.hi >= 0)).sum())
ols_slope = smf.ols("logret ~ log_pm25 + " + CTRL, data=D
                    ).fit(cov_type="HAC", cov_kwds={"maxlags": 5}).params["log_pm25"]

fig, ax = plt.subplots(figsize=(6.0, 4.2))
ax.errorbar(g.x, g.y, yerr=[g.y - g.lo, g.hi - g.y], fmt="o", ms=4,
            color="#2b4c7e", ecolor="#9aa8bd", elw=1, capsize=2)
xs = np.linspace(g.x.min(), g.x.max(), 100)
ax.plot(xs, ols_slope * xs + (g.y.mean() - ols_slope * g.x.mean()),
        color="crimson", lw=1.6, label=f"OLS slope = {ols_slope:.3f}")
ax.axhline(0, color="grey", lw=0.6)
ax.set_xlabel("Residualised log PM$_{2.5}$")
ax.set_ylabel("Residualised daily log return (\\%)")
ax.legend(frameon=False, fontsize=8)
fig.savefig("fig4_binscatter.png"); plt.close(fig)

# =============================================== FIG 5 — specification chart
def iv(df, inst=INST, weather_drop=False):
    k = [c for c in keep if not (weather_drop and c in ("cloud", "radiation"))]
    r = IV2SLS(df["logret"], df[["const"] + k], df["log_pm25"], df[inst]).fit(cov_type="kernel")
    return r.params["log_pm25"], r.std_errors["log_pm25"]

specs = [
    ("Preferred",                 lambda: iv(D)),
    ("Coverage $\\geq$15h",       lambda: iv(D[D.n_daily >= 15])),
    ("Full 11h window",           lambda: iv(D[D.date.isin(W[W.n_window == 11].date)])),
    ("From 2022",                 lambda: iv(D[D.year >= 2022])),
    ("From 2023",                 lambda: iv(D[D.year >= 2023])),
    ("Winter only",               lambda: iv(D[D.month.isin([11, 12, 1, 2, 3])])),
    ("Winsorised 1\\%",           lambda: iv(D.assign(logret=D.logret.clip(*D.logret.quantile([.01, .99]))))),
    ("Excl.\\ $v$",               lambda: iv(D, inst=["log_blh", "u", "pressure"])),
    ("Excl.\\ $u$",               lambda: iv(D, inst=["log_blh", "v", "pressure"])),
    ("BLH + pressure",            lambda: iv(D, inst=["log_blh", "pressure"])),
    ("BLH alone",                 lambda: iv(D, inst=["log_blh"])),
    ("Excl.\\ cloud, radiation",  lambda: iv(D, weather_drop=True)),
    ("Session-only treatment",    lambda: iv(D.assign(log_pm25=np.log(D.pm_session)).replace([np.inf, -np.inf], np.nan).dropna(subset=["log_pm25"]))),
    ("Calendar-day treatment",    lambda: iv(D.assign(log_pm25=np.log(D.pm25_calendar)))),
]
labels, bs, ses = [], [], []
for lab, fn in specs:
    try:
        b_, s_ = fn(); labels.append(lab); bs.append(b_); ses.append(s_)
    except Exception as e:
        print("skipped", lab, e)
bs, ses = np.array(bs), np.array(ses)

fig, ax = plt.subplots(figsize=(6.4, 5.0))
ypos = np.arange(len(labels))[::-1]
cols = ["crimson" if l == "Preferred" else "#2b4c7e" for l in labels]
ax.errorbar(bs, ypos, xerr=1.96 * ses, fmt="o", ms=4, elw=1, capsize=2,
            ecolor="#9aa8bd", linestyle="none")
for x_, y_, c_ in zip(bs, ypos, cols):
    ax.plot(x_, y_, "o", ms=5, color=c_)
ax.axvline(0, color="grey", lw=0.8)
ax.set_yticks(ypos); ax.set_yticklabels(labels, fontsize=7.5)
ax.set_xlabel("Coefficient on log PM$_{2.5}$")
fig.savefig("fig5_specification.png"); plt.close(fig)

# =============================================== CAPTION VALUES
print("=" * 62)
print("VALUES FOR THE CAPTIONS AND %VERIFY MARKERS")
print("=" * 62)
print(f"[Fig 1] mean over filtered set (>=8 in window): {filt_mean:.1f}")
print(f"[Fig 3] bivariate residualised slope: {slope:.3f}")
print(f"[Fig 4] OLS slope: {ols_slope:.4f}   intervals containing zero: {n_zero} of 20")
print(f"[3.4]  sample endpoints: {D.date.min().date()} to {D.date.max().date()}")
print("\n[1.1 / 3.5 / 4.3 / Fig 2] BLH monthly means on the estimation sample:")
for mo in range(1, 13):
    print(f"   {months[mo-1]}: BLH {m.blh.get(mo, float('nan')):7.1f}   "
          f"PM2.5 {m.pm.get(mo, float('nan')):6.2f}   n={int((D.month==mo).sum())}")
lo_mo, hi_mo = m.blh.idxmin(), m.blh.idxmax()
print(f"   minimum {months[lo_mo-1]} {m.blh.min():.1f}   maximum {months[hi_mo-1]} "
      f"{m.blh.max():.1f}   amplitude {m.blh.max()/m.blh.min():.1f}x")
print("\nSaved: fig1_timeseries.png fig2_seasonality.png fig3_firststage.png "
      "fig4_binscatter.png fig5_specification.png")


ModuleNotFoundError: No module named 'linearmodels'

In [ ]:
!pip install linearmodels -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 6.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
!cp /content/drive/MyDrive/pollution_research/*.csv .


Mounted at /content/drive


In [ ]:
"""
REGENERATE ALL FIVE FIGURES on the window panel.
Run after rebuild_window_final.py, in the same directory (needs
panel_window_final.csv and almaty_pm25_daily_window.csv).

Saves fig1_timeseries.png ... fig5_specification.png at 300 dpi with the
exact filenames main_revised.tex expects.

Also prints, at the end, every number the captions and the %VERIFY markers
need: BLH monthly means, sample endpoints, the Figure 3 bivariate slope,
and the Figure 4 zero-containing-interval count. Paste that block back.
"""
import numpy as np, pandas as pd, warnings
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
})

D = pd.read_csv("panel_window_final.csv"); D["date"] = pd.to_datetime(D["date"])
W = pd.read_csv("almaty_pm25_daily_window.csv"); W["date"] = pd.to_datetime(W["date"])

INST = ["log_blh", "u", "v", "pressure"]
keep = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_")) and D[c].nunique() > 1] + \
       [c for c in ["temp", "precip", "wind_speed", "cloud", "radiation",
                    "lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]
kk = []
for c in keep:
    M = D[["const"] + kk + [c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]: kk.append(c)
keep = kk
CTRL = " + ".join(keep)

# =============================================== FIG 1 — daily series
# Full filtered set (>=8 in-window obs), NOT just the estimation panel,
# so the plot shows coverage gaps as gaps.
F = W[W.n_window >= 8].copy()
F = F[(F.date >= "2020-04-08") & (F.date <= "2025-11-14")].sort_values("date")
filt_mean = F.pm25.mean()

# insert NaN rows over gaps so the line breaks instead of interpolating
full = pd.DataFrame({"date": pd.date_range(F.date.min(), F.date.max(), freq="D")})
F = full.merge(F[["date", "pm25"]], on="date", how="left")
gap = F.pm25.isna()
F.loc[gap, "pm25"] = np.nan

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(F.date, F.pm25, lw=0.6, color="#2b4c7e")
ax.axhline(5, ls="--", lw=1, color="green", label="WHO annual guideline (5)")
ax.axhline(filt_mean, ls="--", lw=1, color="crimson",
           label=f"Sample mean ({filt_mean:.1f})")
ax.set_ylabel("PM$_{2.5}$ ($\\mu$g/m$^3$)")
ax.set_xlabel("")
ax.legend(frameon=False, fontsize=8, loc="upper right")
fig.savefig("fig1_timeseries.png"); plt.close(fig)

# =============================================== FIG 2 — seasonality
m = D.groupby(D.date.dt.month).agg(pm=("pm25_raw", "mean"), blh=("blh", "mean"))
months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].bar(range(1, 13), m.pm.reindex(range(1, 13)), color="#2b4c7e")
axes[0].set_xticks(range(1, 13)); axes[0].set_xticklabels(months, fontsize=7)
axes[0].set_ylabel("PM$_{2.5}$ ($\\mu$g/m$^3$)"); axes[0].set_title("Pollution", fontsize=9)
axes[1].bar(range(1, 13), m.blh.reindex(range(1, 13)), color="#7e5a2b")
axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(months, fontsize=7)
axes[1].set_ylabel("Boundary layer height (m)"); axes[1].set_title("Mixing depth", fontsize=9)
fig.tight_layout(); fig.savefig("fig2_seasonality.png"); plt.close(fig)

# =============================================== FIG 3 — first stage
rx = smf.ols("log_blh ~ " + CTRL, data=D).fit().resid
ry = smf.ols("log_pm25 ~ " + CTRL, data=D).fit().resid
slope, intercept = np.polyfit(rx, ry, 1)
fig, ax = plt.subplots(figsize=(5.6, 4.2))
ax.scatter(rx, ry, s=5, alpha=0.35, color="#2b4c7e", edgecolors="none")
xs = np.linspace(rx.min(), rx.max(), 100)
ax.plot(xs, intercept + slope * xs, color="crimson", lw=1.6)
ax.set_xlabel("Residualised log boundary layer height")
ax.set_ylabel("Residualised log PM$_{2.5}$")
ax.text(0.03, 0.05, f"slope = {slope:.3f}", transform=ax.transAxes, fontsize=8)
fig.savefig("fig3_firststage.png"); plt.close(fig)

# =============================================== FIG 4 — binscatter
bx = smf.ols("log_pm25 ~ " + CTRL, data=D).fit().resid
by = smf.ols("logret ~ " + CTRL, data=D).fit().resid
B = pd.DataFrame({"x": bx, "y": by})
B["bin"] = pd.qcut(B.x, 20, labels=False)
g = B.groupby("bin").agg(x=("x", "mean"), y=("y", "mean"),
                         se=("y", lambda s: s.std(ddof=1) / np.sqrt(len(s))))
g["lo"], g["hi"] = g.y - 1.96 * g.se, g.y + 1.96 * g.se
n_zero = int(((g.lo <= 0) & (g.hi >= 0)).sum())
ols_slope = smf.ols("logret ~ log_pm25 + " + CTRL, data=D
                    ).fit(cov_type="HAC", cov_kwds={"maxlags": 5}).params["log_pm25"]

fig, ax = plt.subplots(figsize=(6.0, 4.2))
ax.errorbar(g.x, g.y, yerr=[g.y - g.lo, g.hi - g.y], fmt="o", ms=4,
            color="#2b4c7e", ecolor="#9aa8bd", elw=1, capsize=2)
xs = np.linspace(g.x.min(), g.x.max(), 100)
ax.plot(xs, ols_slope * xs + (g.y.mean() - ols_slope * g.x.mean()),
        color="crimson", lw=1.6, label=f"OLS slope = {ols_slope:.3f}")
ax.axhline(0, color="grey", lw=0.6)
ax.set_xlabel("Residualised log PM$_{2.5}$")
ax.set_ylabel("Residualised daily log return (\\%)")
ax.legend(frameon=False, fontsize=8)
fig.savefig("fig4_binscatter.png"); plt.close(fig)

# =============================================== FIG 5 — specification chart
def iv(df, inst=INST, weather_drop=False):
    k = [c for c in keep if not (weather_drop and c in ("cloud", "radiation"))]
    r = IV2SLS(df["logret"], df[["const"] + k], df["log_pm25"], df[inst]).fit(cov_type="kernel")
    return r.params["log_pm25"], r.std_errors["log_pm25"]

specs = [
    ("Preferred",                 lambda: iv(D)),
    ("Coverage $\\geq$15h",       lambda: iv(D[D.n_daily >= 15])),
    ("Full 11h window",           lambda: iv(D[D.date.isin(W[W.n_window == 11].date)])),
    ("From 2022",                 lambda: iv(D[D.year >= 2022])),
    ("From 2023",                 lambda: iv(D[D.year >= 2023])),
    ("Winter only",               lambda: iv(D[D.month.isin([11, 12, 1, 2, 3])])),
    ("Winsorised 1\\%",           lambda: iv(D.assign(logret=D.logret.clip(*D.logret.quantile([.01, .99]))))),
    ("Excl.\\ $v$",               lambda: iv(D, inst=["log_blh", "u", "pressure"])),
    ("Excl.\\ $u$",               lambda: iv(D, inst=["log_blh", "v", "pressure"])),
    ("BLH + pressure",            lambda: iv(D, inst=["log_blh", "pressure"])),
    ("BLH alone",                 lambda: iv(D, inst=["log_blh"])),
    ("Excl.\\ cloud, radiation",  lambda: iv(D, weather_drop=True)),
    ("Session-only treatment",    lambda: iv(D.assign(log_pm25=np.log(D.pm_session)).replace([np.inf, -np.inf], np.nan).dropna(subset=["log_pm25"]))),
    ("Calendar-day treatment",    lambda: iv(D.assign(log_pm25=np.log(D.pm25_calendar)))),
]
labels, bs, ses = [], [], []
for lab, fn in specs:
    try:
        b_, s_ = fn(); labels.append(lab); bs.append(b_); ses.append(s_)
    except Exception as e:
        print("skipped", lab, e)
bs, ses = np.array(bs), np.array(ses)

fig, ax = plt.subplots(figsize=(6.4, 5.0))
ypos = np.arange(len(labels))[::-1]
cols = ["crimson" if l == "Preferred" else "#2b4c7e" for l in labels]
ax.errorbar(bs, ypos, xerr=1.96 * ses, fmt="o", ms=4, elw=1, capsize=2,
            ecolor="#9aa8bd", linestyle="none")
for x_, y_, c_ in zip(bs, ypos, cols):
    ax.plot(x_, y_, "o", ms=5, color=c_)
ax.axvline(0, color="grey", lw=0.8)
ax.set_yticks(ypos); ax.set_yticklabels(labels, fontsize=7.5)
ax.set_xlabel("Coefficient on log PM$_{2.5}$")
fig.savefig("fig5_specification.png"); plt.close(fig)

# =============================================== CAPTION VALUES
print("=" * 62)
print("VALUES FOR THE CAPTIONS AND %VERIFY MARKERS")
print("=" * 62)
print(f"[Fig 1] mean over filtered set (>=8 in window): {filt_mean:.1f}")
print(f"[Fig 3] bivariate residualised slope: {slope:.3f}")
print(f"[Fig 4] OLS slope: {ols_slope:.4f}   intervals containing zero: {n_zero} of 20")
print(f"[3.4]  sample endpoints: {D.date.min().date()} to {D.date.max().date()}")
print("\n[1.1 / 3.5 / 4.3 / Fig 2] BLH monthly means on the estimation sample:")
for mo in range(1, 13):
    print(f"   {months[mo-1]}: BLH {m.blh.get(mo, float('nan')):7.1f}   "
          f"PM2.5 {m.pm.get(mo, float('nan')):6.2f}   n={int((D.month==mo).sum())}")
lo_mo, hi_mo = m.blh.idxmin(), m.blh.idxmax()
print(f"   minimum {months[lo_mo-1]} {m.blh.min():.1f}   maximum {months[hi_mo-1]} "
      f"{m.blh.max():.1f}   amplitude {m.blh.max()/m.blh.min():.1f}x")
print("\nSaved: fig1_timeseries.png fig2_seasonality.png fig3_firststage.png "
      "fig4_binscatter.png fig5_specification.png")


FileNotFoundError: [Errno 2] No such file or directory: 'panel_window_final.csv'

In [ ]:
import os
os.rename("kase_official_csv.csv", "kase_official.csv")
!ls *.csv


 almaty_pm25_daily.csv	    kase_clean.csv	        panel_analysis.csv
 almaty_pm25_hourly.csv    'KASE Historical Data.csv'   panel_final.csv
 almaty_weather_daily.csv   kase_official.csv	        panel_official.csv


In [ ]:
"""
FINAL REBUILD — window treatment from hourly source + official KASE + full battery
==================================================================================
One script, run top to bottom in Colab. It:

  1. Builds the daily PM2.5 series from almaty_pm25_hourly.csv:
     preferred 07:00-17:59 window mean, session-only (11:00-17:59),
     pre-session (07:00-10:59), and calendar-day means, each with its
     observation count. Hourly zeros (89 obs) are RETAINED — no negatives
     exist in the source, and dropping below-detection readings would bias
     means upward. Prints reconciliation checks against the July analysis
     (expected: 1,668 days with in-window obs; 1,548 passing >= 8).

  2. Builds the panel on the official KASE series with the pre-specified
     rules: >= 8 in-window hourly obs; weekend compensating sessions feed
     the price chain (logret already computed in kase_official.csv) but are
     EXCLUDED as observations; finite-log guard.

  3. Re-runs everything: main spec, first stage, reduced form, Table 1,
     full Table 6 (p-values on every row; session / pre-session /
     calendar-day treatment rows included), placebo leads, randomisation
     inference (1000 draws), threshold indicators, transport tests, S4.7
     FE sensitivity, S3.4 attrition stats, S6.5 monthly counts.

Paste the FULL output back into chat.
"""

# !pip install linearmodels -q
import numpy as np, pandas as pd, warnings
from linearmodels.iv import IV2SLS
import statsmodels.formula.api as smf
from scipy import stats
warnings.filterwarnings("ignore")

# from google.colab import drive
# drive.mount('/content/drive', force_remount=False)
# !cp /content/drive/MyDrive/pollution_research/*.csv .

LN2 = np.log(2)
RNG = np.random.default_rng(20260806)
WINDOW_MIN = 8   # paper's stated rule: >= 8 hourly obs inside 07:00-17:59

# ================================================================== STEP 1
# Build daily treatment series from hourly source
h = pd.read_csv("almaty_pm25_hourly.csv")
assert {"local", "pm25"}.issubset(h.columns), f"unexpected columns: {list(h.columns)}"
h["hour"] = h["local"].str.slice(11, 13).astype(int)
h["date"] = pd.to_datetime(h["local"].str.slice(0, 10))

print("=" * 70)
print("STEP 1 — TREATMENT CONSTRUCTION FROM HOURLY DATA")
print("=" * 70)
print(f"hourly obs: {len(h)}   zeros: {(h.pm25 == 0).sum()}   "
      f"negatives: {(h.pm25 < 0).sum()}   min {h.pm25.min()}   max {h.pm25.max()}")

def agg(mask, name):
    return h[mask].groupby("date").agg(**{f"pm_{name}": ("pm25", "mean"),
                                          f"n_{name}": ("pm25", "size")})

out = (agg((h.hour >= 7) & (h.hour <= 17), "window")
       .join(agg((h.hour >= 11) & (h.hour <= 17), "session"), how="outer")
       .join(agg((h.hour >= 7) & (h.hour <= 10), "presession"), how="outer")
       .join(agg(h.hour >= 0, "daily"), how="outer")
       .reset_index()
       .rename(columns={"pm_window": "pm25", "pm_daily": "pm25_calendar"}))
out.to_csv("almaty_pm25_daily_window.csv", index=False)

print(f"days with any in-window obs: {out.n_window.notna().sum()}   (expect ~1668)")
print(f"days passing >= {WINDOW_MIN} in window: {(out.n_window >= WINDOW_MIN).sum()}"
      f"   (expect ~1548)")
print(f"winter pass rate: "
      f"{(out[out.date.dt.month.isin([11,12,1,2])].n_window >= WINDOW_MIN).mean():.3f}")
print(f"session mean {out.pm_session.mean():.1f}   window mean {out.pm25.mean():.1f}"
      f"   calendar mean {out.pm25_calendar.mean():.1f}")

# ================================================================== STEP 2
# Panel on official KASE
wx = pd.read_csv("almaty_weather_daily.csv"); wx["date"] = pd.to_datetime(wx["date"])
ks = pd.read_csv("kase_official.csv");        ks["date"] = pd.to_datetime(ks["date"])

print("\n" + "=" * 70)
print("STEP 2 — PANEL")
print("=" * 70)
print(f"Official KASE: {len(ks)} rows, {ks.date.min().date()} to {ks.date.max().date()}")
chk = ks[ks.date == "2022-03-09"]
if len(chk): print(f"2022-03-09 logret: {chk.logret.values[0]:.4f}   (target ~4.05)")

p = out[out.n_window >= WINDOW_MIN][
    ["date", "pm25", "pm_session", "pm_presession", "pm25_calendar", "n_daily"]
].rename(columns={"pm25": "pm25_raw"})

D = ks[["date", "logret"]].merge(p, on="date").merge(wx, on="date")
D = D[(D.date >= "2020-04-08") & (D.date <= "2025-11-13")].copy()

# Weekend rule: weekend closes already fed the next session's logret inside
# kase_official.csv; the weekend rows themselves leave the sample here.
n_wk = (D.date.dt.dayofweek >= 5).sum()
D = D[D.date.dt.dayofweek < 5].copy()
print(f"Weekend-session rows dropped from panel (kept in price chain): {n_wk}")

D["log_pm25"] = np.log(D.pm25_raw)
D["log_blh"]  = np.log(D.blh)
D["dow"]   = D.date.dt.dayofweek
D["month"] = D.date.dt.month
D["year"]  = D.date.dt.year
D["lockdown"] = (((D.date >= "2020-03-16") & (D.date <= "2020-05-11")) |
                 ((D.date >= "2020-07-05") & (D.date <= "2020-08-17"))).astype(int)
D["qantar"]   = ((D.date >= "2022-01-04") & (D.date <= "2022-01-19")).astype(int)
D = (D.dropna(subset=["logret", "log_pm25", "log_blh", "u", "v", "pressure"])
       .sort_values("date").reset_index(drop=True))
D = D[np.isfinite(D.log_pm25)].reset_index(drop=True)

WEATHER = ["temp", "precip", "wind_speed", "cloud", "radiation"]
for m in range(1, 13):  D[f"mo_{m}"] = (D.month == m).astype(int)
for d_ in range(0, 5):  D[f"dw_{d_}"] = (D.dow == d_).astype(int)
for y in sorted(D.year.unique()): D[f"yr_{y}"] = (D.year == y).astype(int)
D["const"] = 1

def build_ctrl(df, weather=WEATHER, month_fe=True, year_fe=True):
    ctrl = list(weather)
    if month_fe:
        ctrl += [f"mo_{m}" for m in sorted(df.month.unique())[1:]]
    ctrl += [f"dw_{d_}" for d_ in sorted(df.dow.unique())[1:]]
    if year_fe:
        ctrl += [f"yr_{y}" for y in sorted(df.year.unique())[1:]]
    ctrl += [c for c in ["lockdown", "qantar"] if df[c].nunique() > 1]
    keep = []
    for c in ctrl:
        M = df[["const"] + keep + [c]].astype(float).values
        if np.linalg.matrix_rank(M) == M.shape[1]: keep.append(c)
    return keep

INST = ["log_blh", "u", "v", "pressure"]

def run_iv(df, endog="log_pm25", inst=INST, dep="logret",
           weather=WEATHER, month_fe=True, year_fe=True):
    keep = build_ctrl(df, weather, month_fe, year_fe)
    r = IV2SLS(df[dep], df[["const"] + keep], df[endog], df[inst]).fit(cov_type="kernel")
    b, se = r.params[endog], r.std_errors[endog]
    try:    F = r.first_stage.diagnostics["f.stat"].iloc[0]
    except Exception: F = float("nan")
    try:    sarg = r.sargan.pval if len(inst) > 1 else float("nan")
    except Exception: sarg = float("nan")
    return dict(b=b, se=se, p=r.pvalues[endog], N=int(r.nobs), F=F, sargan=sarg,
                bound=(abs(b) + 1.645 * se) * LN2, res=r, keep=keep)

# ================================================================== STEP 3
M0 = run_iv(D)
b, se = M0["b"], M0["se"]
print("\n" + "=" * 70)
print(f"PREFERRED SPEC — window treatment, weekend rule, >= {WINDOW_MIN} in window")
print("=" * 70)
print(f"  N              {M0['N']:>10}")
print(f"  IV estimate    {b:>10.4f}")
print(f"  SE             {se:>10.4f}")
print(f"  p              {M0['p']:>10.3f}")
print(f"  95% CI         [{b-1.96*se:.4f}, {b+1.96*se:.4f}]")
print(f"  First-stage F  {M0['F']:>10.1f}")
print(f"  Sargan p       {M0['sargan']:>10.4f}")
try: print(f"  Wu-Hausman p   {M0['res'].wu_hausman().pval:>10.4f}")
except Exception: pass
print(f"  TOST bound pp  {M0['bound']:>10.3f}")
print(f"  Per doubling   {b*LN2:>10.4f}  (SE {se*LN2:.4f})")

keep = M0["keep"]
ols = smf.ols("logret ~ log_pm25 + " + " + ".join(keep), data=D
              ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(f"  OLS estimate   {ols.params['log_pm25']:>10.4f}  (SE {ols.bse['log_pm25']:.4f})")

fs = smf.ols("log_pm25 ~ " + " + ".join(INST + keep), data=D
             ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print("\n  FIRST STAGE")
for v_ in INST:
    print(f"    {v_:<10}{fs.params[v_]:>9.4f} ({fs.bse[v_]:.4f})  t={fs.tvalues[v_]:6.2f}")
try:
    print(f"    partial R2 "
          f"{M0['res'].first_stage.diagnostics['partial.rsquared'].iloc[0]:.4f}"
          f"   R2 {fs.rsquared:.4f}")
except Exception: pass

rf   = smf.ols("logret ~ " + " + ".join(INST + keep), data=D
               ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
base = smf.ols("logret ~ " + " + ".join(keep), data=D).fit()
full = smf.ols("logret ~ " + " + ".join(INST + keep), data=D).fit()
print("\n  REDUCED FORM")
for v_ in INST:
    print(f"    {v_:<10}{rf.params[v_]:>9.4f} ({rf.bse[v_]:.4f})  p={rf.pvalues[v_]:.3f}")
print(f"    joint p    {rf.f_test(', '.join(f'{i} = 0' for i in INST)).pvalue:.4f}"
      f"   partial R2 {(full.rsquared-base.rsquared)/(1-base.rsquared):.4f}")
print(f"    BLH TOST bound "
      f"{(abs(rf.params['log_blh'])+1.645*rf.bse['log_blh'])*LN2:.4f} pp")

print(f"\n  TABLE 1 (N = {len(D)})")
for c, l in [("logret", "Log return"), ("pm25_raw", "PM2.5"), ("blh", "BLH"),
             ("temp", "Temp"), ("precip", "Precip"), ("wind_speed", "Wind"),
             ("pressure", "Pressure"), ("cloud", "Cloud"), ("radiation", "Radiation")]:
    print(f"    {l:<11} mean {D[c].mean():9.2f}  sd {D[c].std():8.2f}  "
          f"min {D[c].min():9.2f}  med {D[c].median():8.2f}  max {D[c].max():9.2f}")
print(f"    SD log PM2.5 {np.log(D.pm25_raw).std():.3f}   AR(1) {D.logret.autocorr(1):.3f}")
print(f"    Days >35: {(D.pm25_raw>35).sum()}  >55: {(D.pm25_raw>55).sum()}"
      f"  >75: {(D.pm25_raw>75).sum()}")
print(f"    Window mean / benchmark 11.53 = {D.pm25_raw.mean()/11.53:.2f}"
      f"   (paper currently says 2.6)")
print(f"    Window mean / WHO 5 = {D.pm25_raw.mean()/5:.1f}   (paper says ~6x)")

# ------------------------------------------------------------------ TABLE 6
print("\n" + "=" * 70)
print("TABLE 6 — coef, SE, p, N, F, bound on every row")
print("=" * 70)

def row(name, res):
    print(f"  {name:<34}{res['b']:>8.4f} {res['se']:>7.4f} {res['p']:>6.3f}"
          f" {res['N']:>5} {res['F']:>7.1f} {res['bound']:>7.3f}"
          + (f"  Sarg {res['sargan']:.3f}" if not np.isnan(res['sargan']) else ""))

row("Preferred", M0)
row("Coverage >= 18h (daily)", run_iv(D[D.n_daily >= 18]))
row("Coverage >= 15h (daily)", run_iv(D[D.n_daily >= 15]))
row("Window coverage = 11 (full)", run_iv(D[D.date.isin(
    out[out.n_window == 11].date)]))
row("Sample from 2022",  run_iv(D[D.year >= 2022]))
row("Sample from 2023",  run_iv(D[D.year >= 2023]))
row("Excluding 2025",    run_iv(D[D.year <= 2024]))
row("Pre-2024 clock",    run_iv(D[D.date < "2024-03-01"]))
row("Winter only",       run_iv(D[D.month.isin([11, 12, 1, 2, 3])]))
Dw = D.copy(); lo, hi = Dw.logret.quantile([.01, .99]); Dw["logret"] = Dw.logret.clip(lo, hi)
row("Winsorised 1%",     run_iv(Dw))
row("Instr excl pressure",  run_iv(D, inst=["log_blh", "u", "v"]))
row("Instr excl v",         run_iv(D, inst=["log_blh", "u", "pressure"]))
row("Instr excl u",         run_iv(D, inst=["log_blh", "v", "pressure"]))
row("BLH + u",              run_iv(D, inst=["log_blh", "u"]))
row("BLH + pressure",       run_iv(D, inst=["log_blh", "pressure"]))
row("BLH alone",            run_iv(D, inst=["log_blh"]))
row("Ctrls excl cloud/rad", run_iv(D, weather=["temp", "precip", "wind_speed"]))
row("Excl Jan 2022 unrest", run_iv(D[~((D.date >= "2022-01-04") &
                                       (D.date <= "2022-01-19"))]))
row("Dense (Apr 2023 on)",    run_iv(D[D.date >= "2023-04-01"]))
row("Post-gap (Jul 2024 on)", run_iv(D[D.date >= "2024-07-01"]))

for k in (1, 2, 3):
    Dk = D.copy()
    for c in ["log_pm25"] + INST: Dk[c] = Dk[c].shift(k)
    Dk = Dk.dropna(subset=["log_pm25"] + INST).reset_index(drop=True)
    row(f"Lag {k} day", run_iv(Dk))

Da = D.copy(); Da["abslogret"] = Da.logret.abs()
row("Outcome: |return|", run_iv(Da, dep="abslogret"))

# alternative treatment constructions (S4.2's "reported as robustness")
for alt, lab in [("pm_session", "Session-only treatment"),
                 ("pm_presession", "Pre-session treatment"),
                 ("pm25_calendar", "Calendar-day treatment")]:
    Dv = D.copy(); Dv["log_pm25"] = np.log(Dv[alt])
    Dv = Dv[np.isfinite(Dv.log_pm25)].dropna(subset=["log_pm25"]).reset_index(drop=True)
    row(lab, run_iv(Dv))
pos = D.pm_session > 0
print(f"  log corr window vs session:    "
      f"{np.log(D[pos].pm25_raw).corr(np.log(D[pos].pm_session)):.3f}")
print(f"  log corr window vs presession: "
      f"{np.log(D.pm25_raw).corr(np.log(D.pm_presession)):.3f}")
print(f"  log corr window vs calendar:   "
      f"{np.log(D.pm25_raw).corr(np.log(D.pm25_calendar)):.3f}")

# ------------------------------------------------------------------ PLACEBOS
print("\nPLACEBO LEADS (return realised BEFORE the pollution)")
for k in (1, 2, 3, 5):
    Dk = D.copy(); Dk["lead"] = Dk.logret.shift(-k)
    Dk = Dk.dropna(subset=["lead"]).reset_index(drop=True)
    row(f"{k}-day lead", run_iv(Dk, dep="lead"))
D1 = D.copy(); D1["lead"] = D1.logret.shift(-1); D1["lag_pm"] = D1.log_pm25.shift(1)
D1 = D1.dropna(subset=["lead", "lag_pm"]).reset_index(drop=True)
keep1 = build_ctrl(D1) + ["lag_pm"]
r1 = IV2SLS(D1["lead"], D1[["const"] + keep1], D1["log_pm25"], D1[INST]
            ).fit(cov_type="kernel")
print(f"  {'1-day lead + lagged PM2.5':<34}{r1.params['log_pm25']:>8.4f}"
      f" {r1.std_errors['log_pm25']:>7.4f} {r1.pvalues['log_pm25']:>6.3f}"
      f" {int(r1.nobs):>5}")

# ------------------------------------------------------------------ RANDOMISATION INFERENCE
print("\nRANDOMISATION INFERENCE (1000 circular shifts, multiples of 5)")
exog, endog, instr = D[["const"] + keep], D["log_pm25"], D[INST]
n = len(D); draws = []
for _ in range(1000):
    s = 5 * int(RNG.integers(1, n // 5))
    yb = pd.Series(np.roll(D.logret.values, s), index=D.index)
    draws.append(IV2SLS(yb, exog, endog, instr
                        ).fit(cov_type="unadjusted").params["log_pm25"])
draws = np.array(draws)
print(f"  null SD {draws.std():.4f}  vs HAC SE {se:.4f}  ratio {draws.std()/se:.2f}")
print(f"  95% null range [{np.percentile(draws,2.5):.3f}, "
      f"{np.percentile(draws,97.5):.3f}]")
print(f"  RI p (|b|) {np.mean(np.abs(draws) >= abs(b)):.3f}"
      f"   percentile of estimate {stats.percentileofscore(draws, b):.1f}")

# ------------------------------------------------------------------ THRESHOLDS
print("\nTHRESHOLD INDICATORS (reduced form, OLS)")
for thr in (35, 55, 75):
    Dt = D.copy(); Dt["hi"] = (Dt.pm25_raw > thr).astype(int)
    rt = smf.ols("logret ~ hi + " + " + ".join(keep), data=Dt
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  >{thr}: {rt.params['hi']:.4f} (SE {rt.bse['hi']:.4f},"
          f" p={rt.pvalues['hi']:.3f})   days above: {Dt.hi.sum()}")

# ------------------------------------------------------------------ TRANSPORT
est_pp, se_pp = b * LN2, se * LN2
mean_pm = D.pm25_raw.mean()
lin_pp, lin_se = 0.0168 * mean_pm, 0.0055 * mean_pm   # benchmark linear at OUR mean
print("\nTRANSPORT TESTS (benchmark linear implication recomputed at window mean)")
print(f"  window mean {mean_pm:.2f}  ->  linear implication {lin_pp:.3f} pp"
      f" (SE {lin_se:.3f})   [paper's old figures: 0.500, 0.164]")
for d_, sd_, lab in [(0.194, 0.063, "proportional"), (lin_pp, lin_se, "linear")]:
    t = (d_ + est_pp) / np.sqrt(se_pp ** 2 + sd_ ** 2)
    print(f"  {lab:<14}t = {t:5.2f}   {'rejected' if abs(t) > 1.96 else 'not rejected'}")
print(f"  Per 10 ug/m3 at mean: "
      f"{b*np.log((mean_pm+10)/mean_pm):.4f} (SE {se*np.log((mean_pm+10)/mean_pm):.4f})")

# ------------------------------------------------------------------ S4.7 FE SENSITIVITY
print("\nFIRST-STAGE BLH COEF ACROSS FE SPECS (S4.7)")
for mfe, yfe, lab in [(False, False, "no month/year FE"),
                      (True, False, "month FE only"),
                      (True, True, "month + year FE")]:
    kfe = build_ctrl(D, month_fe=mfe, year_fe=yfe)
    ffe = smf.ols("log_pm25 ~ " + " + ".join(INST + kfe), data=D
                  ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    print(f"  {lab:<18}{ffe.params['log_blh']:>8.4f} ({ffe.bse['log_blh']:.4f})")

# ------------------------------------------------------------------ ATTRITION (S3.4 / S6.5)
print("\nATTRITION — retained vs excluded weekday trading days in window")
allts = ks[(ks.date >= "2020-04-08") & (ks.date <= "2025-11-13")]
allts = allts[allts.date.dt.dayofweek < 5].dropna(subset=["logret"])
ret = allts[allts.date.isin(D.date)]; exc = allts[~allts.date.isin(D.date)]
print(f"  weekday trading days {len(allts)}   retained {len(ret)}   excluded {len(exc)}")
t_, p_ = stats.ttest_ind(ret.logret, exc.logret, equal_var=False)
print(f"  mean ret: t={t_:.2f} p={p_:.3f}"
      f"   KS p={stats.ks_2samp(ret.logret, exc.logret).pvalue:.3f}"
      f"   Levene p={stats.levene(ret.logret, exc.logret).pvalue:.3f}")
print(f"  variance: retained {ret.logret.var():.3f}  excluded {exc.logret.var():.3f}")
print(f"  skew:     retained {ret.logret.skew():.3f}  excluded {exc.logret.skew():.3f}")

print("\nRETAINED DAYS BY MONTH (S6.5)")
print(D.month.value_counts().sort_index().to_string())

D.to_csv("panel_window_final.csv", index=False)
# !cp panel_window_final.csv almaty_pm25_daily_window.csv /content/drive/MyDrive/pollution_research/
print("\nSaved panel_window_final.csv and almaty_pm25_daily_window.csv.")
print("Paste the FULL output back into chat.")


STEP 1 — TREATMENT CONSTRUCTION FROM HOURLY DATA
hourly obs: 37632   zeros: 89   negatives: 0   min 0.0   max 421.0
days with any in-window obs: 1668   (expect ~1668)
days passing >= 8 in window: 1548   (expect ~1548)
winter pass rate: 0.914
session mean 36.1   window mean 31.8   calendar mean 29.5

STEP 2 — PANEL
Official KASE: 1480 rows, 2020-01-05 to 2026-01-09
2022-03-09 logret: 4.0479   (target ~4.05)
Weekend-session rows dropped from panel (kept in price chain): 6

PREFERRED SPEC — window treatment, weekend rule, >= 8 in window
  N                     926
  IV estimate       -0.1053
  SE                 0.0915
  p                   0.250
  95% CI         [-0.2846, 0.0740]
  First-stage F       175.7
  Sargan p           0.1853
  Wu-Hausman p       0.0685
  TOST bound pp       0.177
  Per doubling      -0.0730  (SE 0.0634)
  OLS estimate       0.0126  (SE 0.0377)

  FIRST STAGE
    log_blh     -0.4508 (0.0421)  t=-10.70
    u            0.0314 (0.0095)  t=  3.30
    v           -0

In [53]:
!cp main_final.tex /content/drive/MyDrive/pollution_research/
from google.colab import files
files.download("main_final.tex")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [54]:
import glob; print(glob.glob("*.tex"))

['main_rebuilt.tex', 'main.tex', 'main_final.tex']


In [55]:
#!/usr/bin/env python3
"""
REVIEW FIXES — run on main_final.tex (the file now in Overleaf).

    python3 apply_review_fixes.py

Writes main_v3.tex.

Fixes the scriptable items from the independent review. Four items need
your hands and are listed at the end, because they depend on wording I
cannot see reliably.

NOT CHANGED, deliberately:
  * Table 5's t = 1.60. From unrounded inputs it is 1.601. The reviewer's
    1.59 comes from the table's own rounded display of 0.137/0.086.
  * Table 9's OLS cell 0.047. Computed from unrounded standard
    deviations; 0.046 comes from using the rounded 0.860/0.65.
"""

import re, sys
from pathlib import Path

SRC, OUT = "main_final.tex", "main_v3.tex"

E = [

# --- 1. the OLS-IV gap is 1.10 SE, not 1.33. Four sites. --------------
("gap: abstract / 2.5 / 4.1 / 7",
 "roughly one and a third standard errors",
 "roughly 1.1 standard errors", 4),

# --- 3. common-bandwidth ratio is stale -------------------------------
("5.8 common bandwidth",
 "recomputing both on a common five-lag window gives 2.54",
 "recomputing both on a common five-lag window gives 2.34", 1),

# --- 4. my error: that 0.022 was OUR estimate, not the benchmark's ----
("6.2 conversion",
 r"""The multi-city summary estimate discussed in Section~\ref{sec:transport} corresponds to roughly 0.022 percentage points per doubling at our sample mean---about a seventh of our bound---and an effect of that size would sit well inside our confidence interval.""",
 r"""The multi-city summary estimate discussed in Section~\ref{sec:gap} corresponds to roughly 0.039 percentage points per doubling at our sample mean---about a quarter of our bound---and an effect of that size would sit well inside our confidence interval.""", 1),

# --- 5. my error: six Table 6 rows do exceed 0.194 --------------------
("6.1 unhedged",
 r"""That threshold now sits below the benchmark's \emph{proportional} implication of 0.194---its effect per doubling evaluated at its own sample mean---and remains below it under every specification and inference procedure we examine (Section~\ref{sec:transport}).""",
 r"""That threshold now sits below the benchmark's \emph{proportional} implication of 0.194---its effect per doubling evaluated at its own sample mean---and remains below it under every inference procedure we examine and in every specification other than the smallest subsamples, whose bounds widen with their sample size (Section~\ref{sec:transport}).""", 1),

# --- 6. my error: the five-day lead is 0.71 of contemporaneous --------
("5.7 lead ratio",
 r"""Ours are roughly a tenth to a fifth the size of our contemporaneous estimate, which is comparable.""",
 r"""Ours are roughly a tenth to a fifth the size of our contemporaneous estimate at one to three days, which is comparable; the five-day lead is larger, at seven-tenths, though it is the least informative of the four and is positive.""", 1),

# --- 5.5: name the exceptions instead of excluding them silently ------
("5.5 robustness",
 r"""The just-identified specification using boundary layer height alone yields 0.165, the dispersion-only pair 0.163, and no specification in Table~\ref{tab:robust} that we would defend as our preferred design puts the bound above 0.194.""",
 r"""The just-identified specification using boundary layer height alone yields 0.165 and the dispersion-only pair 0.163. Six rows of Table~\ref{tab:robust} do put the bound above 0.194---the two pandemic-excluding subsamples, the two coverage-restricted subsamples, the pre-session treatment and pressure used alone---and in every case the widening tracks the loss of observations rather than a change in the point estimate: the samples beginning in 2022 and 2023 discard a quarter and two-fifths of the data respectively.""", 1),

# --- 5.2: -0.576 is above -1, smaller in magnitude --------------------
("5.2 elasticity",
 r"""The estimated elasticity of $-0.576$ remains below the value of $-1$""",
 r"""The estimated elasticity of $-0.576$ is smaller in magnitude than the value of $-1$""", 1),

# --- 5.6: "are not confounding" overstates -----------------------------
("5.6 cloud claim",
 "Cloud cover and shortwave radiation are not confounding the estimate.",
 "Cloud cover and shortwave radiation do not drive the estimate.", 1),

# --- 3.2: drop the promise of a Table 6 row that does not exist -------
("3.2 promise",
 r"""We report the windowed construction as a robustness row in Table~\ref{tab:robust}, and note that this is a choice made on instrument relevance, which the second stage does not enter.""",
 r"""That comparison is run on the 926-day sample available before the gap was recovered, holding the days fixed so that only the construction differs. We note that this is a choice made on instrument relevance, which the second stage does not enter.""", 1),

# --- 3.2: why the controls stay with the other source -----------------
("3.2 controls",
 r"""and the remaining meteorological variables, which enter only as controls and for which no gap arises, from Open-Meteo.""",
 r"""and the remaining meteorological variables, which enter only as controls and for which no gap arises, from Open-Meteo. Because those variables enter only as conditioning variables alongside month, day-of-week and year fixed effects, a level distortion of the kind we document for boundary layer height would be absorbed rather than transmitted to the estimate.""", 1),

# --- 6.3: scope the transport claim to what it holds for --------------
("6.3 transport scope",
 r"""Our transport conclusions hold in every specification: linear transport is rejected at the five per cent level throughout.""",
 r"""Our transport conclusions hold across every instrument set we consider: linear transport is rejected at the five per cent level in each, with test statistics between 2.52 and 2.61.""", 1),

# --- 1.1: two endpoints, not "throughout" -----------------------------
("1.1 turnover",
 "have accounted for a majority of secondary equity turnover throughout the period",
 "accounted for a majority of secondary equity turnover at both ends of the period for which figures are published", 1),

# --- 6.2: soften at first statement ------------------------------------
("6.2 foreign",
 "whose institutional ownership is substantially foreign",
 "whose institutional ownership includes a foreign component", 1),

# --- rounding: linear implication ---------------------------------------
("5.5 linear implication",
 "this implies 0.543 percentage points per doubling (SE $0.178$)",
 "this implies 0.542 percentage points per doubling (SE $0.178$)", 1),
("T5 linear row",
 r"""Linear (constant per \ugm{}, at our mean) & $-0.543$ & 0.178 & 0.486 & 0.187 & \textbf{2.60} \\""",
 r"""Linear (constant per \ugm{}, at our mean) & $-0.542$ & 0.178 & 0.485 & 0.187 & \textbf{2.60} \\""", 1),
("6.1 linear implication",
 r"""\emph{linear} implication of 0.543 percentage points""",
 r"""\emph{linear} implication of 0.542 percentage points""", 1),

# --- rounding: pressure-alone p ----------------------------------------
("T6 pressure p",
 r"""Pressure alone & $-0.0984$ & 0.1424 & 0.489 & 1{,}042 & 60.6 & 0.231 \\""",
 r"""Pressure alone & $-0.0984$ & 0.1424 & 0.490 & 1{,}042 & 60.6 & 0.231 \\""", 1),
]

BIB = [("kiihamaki volume", "4011:8628", "11:8628", 1)]


def flex(text, find, repl, label, count, report):
    pat = re.compile(r"\s+".join(re.escape(t) for t in find.split()))
    hits = list(pat.finditer(text))
    if len(hits) == count:
        for m in reversed(hits):
            text = text[:m.start()] + repl + text[m.end():]
        report.append(f"  OK       {label}")
        return text, True
    report.append(f"  {'MISSING ' if not hits else 'COUNT   '}{label}"
                  f"  (found {len(hits)}, expected {count})")
    return text, False


def main():
    p = Path(SRC)
    if not p.exists():
        sys.exit(f"not found: {SRC}")
    t, rep, ok = p.read_text(encoding="utf-8"), ["TEX"], 0
    for label, f, r, c in E:
        t, g = flex(t, f, r, label, c, rep)
        ok += g
    Path(OUT).write_text(t, encoding="utf-8")

    rep.append("\nBIB")
    b = Path("references.bib")
    if b.exists():
        bt = b.read_text(encoding="utf-8")
        for label, f, r, c in BIB:
            bt, _ = flex(bt, f, r, label, c, rep)
        Path("references_v3.bib").write_text(bt, encoding="utf-8")
    else:
        rep.append("  references.bib not here -- fix 4011:8628 by hand")

    print("\n".join(rep))
    print(f"\n{ok}/{len(E)} tex edits.  Wrote {OUT}.")
    print("""
FOUR BY HAND — I cannot see the exact wording well enough to script these

1. FOOTNOTE 4 in Section 4.4. It says the robust-vs-classical Stock-Yogo
   qualification "matters more for the three rows of Table 6 that sit at
   or below the threshold, and those we already describe as weakly
   identified." No Table 6 row does any more. Rewrite along the lines of:
     "...matters more for the weakest specifications. In Table 6 the
      narrowest margin is the pre-session treatment at 25.9 against
      24.58; the instrumented threshold specifications in Table 7 are
      weaker still, at 18.8 and 9.5, and we describe those as weakly
      identified."

2. TABLE 1's NOTE. It still says "\\pmm{} and meteorological variables
   are daily means over the 07:00--17:59 exposure window." Split it:
   \\pmm{} is the window mean, meteorological variables are calendar-day
   means, per Section 3.2.

3. SECTION 3.2, one clause. After "reaching 1,303 metres against 995 in
   the Climate Data Store", add "(both over the full daily series;
   Table 1 reports the estimation subsample)" so it does not read as
   contradicting Table 1's maximum of 923.88.

4. SECTION 3.1 vs 3.4. The monitor reports to 14 November 2025 but the
   sample ends 13 November. Add a clause saying the final day failed the
   coverage filter, so it does not look like an off-by-one.

ALSO WORTH DOING, not scriptable
  * Section 4.5's Brent and uranium figures reconcile only at a third
    digit. Print them as 0.0065 (SE 0.0141) and 0.0173 (SE 0.0096) so a
    reader recomputing the p-values gets your numbers.
  * A footnote in Section 5.5 noting that the exclusion bound is a
    one-sided five per cent construction while the transport test is
    two-sided, so the two are not at a common level. You never claim
    otherwise, but it pre-empts the question.
""")


if __name__ == "__main__":
    main()

TEX
  OK       gap: abstract / 2.5 / 4.1 / 7
  OK       5.8 common bandwidth
  OK       6.2 conversion
  OK       6.1 unhedged
  OK       5.7 lead ratio
  OK       5.5 robustness
  OK       5.2 elasticity
  OK       5.6 cloud claim
  OK       3.2 promise
  OK       3.2 controls
  OK       6.3 transport scope
  OK       1.1 turnover
  OK       6.2 foreign
  OK       5.5 linear implication
  OK       T5 linear row
  OK       6.1 linear implication
  OK       T6 pressure p

BIB
  references.bib not here -- fix 4011:8628 by hand

17/17 tex edits.  Wrote main_v3.tex.

FOUR BY HAND — I cannot see the exact wording well enough to script these

1. FOOTNOTE 4 in Section 4.4. It says the robust-vs-classical Stock-Yogo
   qualification "matters more for the three rows of Table 6 that sit at
   or below the threshold, and those we already describe as weakly
   identified." No Table 6 row does any more. Rewrite along the lines of:
     "...matters more for the weakest specifications. In Table 6 the

In [56]:
!cp main_v3.tex /content/drive/MyDrive/pollution_research/
from google.colab import files
files.download("main_v3.tex")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [57]:
import re
t = open("main_v3.tex").read()

def show(label, pattern, before=0, after=400):
    m = re.search(pattern, t)
    print(f"\n===== {label} =====")
    print(t[max(0,m.start()-before):m.end()+after] if m else "NOT FOUND")

show("footnote 4", r"Stock--Yogo critical values are computed under")
show("Table 1 note", r"exposure window\. Meteorological|are daily means over the 07")
show("3.2 CDS sentence", r"reaching 1,303 metres")
show("3.1 monitor end date", r"14 November 2025|November 2025")


===== footnote 4 =====
NOT FOUND

===== Table 1 note =====
are daily means over the 07:00--17:59 exposure window defined in Section~\ref{sec:window}. The standard deviation of log \pmm{}, 0.860, is reported in the text below and used in every conversion between coefficients and interpretable magnitudes. The maximum log return of 4.05 is the session of 9 March 2022, during the market dislocation that followed the February 2022 invasion of Ukraine; Table~\ref{tab:robust} reports a wi

===== 3.2 CDS sentence =====
reaching 1,303 metres against 995 in the Climate Data Store, and a regression of one on the other has a slope of 0.85. Open-Meteo does not document the processing that produces the difference. We therefore report boundary layer height from the authoritative source throughout rather than splicing two series within one variable, and the remaining meteorological variables, which enter only as controls and for which no ga

===== 3.1 monitor end date =====
November 2025.

\subsectio

In [58]:
import re
t = open("main_v3.tex").read()

print("=== footnotes mentioning Stock-Yogo or weakly identified ===")
for m in re.finditer(r"\\footnote\{[^}]{0,600}(Stock|weakly identified|homoskedas)[^}]{0,600}\}", t):
    print("\n---\n", m.group(0)[:800])

print("\n=== 3.1 monitor sentence ===")
i = t.find("November 2025.\n\n\\subsection{Why Kazakhstan}")
print(t[max(0,i-600):i+20])

=== footnotes mentioning Stock-Yogo or weakly identified ===

---
 \footnote{Kazakhstan Stock Exchange, \emph{Annual Report 2019}

---
 \footnote{One qualification. The Stock--Yogo critical values are derived under homoskedastic, serially uncorrelated errors, whereas the statistic we compare against them is heteroskedasticity- and autocorrelation-robust. The appropriate analogue in a robust setting is the effective $F$ of \citet{montielolea2013}

---
 \footnote{The Sargan statistic assumes homoskedastic, serially uncorrelated errors, while the standard errors reported throughout are robust. We therefore also computed heteroskedasticity-robust analogues: Wooldridge's score test of overidentification \citep{wooldridge2010}

=== 3.1 monitor sentence ===
recondition rarely stated explicitly: \emph{the marginal price-setter must be exposed to the pollution being measured.} The existing evidence comes predominantly from deep, domestically-held markets where that condition holds by constructi

In [59]:
t = open("main_v3.tex").read()

i = t.find("The appropriate analogue in a robust setting is the effective")
print("=== FOOTNOTE 4 ===")
print(t[i:i+700])

print("\n=== MONITOR COVERAGE (3.1) ===")
for k in ["monitor reports", "reports through", "coverage runs", "series runs",
          "14 November", "13 November", "final observation"]:
    j = t.find(k)
    if j > -1:
        print(f"\n--- {k} ---")
        print(t[max(0,j-350):j+350])

=== FOOTNOTE 4 ===
The appropriate analogue in a robust setting is the effective $F$ of \citet{montielolea2013}, which we do not compute. Our preferred specification clears the classical threshold by a wide margin, so the qualification is unlikely to matter here; it matters more for the three rows of Table~\ref{tab:robust} that sit at or below the threshold, and those we already describe as weakly identified.}

The economic interpretation of the individual coefficients is discussed in Section~\ref{sec:firststage}.

\subsection{The exclusion restriction}\label{sec:exclusion}

Identification requires that these meteorological variables affect returns only through pollution. The assumption faces one well-document

=== MONITOR COVERAGE (3.1) ===

--- monitor reports ---
24. Hourly coverage across the day is close to uniform, ranging between 1,505 and 1,589 observations in each hour of the twenty-four, so the eight-hour threshold does not select systematically on time of day. Hourly values 

In [60]:
t = open("main_v3.tex").read()
n = 0

# 1. Footnote 4 — no Table 6 row sits at or below its threshold now
a = "it matters more for the three rows of Table~\\ref{tab:robust} that sit at or below the threshold, and those we already describe as weakly identified."
b = "it matters more for the weakest specifications. The narrowest margin in Table~\\ref{tab:robust} is the pre-session treatment, at 25.9 against a critical value of 24.58; the instrumented threshold specifications of Table~\\ref{tab:thresholds} are weaker still, at 18.8 and 9.5, and those we describe as weakly identified."
assert t.count(a) == 1; t = t.replace(a, b); n += 1

# 2. Table 1 note — meteorological variables are calendar-day means
a = "are daily means over the 07:00--17:59 exposure window defined in Section~\\ref{sec:window}."
b = "are daily means over the 07:00--17:59 exposure window defined in Section~\\ref{sec:window}; meteorological variables are calendar-day means, for the reasons given in Section~\\ref{sec:met}."
assert t.count(a) == 1; t = t.replace(a, b); n += 1

# 3. §3.2 — say which sample the 995 refers to
a = "reaching 1,303 metres against 995 in the Climate Data Store,"
b = "reaching 1,303 metres against 995 in the Climate Data Store---both over the full daily series, where Table~\\ref{tab:descriptives} reports the estimation subsample---"
assert t.count(a) == 1; t = t.replace(a, b); n += 1

# 4. the 14 vs 13 November gap
a = "yields a final estimation sample of \\textbf{1,042 trading days}, running from 10 April 2020 to 13 November 2025."
b = "yields a final estimation sample of \\textbf{1,042 trading days}, running from 10 April 2020 to 13 November 2025. The sample ends one day before the monitor's final observation because 14 November 2025, a Friday, carries too few in-window hourly readings to pass the coverage filter."
assert t.count(a) == 1; t = t.replace(a, b); n += 1

# 5. §4.5 — third digit so the p-values reconcile
for a, b in [("(0.007, SE $0.014$, $p = 0.644$)", "(0.0065, SE $0.0141$, $p = 0.644$)"),
             ("(0.017, SE $0.010$, $p = 0.071$)", "(0.0173, SE $0.0096$, $p = 0.071$)")]:
    if t.count(a) == 1: t = t.replace(a, b); n += 1
    else: print(f"skipped: {a}")

open("main_v3.tex","w").write(t)
print(f"{n} fixes applied")

6 fixes applied


In [62]:
!cp main_v3.tex /content/drive/MyDrive/pollution_research/
from google.colab import files
files.download("main_v3.tex")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [63]:
t = open("main_v3.tex").read()
n = 0
def sub(a, b):
    global t, n
    if t.count(a) == 1: t = t.replace(a, b); n += 1
    else: print(f"CHECK ({t.count(a)}x): {a[:60]}")

# 1. Table 1 note contradicted itself
sub("\\pmm{} and meteorological variables are daily means over the 07:00--17:59 exposure window",
    "\\pmm{} is a daily mean over the 07:00--17:59 exposure window")

# 2. §5.5 — the six rows, correctly explained
sub("Six rows of Table~\\ref{tab:robust} do put the bound above 0.194---the two pandemic-excluding subsamples, the two coverage-restricted subsamples, the pre-session treatment and pressure used alone---and in every case the widening tracks the loss of observations rather than a change in the point estimate: the samples beginning in 2022 and 2023 discard a quarter and two-fifths of the data respectively.",
    "Six rows of Table~\\ref{tab:robust} do put the bound above 0.194. In the four subsample rows the widening reflects lost observations---the samples beginning in 2022 and 2023 discard a quarter and two-fifths of the data---and in two of them a more negative point estimate as well. In the remaining two, the pre-session treatment and pressure used alone, the sample is unchanged and the widening reflects weaker identification. None reflects a point estimate that has moved toward zero.")

# 3. §6.1 inherited the same error
sub("in every specification other than the smallest subsamples, whose bounds widen with their sample size",
    "in every specification other than the least precisely identified, whose bounds widen with their standard errors")

# 4. §6.3 range understated
sub("with test statistics between 2.52 and 2.61",
    "with test statistics between 2.33 and 2.61, the weakest being pressure used alone")

# 5. §5.8 inconsistent with the abstract
sub("roughly two-and-a-half-fold", "roughly 2.4-fold")

# 6. §3.2 — the controls sentence overclaimed on measurement error
sub("Because those variables enter only as conditioning variables alongside month, day-of-week and year fixed effects, a level distortion of the kind we document for boundary layer height would be absorbed rather than transmitted to the estimate.",
    "Because those variables enter only as conditioning variables, a level distortion or rescaling of the kind we document for boundary layer height would be absorbed by the coefficient on the control. Table~\\ref{tab:robust} offers the more direct reassurance: dropping cloud cover and radiation, and varying the functional form in temperature, leave the estimate within one standard error.")

open("main_v3.tex","w").write(t)
print(f"{n}/6 applied")

6/6 applied


In [64]:
!cp main_v3.tex /content/drive/MyDrive/pollution_research/
from google.colab import files
files.download("main_v3.tex")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [65]:
t = open("main_v3.tex").read()
n = 0
def sub(a, b, expect=1):
    global t, n
    if t.count(a) == expect: t = t.replace(a, b); n += 1
    else: print(f"CHECK ({t.count(a)}x, expected {expect}): {a[:70]}")

# §6.3 — the excl-u sentence: 0.153, and it is a wash not a gain
sub("the specification excluding it yields $-0.077$ with an exclusion bound of 0.147 percentage points, tighter than the bound we report: our conclusions do not depend on the instrument about which we have the most doubt.",
    "the specification excluding it yields $-0.077$ with an exclusion bound of 0.153 percentage points, essentially identical to the one we report and still below the benchmark's proportional implication: our conclusions do not depend on the instrument about which we have the most doubt.")

# §4.5 — same number
sub("with an exclusion bound of 0.147 percentage points against 0.152",
    "with an exclusion bound of 0.153 percentage points against 0.152")

# §6.3 — the drop-u figure in the instrument-sensitivity paragraph
sub("dropping $u$ to $-0.077$ with a bound of 0.147.",
    "dropping $u$ to $-0.077$ with a bound of 0.153.")

# §5.5 — all four subsample rows are more negative, not two
sub("and in two of them a more negative point estimate as well",
    "and in each a more negative point estimate as well")

open("main_v3.tex","w").write(t)
print(f"{n}/4 applied")
print("remaining 0.147:", t.count("0.147"))

4/4 applied
remaining 0.147: 0


In [66]:
!cp main_v3.tex /content/drive/MyDrive/pollution_research/
from google.colab import files
files.download("main_v3.tex")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!cp panel_window_final.csv almaty_pm25_daily_window.csv kase_official.csv /content/drive/MyDrive/pollution_research/
!ls /content/drive/MyDrive/pollution_research/


 almaty_pm25_daily.csv	        fig5_specification.png
 almaty_pm25_daily_window.csv   kase_clean.csv
 almaty_pm25_hourly.csv        'KASE Historical Data.csv'
 almaty_weather_daily.csv       kase_official.csv
 fig1_timeseries.png	        panel_analysis.csv
 fig2_seasonality.png	        panel_final.csv
 fig3_firststage.png	        panel_official.csv
 fig4_binscatter.png	        panel_window_final.csv


In [1]:
import pandas as pd
D = pd.read_csv("/content/drive/MyDrive/pollution_research/panel_cds.csv")

print("lockdown days:", int(D.lockdown.sum()))
print("  dates:", D[D.lockdown==1].date.min(), "to", D[D.lockdown==1].date.max())
print("qantar days:", int(D.qantar.sum()))
print("  dates:", D[D.qantar==1].date.min(), "to", D[D.qantar==1].date.max())

lockdown days: 36
  dates: 2020-04-10 to 2020-08-14
qantar days: 6
  dates: 2022-01-11 to 2022-01-19


In [2]:
from linearmodels.iv import IV2SLS
INST = ["log_blh", "u", "v", "pressure"]
ctrl = [c for c in D.columns
        if c.startswith(("mo_", "dw_", "yr_")) and D[c].nunique() > 1]
ctrl += ["temp", "precip", "wind_speed", "cloud", "radiation", "lockdown", "qantar"]

r = IV2SLS(D["logret"], D[["const"] + ctrl], D["log_pm25"], D[INST]).fit(cov_type="kernel")
print(round(r.params["log_pm25"], 4), round(r.std_errors["log_pm25"], 4))

ModuleNotFoundError: No module named 'linearmodels'

In [3]:
!pip install linearmodels -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 10.2 MB/s eta 0:00:00


In [4]:
from linearmodels.iv import IV2SLS
INST = ["log_blh", "u", "v", "pressure"]
ctrl = [c for c in D.columns
        if c.startswith(("mo_", "dw_", "yr_")) and D[c].nunique() > 1]
ctrl += ["temp", "precip", "wind_speed", "cloud", "radiation", "lockdown", "qantar"]

r = IV2SLS(D["logret"], D[["const"] + ctrl], D["log_pm25"], D[INST]).fit(cov_type="kernel")
print(round(r.params["log_pm25"], 4), round(r.std_errors["log_pm25"], 4))

ValueError: regressors [exog endog] do not have full column rank

In [5]:
import numpy as np
from linearmodels.iv import IV2SLS

INST = ["log_blh", "u", "v", "pressure"]
cand = [c for c in D.columns
        if c.startswith(("mo_", "dw_", "yr_")) and D[c].nunique() > 1]
cand += ["temp", "precip", "wind_speed", "cloud", "radiation", "lockdown", "qantar"]

ctrl = []
for c in cand:
    M = D[["const"] + ctrl + [c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]:
        ctrl.append(c)

r = IV2SLS(D["logret"], D[["const"] + ctrl], D["log_pm25"], D[INST]).fit(cov_type="kernel")
print(round(r.params["log_pm25"], 4), round(r.std_errors["log_pm25"], 4))

-0.0826 0.0832


In [1]:
import shutil, os
from google.colab import files

os.makedirs("deposit", exist_ok=True)
D = "/content/drive/MyDrive/pollution_research"

for f in ["panel_cds.csv", "almaty_weather_daily_cds.csv",
          "almaty_pm25_hourly.csv", "almaty_pm25_daily_window.csv"]:
    shutil.copy(f"{D}/{f}", f"deposit/{f}")

print("copied:", os.listdir("deposit"))
shutil.make_archive("kase_deposit", "zip", "deposit")
files.download("kase_deposit.zip")

copied: ['almaty_pm25_daily_window.csv', 'panel_cds.csv', 'almaty_weather_daily_cds.csv', 'almaty_pm25_hourly.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
!cp kase_deposit.zip /content/drive/MyDrive/pollution_research/
print("on Drive now")


on Drive now


In [ ]:
"""
REGENERATE ALL FIVE FIGURES on the window panel.
Run after rebuild_window_final.py, in the same directory (needs
panel_window_final.csv and almaty_pm25_daily_window.csv).

Saves fig1_timeseries.png ... fig5_specification.png at 300 dpi with the
exact filenames main_revised.tex expects.

Also prints, at the end, every number the captions and the %VERIFY markers
need: BLH monthly means, sample endpoints, the Figure 3 bivariate slope,
and the Figure 4 zero-containing-interval count. Paste that block back.
"""
import numpy as np, pandas as pd, warnings
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
})

D = pd.read_csv("panel_window_final.csv"); D["date"] = pd.to_datetime(D["date"])
W = pd.read_csv("almaty_pm25_daily_window.csv"); W["date"] = pd.to_datetime(W["date"])

INST = ["log_blh", "u", "v", "pressure"]
keep = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_")) and D[c].nunique() > 1] + \
       [c for c in ["temp", "precip", "wind_speed", "cloud", "radiation",
                    "lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]
kk = []
for c in keep:
    M = D[["const"] + kk + [c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]: kk.append(c)
keep = kk
CTRL = " + ".join(keep)

# =============================================== FIG 1 — daily series
# Full filtered set (>=8 in-window obs), NOT just the estimation panel,
# so the plot shows coverage gaps as gaps.
F = W[W.n_window >= 8].copy()
F = F[(F.date >= "2020-04-08") & (F.date <= "2025-11-14")].sort_values("date")
filt_mean = F.pm25.mean()

# insert NaN rows over gaps so the line breaks instead of interpolating
full = pd.DataFrame({"date": pd.date_range(F.date.min(), F.date.max(), freq="D")})
F = full.merge(F[["date", "pm25"]], on="date", how="left")
gap = F.pm25.isna()
F.loc[gap, "pm25"] = np.nan

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(F.date, F.pm25, lw=0.6, color="#2b4c7e")
ax.axhline(5, ls="--", lw=1, color="green", label="WHO annual guideline (5)")
ax.axhline(filt_mean, ls="--", lw=1, color="crimson",
           label=f"Sample mean ({filt_mean:.1f})")
ax.set_ylabel("PM$_{2.5}$ ($\\mu$g/m$^3$)")
ax.set_xlabel("")
ax.legend(frameon=False, fontsize=8, loc="upper right")
fig.savefig("fig1_timeseries.png"); plt.close(fig)

# =============================================== FIG 2 — seasonality
m = D.groupby(D.date.dt.month).agg(pm=("pm25_raw", "mean"), blh=("blh", "mean"))
months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].bar(range(1, 13), m.pm.reindex(range(1, 13)), color="#2b4c7e")
axes[0].set_xticks(range(1, 13)); axes[0].set_xticklabels(months, fontsize=7)
axes[0].set_ylabel("PM$_{2.5}$ ($\\mu$g/m$^3$)"); axes[0].set_title("Pollution", fontsize=9)
axes[1].bar(range(1, 13), m.blh.reindex(range(1, 13)), color="#7e5a2b")
axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(months, fontsize=7)
axes[1].set_ylabel("Boundary layer height (m)"); axes[1].set_title("Mixing depth", fontsize=9)
fig.tight_layout(); fig.savefig("fig2_seasonality.png"); plt.close(fig)

# =============================================== FIG 3 — first stage
rx = smf.ols("log_blh ~ " + CTRL, data=D).fit().resid
ry = smf.ols("log_pm25 ~ " + CTRL, data=D).fit().resid
slope, intercept = np.polyfit(rx, ry, 1)
fig, ax = plt.subplots(figsize=(5.6, 4.2))
ax.scatter(rx, ry, s=5, alpha=0.35, color="#2b4c7e", edgecolors="none")
xs = np.linspace(rx.min(), rx.max(), 100)
ax.plot(xs, intercept + slope * xs, color="crimson", lw=1.6)
ax.set_xlabel("Residualised log boundary layer height")
ax.set_ylabel("Residualised log PM$_{2.5}$")
ax.text(0.03, 0.05, f"slope = {slope:.3f}", transform=ax.transAxes, fontsize=8)
fig.savefig("fig3_firststage.png"); plt.close(fig)

# =============================================== FIG 4 — binscatter
bx = smf.ols("log_pm25 ~ " + CTRL, data=D).fit().resid
by = smf.ols("logret ~ " + CTRL, data=D).fit().resid
B = pd.DataFrame({"x": bx, "y": by})
B["bin"] = pd.qcut(B.x, 20, labels=False)
g = B.groupby("bin").agg(x=("x", "mean"), y=("y", "mean"),
                         se=("y", lambda s: s.std(ddof=1) / np.sqrt(len(s))))
g["lo"], g["hi"] = g.y - 1.96 * g.se, g.y + 1.96 * g.se
n_zero = int(((g.lo <= 0) & (g.hi >= 0)).sum())
ols_slope = smf.ols("logret ~ log_pm25 + " + CTRL, data=D
                    ).fit(cov_type="HAC", cov_kwds={"maxlags": 5}).params["log_pm25"]

fig, ax = plt.subplots(figsize=(6.0, 4.2))
ax.errorbar(g.x, g.y, yerr=[g.y - g.lo, g.hi - g.y], fmt="o", ms=4,
            color="#2b4c7e", ecolor="#9aa8bd", elw=1, capsize=2)
xs = np.linspace(g.x.min(), g.x.max(), 100)
ax.plot(xs, ols_slope * xs + (g.y.mean() - ols_slope * g.x.mean()),
        color="crimson", lw=1.6, label=f"OLS slope = {ols_slope:.3f}")
ax.axhline(0, color="grey", lw=0.6)
ax.set_xlabel("Residualised log PM$_{2.5}$")
ax.set_ylabel("Residualised daily log return (\\%)")
ax.legend(frameon=False, fontsize=8)
fig.savefig("fig4_binscatter.png"); plt.close(fig)

# =============================================== FIG 5 — specification chart
def iv(df, inst=INST, weather_drop=False):
    k = [c for c in keep if not (weather_drop and c in ("cloud", "radiation"))]
    r = IV2SLS(df["logret"], df[["const"] + k], df["log_pm25"], df[inst]).fit(cov_type="kernel")
    return r.params["log_pm25"], r.std_errors["log_pm25"]

specs = [
    ("Preferred",                 lambda: iv(D)),
    ("Coverage $\\geq$15h",       lambda: iv(D[D.n_daily >= 15])),
    ("Full 11h window",           lambda: iv(D[D.date.isin(W[W.n_window == 11].date)])),
    ("From 2022",                 lambda: iv(D[D.year >= 2022])),
    ("From 2023",                 lambda: iv(D[D.year >= 2023])),
    ("Winter only",               lambda: iv(D[D.month.isin([11, 12, 1, 2, 3])])),
    ("Winsorised 1\\%",           lambda: iv(D.assign(logret=D.logret.clip(*D.logret.quantile([.01, .99]))))),
    ("Excl.\\ $v$",               lambda: iv(D, inst=["log_blh", "u", "pressure"])),
    ("Excl.\\ $u$",               lambda: iv(D, inst=["log_blh", "v", "pressure"])),
    ("BLH + pressure",            lambda: iv(D, inst=["log_blh", "pressure"])),
    ("BLH alone",                 lambda: iv(D, inst=["log_blh"])),
    ("Excl.\\ cloud, radiation",  lambda: iv(D, weather_drop=True)),
    ("Session-only treatment",    lambda: iv(D.assign(log_pm25=np.log(D.pm_session)).replace([np.inf, -np.inf], np.nan).dropna(subset=["log_pm25"]))),
    ("Calendar-day treatment",    lambda: iv(D.assign(log_pm25=np.log(D.pm25_calendar)))),
]
labels, bs, ses = [], [], []
for lab, fn in specs:
    try:
        b_, s_ = fn(); labels.append(lab); bs.append(b_); ses.append(s_)
    except Exception as e:
        print("skipped", lab, e)
bs, ses = np.array(bs), np.array(ses)

fig, ax = plt.subplots(figsize=(6.4, 5.0))
ypos = np.arange(len(labels))[::-1]
cols = ["crimson" if l == "Preferred" else "#2b4c7e" for l in labels]
ax.errorbar(bs, ypos, xerr=1.96 * ses, fmt="o", ms=4, elw=1, capsize=2,
            ecolor="#9aa8bd", linestyle="none")
for x_, y_, c_ in zip(bs, ypos, cols):
    ax.plot(x_, y_, "o", ms=5, color=c_)
ax.axvline(0, color="grey", lw=0.8)
ax.set_yticks(ypos); ax.set_yticklabels(labels, fontsize=7.5)
ax.set_xlabel("Coefficient on log PM$_{2.5}$")
fig.savefig("fig5_specification.png"); plt.close(fig)

# =============================================== CAPTION VALUES
print("=" * 62)
print("VALUES FOR THE CAPTIONS AND %VERIFY MARKERS")
print("=" * 62)
print(f"[Fig 1] mean over filtered set (>=8 in window): {filt_mean:.1f}")
print(f"[Fig 3] bivariate residualised slope: {slope:.3f}")
print(f"[Fig 4] OLS slope: {ols_slope:.4f}   intervals containing zero: {n_zero} of 20")
print(f"[3.4]  sample endpoints: {D.date.min().date()} to {D.date.max().date()}")
print("\n[1.1 / 3.5 / 4.3 / Fig 2] BLH monthly means on the estimation sample:")
for mo in range(1, 13):
    print(f"   {months[mo-1]}: BLH {m.blh.get(mo, float('nan')):7.1f}   "
          f"PM2.5 {m.pm.get(mo, float('nan')):6.2f}   n={int((D.month==mo).sum())}")
lo_mo, hi_mo = m.blh.idxmin(), m.blh.idxmax()
print(f"   minimum {months[lo_mo-1]} {m.blh.min():.1f}   maximum {months[hi_mo-1]} "
      f"{m.blh.max():.1f}   amplitude {m.blh.max()/m.blh.min():.1f}x")
print("\nSaved: fig1_timeseries.png fig2_seasonality.png fig3_firststage.png "
      "fig4_binscatter.png fig5_specification.png")


AttributeError: Line2D.set() got an unexpected keyword argument 'elw'

In [ ]:
"""
REGENERATE ALL FIVE FIGURES — corrected, drop-in for Colab.
Needs panel_window_final.csv and almaty_pm25_daily_window.csv in the
working directory (both written by rebuild_window_final.py).

Saves fig1_timeseries.png ... fig5_specification.png at 300 dpi under the
exact filenames main_revised.tex expects, then prints every number the
captions and %VERIFY markers still need. Paste that final block back.

Fixes over the previous version: elinewidth (not elw); each figure wrapped
so one plotting failure cannot lose the others; explicit float casts around
the binscatter aggregation.
"""
# !pip install linearmodels -q

import numpy as np, pandas as pd, warnings, traceback
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
})

D = pd.read_csv("panel_window_final.csv"); D["date"] = pd.to_datetime(D["date"])
W = pd.read_csv("almaty_pm25_daily_window.csv"); W["date"] = pd.to_datetime(W["date"])

INST = ["log_blh", "u", "v", "pressure"]
keep = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_")) and D[c].nunique() > 1] + \
       [c for c in ["temp", "precip", "wind_speed", "cloud", "radiation",
                    "lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]
kk = []
for c in keep:
    M = D[["const"] + kk + [c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]: kk.append(c)
keep = kk
CTRL = " + ".join(keep)

MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
out = {}                      # collects caption values

# ============================================================ FIG 1
try:
    F = W[W.n_window >= 8].copy()
    F = F[(F.date >= "2020-04-08") & (F.date <= "2025-11-14")].sort_values("date")
    out["fig1_mean"] = float(F.pm25.mean())

    full = pd.DataFrame({"date": pd.date_range(F.date.min(), F.date.max(), freq="D")})
    P = full.merge(F[["date", "pm25"]], on="date", how="left")   # gaps stay NaN

    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.plot(P.date, P.pm25, lw=0.6, color="#2b4c7e")
    ax.axhline(5, ls="--", lw=1, color="green", label="WHO annual guideline (5)")
    ax.axhline(out["fig1_mean"], ls="--", lw=1, color="crimson",
               label=f"Sample mean ({out['fig1_mean']:.1f})")
    ax.set_ylabel("PM$_{2.5}$ ($\\mu$g/m$^3$)")
    ax.legend(frameon=False, fontsize=8, loc="upper right")
    fig.savefig("fig1_timeseries.png"); plt.close(fig)
    print("fig1 saved")
except Exception:
    traceback.print_exc()

# ============================================================ FIG 2
try:
    m = D.groupby(D.date.dt.month).agg(pm=("pm25_raw", "mean"), blh=("blh", "mean"))
    out["monthly"] = m
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    axes[0].bar(range(1, 13), m.pm.reindex(range(1, 13)).values, color="#2b4c7e")
    axes[0].set_xticks(range(1, 13)); axes[0].set_xticklabels(MONTHS, fontsize=7)
    axes[0].set_ylabel("PM$_{2.5}$ ($\\mu$g/m$^3$)")
    axes[0].set_title("Pollution", fontsize=9)
    axes[1].bar(range(1, 13), m.blh.reindex(range(1, 13)).values, color="#7e5a2b")
    axes[1].set_xticks(range(1, 13)); axes[1].set_xticklabels(MONTHS, fontsize=7)
    axes[1].set_ylabel("Boundary layer height (m)")
    axes[1].set_title("Mixing depth", fontsize=9)
    fig.tight_layout(); fig.savefig("fig2_seasonality.png"); plt.close(fig)
    print("fig2 saved")
except Exception:
    traceback.print_exc()

# ============================================================ FIG 3
try:
    rx = np.asarray(smf.ols("log_blh ~ " + CTRL, data=D).fit().resid, dtype=float)
    ry = np.asarray(smf.ols("log_pm25 ~ " + CTRL, data=D).fit().resid, dtype=float)
    slope, intercept = np.polyfit(rx, ry, 1)
    out["fig3_slope"] = float(slope)

    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    ax.scatter(rx, ry, s=5, alpha=0.35, color="#2b4c7e", edgecolors="none")
    xs = np.linspace(rx.min(), rx.max(), 100)
    ax.plot(xs, intercept + slope * xs, color="crimson", lw=1.6)
    ax.set_xlabel("Residualised log boundary layer height")
    ax.set_ylabel("Residualised log PM$_{2.5}$")
    ax.text(0.03, 0.05, f"slope = {slope:.3f}", transform=ax.transAxes, fontsize=8)
    fig.savefig("fig3_firststage.png"); plt.close(fig)
    print("fig3 saved")
except Exception:
    traceback.print_exc()

# ============================================================ FIG 4
try:
    bx = np.asarray(smf.ols("log_pm25 ~ " + CTRL, data=D).fit().resid, dtype=float)
    by = np.asarray(smf.ols("logret ~ " + CTRL, data=D).fit().resid, dtype=float)
    B = pd.DataFrame({"x": bx, "y": by})
    B["bin"] = pd.qcut(B.x, 20, labels=False)
    g = B.groupby("bin").agg(x=("x", "mean"), y=("y", "mean"),
                             sd=("y", "std"), n=("y", "size"))
    g["se"] = g.sd / np.sqrt(g.n)
    g["lo"], g["hi"] = g.y - 1.96 * g.se, g.y + 1.96 * g.se
    out["fig4_zero"] = int(((g.lo <= 0) & (g.hi >= 0)).sum())

    ols_slope = float(smf.ols("logret ~ log_pm25 + " + CTRL, data=D
                              ).fit(cov_type="HAC", cov_kwds={"maxlags": 5}
                                    ).params["log_pm25"])
    out["fig4_slope"] = ols_slope

    xv = g.x.to_numpy(dtype=float); yv = g.y.to_numpy(dtype=float)
    lo = (yv - g.lo.to_numpy(dtype=float)); hi = (g.hi.to_numpy(dtype=float) - yv)

    fig, ax = plt.subplots(figsize=(6.0, 4.2))
    ax.errorbar(xv, yv, yerr=[lo, hi], fmt="o", ms=4, color="#2b4c7e",
                ecolor="#9aa8bd", elinewidth=1, capsize=2, linestyle="none")
    xs = np.linspace(xv.min(), xv.max(), 100)
    ax.plot(xs, ols_slope * xs + (yv.mean() - ols_slope * xv.mean()),
            color="crimson", lw=1.6, label=f"OLS slope = {ols_slope:.3f}")
    ax.axhline(0, color="grey", lw=0.6)
    ax.set_xlabel("Residualised log PM$_{2.5}$")
    ax.set_ylabel("Residualised daily log return (%)")
    ax.legend(frameon=False, fontsize=8)
    fig.savefig("fig4_binscatter.png"); plt.close(fig)
    print("fig4 saved")
except Exception:
    traceback.print_exc()

# ============================================================ FIG 5
try:
    def iv(df, inst=INST, drop_weather=False):
        k = [c for c in keep if not (drop_weather and c in ("cloud", "radiation"))]
        r = IV2SLS(df["logret"], df[["const"] + k], df["log_pm25"], df[inst]
                   ).fit(cov_type="kernel")
        return float(r.params["log_pm25"]), float(r.std_errors["log_pm25"])

    lo_q, hi_q = D.logret.quantile([.01, .99])
    specs = [
        ("Preferred",                lambda: iv(D)),
        ("Coverage $\\geq$15h",      lambda: iv(D[D.n_daily >= 15])),
        ("Full 11h window",          lambda: iv(D[D.date.isin(W[W.n_window == 11].date)])),
        ("From 2022",                lambda: iv(D[D.year >= 2022])),
        ("From 2023",                lambda: iv(D[D.year >= 2023])),
        ("Winter only",              lambda: iv(D[D.month.isin([11, 12, 1, 2, 3])])),
        ("Winsorised 1%",            lambda: iv(D.assign(logret=D.logret.clip(lo_q, hi_q)))),
        ("Excl. $v$",                lambda: iv(D, inst=["log_blh", "u", "pressure"])),
        ("Excl. $u$",                lambda: iv(D, inst=["log_blh", "v", "pressure"])),
        ("BLH + pressure",           lambda: iv(D, inst=["log_blh", "pressure"])),
        ("BLH alone",                lambda: iv(D, inst=["log_blh"])),
        ("Excl. cloud, radiation",   lambda: iv(D, drop_weather=True)),
        ("Session-only treatment",   lambda: iv(D.assign(log_pm25=np.log(D.pm_session.where(D.pm_session > 0))).dropna(subset=["log_pm25"]))),
        ("Calendar-day treatment",   lambda: iv(D.assign(log_pm25=np.log(D.pm25_calendar)))),
    ]
    labels, bs, ses = [], [], []
    for lab, fn in specs:
        try:
            b_, s_ = fn(); labels.append(lab); bs.append(b_); ses.append(s_)
        except Exception as e:
            print("  skipped:", lab, e)
    bs, ses = np.array(bs, dtype=float), np.array(ses, dtype=float)
    ypos = np.arange(len(labels))[::-1].astype(float)

    fig, ax = plt.subplots(figsize=(6.4, 5.0))
    ax.errorbar(bs, ypos, xerr=1.96 * ses, fmt="none",
                ecolor="#9aa8bd", elinewidth=1, capsize=2)
    for x_, y_, l_ in zip(bs, ypos, labels):
        ax.plot(x_, y_, "o", ms=5,
                color="crimson" if l_ == "Preferred" else "#2b4c7e")
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_yticks(ypos); ax.set_yticklabels(labels, fontsize=7.5)
    ax.set_xlabel("Coefficient on log PM$_{2.5}$")
    fig.savefig("fig5_specification.png"); plt.close(fig)
    print("fig5 saved")
except Exception:
    traceback.print_exc()

# ============================================================ CAPTION VALUES
print("\n" + "=" * 62)
print("VALUES FOR THE CAPTIONS AND %VERIFY MARKERS")
print("=" * 62)
if "fig1_mean" in out:
    print(f"[Fig 1] mean over filtered set (>=8 in window): {out['fig1_mean']:.1f}")
if "fig3_slope" in out:
    print(f"[Fig 3] bivariate residualised slope: {out['fig3_slope']:.3f}")
if "fig4_slope" in out:
    print(f"[Fig 4] OLS slope: {out['fig4_slope']:.4f}   "
          f"intervals containing zero: {out['fig4_zero']} of 20")
print(f"[3.4]  sample endpoints: {D.date.min().date()} to {D.date.max().date()}")
if "monthly" in out:
    m = out["monthly"]
    print("\n[1.1 / 3.5 / 4.3 / Fig 2] monthly means on the estimation sample:")
    for mo in range(1, 13):
        print(f"   {MONTHS[mo-1]}: BLH {m.blh.get(mo, float('nan')):7.1f}   "
              f"PM2.5 {m.pm.get(mo, float('nan')):6.2f}   n={int((D.month == mo).sum())}")
    print(f"   minimum {MONTHS[int(m.blh.idxmin())-1]} {m.blh.min():.1f}   "
          f"maximum {MONTHS[int(m.blh.idxmax())-1]} {m.blh.max():.1f}   "
          f"amplitude {m.blh.max()/m.blh.min():.1f}x")
print("\nDone. Download the five PNGs and upload them to Overleaf.")


fig1 saved
fig2 saved
fig3 saved
fig4 saved
  skipped: From 2022 regressors [exog endog] do not have full column rank
  skipped: From 2023 regressors [exog endog] do not have full column rank
  skipped: Winter only regressors [exog endog] do not have full column rank
fig5 saved

VALUES FOR THE CAPTIONS AND %VERIFY MARKERS
[Fig 1] mean over filtered set (>=8 in window): 31.9
[Fig 3] bivariate residualised slope: -0.501
[Fig 4] OLS slope: 0.0126   intervals containing zero: 19 of 20
[3.4]  sample endpoints: 2020-04-10 to 2025-11-13

[1.1 / 3.5 / 4.3 / Fig 2] monthly means on the estimation sample:
   Jan: BLH    61.4   PM2.5  77.61   n=62
   Feb: BLH    65.8   PM2.5  72.79   n=59
   Mar: BLH   128.7   PM2.5  28.99   n=50
   Apr: BLH   238.9   PM2.5  18.49   n=85
   May: BLH   367.0   PM2.5  12.91   n=58
   Jun: BLH   547.0   PM2.5  12.57   n=58
   Jul: BLH   560.4   PM2.5  11.15   n=77
   Aug: BLH   509.0   PM2.5  10.68   n=93
   Sep: BLH   362.0   PM2.5  16.17   n=101
   Oct: BLH   195.

In [ ]:
from google.colab import files
for f in ["fig1_timeseries.png","fig2_seasonality.png","fig3_firststage.png",
          "fig4_binscatter.png","fig5_specification.png"]:
    files.download(f)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("fig3_firststage.png")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
"""
FIGURE 5 ONLY — corrected so subsample rows no longer drop out.

The previous version built the control list once on the full panel, so
subsetting (from 2022, from 2023, winter only) left constant dummy columns
and linearmodels refused the regression. This rebuilds the control set
inside each specification.

Run in the same session as make_figures_v2.py (needs D, W, INST, plt, np,
pd, IV2SLS already defined). If starting fresh, run the load block at the
top of make_figures_v2.py first.

Expected restored rows: From 2022 = -0.139, From 2023 = -0.166,
Winter only = -0.081 (matching Table 6).
"""
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from linearmodels.iv import IV2SLS

def ctrl_for(df):
    """Rebuild a non-collinear control list for whatever subsample is passed."""
    base = [c for c in df.columns
            if c.startswith(("mo_", "dw_", "yr_")) and df[c].nunique() > 1] + \
           [c for c in ["temp", "precip", "wind_speed", "cloud", "radiation",
                        "lockdown", "qantar"]
            if c in df.columns and df[c].nunique() > 1]
    k = []
    for c in base:
        M = df[["const"] + k + [c]].astype(float).values
        if np.linalg.matrix_rank(M) == M.shape[1]:
            k.append(c)
    return k

def iv(df, inst=INST, drop_weather=False):
    k = [c for c in ctrl_for(df) if not (drop_weather and c in ("cloud", "radiation"))]
    r = IV2SLS(df["logret"], df[["const"] + k], df["log_pm25"], df[inst]
               ).fit(cov_type="kernel")
    return float(r.params["log_pm25"]), float(r.std_errors["log_pm25"])

lo_q, hi_q = D.logret.quantile([.01, .99])
specs = [
    ("Preferred",              lambda: iv(D)),
    ("Coverage $\\geq$15h",    lambda: iv(D[D.n_daily >= 15])),
    ("Full 11h window",        lambda: iv(D[D.date.isin(W[W.n_window == 11].date)])),
    ("From 2022",              lambda: iv(D[D.year >= 2022])),
    ("From 2023",              lambda: iv(D[D.year >= 2023])),
    ("Winter only",            lambda: iv(D[D.month.isin([11, 12, 1, 2, 3])])),
    ("Winsorised 1%",          lambda: iv(D.assign(logret=D.logret.clip(lo_q, hi_q)))),
    ("Excl. $v$",              lambda: iv(D, inst=["log_blh", "u", "pressure"])),
    ("Excl. $u$",              lambda: iv(D, inst=["log_blh", "v", "pressure"])),
    ("BLH + pressure",         lambda: iv(D, inst=["log_blh", "pressure"])),
    ("BLH alone",              lambda: iv(D, inst=["log_blh"])),
    ("Excl. cloud, radiation", lambda: iv(D, drop_weather=True)),
    ("Session-only treatment", lambda: iv(D.assign(
        log_pm25=np.log(D.pm_session.where(D.pm_session > 0))).dropna(subset=["log_pm25"]))),
    ("Calendar-day treatment", lambda: iv(D.assign(log_pm25=np.log(D.pm25_calendar)))),
]

labels, bs, ses = [], [], []
for lab, fn in specs:
    try:
        b_, s_ = fn()
        labels.append(lab); bs.append(b_); ses.append(s_)
        print(f"  {lab:<24}{b_:8.4f} ({s_:.4f})")
    except Exception as ex:
        print("  SKIPPED:", lab, ex)

bs, ses = np.array(bs, dtype=float), np.array(ses, dtype=float)
ypos = np.arange(len(labels))[::-1].astype(float)

fig, ax = plt.subplots(figsize=(6.4, 5.0))
ax.errorbar(bs, ypos, xerr=1.96 * ses, fmt="none",
            ecolor="#9aa8bd", elinewidth=1, capsize=2)
for x_, y_, l_ in zip(bs, ypos, labels):
    ax.plot(x_, y_, "o", ms=5, color="crimson" if l_ == "Preferred" else "#2b4c7e")
ax.axvline(0, color="grey", lw=0.8)
ax.set_yticks(ypos); ax.set_yticklabels(labels, fontsize=7.5)
ax.set_xlabel("Coefficient on log PM$_{2.5}$")
ax.grid(True, alpha=0.25, linewidth=0.5)
fig.savefig("fig5_specification.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"\nfig5 saved with {len(labels)} of {len(specs)} specifications")


  Preferred                -0.1053 (0.0915)
  Coverage $\geq$15h       -0.1077 (0.0926)
  Full 11h window          -0.0794 (0.1019)
  From 2022                -0.1390 (0.1141)
  From 2023                -0.1658 (0.1369)
  Winter only              -0.0809 (0.1134)
  Winsorised 1%            -0.1034 (0.0848)
  Excl. $v$                -0.1337 (0.0930)
  Excl. $u$                -0.0953 (0.0967)
  BLH + pressure           -0.1161 (0.0979)
  BLH alone                -0.1168 (0.1006)
  Excl. cloud, radiation   -0.0667 (0.0881)
  Session-only treatment   -0.0885 (0.0844)
  Calendar-day treatment   -0.0990 (0.1138)

fig5 saved with 14 of 14 specifications


In [ ]:
from google.colab import files
files.download("fig5_specification.png")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
!ls -la /content/drive/MyDrive/pollution_research/


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 3993
-rw------- 1 root root   43725 Jul 26 17:41  almaty_pm25_daily.csv
-rw------- 1 root root  133175 Aug  7 04:04  almaty_pm25_daily_window.csv
-rw------- 1 root root 1669844 Jul 26 17:41  almaty_pm25_hourly.csv
-rw------- 1 root root  338712 Jul 26 17:41  almaty_weather_daily.csv
-rw------- 1 root root  131212 Aug  5 12:21  fig1_timeseries.png
-rw------- 1 root root   76995 Aug  5 12:21  fig2_seasonality.png
-rw------- 1 root root  207482 Aug  5 12:21  fig3_firststage.png
-rw------- 1 root root   91950 Aug  5 12:21  fig4_binscatter.png
-rw------- 1 root root   97831 Aug  5 12:21  fig5_specification.png
-rw------- 1 root root   61916 Jul 26 17:41  kase_clean.csv
-rw------- 1 root root  112646 Jul 26 17:41 'KASE Historical Data.csv'
-rw------- 1 root root   44728 Aug  7 04:04  kase_official.csv
-rw------- 1 root root  277264 Jul 26 17:41  panel_analysi

In [ ]:
!cp /content/drive/MyDrive/pollution_research/*.csv .
!ls *.csv



 almaty_pm25_daily.csv	        kase_official.csv
 almaty_pm25_daily_window.csv   panel_analysis.csv
 almaty_pm25_hourly.csv         panel_final.csv
 almaty_weather_daily.csv       panel_official.csv
 kase_clean.csv		        panel_window_final.csv
'KASE Historical Data.csv'


In [ ]:
"""
ROBUST DIAGNOSTICS — the last three computations the paper needs.

Run in Colab with panel_window_final.csv present.

Produces:
  1. Kleibergen-Paap rk Wald F and Hansen's J (heteroskedasticity- and
     autocorrelation-robust analogues of the first-stage F and Sargan test
     the paper currently reports). Replaces the concession in footnote 4.
  2. IV standard error at 0, 5 and 20 HAC lags, with the resulting TOST
     bounds -- every bound in the paper rests on this number, but §4.7
     currently reports lag sensitivity for OLS only.
  3. Descriptive statistics for the wind components u and v, and for
     log PM2.5, for the Table 1 additions.

Paste the printed block back.
"""
# !pip install linearmodels -q
import numpy as np, pandas as pd, warnings
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

D = pd.read_csv("panel_window_final.csv"); D["date"] = pd.to_datetime(D["date"])
INST = ["log_blh", "u", "v", "pressure"]
keep = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_")) and D[c].nunique() > 1] + \
       [c for c in ["temp", "precip", "wind_speed", "cloud", "radiation",
                    "lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]
kk = []
for c in keep:
    M = D[["const"] + kk + [c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]: kk.append(c)
keep = kk
LN2 = np.log(2)

print("=" * 64)
print("1. ROBUST WEAK-INSTRUMENT AND OVERIDENTIFICATION STATISTICS")
print("=" * 64)
r_k = IV2SLS(D["logret"], D[["const"] + keep], D["log_pm25"], D[INST]).fit(cov_type="kernel")

# linearmodels reports Kleibergen-Paap when cov is robust/kernel
try:
    diag = r_k.first_stage.diagnostics
    print(diag.T.to_string())
except Exception as ex:
    print("first-stage diagnostics unavailable:", ex)

for name, attr in [("Sargan (classical)", "sargan"), ("Wooldridge score (robust)", "wooldridge_overid"),
                   ("Basmann", "basmann"), ("Anderson-Rubin overid", "anderson_rubin"),
                   ("Wooldridge regression (robust exogeneity)", "wooldridge_regression"),
                   ("Wu-Hausman (classical)", "wu_hausman"), ("Durbin", "durbin")]:
    try:
        obj = getattr(r_k, attr)
        obj = obj() if callable(obj) else obj
        print(f"  {name:<42}{float(obj.stat):8.3f}   p = {float(obj.pval):.4f}")
    except Exception:
        pass

# Kleibergen-Paap rk Wald F explicitly, via robust first-stage Wald test
try:
    import statsmodels.formula.api as smf
    fs = smf.ols("log_pm25 ~ " + " + ".join(INST + keep), data=D
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    W = fs.f_test(", ".join(f"{i} = 0" for i in INST))
    print(f"\n  HAC-robust first-stage Wald F on excluded instruments: {float(W.fvalue):.1f}"
          f"   p = {float(W.pvalue):.3g}")
    print("  (report this alongside, or instead of, the classical F of 175.7)")
except Exception as ex:
    print("robust first-stage F failed:", ex)

print("\n" + "=" * 64)
print("2. IV STANDARD ERROR AND BOUND BY HAC LAG LENGTH")
print("=" * 64)
for L in (0, 5, 20):
    try:
        if L == 0:
            r = IV2SLS(D["logret"], D[["const"] + keep], D["log_pm25"], D[INST]
                       ).fit(cov_type="robust")
            lab = "robust, no lags"
        else:
            r = IV2SLS(D["logret"], D[["const"] + keep], D["log_pm25"], D[INST]
                       ).fit(cov_type="kernel", kernel="bartlett", bandwidth=L)
            lab = f"kernel, {L} lags"
        b_, s_ = float(r.params["log_pm25"]), float(r.std_errors["log_pm25"])
        print(f"  {lab:<20}coef {b_:8.4f}   SE {s_:.4f}   "
              f"bound {(abs(b_) + 1.645 * s_) * LN2:.3f} pp")
    except Exception as ex:
        print(f"  lag {L}: {ex}")
print("  (paper's preferred: coef -0.1053, SE 0.0915, bound 0.177)")

print("\n" + "=" * 64)
print("3. TABLE 1 ADDITIONS")
print("=" * 64)
D["log_pm25_v"] = np.log(D.pm25_raw)
for col, lab in [("u", "Wind component u (E-W, m/s)"),
                 ("v", "Wind component v (N-S, m/s)"),
                 ("log_pm25_v", "Log PM2.5")]:
    x = D[col]
    print(f"  {lab:<30}mean {x.mean():8.2f}  sd {x.std():7.2f}  min {x.min():8.2f}  "
          f"med {x.median():7.2f}  max {x.max():8.2f}")


ModuleNotFoundError: No module named 'linearmodels'

In [ ]:
!pip install linearmodels -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 4.7 MB/s eta 0:00:00


In [ ]:
"""
ROBUST DIAGNOSTICS — the last three computations the paper needs.

Run in Colab with panel_window_final.csv present.

Produces:
  1. Kleibergen-Paap rk Wald F and Hansen's J (heteroskedasticity- and
     autocorrelation-robust analogues of the first-stage F and Sargan test
     the paper currently reports). Replaces the concession in footnote 4.
  2. IV standard error at 0, 5 and 20 HAC lags, with the resulting TOST
     bounds -- every bound in the paper rests on this number, but §4.7
     currently reports lag sensitivity for OLS only.
  3. Descriptive statistics for the wind components u and v, and for
     log PM2.5, for the Table 1 additions.

Paste the printed block back.
"""
# !pip install linearmodels -q
import numpy as np, pandas as pd, warnings
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

D = pd.read_csv("panel_window_final.csv"); D["date"] = pd.to_datetime(D["date"])
INST = ["log_blh", "u", "v", "pressure"]
keep = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_")) and D[c].nunique() > 1] + \
       [c for c in ["temp", "precip", "wind_speed", "cloud", "radiation",
                    "lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]
kk = []
for c in keep:
    M = D[["const"] + kk + [c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]: kk.append(c)
keep = kk
LN2 = np.log(2)

print("=" * 64)
print("1. ROBUST WEAK-INSTRUMENT AND OVERIDENTIFICATION STATISTICS")
print("=" * 64)
r_k = IV2SLS(D["logret"], D[["const"] + keep], D["log_pm25"], D[INST]).fit(cov_type="kernel")

# linearmodels reports Kleibergen-Paap when cov is robust/kernel
try:
    diag = r_k.first_stage.diagnostics
    print(diag.T.to_string())
except Exception as ex:
    print("first-stage diagnostics unavailable:", ex)

for name, attr in [("Sargan (classical)", "sargan"), ("Wooldridge score (robust)", "wooldridge_overid"),
                   ("Basmann", "basmann"), ("Anderson-Rubin overid", "anderson_rubin"),
                   ("Wooldridge regression (robust exogeneity)", "wooldridge_regression"),
                   ("Wu-Hausman (classical)", "wu_hausman"), ("Durbin", "durbin")]:
    try:
        obj = getattr(r_k, attr)
        obj = obj() if callable(obj) else obj
        print(f"  {name:<42}{float(obj.stat):8.3f}   p = {float(obj.pval):.4f}")
    except Exception:
        pass

# Kleibergen-Paap rk Wald F explicitly, via robust first-stage Wald test
try:
    import statsmodels.formula.api as smf
    fs = smf.ols("log_pm25 ~ " + " + ".join(INST + keep), data=D
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    W = fs.f_test(", ".join(f"{i} = 0" for i in INST))
    print(f"\n  HAC-robust first-stage Wald F on excluded instruments: {float(W.fvalue):.1f}"
          f"   p = {float(W.pvalue):.3g}")
    print("  (report this alongside, or instead of, the classical F of 175.7)")
except Exception as ex:
    print("robust first-stage F failed:", ex)

print("\n" + "=" * 64)
print("2. IV STANDARD ERROR AND BOUND BY HAC LAG LENGTH")
print("=" * 64)
for L in (0, 5, 20):
    try:
        if L == 0:
            r = IV2SLS(D["logret"], D[["const"] + keep], D["log_pm25"], D[INST]
                       ).fit(cov_type="robust")
            lab = "robust, no lags"
        else:
            r = IV2SLS(D["logret"], D[["const"] + keep], D["log_pm25"], D[INST]
                       ).fit(cov_type="kernel", kernel="bartlett", bandwidth=L)
            lab = f"kernel, {L} lags"
        b_, s_ = float(r.params["log_pm25"]), float(r.std_errors["log_pm25"])
        print(f"  {lab:<20}coef {b_:8.4f}   SE {s_:.4f}   "
              f"bound {(abs(b_) + 1.645 * s_) * LN2:.3f} pp")
    except Exception as ex:
        print(f"  lag {L}: {ex}")
print("  (paper's preferred: coef -0.1053, SE 0.0915, bound 0.177)")

print("\n" + "=" * 64)
print("3. TABLE 1 ADDITIONS")
print("=" * 64)
D["log_pm25_v"] = np.log(D.pm25_raw)
for col, lab in [("u", "Wind component u (E-W, m/s)"),
                 ("v", "Wind component v (N-S, m/s)"),
                 ("log_pm25_v", "Log PM2.5")]:
    x = D[col]
    print(f"  {lab:<30}mean {x.mean():8.2f}  sd {x.std():7.2f}  min {x.min():8.2f}  "
          f"med {x.median():7.2f}  max {x.max():8.2f}")


1. ROBUST WEAK-INSTRUMENT AND OVERIDENTIFICATION STATISTICS
                    log_pm25
rsquared            0.728493
partial.rsquared    0.214953
shea.rsquared       0.214953
f.stat            175.675009
f.pval                   0.0
f.dist               chi2(4)
  Sargan (classical)                           4.822   p = 0.1853
  Wooldridge score (robust)                    4.055   p = 0.2556
  Basmann                                      4.680   p = 0.1968
  Anderson-Rubin overid                        4.834   p = 0.1844
  Wooldridge regression (robust exogeneity)    1.863   p = 0.1722
  Wu-Hausman (classical)                       3.327   p = 0.0685
  Durbin                                       3.426   p = 0.0642

  HAC-robust first-stage Wald F on excluded instruments: 44.9   p = 2.13e-34
  (report this alongside, or instead of, the classical F of 175.7)

2. IV STANDARD ERROR AND BOUND BY HAC LAG LENGTH
  robust, no lags     coef  -0.1053   SE 0.0937   bound 0.180 pp
  kernel, 5 lag

In [ ]:
"""
FIRST-STAGE F ON THE CORRECT BASIS — every specification in the paper.

Why this is needed: linearmodels' first_stage.diagnostics reports 'f.stat'
with f.dist = chi2(k), i.e. a Wald chi-squared on k excluded instruments,
NOT an F. Stock-Yogo critical values are F-based. Dividing by k gives the
F-equivalent, and the HAC-robust Wald F computed directly from the first
stage is the statistic that should actually be reported alongside HAC
standard errors.

This prints, for every row of Table 6 and for the main and placebo specs:
  k          number of excluded instruments
  chi2       what the paper currently prints as "F"
  chi2/k     the F-equivalent of that statistic
  robust F   HAC-robust Wald F on the excluded instruments (REPORT THIS)

Paste the whole output back.
"""
# !pip install linearmodels -q
import numpy as np, pandas as pd, warnings
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

D = pd.read_csv("panel_window_final.csv"); D["date"] = pd.to_datetime(D["date"])
W = pd.read_csv("almaty_pm25_daily_window.csv"); W["date"] = pd.to_datetime(W["date"])
INST = ["log_blh", "u", "v", "pressure"]
WEATHER = ["temp", "precip", "wind_speed", "cloud", "radiation"]

def ctrl_for(df, weather=WEATHER):
    base = list(weather) + \
           [c for c in df.columns if c.startswith(("mo_", "dw_", "yr_")) and df[c].nunique() > 1] + \
           [c for c in ["lockdown", "qantar"] if c in df.columns and df[c].nunique() > 1]
    base = [c for c in base if c in df.columns and df[c].nunique() > 1]
    k = []
    for c in base:
        M = df[["const"] + k + [c]].astype(float).values
        if np.linalg.matrix_rank(M) == M.shape[1]: k.append(c)
    return k

def fstats(df, inst=INST, endog="log_pm25", weather=WEATHER, lags=5):
    """Return (chi2 as linearmodels prints it, chi2/k, HAC-robust Wald F, k)."""
    keep = ctrl_for(df, weather)
    r = IV2SLS(df["logret"], df[["const"] + keep], df[endog], df[inst]).fit(cov_type="kernel")
    try:
        chi2 = float(r.first_stage.diagnostics["f.stat"].iloc[0])
    except Exception:
        chi2 = float("nan")
    fs = smf.ols(f"{endog} ~ " + " + ".join(inst + keep), data=df
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    Wt = fs.f_test(", ".join(f"{i} = 0" for i in inst))
    return chi2, chi2 / len(inst), float(Wt.fvalue), len(inst)

rows = [
    ("Preferred",                    lambda: fstats(D)),
    ("Coverage >= 18h (daily)",      lambda: fstats(D[D.n_daily >= 18])),
    ("Coverage >= 15h (daily)",      lambda: fstats(D[D.n_daily >= 15])),
    ("Full 11h window",              lambda: fstats(D[D.date.isin(W[W.n_window == 11].date)])),
    ("Sample from 2022",             lambda: fstats(D[D.year >= 2022])),
    ("Sample from 2023",             lambda: fstats(D[D.year >= 2023])),
    ("Excluding 2025",               lambda: fstats(D[D.year <= 2024])),
    ("Pre-2024 clock",               lambda: fstats(D[D.date < "2024-03-01"])),
    ("Winter only",                  lambda: fstats(D[D.month.isin([11, 12, 1, 2, 3])])),
    ("Winsorised 1%",                lambda: fstats(D.assign(
                                        logret=D.logret.clip(*D.logret.quantile([.01, .99]))))),
    ("Instr excl pressure",          lambda: fstats(D, inst=["log_blh", "u", "v"])),
    ("Instr excl v",                 lambda: fstats(D, inst=["log_blh", "u", "pressure"])),
    ("Instr excl u",                 lambda: fstats(D, inst=["log_blh", "v", "pressure"])),
    ("BLH + u",                      lambda: fstats(D, inst=["log_blh", "u"])),
    ("BLH + pressure",               lambda: fstats(D, inst=["log_blh", "pressure"])),
    ("BLH alone",                    lambda: fstats(D, inst=["log_blh"])),
    ("Ctrls excl cloud/rad",         lambda: fstats(D, weather=["temp", "precip", "wind_speed"])),
    ("Excl Jan 2022 unrest",         lambda: fstats(D[~((D.date >= "2022-01-04") &
                                                        (D.date <= "2022-01-19"))])),
    ("Dense (Apr 2023 on)",          lambda: fstats(D[D.date >= "2023-04-01"])),
    ("Post-gap (Jul 2024 on)",       lambda: fstats(D[D.date >= "2024-07-01"])),
]

# lag rows
for k in (1, 2, 3):
    def mk(k=k):
        Dk = D.copy()
        for c in ["log_pm25"] + INST: Dk[c] = Dk[c].shift(k)
        Dk = Dk.dropna(subset=["log_pm25"] + INST).reset_index(drop=True)
        return fstats(Dk)
    rows.append((f"Lag {k} day", mk))

rows.append(("Outcome: |return|", lambda: fstats(D.assign(logret=D.logret.abs()))))
rows.append(("Session-only treatment", lambda: fstats(
    D.assign(log_pm25=np.log(D.pm_session.where(D.pm_session > 0))).dropna(subset=["log_pm25"]))))
rows.append(("Pre-session treatment", lambda: fstats(
    D.assign(log_pm25=np.log(D.pm_presession)))))
rows.append(("Calendar-day treatment", lambda: fstats(
    D.assign(log_pm25=np.log(D.pm25_calendar)))))

# placebo leads
for k in (1, 2, 3, 5):
    def mkl(k=k):
        Dk = D.copy(); Dk["lead"] = Dk.logret.shift(-k)
        Dk = Dk.dropna(subset=["lead"]).reset_index(drop=True)
        keep = ctrl_for(Dk)
        r = IV2SLS(Dk["lead"], Dk[["const"] + keep], Dk["log_pm25"], Dk[INST]).fit(cov_type="kernel")
        chi2 = float(r.first_stage.diagnostics["f.stat"].iloc[0])
        fs = smf.ols("log_pm25 ~ " + " + ".join(INST + keep), data=Dk
                     ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
        Wt = fs.f_test(", ".join(f"{i} = 0" for i in INST))
        return chi2, chi2 / len(INST), float(Wt.fvalue), len(INST)
    rows.append((f"{k}-day lead", mkl))

print(f"{'Specification':<28}{'k':>3}{'chi2 (as printed)':>19}{'chi2/k':>10}{'robust F':>11}")
print("-" * 71)
for lab, fn in rows:
    try:
        chi2, fequiv, robF, k = fn()
        print(f"{lab:<28}{k:>3}{chi2:>19.1f}{fequiv:>10.1f}{robF:>11.1f}")
    except Exception as ex:
        print(f"{lab:<28}  failed: {ex}")

print("\nJust-identified per-instrument (Section 6.3):")
for z in INST:
    try:
        chi2, fequiv, robF, k = fstats(D, inst=[z])
        print(f"  {z:<10}chi2 {chi2:8.1f}   chi2/k {fequiv:7.1f}   robust F {robF:7.1f}")
    except Exception as ex:
        print(f"  {z}: {ex}")

print("\nStock-Yogo 10% maximal-size critical values (1 endogenous regressor):")
print("  k=1: 16.38   k=2: 19.93   k=3: 22.30   k=4: 24.58")


Specification                 k  chi2 (as printed)    chi2/k   robust F
-----------------------------------------------------------------------
Preferred                     4              175.7      43.9       44.9
Coverage >= 18h (daily)       4              164.6      41.1       42.8
Coverage >= 15h (daily)       4              176.1      44.0       45.3
Full 11h window               4              130.1      32.5       38.2
Sample from 2022              4              137.0      34.2       35.2
Sample from 2023              4              101.4      25.4       25.9
Excluding 2025                4              156.0      39.0       39.1
Pre-2024 clock                4              183.8      46.0       39.3
Winter only                   4              111.9      28.0       27.6
Winsorised 1%                 4              177.7      44.4       44.9
Instr excl pressure           3              164.7      54.9       56.6
Instr excl v                  3              169.1      56.4    

In [ ]:
"""
FINAL REFEREE CHECKS — four computations, one cell.

Needs panel_window_final.csv.

  1. Randomisation inference redone with shifts in multiples of ONE
     observation (925 distinct shifts, no replacement needed), reporting
     the null MEAN as well as its SD -- the current null's 95% range is
     off-centre (+0.017 midpoint) and the mean is not reported anywhere.
  2. Temperature control functional form: linear (current) vs 2.5-degree
     bins (the benchmark's choice) vs a cubic spline. The exclusion
     restriction is most exposed here, since cold snaps and inversions
     coincide.
  3. Year-by-month fixed effects instead of month + year separately.
  4. Threshold specifications instrumented rather than OLS -- the paper
     currently reports 35/55/75 cut-offs by OLS, which under its own
     argument are not identified.

Paste the whole output back.
"""
# !pip install linearmodels -q
import numpy as np, pandas as pd, warnings
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

D = pd.read_csv("panel_window_final.csv"); D["date"] = pd.to_datetime(D["date"])
INST = ["log_blh", "u", "v", "pressure"]
WEATHER = ["temp", "precip", "wind_speed", "cloud", "radiation"]
LN2 = np.log(2)
RNG = np.random.default_rng(20260807)

def ctrl(df, weather=WEATHER, month_fe=True, year_fe=True, ym_fe=False):
    base = list(weather)
    if ym_fe:
        base += [c for c in df.columns if c.startswith("ym_") and df[c].nunique() > 1]
    else:
        if month_fe: base += [c for c in df.columns if c.startswith("mo_") and df[c].nunique() > 1]
        if year_fe:  base += [c for c in df.columns if c.startswith("yr_") and df[c].nunique() > 1]
    base += [c for c in df.columns if c.startswith("dw_") and df[c].nunique() > 1]
    base += [c for c in ["lockdown", "qantar"] if c in df.columns and df[c].nunique() > 1]
    base += [c for c in df.columns if c.startswith(("tbin_", "tspl_")) and df[c].nunique() > 1]
    k = []
    for c in base:
        M = df[["const"] + k + [c]].astype(float).values
        if np.linalg.matrix_rank(M) == M.shape[1]: k.append(c)
    return k

def run(df, **kw):
    keep = ctrl(df, **kw)
    r = IV2SLS(df["logret"], df[["const"] + keep], df["log_pm25"], df[INST]).fit(cov_type="kernel")
    b, se = float(r.params["log_pm25"]), float(r.std_errors["log_pm25"])
    fs = smf.ols("log_pm25 ~ " + " + ".join(INST + keep), data=df
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    F = float(fs.f_test(", ".join(f"{i} = 0" for i in INST)).fvalue)
    return b, se, (abs(b) + 1.645 * se) * LN2, F, int(r.nobs)

print("=" * 66)
print("1. RANDOMISATION INFERENCE, SHIFTS IN MULTIPLES OF ONE")
print("=" * 66)
keep0 = ctrl(D)
exog, endog, instr = D[["const"] + keep0], D["log_pm25"], D[INST]
b0 = float(IV2SLS(D["logret"], exog, endog, instr).fit(cov_type="kernel").params["log_pm25"])
n = len(D)
draws = []
for s_ in RNG.choice(np.arange(1, n), size=1000, replace=False):
    yb = pd.Series(np.roll(D.logret.values, int(s_)), index=D.index)
    draws.append(float(IV2SLS(yb, exog, endog, instr
                              ).fit(cov_type="unadjusted").params["log_pm25"]))
draws = np.array(draws)
print(f"  distinct shifts available: {n-1}   draws used: {len(draws)} (without replacement)")
print(f"  null mean   {draws.mean():+.4f}   median {np.median(draws):+.4f}")
print(f"  null SD     {draws.std():.4f}    vs HAC SE 0.0915   ratio {draws.std()/0.0915:.2f}")
print(f"  95% range   [{np.percentile(draws,2.5):.3f}, {np.percentile(draws,97.5):.3f}]"
      f"   midpoint {(np.percentile(draws,2.5)+np.percentile(draws,97.5))/2:+.4f}")
print(f"  RI p (|b|)  {np.mean(np.abs(draws) >= abs(b0)):.3f}"
      f"   estimate at percentile {(draws < b0).mean()*100:.1f}")
print("  (five-observation version reported in the paper: mean not reported,"
      " SD 0.0909, range [-0.164, 0.197], p 0.275)")

print("\n" + "=" * 66)
print("2. TEMPERATURE CONTROL FUNCTIONAL FORM")
print("=" * 66)
b, se, bd, F, N = run(D)
print(f"  linear temperature (current)   {b:8.4f} ({se:.4f})  bound {bd:.3f}  F {F:5.1f}  N {N}")

Db = D.copy()
edges = np.arange(np.floor(Db.temp.min()/2.5)*2.5, Db.temp.max()+2.5, 2.5)
lab = pd.cut(Db.temp, bins=edges, labels=False, include_lowest=True)
for j in sorted(pd.Series(lab).dropna().unique())[1:]:
    Db[f"tbin_{int(j)}"] = (lab == j).astype(int)
b, se, bd, F, N = run(Db, weather=["precip", "wind_speed", "cloud", "radiation"])
print(f"  2.5-degree bins (benchmark)    {b:8.4f} ({se:.4f})  bound {bd:.3f}  F {F:5.1f}  N {N}"
      f"   [{sum(c.startswith('tbin_') for c in Db.columns)} bins]")

Ds = D.copy()
for p, q in enumerate(Ds.temp.quantile([.2, .4, .6, .8]).values, start=1):
    Ds[f"tspl_{p}"] = np.clip(Ds.temp - q, 0, None) ** 3
Ds["tspl_0"] = Ds.temp ** 2
b, se, bd, F, N = run(Ds)
print(f"  cubic spline in temperature    {b:8.4f} ({se:.4f})  bound {bd:.3f}  F {F:5.1f}  N {N}")

print("\n" + "=" * 66)
print("3. YEAR-BY-MONTH FIXED EFFECTS")
print("=" * 66)
Dy = D.copy()
ym = Dy.date.dt.year.astype(str) + "-" + Dy.date.dt.month.astype(str).str.zfill(2)
for j, v in enumerate(sorted(ym.unique())[1:]):
    Dy[f"ym_{j}"] = (ym == v).astype(int)
try:
    b, se, bd, F, N = run(Dy, ym_fe=True)
    print(f"  year-by-month FE               {b:8.4f} ({se:.4f})  bound {bd:.3f}  F {F:5.1f}  N {N}"
          f"   [{sum(c.startswith('ym_') for c in Dy.columns)} dummies]")
except Exception as ex:
    print("  year-by-month FE failed:", ex)
print("  (month + year FE, as reported: -0.1053 (0.0915), bound 0.177, F 44.9)")

print("\n" + "=" * 66)
print("4. THRESHOLD SPECIFICATIONS, INSTRUMENTED")
print("=" * 66)
for thr in (35, 55, 75):
    Dt = D.copy(); Dt["hi"] = (Dt.pm25_raw > thr).astype(int)
    keep = ctrl(Dt)
    try:
        r = IV2SLS(Dt["logret"], Dt[["const"] + keep], Dt["hi"], Dt[INST]).fit(cov_type="kernel")
        fs = smf.ols("hi ~ " + " + ".join(INST + keep), data=Dt
                     ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
        F = float(fs.f_test(", ".join(f"{i} = 0" for i in INST)).fvalue)
        print(f"  >{thr}: IV {float(r.params['hi']):8.4f} ({float(r.std_errors['hi']):.4f})"
              f"  p {float(r.pvalues['hi']):.3f}   first-stage F {F:5.1f}   days above {int(Dt.hi.sum())}")
    except Exception as ex:
        print(f"  >{thr}: failed — {ex}")
print("  (OLS versions in the paper: 0.009 / 0.058 / 0.079)")


1. RANDOMISATION INFERENCE, SHIFTS IN MULTIPLES OF ONE


ValueError: Cannot take a larger sample than population when replace is False

In [ ]:
n = len(D)
draws = []
for s_ in RNG.choice(np.arange(1, n), size=1000, replace=False):


SyntaxError: incomplete input (1190584328.py, line 3)

In [ ]:
"""
FINAL REFEREE CHECKS — four computations, one cell.

Needs panel_window_final.csv.

  1. Randomisation inference redone with shifts in multiples of ONE
     observation (925 distinct shifts, no replacement needed), reporting
     the null MEAN as well as its SD -- the current null's 95% range is
     off-centre (+0.017 midpoint) and the mean is not reported anywhere.
  2. Temperature control functional form: linear (current) vs 2.5-degree
     bins (the benchmark's choice) vs a cubic spline. The exclusion
     restriction is most exposed here, since cold snaps and inversions
     coincide.
  3. Year-by-month fixed effects instead of month + year separately.
  4. Threshold specifications instrumented rather than OLS -- the paper
     currently reports 35/55/75 cut-offs by OLS, which under its own
     argument are not identified.

Paste the whole output back.
"""
# !pip install linearmodels -q
import numpy as np, pandas as pd, warnings
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
warnings.filterwarnings("ignore")

D = pd.read_csv("panel_window_final.csv"); D["date"] = pd.to_datetime(D["date"])
INST = ["log_blh", "u", "v", "pressure"]
WEATHER = ["temp", "precip", "wind_speed", "cloud", "radiation"]
LN2 = np.log(2)
RNG = np.random.default_rng(20260807)

def ctrl(df, weather=WEATHER, month_fe=True, year_fe=True, ym_fe=False):
    base = list(weather)
    if ym_fe:
        base += [c for c in df.columns if c.startswith("ym_") and df[c].nunique() > 1]
    else:
        if month_fe: base += [c for c in df.columns if c.startswith("mo_") and df[c].nunique() > 1]
        if year_fe:  base += [c for c in df.columns if c.startswith("yr_") and df[c].nunique() > 1]
    base += [c for c in df.columns if c.startswith("dw_") and df[c].nunique() > 1]
    base += [c for c in ["lockdown", "qantar"] if c in df.columns and df[c].nunique() > 1]
    base += [c for c in df.columns if c.startswith(("tbin_", "tspl_")) and df[c].nunique() > 1]
    k = []
    for c in base:
        M = df[["const"] + k + [c]].astype(float).values
        if np.linalg.matrix_rank(M) == M.shape[1]: k.append(c)
    return k

def run(df, **kw):
    keep = ctrl(df, **kw)
    r = IV2SLS(df["logret"], df[["const"] + keep], df["log_pm25"], df[INST]).fit(cov_type="kernel")
    b, se = float(r.params["log_pm25"]), float(r.std_errors["log_pm25"])
    fs = smf.ols("log_pm25 ~ " + " + ".join(INST + keep), data=df
                 ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    F = float(fs.f_test(", ".join(f"{i} = 0" for i in INST)).fvalue)
    return b, se, (abs(b) + 1.645 * se) * LN2, F, int(r.nobs)

print("=" * 66)
print("1. RANDOMISATION INFERENCE, SHIFTS IN MULTIPLES OF ONE")
print("=" * 66)
keep0 = ctrl(D)
exog, endog, instr = D[["const"] + keep0], D["log_pm25"], D[INST]
b0 = float(IV2SLS(D["logret"], exog, endog, instr).fit(cov_type="kernel").params["log_pm25"])
n = len(D)
draws = []
shifts = np.arange(1, n)          # all 925 distinct shifts — exhaustive, not sampled
for s_ in shifts:

    yb = pd.Series(np.roll(D.logret.values, int(s_)), index=D.index)
    draws.append(float(IV2SLS(yb, exog, endog, instr
                              ).fit(cov_type="unadjusted").params["log_pm25"]))
draws = np.array(draws)
print(f"  distinct shifts: {n-1}   all enumerated (exact randomisation null)")
print(f"  null mean   {draws.mean():+.4f}   median {np.median(draws):+.4f}")
print(f"  null SD     {draws.std():.4f}    vs HAC SE 0.0915   ratio {draws.std()/0.0915:.2f}")
print(f"  95% range   [{np.percentile(draws,2.5):.3f}, {np.percentile(draws,97.5):.3f}]"
      f"   midpoint {(np.percentile(draws,2.5)+np.percentile(draws,97.5))/2:+.4f}")
print(f"  RI p (|b|)  {np.mean(np.abs(draws) >= abs(b0)):.3f}"
      f"   estimate at percentile {(draws < b0).mean()*100:.1f}")
print("  (five-observation version reported in the paper: mean not reported,"
      " SD 0.0909, range [-0.164, 0.197], p 0.275)")

print("\n" + "=" * 66)
print("2. TEMPERATURE CONTROL FUNCTIONAL FORM")
print("=" * 66)
b, se, bd, F, N = run(D)
print(f"  linear temperature (current)   {b:8.4f} ({se:.4f})  bound {bd:.3f}  F {F:5.1f}  N {N}")

Db = D.copy()
edges = np.arange(np.floor(Db.temp.min()/2.5)*2.5, Db.temp.max()+2.5, 2.5)
lab = pd.cut(Db.temp, bins=edges, labels=False, include_lowest=True)
for j in sorted(pd.Series(lab).dropna().unique())[1:]:
    Db[f"tbin_{int(j)}"] = (lab == j).astype(int)
b, se, bd, F, N = run(Db, weather=["precip", "wind_speed", "cloud", "radiation"])
print(f"  2.5-degree bins (benchmark)    {b:8.4f} ({se:.4f})  bound {bd:.3f}  F {F:5.1f}  N {N}"
      f"   [{sum(c.startswith('tbin_') for c in Db.columns)} bins]")

Ds = D.copy()
for p, q in enumerate(Ds.temp.quantile([.2, .4, .6, .8]).values, start=1):
    Ds[f"tspl_{p}"] = np.clip(Ds.temp - q, 0, None) ** 3
Ds["tspl_0"] = Ds.temp ** 2
b, se, bd, F, N = run(Ds)
print(f"  cubic spline in temperature    {b:8.4f} ({se:.4f})  bound {bd:.3f}  F {F:5.1f}  N {N}")

print("\n" + "=" * 66)
print("3. YEAR-BY-MONTH FIXED EFFECTS")
print("=" * 66)
Dy = D.copy()
ym = Dy.date.dt.year.astype(str) + "-" + Dy.date.dt.month.astype(str).str.zfill(2)
for j, v in enumerate(sorted(ym.unique())[1:]):
    Dy[f"ym_{j}"] = (ym == v).astype(int)
try:
    b, se, bd, F, N = run(Dy, ym_fe=True)
    print(f"  year-by-month FE               {b:8.4f} ({se:.4f})  bound {bd:.3f}  F {F:5.1f}  N {N}"
          f"   [{sum(c.startswith('ym_') for c in Dy.columns)} dummies]")
except Exception as ex:
    print("  year-by-month FE failed:", ex)
print("  (month + year FE, as reported: -0.1053 (0.0915), bound 0.177, F 44.9)")

print("\n" + "=" * 66)
print("4. THRESHOLD SPECIFICATIONS, INSTRUMENTED")
print("=" * 66)
for thr in (35, 55, 75):
    Dt = D.copy(); Dt["hi"] = (Dt.pm25_raw > thr).astype(int)
    keep = ctrl(Dt)
    try:
        r = IV2SLS(Dt["logret"], Dt[["const"] + keep], Dt["hi"], Dt[INST]).fit(cov_type="kernel")
        fs = smf.ols("hi ~ " + " + ".join(INST + keep), data=Dt
                     ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
        F = float(fs.f_test(", ".join(f"{i} = 0" for i in INST)).fvalue)
        print(f"  >{thr}: IV {float(r.params['hi']):8.4f} ({float(r.std_errors['hi']):.4f})"
              f"  p {float(r.pvalues['hi']):.3f}   first-stage F {F:5.1f}   days above {int(Dt.hi.sum())}")
    except Exception as ex:
        print(f"  >{thr}: failed — {ex}")
print("  (OLS versions in the paper: 0.009 / 0.058 / 0.079)")


1. RANDOMISATION INFERENCE, SHIFTS IN MULTIPLES OF ONE
  distinct shifts: 925   all enumerated (exact randomisation null)
  null mean   +0.0001   median +0.0031
  null SD     0.0918    vs HAC SE 0.0915   ratio 1.00
  95% range   [-0.175, 0.180]   midpoint +0.0024
  RI p (|b|)  0.243   estimate at percentile 12.6
  (five-observation version reported in the paper: mean not reported, SD 0.0909, range [-0.164, 0.197], p 0.275)

2. TEMPERATURE CONTROL FUNCTIONAL FORM
  linear temperature (current)    -0.1053 (0.0915)  bound 0.177  F  44.9  N 926
  2.5-degree bins (benchmark)     -0.1338 (0.0873)  bound 0.192  F  49.2  N 926   [21 bins]
  cubic spline in temperature     -0.0961 (0.0837)  bound 0.162  F  49.0  N 926

3. YEAR-BY-MONTH FIXED EFFECTS
  year-by-month FE                -0.0674 (0.0961)  bound 0.156  F  64.0  N 926   [57 dummies]
  (month + year FE, as reported: -0.1053 (0.0915), bound 0.177, F 44.9)

4. THRESHOLD SPECIFICATIONS, INSTRUMENTED
  >35: IV  -0.2393 (0.1667)  p 0.151   

In [ ]:
"""
TABLE 7 PERCENTILES against the exact 925-shift randomisation null.

Self-contained: rebuilds the null from panel_window_final.csv, so it works
in a fresh session. Takes a couple of minutes (925 IV fits).

Prints the six percentiles Table 7 needs, plus the null's mean, SD and
95% range so you can confirm they match what the paper now states
(+0.0001, 0.0918, [-0.175, 0.180]).
"""
# !pip install linearmodels -q
import numpy as np, pandas as pd, warnings
from linearmodels.iv import IV2SLS
from scipy import stats
warnings.filterwarnings("ignore")

D = pd.read_csv("panel_window_final.csv"); D["date"] = pd.to_datetime(D["date"])
INST = ["log_blh", "u", "v", "pressure"]
keep = [c for c in D.columns if c.startswith(("mo_", "dw_", "yr_")) and D[c].nunique() > 1] + \
       [c for c in ["temp", "precip", "wind_speed", "cloud", "radiation",
                    "lockdown", "qantar"] if c in D.columns and D[c].nunique() > 1]
kk = []
for c in keep:
    M = D[["const"] + kk + [c]].astype(float).values
    if np.linalg.matrix_rank(M) == M.shape[1]: kk.append(c)
keep = kk

exog, endog, instr = D[["const"] + keep], D["log_pm25"], D[INST]
n = len(D)

print("building exact null over", n - 1, "shifts ...")
draws = []
for s_ in range(1, n):
    yb = pd.Series(np.roll(D.logret.values, s_), index=D.index)
    draws.append(float(IV2SLS(yb, exog, endog, instr
                              ).fit(cov_type="unadjusted").params["log_pm25"]))
draws = np.array(draws)

print(f"\nnull mean {draws.mean():+.4f}   SD {draws.std():.4f}   "
      f"95% range [{np.percentile(draws,2.5):.3f}, {np.percentile(draws,97.5):.3f}]")
print("  (paper states +0.0001, 0.0918, [-0.175, 0.180])\n")

print("TABLE 7 PERCENTILE COLUMN — replace the existing values with these:")
rows = [("Contemporaneous (main estimate)", -0.1053),
        ("1-day lead return",               -0.0840),
        ("1-day lead, cond. on lagged PM",  -0.1153),
        ("2-day lead return",               -0.0724),
        ("3-day lead return",               -0.0727),
        ("5-day lead return",                0.0666)]
for lab, v in rows:
    print(f"  {lab:<34}{stats.percentileofscore(draws, v):5.1f}")
print("\nThe 1-day-lead-conditioned percentile is also quoted in the §5.7 text"
      " (currently 9.5) — update it there too.")


building exact null over 925 shifts ...

null mean +0.0001   SD 0.0918   95% range [-0.175, 0.180]
  (paper states +0.0001, 0.0918, [-0.175, 0.180])

TABLE 7 PERCENTILE COLUMN — replace the existing values with these:
  Contemporaneous (main estimate)    12.6
  1-day lead return                  17.0
  1-day lead, cond. on lagged PM      9.8
  2-day lead return                  21.6
  3-day lead return                  21.4
  5-day lead return                  76.4

The 1-day-lead-conditioned percentile is also quoted in the §5.7 text (currently 9.5) — update it there too.


In [ ]:
import pandas as pd
h = pd.read_csv("almaty_pm25_hourly.csv")
h["hour"] = h["local"].str.slice(11,13).astype(int)
h["date"] = pd.to_datetime(h["local"].str.slice(0,10))
w = h[(h.hour>=7)&(h.hour<=17)]
g = w.groupby("date")["pm25"].agg(n="size", pm="mean")
g = g[(g.index>="2020-04-09")&(g.index<="2025-11-13")&(g.n>=8)]
print(len(g), round(g.pm.mean(), 3))


FileNotFoundError: [Errno 2] No such file or directory: 'almaty_pm25_hourly.csv'

In [ ]:
"""
ERA5 boundary layer height: pull from Copernicus CDS and validate against Open-Meteo.

Purpose: the Jan-Jun 2024 gap in the paper's BLH series is almost certainly an
Open-Meteo archive limitation, not an ERA5 one. This script (a) pulls BLH directly
from the CDS for the full sample period, (b) rebuilds the 07:00-17:59 local-time
daily mean using the same timezone rule as the paper, and (c) compares it against
the existing Open-Meteo series on overlapping dates.

Decision rule: if correlation on overlapping days is > 0.98 and the regression of
CDS on Open-Meteo has slope near 1 with small intercept, the two are the same
quantity and you can rebuild the panel on the CDS series with the gap filled.
If not, you have learned something that needs explaining before any rebuild.

SETUP (one time):
  1. Register at https://cds.climate.copernicus.eu and log in.
  2. Accept the licence on the "ERA5 hourly data on single levels" dataset page.
     Downloads silently fail with a 403 if you skip this.
  3. Copy your API token from your CDS profile page.
  4. In Colab:  !pip install cdsapi xarray netcdf4
  5. Paste the token below.

NOTE ON API DETAILS: the CDS migrated to a new backend and the endpoint URL,
token format and dataset naming have changed at least once. If the request below
errors on authentication or dataset name, check the current example request on
the dataset's "Download" tab, which always shows working code.
"""

import os
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# 0. Configuration
# ----------------------------------------------------------------------------

CDS_URL = "https://cds.climate.copernicus.eu/api"
CDS_KEY = "PASTE_YOUR_TOKEN_HERE"

# Embassy monitor coordinates (paper section 3.1)
LAT, LON = 43.2529, 76.9312

# ERA5 is on a 0.25 deg grid; take a small box around the point so we can
# compare nearest-gridpoint against bilinear interpolation.
AREA = [43.75, 76.50, 42.75, 77.50]   # N, W, S, E

YEARS = [str(y) for y in range(2020, 2026)]
OUTDIR = "era5_blh"
os.makedirs(OUTDIR, exist_ok=True)

# Path to your existing panel with the Open-Meteo BLH daily means.
# Expected columns: a date column and a boundary-layer-height column.
OPENMETEO_PANEL = "panel_window_final.csv"
OM_DATE_COL = "date"
OM_BLH_COL = "blh"          # rename if yours differs

# ----------------------------------------------------------------------------
# 1. Download from CDS, one year at a time
# ----------------------------------------------------------------------------

def download():
    import cdsapi
    c = cdsapi.Client(url=CDS_URL, key=CDS_KEY)

    for year in YEARS:
        target = f"{OUTDIR}/blh_{year}.nc"
        if os.path.exists(target):
            print(f"{target} already present, skipping")
            continue
        print(f"requesting {year} ...")
        c.retrieve(
            "reanalysis-era5-single-levels",
            {
                "product_type": ["reanalysis"],
                "variable": ["boundary_layer_height"],
                "year": [year],
                "month": [f"{m:02d}" for m in range(1, 13)],
                "day": [f"{d:02d}" for d in range(1, 32)],
                "time": [f"{h:02d}:00" for h in range(24)],
                "area": AREA,
                "data_format": "netcdf",
                "download_format": "unarchived",
            },
            target,
        )
        print(f"  -> {target}")

# ----------------------------------------------------------------------------
# 2. Build the daily series on the paper's exposure window
# ----------------------------------------------------------------------------

def local_offset_hours(utc_index):
    """Kazakhstan unified its time zones on 1 March 2024: Almaty moved from
    UTC+6 to UTC+5. The paper reads local clock time directly, so we replicate
    the same break rather than applying a single fixed offset."""
    switch = pd.Timestamp("2024-03-01")
    return np.where(utc_index < switch, 6, 5)

def build_daily(method="nearest"):
    import xarray as xr

    ds = xr.open_mfdataset(f"{OUTDIR}/blh_*.nc", combine="by_coords")

    # ERA5 names this variable 'blh'; guard in case the backend returns
    # the long name instead.
    varname = "blh" if "blh" in ds else list(ds.data_vars)[0]
    da = ds[varname]

    # Time coordinate is 'valid_time' on the new backend, 'time' on the old.
    tname = "valid_time" if "valid_time" in da.dims else "time"

    if method == "nearest":
        series = da.sel(latitude=LAT, longitude=LON, method="nearest")
    else:
        series = da.interp(latitude=LAT, longitude=LON)

    df = series.to_dataframe(name="blh_cds").reset_index()
    df = df[[tname, "blh_cds"]].rename(columns={tname: "utc"})
    df["utc"] = pd.to_datetime(df["utc"])
    df = df.dropna().sort_values("utc")

    # UTC -> local clock time, honouring the 2024 break
    df["local"] = df["utc"] + pd.to_timedelta(local_offset_hours(df["utc"]), unit="h")
    df["date"] = df["local"].dt.normalize()
    df["hour"] = df["local"].dt.hour

    # Paper's exposure window: 07:00-17:59 inclusive (11 hourly slots)
    win = df[(df["hour"] >= 7) & (df["hour"] <= 17)]

    daily = (win.groupby("date")
                .agg(blh_cds=("blh_cds", "mean"), n_hours=("blh_cds", "size"))
                .reset_index())

    # Keep only complete days so the comparison is like-for-like
    daily = daily[daily["n_hours"] == 11].drop(columns="n_hours")
    return daily

# ----------------------------------------------------------------------------
# 3. Compare against Open-Meteo
# ----------------------------------------------------------------------------

def compare(daily_cds):
    om = pd.read_csv(OPENMETEO_PANEL)
    om[OM_DATE_COL] = pd.to_datetime(om[OM_DATE_COL]).dt.normalize()
    om = om[[OM_DATE_COL, OM_BLH_COL]].rename(
        columns={OM_DATE_COL: "date", OM_BLH_COL: "blh_om"}).dropna()

    m = daily_cds.merge(om, on="date", how="inner")
    print(f"\noverlapping days: {len(m)}")

    if len(m) == 0:
        print("No overlap. Check the date column name and dtype.")
        return m

    r_lev = m["blh_cds"].corr(m["blh_om"])
    r_log = np.log(m["blh_cds"]).corr(np.log(m["blh_om"]))

    # Regression of CDS on Open-Meteo
    x = np.c_[np.ones(len(m)), m["blh_om"].values]
    beta = np.linalg.lstsq(x, m["blh_cds"].values, rcond=None)[0]

    diff = m["blh_cds"] - m["blh_om"]

    print(f"correlation, levels         : {r_lev:.6f}")
    print(f"correlation, logs           : {r_log:.6f}   <- this is what the first stage uses")
    print(f"regression slope            : {beta[1]:.4f}")
    print(f"regression intercept        : {beta[0]:.2f} m")
    print(f"mean difference             : {diff.mean():.2f} m")
    print(f"median absolute difference  : {diff.abs().median():.2f} m")
    print(f"CDS mean / Open-Meteo mean  : {m['blh_cds'].mean():.1f} / {m['blh_om'].mean():.1f}")
    print(f"CDS min  / Open-Meteo min   : {m['blh_cds'].min():.1f} / {m['blh_om'].min():.1f}")
    print(f"CDS max  / Open-Meteo max   : {m['blh_cds'].max():.1f} / {m['blh_om'].max():.1f}")

    # Winter days matter most: they carry the identifying variation
    w = m[m["date"].dt.month.isin([11, 12, 1, 2, 3])]
    if len(w):
        print(f"\nwinter days only (Nov-Mar), n = {len(w)}")
        print(f"  correlation, logs         : {np.log(w['blh_cds']).corr(np.log(w['blh_om'])):.6f}")
        print(f"  median absolute difference: {(w['blh_cds'] - w['blh_om']).abs().median():.2f} m")

    return m

def gap_coverage(daily_cds):
    """The whole point: does the CDS series cover 2 Jan - 30 Jun 2024?"""
    gap = daily_cds[(daily_cds["date"] >= "2024-01-02") &
                    (daily_cds["date"] <= "2024-06-30")]
    print(f"\ndays recovered in the 2024 gap window: {len(gap)} of 181")
    if len(gap):
        print(f"  mean BLH over the gap: {gap['blh_cds'].mean():.1f} m")
        print(f"  Jan-Feb mean         : "
              f"{gap[gap['date'].dt.month <= 2]['blh_cds'].mean():.1f} m")

# ----------------------------------------------------------------------------

if __name__ == "__main__":
    download()
    daily = build_daily(method="nearest")
    daily.to_csv("blh_cds_daily_window.csv", index=False)
    print(f"built {len(daily)} complete-window days")
    gap_coverage(daily)
    m = compare(daily)

    # Optional: check whether bilinear interpolation matches Open-Meteo better,
    # which would tell you Open-Meteo interpolates rather than snapping to grid.
    # daily_bilinear = build_daily(method="linear")
    # compare(daily_bilinear)


ModuleNotFoundError: No module named 'cdsapi'

In [ ]:
!pip install cdsapi xarray netcdf4 -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.6 MB/s eta 0:00:00


In [ ]:
"""
ERA5 boundary layer height: pull from Copernicus CDS and validate against Open-Meteo.

Purpose: the Jan-Jun 2024 gap in the paper's BLH series is almost certainly an
Open-Meteo archive limitation, not an ERA5 one. This script (a) pulls BLH directly
from the CDS for the full sample period, (b) rebuilds the 07:00-17:59 local-time
daily mean using the same timezone rule as the paper, and (c) compares it against
the existing Open-Meteo series on overlapping dates.

Decision rule: if correlation on overlapping days is > 0.98 and the regression of
CDS on Open-Meteo has slope near 1 with small intercept, the two are the same
quantity and you can rebuild the panel on the CDS series with the gap filled.
If not, you have learned something that needs explaining before any rebuild.

SETUP (one time):
  1. Register at https://cds.climate.copernicus.eu and log in.
  2. Accept the licence on the "ERA5 hourly data on single levels" dataset page.
     Downloads silently fail with a 403 if you skip this.
  3. Copy your API token from your CDS profile page.
  4. In Colab:  !pip install cdsapi xarray netcdf4
  5. Paste the token below.

NOTE ON API DETAILS: the CDS migrated to a new backend and the endpoint URL,
token format and dataset naming have changed at least once. If the request below
errors on authentication or dataset name, check the current example request on
the dataset's "Download" tab, which always shows working code.
"""

import os
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# 0. Configuration
# ----------------------------------------------------------------------------

CDS_URL = "https://cds.climate.copernicus.eu/api"
CDS_KEY = "PASTE_YOUR_TOKEN_HERE"

# Embassy monitor coordinates (paper section 3.1)
LAT, LON = 43.2529, 76.9312

# ERA5 is on a 0.25 deg grid; take a small box around the point so we can
# compare nearest-gridpoint against bilinear interpolation.
AREA = [43.75, 76.50, 42.75, 77.50]   # N, W, S, E

YEARS = [str(y) for y in range(2020, 2026)]
OUTDIR = "era5_blh"
os.makedirs(OUTDIR, exist_ok=True)

# Path to your existing panel with the Open-Meteo BLH daily means.
# Expected columns: a date column and a boundary-layer-height column.
OPENMETEO_PANEL = "panel_window_final.csv"
OM_DATE_COL = "date"
OM_BLH_COL = "blh"          # rename if yours differs

# ----------------------------------------------------------------------------
# 1. Download from CDS, one year at a time
# ----------------------------------------------------------------------------

def download():
    import cdsapi
    c = cdsapi.Client(url=CDS_URL, key=CDS_KEY)

    for year in YEARS:
        target = f"{OUTDIR}/blh_{year}.nc"
        if os.path.exists(target):
            print(f"{target} already present, skipping")
            continue
        print(f"requesting {year} ...")
        c.retrieve(
            "reanalysis-era5-single-levels",
            {
                "product_type": ["reanalysis"],
                "variable": ["boundary_layer_height"],
                "year": [year],
                "month": [f"{m:02d}" for m in range(1, 13)],
                "day": [f"{d:02d}" for d in range(1, 32)],
                "time": [f"{h:02d}:00" for h in range(24)],
                "area": AREA,
                "data_format": "netcdf",
                "download_format": "unarchived",
            },
            target,
        )
        print(f"  -> {target}")

# ----------------------------------------------------------------------------
# 2. Build the daily series on the paper's exposure window
# ----------------------------------------------------------------------------

def local_offset_hours(utc_index):
    """Kazakhstan unified its time zones on 1 March 2024: Almaty moved from
    UTC+6 to UTC+5. The paper reads local clock time directly, so we replicate
    the same break rather than applying a single fixed offset."""
    switch = pd.Timestamp("2024-03-01")
    return np.where(utc_index < switch, 6, 5)

def build_daily(method="nearest"):
    import xarray as xr

    ds = xr.open_mfdataset(f"{OUTDIR}/blh_*.nc", combine="by_coords")

    # ERA5 names this variable 'blh'; guard in case the backend returns
    # the long name instead.
    varname = "blh" if "blh" in ds else list(ds.data_vars)[0]
    da = ds[varname]

    # Time coordinate is 'valid_time' on the new backend, 'time' on the old.
    tname = "valid_time" if "valid_time" in da.dims else "time"

    if method == "nearest":
        series = da.sel(latitude=LAT, longitude=LON, method="nearest")
    else:
        series = da.interp(latitude=LAT, longitude=LON)

    df = series.to_dataframe(name="blh_cds").reset_index()
    df = df[[tname, "blh_cds"]].rename(columns={tname: "utc"})
    df["utc"] = pd.to_datetime(df["utc"])
    df = df.dropna().sort_values("utc")

    # UTC -> local clock time, honouring the 2024 break
    df["local"] = df["utc"] + pd.to_timedelta(local_offset_hours(df["utc"]), unit="h")
    df["date"] = df["local"].dt.normalize()
    df["hour"] = df["local"].dt.hour

    # Paper's exposure window: 07:00-17:59 inclusive (11 hourly slots)
    win = df[(df["hour"] >= 7) & (df["hour"] <= 17)]

    daily = (win.groupby("date")
                .agg(blh_cds=("blh_cds", "mean"), n_hours=("blh_cds", "size"))
                .reset_index())

    # Keep only complete days so the comparison is like-for-like
    daily = daily[daily["n_hours"] == 11].drop(columns="n_hours")
    return daily

# ----------------------------------------------------------------------------
# 3. Compare against Open-Meteo
# ----------------------------------------------------------------------------

def compare(daily_cds):
    om = pd.read_csv(OPENMETEO_PANEL)
    om[OM_DATE_COL] = pd.to_datetime(om[OM_DATE_COL]).dt.normalize()
    om = om[[OM_DATE_COL, OM_BLH_COL]].rename(
        columns={OM_DATE_COL: "date", OM_BLH_COL: "blh_om"}).dropna()

    m = daily_cds.merge(om, on="date", how="inner")
    print(f"\noverlapping days: {len(m)}")

    if len(m) == 0:
        print("No overlap. Check the date column name and dtype.")
        return m

    r_lev = m["blh_cds"].corr(m["blh_om"])
    r_log = np.log(m["blh_cds"]).corr(np.log(m["blh_om"]))

    # Regression of CDS on Open-Meteo
    x = np.c_[np.ones(len(m)), m["blh_om"].values]
    beta = np.linalg.lstsq(x, m["blh_cds"].values, rcond=None)[0]

    diff = m["blh_cds"] - m["blh_om"]

    print(f"correlation, levels         : {r_lev:.6f}")
    print(f"correlation, logs           : {r_log:.6f}   <- this is what the first stage uses")
    print(f"regression slope            : {beta[1]:.4f}")
    print(f"regression intercept        : {beta[0]:.2f} m")
    print(f"mean difference             : {diff.mean():.2f} m")
    print(f"median absolute difference  : {diff.abs().median():.2f} m")
    print(f"CDS mean / Open-Meteo mean  : {m['blh_cds'].mean():.1f} / {m['blh_om'].mean():.1f}")
    print(f"CDS min  / Open-Meteo min   : {m['blh_cds'].min():.1f} / {m['blh_om'].min():.1f}")
    print(f"CDS max  / Open-Meteo max   : {m['blh_cds'].max():.1f} / {m['blh_om'].max():.1f}")

    # Winter days matter most: they carry the identifying variation
    w = m[m["date"].dt.month.isin([11, 12, 1, 2, 3])]
    if len(w):
        print(f"\nwinter days only (Nov-Mar), n = {len(w)}")
        print(f"  correlation, logs         : {np.log(w['blh_cds']).corr(np.log(w['blh_om'])):.6f}")
        print(f"  median absolute difference: {(w['blh_cds'] - w['blh_om']).abs().median():.2f} m")

    return m

def gap_coverage(daily_cds):
    """The whole point: does the CDS series cover 2 Jan - 30 Jun 2024?"""
    gap = daily_cds[(daily_cds["date"] >= "2024-01-02") &
                    (daily_cds["date"] <= "2024-06-30")]
    print(f"\ndays recovered in the 2024 gap window: {len(gap)} of 181")
    if len(gap):
        print(f"  mean BLH over the gap: {gap['blh_cds'].mean():.1f} m")
        print(f"  Jan-Feb mean         : "
              f"{gap[gap['date'].dt.month <= 2]['blh_cds'].mean():.1f} m")

# ----------------------------------------------------------------------------

if __name__ == "__main__":
    download()
    daily = build_daily(method="nearest")
    daily.to_csv("blh_cds_daily_window.csv", index=False)
    print(f"built {len(daily)} complete-window days")
    gap_coverage(daily)
    m = compare(daily)

    # Optional: check whether bilinear interpolation matches Open-Meteo better,
    # which would tell you Open-Meteo interpolates rather than snapping to grid.
    # daily_bilinear = build_daily(method="linear")
    # compare(daily_bilinear)


requesting 2020 ...


HTTPError: 401 Client Error: Unauthorized for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-single-levels/execution
Authentication failed
operation not allowed

In [ ]:
#!/usr/bin/env python3
"""
Tier 1 fixes for Pollution_kase.tex

Applies seven edits:
  1. Section 2.2  "therefore" -> "nonetheless" (logic error)
  2. Section 6.5  removes the surviving not-X-but-Y antithesis
  3. Section 5.8  "What binds causal identification..." -> plainer phrasing
  4. Section 2.5  clarifies the referent of "the two"
  5. Bibliography five missing entries printed for you to paste
  6. Bibliography brace-protects lowercase proper nouns in existing entries
  7. Table 6 + threshold table: printed for you to paste (positional, not automated)

USAGE
    python tier1_fixes.py paper.tex

Writes paper_tier1.tex alongside the original and leaves the original untouched.
Every text replacement is checked for exactly one match; anything that does not
match, or matches more than once, is reported and skipped so nothing is applied
blind. If your .tex wording differs from the PDF (line breaks, ~ ties, \\% escapes),
the report tells you which ones to do by hand.

The .bib fixes run only if a .bib file is found next to the .tex.
"""

import re
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# The four prose edits. (label, find, replace)
# Kept short so LaTeX line-wrapping in the source is less likely to break them.
# ---------------------------------------------------------------------------

PROSE_EDITS = [
    (
        "1. Sec 2.2 therefore -> nonetheless",
        "Their finding is therefore a transitory mispricing",
        "Their finding is nonetheless a transitory mispricing",
    ),
    (
        "2. Sec 6.5 antithesis removed",
        "suggests that the binding constraint in this literature is not data frequency "
        "but the cost of credible identification.",
        "suggests that what constrains this literature is the cost of credible "
        "identification.",
    ),
    (
        "3. Sec 5.8 plainer phrasing",
        "What binds causal identification of pollution effects in\nfinancial markets is "
        "the price of instrumentation. Daily data are abundant; credible\nidentification "
        "is what is scarce.",
        "The binding constraint on causal identification of pollution effects in "
        "financial markets is the price of instrumentation. Daily data are abundant; "
        "credible identification is what is scarce.",
    ),
    (
        "4. Sec 2.5 referent of 'the two'",
        "the accurate statement\nis that we cannot distinguish zero from the average "
        "global effect. Agreement between the two\nwould be too strong a reading.",
        "the accurate statement is that we cannot distinguish zero from the average "
        "global effect. Claiming agreement with their estimate would be too strong a "
        "reading.",
    ),
]

# Whitespace-insensitive matching: LaTeX sources wrap lines wherever they like,
# so we compare on collapsed whitespace and map back to the original span.


def flexible_replace(text, find, repl, label, report):
    """Replace `find` in `text` ignoring differences in whitespace/line breaks.

    Requires exactly one match. Returns (new_text, ok)."""
    pattern = re.compile(r"\s+".join(re.escape(tok) for tok in find.split()))
    matches = list(pattern.finditer(text))

    if len(matches) == 1:
        m = matches[0]
        report.append(f"  OK      {label}")
        return text[: m.start()] + repl + text[m.end():], True
    elif len(matches) == 0:
        report.append(f"  MISSING {label}  -- no match, apply by hand")
    else:
        report.append(f"  AMBIG   {label}  -- {len(matches)} matches, apply by hand")
    return text, False


# ---------------------------------------------------------------------------
# 6. Brace-protect lowercase proper nouns in .bib titles.
# BibTeX lowercases title words unless braced, which is why "almaty" and
# "china" render wrong in the reference list.
# ---------------------------------------------------------------------------

PROPER_NOUNS = [
    "Almaty", "Kazakhstan", "China", "Manhattan", "Italy", "Milan",
    "New York", "Wall Street", "US", "United States", "ERA5", "PM2.5",
    "S&P", "SAD", "IV",
]


def fix_bib_capitalisation(bib_text, report):
    """Wrap proper nouns in braces inside title = {...} fields."""
    n = 0

    def protect_title(match):
        nonlocal n
        head, body, tail = match.group(1), match.group(2), match.group(3)
        for noun in PROPER_NOUNS:
            # skip if already braced
            already = re.compile(r"\{" + re.escape(noun) + r"\}", re.IGNORECASE)
            if already.search(body):
                continue
            pat = re.compile(r"(?<![\w{])" + re.escape(noun) + r"(?![\w}])",
                             re.IGNORECASE)
            body, k = pat.subn("{" + noun + "}", body)
            n += k
        return head + body + tail

    title_pat = re.compile(r"(title\s*=\s*\{)(.*?)(\},?\s*\n)",
                           re.IGNORECASE | re.DOTALL)
    out = title_pat.sub(protect_title, bib_text)
    report.append(f"  {n} proper noun(s) brace-protected in .bib titles")
    return out


# ---------------------------------------------------------------------------
# 5. Missing bibliography entries, printed for pasting.
# ---------------------------------------------------------------------------

MISSING_BIB = r"""
% ---- paste these into your .bib -------------------------------------------

@article{basmann1960,
  author  = {Basmann, R. L.},
  title   = {On finite sample distributions of generalized classical linear
             identifiability test statistics},
  journal = {Journal of the American Statistical Association},
  year    = {1960},
  volume  = {55},
  number  = {292},
  pages   = {650--659}
}

@article{montielolea2013,
  author  = {Montiel Olea, Jos\'e Luis and Pflueger, Carolin},
  title   = {A robust test for weak instruments},
  journal = {Journal of Business \& Economic Statistics},
  year    = {2013},
  volume  = {31},
  number  = {3},
  pages   = {358--369}
}

@book{wooldridge2010,
  author    = {Wooldridge, Jeffrey M.},
  title     = {Econometric Analysis of Cross Section and Panel Data},
  publisher = {MIT Press},
  address   = {Cambridge, MA},
  year      = {2010},
  edition   = {2nd}
}

@misc{openmeteo2025,
  author       = {{Open-Meteo}},
  title        = {Historical weather {API}},
  year         = {2025},
  howpublished = {\url{https://open-meteo.com/en/docs/historical-weather-api}},
  note         = {Accessed July 2026}
}

@techreport{who2021,
  author      = {{World Health Organization}},
  title       = {{WHO} global air quality guidelines: particulate matter
                 ({PM2.5} and {PM10}), ozone, nitrogen dioxide, sulfur dioxide
                 and carbon monoxide},
  institution = {World Health Organization},
  address     = {Geneva},
  year        = {2021}
}

% Check the Wooldridge citation: your footnote 5 refers to "Wooldridge's score
% test of overidentification", which is the robust score test in the 2010 text
% (also in his 1995 chapter). Cite whichever you actually used.
% ---------------------------------------------------------------------------
"""

# ---------------------------------------------------------------------------
# 7. Table additions, printed for pasting.
# ---------------------------------------------------------------------------

TABLE_ADDITIONS = r"""
% ---- 7a. add to Table 6, directly after the "BLH alone" row ----------------

Pressure alone & $-0.1140$ & 0.158 & 0.470 & 926 & 56.1 & 0.259 \\

% Check the p-value and bound against your output before pasting: the p is
% computed from -0.1140/0.158, and the bound as (0.1140 + 1.645*0.158)*ln 2.
% Both follow the paper's own conventions but were never in a table, so verify
% against the regression output rather than trusting this line.


% ---- 7b. new table for the threshold specifications -----------------------
% Replaces the prose paragraph at the end of Section 5.6. Keep one sentence
% of prose pointing at it, and keep the caveat about identification.

\begin{table}[htbp]
\centering
\caption{Threshold specifications}
\label{tab:thresholds}
\begin{tabular}{lcccc}
\hline
Cut-off & OLS coef. & (SE) & IV coef. & (SE) \\
\hline
Above 35 $\mu$g/m$^3$ & 0.009 & (0.069) & $-0.239$ & (0.167) \\
Above 55 $\mu$g/m$^3$ & 0.058 & (0.079) & $-0.335$ & (0.232) \\
Above 75 $\mu$g/m$^3$ & 0.079 & (0.085) & $-0.486$ & (0.381) \\
\hline
\end{tabular}

\begin{minipage}{0.9\textwidth}
\vspace{0.5em}
\footnotesize
\textit{Notes:} Log concentration is replaced by an indicator for PM$_{2.5}$
exceeding the stated cut-off. Controls as in Table 2. The ordinary least
squares columns are structural specifications in the treatment, so under this
paper's own maintained argument they are not identified; we report them as
descriptive. Robust first-stage $F$ statistics for the instrumented columns are
32.5, 18.3 and 8.5, so the two higher cut-offs are weakly identified. None of
the six estimates attains significance at conventional levels.
\end{minipage}
\end{table}
% ---------------------------------------------------------------------------
"""


def main():
    if len(sys.argv) != 2:
        sys.exit("usage: python tier1_fixes.py paper.tex")

    tex_path = Path(sys.argv[1])
    if not tex_path.exists():
        sys.exit(f"not found: {tex_path}")

    text = tex_path.read_text(encoding="utf-8")
    report = ["\nPROSE EDITS"]

    applied = 0
    for label, find, repl in PROSE_EDITS:
        text, ok = flexible_replace(text, find, repl, label, report)
        applied += ok

    out_path = tex_path.with_name(tex_path.stem + "_tier1" + tex_path.suffix)
    out_path.write_text(text, encoding="utf-8")

    # .bib handling
    report.append("\nBIBLIOGRAPHY")
    bibs = list(tex_path.parent.glob("*.bib"))
    if bibs:
        bib_path = bibs[0]
        bib_text = bib_path.read_text(encoding="utf-8")
        bib_text = fix_bib_capitalisation(bib_text, report)
        bib_out = bib_path.with_name(bib_path.stem + "_tier1" + bib_path.suffix)
        bib_out.write_text(bib_text, encoding="utf-8")
        report.append(f"  wrote {bib_out.name}")
    else:
        report.append("  no .bib found next to the .tex -- brace-protect by hand")

    print("\n".join(report))
    print(f"\nwrote {out_path.name}  ({applied}/{len(PROSE_EDITS)} prose edits applied)")
    print(MISSING_BIB)
    print(TABLE_ADDITIONS)
    print("""
REMAINING BY HAND
  - Paste the five .bib entries above, then \\citep them at their use sites:
      footnote 4  -> Montiel Olea and Pflueger
      footnote 5  -> Wooldridge, Basmann
      Section 3.2 -> Open-Meteo
      Section 1   -> WHO (first mention of the 5 ug/m3 guideline)
  - Paste the Table 6 row and the threshold table.
  - Rewrite the Section 5.6 threshold paragraph down to one pointing sentence.
  - Diff paper_tier1.tex against paper.tex before compiling.
""")


if __name__ == "__main__":
    main()


SystemExit: usage: python tier1_fixes.py paper.tex

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
!pip install xarray netcdf4 cfgrib -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.6/91.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 100.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
print(pd.read_csv('panel_window_final.csv').columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'panel_window_final.csv'

In [4]:
import glob, os
print("cwd:", os.getcwd())
for f in sorted(glob.glob('**/*', recursive=True)):
    if os.path.isfile(f):
        print(f"{os.path.getsize(f)/1e6:8.2f} MB  {f}")

cwd: /content
    0.00 MB  drive/MyDrive/ The importance of empathy and curiosity in business.  .gdoc
    0.00 MB  drive/MyDrive/1st essay.gdoc
    0.00 MB  drive/MyDrive/2nd essay.gdoc
    0.00 MB  drive/MyDrive/About Page VFS.gdoc
    0.00 MB  drive/MyDrive/About SHORT VFS.gdoc
    0.00 MB  drive/MyDrive/About Ventorra FS.gdoc
    0.00 MB  drive/MyDrive/Activities List.gdoc
    0.00 MB  drive/MyDrive/Akzhan’s Essay.gdoc
    0.00 MB  drive/MyDrive/Andrew Carnegie: The Tycoon Who Gave It All Away .gdoc
    0.17 MB  drive/MyDrive/Arnay Mukhtar CV (1).pdf
    0.15 MB  drive/MyDrive/Arnay Mukhtar CV.pdf
    0.87 MB  drive/MyDrive/Arnay Mukhtar’s CV.pdf
    0.15 MB  drive/MyDrive/Arnay's Selfie.jpeg
    0.13 MB  drive/MyDrive/Arnay's passport.jpeg
    0.00 MB  drive/MyDrive/BU Supp Essay.gdoc
    0.00 MB  drive/MyDrive/Bill Ackman: The Comeback Capitalist.gdoc
    0.33 MB  drive/MyDrive/Boyan, Nikola- Ventorra affiliation 3 (1).pdf
    0.33 MB  drive/MyDrive/Boyan, Nikola- Ventorra affilia

In [5]:
NC_GLOB = "/content/drive/MyDrive/pollution_research/blh_*.nc"
PANEL = "/content/drive/MyDrive/pollution_research/panel_window_final.csv"

In [6]:
import pandas as pd
print(pd.read_csv('/content/drive/MyDrive/pollution_research/panel_window_final.csv').columns.tolist())

['date', 'logret', 'pm25_raw', 'pm_session', 'pm_presession', 'pm25_calendar', 'n_daily', 'temp', 'temp_min', 'precip', 'wind_speed', 'blh', 'blh_min', 'pressure', 'cloud', 'radiation', 'u', 'v', 'log_pm25', 'log_blh', 'dow', 'month', 'year', 'lockdown', 'qantar', 'mo_1', 'mo_2', 'mo_3', 'mo_4', 'mo_5', 'mo_6', 'mo_7', 'mo_8', 'mo_9', 'mo_10', 'mo_11', 'mo_12', 'dw_0', 'dw_1', 'dw_2', 'dw_3', 'dw_4', 'yr_2020', 'yr_2021', 'yr_2022', 'yr_2023', 'yr_2024', 'yr_2025', 'const']


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
"""
ERA5 boundary layer height: validate the CDS series against Open-Meteo.

Paths are set for Arnay's Drive layout. Run after mounting Drive:
    from google.colab import drive
    drive.mount('/content/drive')

The six blh_20XX.nc files must be in pollution_research/ alongside the panel.
"""

import glob
import numpy as np
import pandas as pd
import xarray as xr

# ---------------------------------------------------------------------------

DIR = "/content/drive/MyDrive/pollution_research"
NC_GLOB = f"{DIR}/blh_*.nc"
PANEL = f"{DIR}/panel_window_final.csv"
PANEL_DATE_COL = "date"
PANEL_BLH_COL = "blh"

LAT, LON = 43.2529, 76.9312          # Embassy monitor, section 3.1

# ---------------------------------------------------------------------------
# 1. Load
# ---------------------------------------------------------------------------

files = sorted(glob.glob(NC_GLOB))
if not files:
    raise SystemExit(f"No files matching {NC_GLOB}\n"
                     f"Check the .nc files are in {DIR}")
print(f"found {len(files)} file(s):")
for f in files:
    print("   ", f.split("/")[-1])
print()

ds = xr.open_mfdataset(files, combine="by_coords") if len(files) > 1 \
     else xr.open_dataset(files[0])

VAR = "blh" if "blh" in ds.data_vars else list(ds.data_vars)[0]
TIME = "valid_time" if "valid_time" in ds.coords else "time"
print(f"variable '{VAR}', time coord '{TIME}', dims {dict(ds.sizes)}\n")

# ---------------------------------------------------------------------------
# 2. Point series -> daily mean over the 07:00-17:59 local window
# ---------------------------------------------------------------------------

def build_daily(method="nearest"):
    da = ds[VAR]
    if method == "nearest":
        pt = da.sel(latitude=LAT, longitude=LON, method="nearest")
        print(f"[{method}] grid cell {float(pt.latitude):.2f}N "
              f"{float(pt.longitude):.2f}E")
    else:
        pt = da.interp(latitude=LAT, longitude=LON)
        print(f"[{method}] bilinear to {LAT}N {LON}E")

    df = pt.to_dataframe(name="blh_cds").reset_index()
    df = df[[TIME, "blh_cds"]].rename(columns={TIME: "utc"})
    df["utc"] = pd.to_datetime(df["utc"])
    df = df.dropna().sort_values("utc")

    # Almaty: UTC+6 before 1 March 2024, UTC+5 after
    offset = np.where(df["utc"] < pd.Timestamp("2024-03-01"), 6, 5)
    df["local"] = df["utc"] + pd.to_timedelta(offset, unit="h")
    df["date"] = df["local"].dt.normalize()
    df["hour"] = df["local"].dt.hour

    win = df[(df["hour"] >= 7) & (df["hour"] <= 17)]
    daily = (win.groupby("date")
                .agg(blh_cds=("blh_cds", "mean"), n=("blh_cds", "size"))
                .reset_index())
    complete = daily[daily["n"] == 11].drop(columns="n")
    print(f"[{method}] {len(complete)} complete-window days "
          f"({len(daily) - len(complete)} partial dropped)\n")
    return complete

daily = build_daily("nearest")

# ---------------------------------------------------------------------------
# 3. The decisive number: is the 2024 gap covered?
# ---------------------------------------------------------------------------

gap = daily[(daily["date"] >= "2024-01-02") & (daily["date"] <= "2024-06-30")]
print("=" * 62)
print(f"DAYS RECOVERED IN THE 2024 GAP: {len(gap)} of 181")
print("=" * 62)
if len(gap):
    jf = gap[gap["date"].dt.month <= 2]
    print(f"  mean BLH over the gap : {gap['blh_cds'].mean():.1f} m")
    print(f"  Jan-Feb mean          : {jf['blh_cds'].mean():.1f} m (n={len(jf)})")
    print("  Paper reports 61 m for January. A Jan-Feb mean near that")
    print("  confirms the extraction is pointing at the right place.")
print()

# ---------------------------------------------------------------------------
# 4. Agreement with Open-Meteo
# ---------------------------------------------------------------------------

om = pd.read_csv(PANEL)
om[PANEL_DATE_COL] = pd.to_datetime(om[PANEL_DATE_COL]).dt.normalize()
om = (om[[PANEL_DATE_COL, PANEL_BLH_COL]]
      .rename(columns={PANEL_DATE_COL: "date", PANEL_BLH_COL: "blh_om"})
      .dropna())

def compare(d, label):
    m = d.merge(om, on="date", how="inner")
    print("-" * 62)
    print(f"AGREEMENT WITH OPEN-METEO  [{label}]")
    print("-" * 62)
    print(f"overlapping days: {len(m)}")
    if len(m) == 0:
        print("No overlap -- check date parsing.")
        return

    r_log = np.log(m["blh_cds"]).corr(np.log(m["blh_om"]))
    x = np.c_[np.ones(len(m)), m["blh_om"].values]
    b0, b1 = np.linalg.lstsq(x, m["blh_cds"].values, rcond=None)[0]
    d_abs = (m["blh_cds"] - m["blh_om"]).abs()

    print(f"correlation, LOGS       : {r_log:.6f}   <- the decisive number")
    print(f"correlation, levels     : {m['blh_cds'].corr(m['blh_om']):.6f}")
    print(f"slope / intercept       : {b1:.4f} / {b0:.2f} m")
    print(f"median abs difference   : {d_abs.median():.2f} m")
    print(f"means  CDS / Open-Meteo : {m['blh_cds'].mean():.1f} / "
          f"{m['blh_om'].mean():.1f}")
    print(f"mins                    : {m['blh_cds'].min():.1f} / "
          f"{m['blh_om'].min():.1f}")
    print(f"maxes                   : {m['blh_cds'].max():.1f} / "
          f"{m['blh_om'].max():.1f}")

    w = m[m["date"].dt.month.isin([11, 12, 1, 2, 3])]
    if len(w):
        print(f"\nWINTER ONLY (Nov-Mar), n={len(w)}  <- where identification lives")
        print(f"  correlation, logs     : "
              f"{np.log(w['blh_cds']).corr(np.log(w['blh_om'])):.6f}")
        print(f"  median abs difference : "
              f"{(w['blh_cds'] - w['blh_om']).abs().median():.2f} m")
    print()

compare(daily, "nearest grid cell")

# Open-Meteo may interpolate rather than snap. Whichever matches better is the
# one to use for the rebuild, so pre-gap and post-gap are built identically.
daily_bil = build_daily("linear")
compare(daily_bil, "bilinear")

daily.to_csv(f"{DIR}/blh_cds_daily_nearest.csv", index=False)
daily_bil.to_csv(f"{DIR}/blh_cds_daily_bilinear.csv", index=False)
print("saved both daily series to Drive")

found 6 file(s):
    blh_2020.nc
    blh_2021.nc
    blh_2022.nc
    blh_2023.nc
    blh_2024.nc
    blh_2025.nc

variable 'blh', time coord 'valid_time', dims {'valid_time': 52608, 'latitude': 5, 'longitude': 5}

[nearest] grid cell 43.25N 77.00E
[nearest] 2192 complete-window days (0 partial dropped)

DAYS RECOVERED IN THE 2024 GAP: 181 of 181
  mean BLH over the gap : 318.8 m
  Jan-Feb mean          : 66.8 m (n=59)
  Paper reports 61 m for January. A Jan-Feb mean near that
  confirms the extraction is pointing at the right place.

--------------------------------------------------------------
AGREEMENT WITH OPEN-METEO  [nearest grid cell]
--------------------------------------------------------------
overlapping days: 926
correlation, LOGS       : 0.955719   <- the decisive number
correlation, levels     : 0.916732
slope / intercept       : 1.5470 / 37.75 m
median abs difference   : 138.76 m
means  CDS / Open-Meteo : 451.6 / 267.5
mins                    : 20.7 / 19.4
maxes         

In [9]:
# Rebuild the CDS series as a CALENDAR-DAY mean and compare
pt = ds[VAR].interp(latitude=LAT, longitude=LON)
d = pt.to_dataframe(name="blh_cds").reset_index()[[TIME, "blh_cds"]]
d.columns = ["utc", "blh_cds"]
d["utc"] = pd.to_datetime(d["utc"]); d = d.dropna()
off = np.where(d["utc"] < pd.Timestamp("2024-03-01"), 6, 5)
d["date"] = (d["utc"] + pd.to_timedelta(off, unit="h")).dt.normalize()
cal = d.groupby("date").agg(blh_cds=("blh_cds","mean"), n=("blh_cds","size"))
cal = cal[cal["n"] == 24].drop(columns="n").reset_index()

m = cal.merge(om, on="date")
print(f"n = {len(m)}")
print(f"log corr : {np.log(m.blh_cds).corr(np.log(m.blh_om)):.6f}")
print(f"slope    : {np.polyfit(m.blh_om, m.blh_cds, 1)[0]:.4f}")
print(f"means    : {m.blh_cds.mean():.1f} / {m.blh_om.mean():.1f}")
print(f"med diff : {(m.blh_cds-m.blh_om).abs().median():.2f} m")
print("\nmonthly means, CDS calendar vs panel:")
print(m.assign(mo=m.date.dt.month).groupby("mo")[["blh_cds","blh_om"]].mean().round(1))

n = 926
log corr : 0.967731
slope    : 0.8304
means    : 258.8 / 267.5
med diff : 16.02 m

monthly means, CDS calendar vs panel:
    blh_cds  blh_om
mo                 
1      58.3    61.4
2      69.4    65.8
3     122.9   128.7
4     236.4   238.9
5     377.2   367.0
6     482.4   547.0
7     520.2   560.4
8     500.6   509.0
9     363.7   362.0
10    200.2   195.0
11     90.3    99.3
12     49.9    53.1


In [11]:
"""
Stage 1: build both BLH constructions from the CDS files, splice the 2024 gap,
and let the first stage decide which one the paper should use.

Run in Colab with Drive mounted. The six blh_20XX.nc files and the panel must
be in pollution_research/.

WHAT THIS DOES
  A. Builds calendar-day and 07:00-17:59 windowed BLH for all 2,192 days.
  B. Saves both, so nothing has to be recomputed later.
  C. Merges each onto the existing panel and runs the full first stage.
  D. Reports F, partial R-squared and the BLH elasticity side by side.

WHAT IT DOES NOT DO
  It does not touch the second stage. The comparison is on instrument
  relevance only, which is the thing that is legitimate to choose on -- and
  it still has to be disclosed in the paper, alongside the losing
  construction as a robustness row.

CAVEAT ON THE COMPARISON
  This runs on the 926-day panel, so both constructions are evaluated on the
  SAME days -- the gap is not yet spliced in. That is deliberate: it isolates
  the construction question from the sample-size question. The splice happens
  in step B and gets used once the construction is chosen.
"""

import glob
import numpy as np
import pandas as pd
import xarray as xr
import statsmodels.api as sm

DIR = "/content/drive/MyDrive/pollution_research"
LAT, LON = 43.2529, 76.9312

# ---------------------------------------------------------------------------
# A. Build both constructions
# ---------------------------------------------------------------------------

ds = xr.open_mfdataset(sorted(glob.glob(f"{DIR}/blh_*.nc")), combine="by_coords")
VAR = "blh" if "blh" in ds.data_vars else list(ds.data_vars)[0]
TIME = "valid_time" if "valid_time" in ds.coords else "time"

# Bilinear matched Open-Meteo marginally better than nearest, so use it
pt = ds[VAR].interp(latitude=LAT, longitude=LON)
h = pt.to_dataframe(name="blh").reset_index()[[TIME, "blh"]]
h.columns = ["utc", "blh"]
h["utc"] = pd.to_datetime(h["utc"])
h = h.dropna().sort_values("utc")

# Almaty unified its time zones on 1 March 2024: UTC+6 -> UTC+5
off = np.where(h["utc"] < pd.Timestamp("2024-03-01"), 6, 5)
h["local"] = h["utc"] + pd.to_timedelta(off, unit="h")
h["date"] = h["local"].dt.normalize()
h["hour"] = h["local"].dt.hour

# Calendar day: all 24 hours
cal = (h.groupby("date")
         .agg(blh_cal=("blh", "mean"), blh_cal_min=("blh", "min"), n=("blh", "size"))
         .reset_index())
cal = cal[cal["n"] == 24].drop(columns="n")

# Window: 07:00-17:59, eleven hours
w = h[(h["hour"] >= 7) & (h["hour"] <= 17)]
win = (w.groupby("date")
         .agg(blh_win=("blh", "mean"), blh_win_min=("blh", "min"), n=("blh", "size"))
         .reset_index())
win = win[win["n"] == 11].drop(columns="n")

both = cal.merge(win, on="date", how="outer").sort_values("date")
both.to_csv(f"{DIR}/blh_cds_both_constructions.csv", index=False)

print(f"built {len(cal)} calendar-day and {len(win)} windowed days")
print(f"calendar mean {cal.blh_cal.mean():.1f} m,  window mean {win.blh_win.mean():.1f} m")
gap = both[(both.date >= "2024-01-02") & (both.date <= "2024-06-30")]
print(f"2024 gap coverage: {gap.blh_cal.notna().sum()} of 181 days\n")

# ---------------------------------------------------------------------------
# C. First stage under each construction, on the SAME 926 days
# ---------------------------------------------------------------------------

panel = pd.read_csv(f"{DIR}/panel_window_final.csv")
panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()

CONTROLS = ["temp", "precip", "wind_speed", "cloud", "radiation", "lockdown", "qantar"]
FE = ([f"mo_{i}" for i in range(2, 13)]
      + [f"dw_{i}" for i in range(1, 5)]
      + [f"yr_{y}" for y in range(2021, 2026)])

def first_stage(blh_col, label):
    d = panel.merge(both[["date", blh_col]], on="date", how="inner").dropna(
        subset=[blh_col, "log_pm25", "u", "v", "pressure"] + CONTROLS + FE)

    d = d.assign(log_blh_new=np.log(d[blh_col]))
    excluded = ["log_blh_new", "u", "v", "pressure"]
    X = sm.add_constant(d[excluded + CONTROLS + FE])
    y = d["log_pm25"]

    fit = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

    # Joint robust Wald on the excluded instruments, expressed as an F
    R = np.zeros((len(excluded), X.shape[1]))
    for i, name in enumerate(excluded):
        R[i, list(X.columns).index(name)] = 1
    wald = fit.wald_test(R, scalar=True)
    F = float(wald.statistic) / len(excluded)

    # Partial R-squared: residualise treatment and instruments on controls
    Z = sm.add_constant(d[CONTROLS + FE])
    y_r = sm.OLS(y, Z).fit().resid
    inst_r = np.column_stack([sm.OLS(d[c], Z).fit().resid for c in excluded])
    partial = sm.OLS(y_r, sm.add_constant(inst_r)).fit().rsquared

    print(f"--- {label} ---")
    print(f"  N                    : {len(d)}")
    print(f"  robust first-stage F : {F:.1f}")
    print(f"  partial R-squared    : {partial:.4f}")
    print(f"  log BLH coefficient  : {fit.params['log_blh_new']:.4f} "
          f"(SE {fit.bse['log_blh_new']:.4f}, t {fit.tvalues['log_blh_new']:.2f})")
    print(f"  u / v / pressure t   : {fit.tvalues['u']:.2f} / "
          f"{fit.tvalues['v']:.2f} / {fit.tvalues['pressure']:.2f}")
    print()
    return F, partial

print("=" * 64)
print("FIRST STAGE COMPARISON  (same 926 days, only BLH construction differs)")
print("=" * 64)
f_cal, r_cal = first_stage("blh_cal", "CALENDAR-DAY BLH")
f_win, r_win = first_stage("blh_win", "WINDOWED BLH (07:00-17:59)")

print("=" * 64)
winner = "CALENDAR-DAY" if f_cal > f_win else "WINDOWED"
print(f"Stronger first stage: {winner}")
print(f"  F        {f_cal:.1f} vs {f_win:.1f}")
print(f"  partial R2  {r_cal:.4f} vs {r_win:.4f}")
print("=" * 64)
print("""
Read it this way. A clear win for calendar-day supports keeping the existing
met construction, correcting Section 3.2 to describe it accurately, and
splicing only the 2024 gap. A clear win for windowed argues for pulling u, v
and pressure on the window too -- nine more CDS requests, not twenty-four.
If the two are close, sample size is doing more work than construction, and
the cheaper path is the better one.
""")

built 2190 calendar-day and 2192 windowed days
calendar mean 251.1 m,  window mean 458.3 m
2024 gap coverage: 180 of 181 days

FIRST STAGE COMPARISON  (same 926 days, only BLH construction differs)
--- CALENDAR-DAY BLH ---
  N                    : 926
  robust first-stage F : 67.6
  partial R-squared    : 0.2724
  log BLH coefficient  : -0.6097 (SE 0.0433, t -14.07)
  u / v / pressure t   : 3.72 / -2.61 / -4.55

--- WINDOWED BLH (07:00-17:59) ---
  N                    : 926
  robust first-stage F : 56.4
  partial R-squared    : 0.2560
  log BLH coefficient  : -0.5304 (SE 0.0421, t -12.61)
  u / v / pressure t   : 3.64 / -2.37 / -4.20

Stronger first stage: CALENDAR-DAY
  F        67.6 vs 56.4
  partial R2  0.2724 vs 0.2560

Read it this way. A clear win for calendar-day supports keeping the existing
met construction, correcting Section 3.2 to describe it accurately, and
splicing only the 2024 gap. A clear win for windowed argues for pulling u, v
and pressure on the window too -- nine 

In [12]:
both[both.blh_cal.isna() & both.blh_win.notna()].date

,date
0,2020-01-01
1521,2024-03-01


In [13]:
cal = cal[cal["n"].isin([23, 24, 25])].drop(columns="n")

KeyError: 'n'

In [17]:
# ── Rebuild BLH with the 25-hour day accepted, then splice into the met file ──
import glob
import numpy as np
import pandas as pd
import xarray as xr

DIR = "/content/drive/MyDrive/pollution_research"
LAT, LON = 43.2529, 76.9312

ds = xr.open_mfdataset(sorted(glob.glob(f"{DIR}/blh_*.nc")), combine="by_coords")
VAR  = "blh" if "blh" in ds.data_vars else list(ds.data_vars)[0]
TIME = "valid_time" if "valid_time" in ds.coords else "time"

pt = ds[VAR].sel(latitude=LAT, longitude=LON, method="nearest")
h = pt.to_dataframe(name="blh").reset_index()[[TIME, "blh"]]
h.columns = ["utc", "blh"]
h["utc"] = pd.to_datetime(h["utc"])
h = h.dropna().sort_values("utc")

off = np.where(h["utc"] < pd.Timestamp("2024-03-01"), 6, 5)
h["date"] = (h["utc"] + pd.to_timedelta(off, unit="h")).dt.normalize()

cal = (h.groupby("date")
         .agg(blh_new=("blh", "mean"), blh_min_new=("blh", "min"), n=("blh", "size"))
         .reset_index())
cal = cal[cal["n"].isin([23, 24, 25])].drop(columns="n")

gap = cal[(cal.date >= "2024-01-02") & (cal.date <= "2024-06-30")]
print(f"calendar-day BLH built for {len(cal)} days")
print(f"2024 gap coverage: {len(gap)} of 181")

# ── Splice: replace blh and blh_min, leave every other column untouched ──
met = pd.read_csv(f"{DIR}/almaty_weather_daily.csv")
met["date"] = pd.to_datetime(met["date"]).dt.normalize()

before = met["blh"].notna().sum()
met = met.merge(cal, on="date", how="left")
met["blh"] = met["blh_new"]
met["blh_min"] = met["blh_min_new"]
met = met.drop(columns=["blh_new", "blh_min_new"])

print(f"\nblh non-null: {before} -> {met['blh'].notna().sum()} (of {len(met)})")
print(f"blh mean {met['blh'].mean():.2f}, min {met['blh'].min():.2f}, "
      f"max {met['blh'].max():.2f}")
print(f"columns unchanged: {met.columns.tolist()}")

met.to_csv(f"{DIR}/almaty_weather_daily_cds.csv", index=False)
print(f"\nwrote {DIR}/almaty_weather_daily_cds.csv")

calendar-day BLH built for 2191 days
2024 gap coverage: 181 of 181

blh non-null: 2011 -> 2191 (of 2192)
blh mean 237.89, min 21.15, max 995.48
columns unchanged: ['date', 'temp', 'temp_min', 'precip', 'wind_speed', 'blh', 'blh_min', 'pressure', 'cloud', 'radiation', 'u', 'v']

wrote /content/drive/MyDrive/pollution_research/almaty_weather_daily_cds.csv


In [15]:
pt_n = ds[VAR].sel(latitude=LAT, longitude=LON, method="nearest")
hn = pt_n.to_dataframe(name="blh").reset_index()[[TIME, "blh"]]
hn.columns = ["utc", "blh"]
hn["utc"] = pd.to_datetime(hn["utc"]); hn = hn.dropna()
o = np.where(hn["utc"] < pd.Timestamp("2024-03-01"), 6, 5)
hn["date"] = (hn["utc"] + pd.to_timedelta(o, unit="h")).dt.normalize()
cn = (hn.groupby("date").agg(b=("blh","mean"), n=("blh","size")).reset_index())
cn = cn[cn.n.isin([23,24,25])]

om = pd.read_csv(f"{DIR}/almaty_weather_daily.csv")
om["date"] = pd.to_datetime(om["date"]).dt.normalize()
m = cn.merge(om[["date","blh"]].dropna(), on="date")
print(f"NEAREST  mean {cn.b.mean():.1f}  min {cn.b.min():.1f}  max {cn.b.max():.1f}")
print(f"  log corr with Open-Meteo {np.log(m.b).corr(np.log(m.blh)):.4f}, "
      f"slope {np.polyfit(m.blh, m.b, 1)[0]:.3f}")
print(f"OPEN-METEO  mean {om.blh.mean():.1f}  min {om.blh.min():.1f}  "
      f"max {om.blh.max():.1f}")

NEAREST  mean 237.9  min 21.1  max 995.5
  log corr with Open-Meteo 0.9731, slope 0.852
OPEN-METEO  mean 258.0  min 19.4  max 1303.1


In [16]:
pt = ds[VAR].sel(latitude=LAT, longitude=LON, method="nearest")
hn = pt_n.to_dataframe(name="blh").reset_index()[[TIME, "blh"]]
hn.columns = ["utc", "blh"]
hn["utc"] = pd.to_datetime(hn["utc"]); hn = hn.dropna()
o = np.where(hn["utc"] < pd.Timestamp("2024-03-01"), 6, 5)
hn["date"] = (hn["utc"] + pd.to_timedelta(o, unit="h")).dt.normalize()
cn = (hn.groupby("date").agg(b=("blh","mean"), n=("blh","size")).reset_index())
cn = cn[cn.n.isin([23,24,25])]

om = pd.read_csv(f"{DIR}/almaty_weather_daily.csv")
om["date"] = pd.to_datetime(om["date"]).dt.normalize()
m = cn.merge(om[["date","blh"]].dropna(), on="date")
print(f"NEAREST  mean {cn.b.mean():.1f}  min {cn.b.min():.1f}  max {cn.b.max():.1f}")
print(f"  log corr with Open-Meteo {np.log(m.b).corr(np.log(m.blh)):.4f}, "
      f"slope {np.polyfit(m.blh, m.b, 1)[0]:.3f}")
print(f"OPEN-METEO  mean {om.blh.mean():.1f}  min {om.blh.min():.1f}  "
      f"max {om.blh.max():.1f}")

NEAREST  mean 237.9  min 21.1  max 995.5
  log corr with Open-Meteo 0.9731, slope 0.852
OPEN-METEO  mean 258.0  min 19.4  max 1303.1


In [18]:
met = pd.read_csv(f"{DIR}/almaty_weather_daily_cds.csv")
met["date"] = pd.to_datetime(met["date"])
mm = met.groupby(met.date.dt.month)["blh"].mean().round(1)
print(mm)
print(f"amplitude: {mm.max()/mm.min():.1f}x")
print(f"range: {met.blh.min():.1f} to {met.blh.max():.1f}")

date
1      46.6
2      57.7
3     117.1
4     224.0
5     331.9
6     433.0
7     493.0
8     468.2
9     352.8
10    192.1
11     79.5
12     46.7
Name: blh, dtype: float64
amplitude: 10.6x
range: 21.1 to 995.5
